# US-Aktien-Bot V1.6.1 – Backtest, historische SIP-Daten und Research-Export

V1.6 baut auf V1.5 auf; V1.5 bleibt unverändert.

**Bedienung:** alles im Cockpit unter der Startzelle. Einmal „Laufzeit → Alle ausführen“, danach nur noch Knöpfe.

Die Strategie, das Risiko und die Scanner-Parameter sind **unverändert**. Sie werden weiterhin aus dem geprüften Scannerlauf geladen (`PREVIOUS_SCANNER_RUN`, Hash-Prüfung von v02/v03). Es gibt keine Optimierung und keine Parameter-Sweeps.

## Modi: immer genau einer

| Modus | Was läuft | Orders |
|---|---|---|
| **OBSERVE** | BOT 2: heutige SIP-Daten, 16 Min. verzögert, simuliert (wie V1.5 `SIP_ONLY`) | keine |
| **DRY RUN** | BOT 1: IEX in Echtzeit mit interner Simulation (wie V1.5 `IEX_ONLY` ohne Freigabe) | keine |
| **PAPER** | BOT 1 wie DRY RUN, dazu Alpaca-**Paper**-Orders mit der Tagesfreigabe `PAPER-ORDERS-OK <Konto> <Datum>` | nur BOT 1, nur Paper |
| **BACKTEST** | vergangene Handelstage mit historischen SIP-Daten, simulierter Uhr und derselben Strategie-Schleife | **keine** (technisch gesperrt) |

- **SIP-Vergleichsbot parallel:** Häkchen bei DRY RUN und PAPER (entspricht V1.5 `DUAL_MODE`). Der SIP-Bot bekommt keinen Order-Client und kann nie Orders auslösen.
- **Order-Sperre:** Die Sperre sitzt im Order-Layer selbst, nicht nur in der Oberfläche. `paper_orders_v16` und `orb_sim_v16` prüfen vor jeder Order, ob der aktive Modus PAPER ist.
- **Backtest-Zugriff:** Im Backtest bekommt der Datenlader nur einen Nur-Lese-Zugang (Kalender und Aktienliste).

## BACKTEST

1. **Datenzugang prüfen** (einmalig): prüft für AAPL separat SIP-Bars, SIP-Quotes und SIP-Trades.
   - Ergebnis je Datentyp: `AVAILABLE`, `EMPTY RESULT` (keine Aussage über Berechtigung), `NOT AVAILABLE` (HTTP 403), `RATE LIMITED` (429) oder `INVALID REQUEST`.
   - Dazu die tatsächlich erreichbare Historie.
   - Es werden keine Zugangsdaten angezeigt oder gespeichert.
2. **Eingaben:** Startdatum, Enddatum, Startkapital. Ein einzelner Tag ist möglich (Start = Ende).
3. **▶ START BACKTEST:** Cache prüfen → fehlende SIP-Daten laden → prüfen → speichern → Replay → Research-Export → Analysis-Bundle.
4. **Nach dem Lauf:** die Knöpfe ERGEBNIS, TRADES, SIGNALE, TAGE und ANALYSIS-BUNDLE (Download in Colab).

Regeln im Backtest:
- **Zeitraum:** Wochenenden und Feiertage entfallen laut Alpaca-Börsenkalender, verkürzte Handelstage werden erkannt.
- **Zeiten:** alle in America/New_York.
- **Kapital:** wird über die Tage fortgeschrieben.
- **Warmup:** 20 Vortage für RVOL und den Liquiditätsfilter, abgeleitet aus der Strategie. Warmup-Tage werden nur geladen, nie simuliert.
- **Kein Blick in die Zukunft:** Eine Minutenkerze ist erst 2 s nach Minutenende sichtbar, Quotes nur bis zur simulierten Uhrzeit.

## V1.6.1 – schneller, kompakter Fortschritt, fortsetzbar

- **Minute für Minute intern, Tag für Tag nach außen.** Die Strategie rechnet unverändert jede Minute. Das Cockpit zeigt nur noch eine kompakte Übersicht:
  - Zeitraum, Handelstage x / y, aktueller Tag, Signale, Trades, Equity, Datenabdeckung, Laufzeit
  - Fortschrittsbalken und aktive Phase (1 Cache prüfen … 6 Research-Output)
  - „Letzte Aktualisierung“, alle 5 Sekunden neu
  - eine Zeile je fertigem Handelstag
- **Auto-Save nach jedem Handelstag:** `checkpoint.json` und die Research-Teile des Tages (`parts/<Datum>/`) werden geschrieben. Ein Lebenszeichen wird alle 30 s gespeichert.
- **Fortsetzen statt neu rechnen:**
  - Beim Öffnen erscheint bei einem unterbrochenen Lauf **INCOMPLETE BACKTEST FOUND** mit **▶ RESUME BACKTEST** und **DISCARD**.
  - Fertige Tage werden nicht noch einmal gerechnet.
  - **⏸ PAUSE NACH TAG** hält nach dem laufenden Handelstag an.
- **Schneller, ohne andere Entscheidungen** (per Tests gegen die bisherige Rechnung abgesichert):
  - Signal-Scan auf numpy-Basis
  - RVOL-Vergleich aus vorbereiteten Tagesdaten
  - Minutendaten je Tag einmal vorbereitet
  - Dateien einmal pro Tag statt jede Minute geschrieben; das spart vor allem auf Google Drive viel Zeit

## Historische Daten

- **Ablage:** `MyDrive/US_Aktien_Bot/data/historical/`, Parquet je Feed, Datentyp, Symbol und Monat, mit `historical_data_manifest.json`.
- **Nur einmal laden:** Beim zweiten Backtest desselben Zeitraums wird nichts erneut geladen. Der alte `replay_cache_sip/` wird beim ersten Mal geprüft übernommen; Tage, die unvollständig sein könnten, werden neu geladen.
- **Nur SIP:** Es gibt keinen stillen Wechsel auf IEX. Fehlen SIP-Daten, erscheint `HISTORICAL DATA WARNING` und der Lauf stoppt.
- **Quotes:** nur an den Zeitpunkten, die der Filter braucht (Signal, Einstieg, Ausstieg).
- **Trades:** nur für ungeklärte RVOL-Vergleichsminuten. Alle Trades der Signal-Aktien werden nur mit dem Häkchen „FULL_MARKET_DATA“ geladen.
- **Datenqualität:**
  - Fehlende Minuten werden nie aufgefüllt; die bestehenden Regeln lehnen das betroffene Signal ab.
  - Ein Symbol ohne brauchbare Daten fehlt nur an diesem Tag.
  - Bei einem systemischen Problem (Download fehlgeschlagen, keine QQQ-Daten, weniger als 50 % des Universums mit Kerzen) wird der Tag nicht simuliert. Der Backtest stoppt mit Status `INCOMPLETE_DATA`; alles bis dahin wird gespeichert.
  - Jeder Handelstag steht in `daily`, auch wenn er nicht simuliert wurde.

## Research-Paket je Lauf

Ablage: `MyDrive/US_Aktien_Bot/backtests/<RUN_ID>/`

- **Beschreibung:** `run_manifest.json` (alle Einstellungen), `summary.json` (Kennzahlen), `schema.json` (jede Spalte erklärt)
- **Tabellen:** `trades`, `signals` (**alle** Kandidatensignale inklusive Ablehnungsgrund und Filter-Flags), `daily` (inklusive Datenabdeckung), `equity_curve`, `events`, `data_gaps`
- **Formate:** Parquet, dazu CSV
- **Bundle:** `<RUN_ID>_ANALYSIS.zip` zum direkten Hochladen in ein Analysewerkzeug
- **Index aller Läufe:** `backtests/backtest_runs.csv`

Die Felder `future_*` in `signals` zeigen, was nach einem Signal passiert wäre. Sie entstehen **erst nach dem Lauf** in `research_export_v16` und fließen nie in eine Entscheidung ein.

## Grenzen

- **Modellwerte:** Die Ergebnisse sind Modellrechnungen, kein Rentabilitätsnachweis.
- **Survivorship-Bias:** Für alle Tage gilt der aktuelle Nasdaq-100-Stand.
- **Ausführung:** Ask/Bid-Proxy und ein 10-bp-Stressszenario. Kommission und Marktauswirkung sind nicht modelliert.
- **Kerzen:** Historische Kerzen können spätere Korrekturen enthalten.
- **Stop in der Minute:** Die Reihenfolge von Hoch und Tief innerhalb einer Minute ist unbekannt; betroffene Stops sind als `intrabar_ambiguous` gekennzeichnet.


In [ ]:
%pip -q install alpaca-py requests


In [ ]:
PREVIOUS_SCANNER_RUN = '20260923T124355Z_838f60df'

# V1.6: Alle Einstellungen (Modus, Zeitraum, Startkapital, Freigabe) stehen im Cockpit
# unter der Startzelle. Hier muss im normalen Betrieb nichts geändert werden.


In [ ]:
%%writefile /content/us_orb_test_v02.py
"""US-ORB V0.2: historisches Testlabor. Keine Ordermethoden enthalten."""
from dataclasses import dataclass, asdict
from decimal import Decimal, ROUND_FLOOR
from datetime import datetime, timedelta, timezone
from pathlib import Path
from zoneinfo import ZoneInfo
import hashlib
import importlib.metadata
import json
import platform
import uuid
import numpy as np
import pandas as pd

NY = ZoneInfo("America/New_York")
VERSION = "0.2-testlabor"
PENDING = [
    "Aktuelles Nasdaq-100-Universum und zeitgerechte historische Mitglieder",
    "Mindestliquidität, historische Quotes/Spread, Ranking und relative Stärke",
    "Signalgültigkeit und Wiedereinstiege",
    "Gemeinsame Portfolio-/Risikosimulation mit Orderreservierungen",
    "Ausführungsmodell, Kosten, Slippage, MFE/MAE und Benchmark",
    "Broker-Trailing-Stop, Teilfüllungen, Abbruch- und Neustartabgleich",
    "Persistente operative Tagessperre und Maßnahmen für offene Positionen",
    "Echtzeit-Feed-Entscheidung und sieben Handelstage Paper Trading",
]

@dataclass(frozen=True)
class Config:
    test_date: str = "2026-09-22"
    symbols: tuple = ("AAPL", "MSFT", "NVDA")
    opening_minutes: int = 15
    breakout_buffer: str = "0.0005"
    entry_cap: str = "0.003"
    trailing: str = "0.01"
    rvol_days: int = 20
    minimum_rvol: float = 1.5
    max_position_capital: str = "0.30"
    max_trade_risk: str = "0.01"
    max_total_risk: str = "0.03"
    max_positions: int = 3
    max_per_sector: int = 2
    daily_loss: str = "0.02"
    entry_cutoff_minutes: int = 45
    close_before_minutes: int = 30
    minimum_price: float = 20.0

def D(value):
    value = Decimal(str(value))
    if not value.is_finite():
        raise ValueError("Nicht endlicher Zahlenwert")
    return value

def utc(value):
    stamp = pd.Timestamp(value)
    if stamp.tzinfo is None:
        stamp = stamp.tz_localize(NY)
    return stamp.tz_convert("UTC")

def validate_config(c):
    datetime.strptime(c.test_date, "%Y-%m-%d")
    if not c.symbols or len(set(c.symbols)) != len(c.symbols):
        raise ValueError("Aktienliste ist leer oder enthält Duplikate")
    if any(not isinstance(s, str) or not s.strip() for s in c.symbols):
        raise ValueError("Ungültiges Symbol")
    for name in ("trailing", "max_position_capital", "max_trade_risk",
                 "max_total_risk", "daily_loss"):
        if not 0 < D(getattr(c, name)) < 1:
            raise ValueError(f"Ungültiger Parameter: {name}")
    if not 0 <= D(c.breakout_buffer) <= D(c.entry_cap) < 1:
        raise ValueError("Ungültiger Ausbruchsabstand")
    if not 0 < c.close_before_minutes < c.entry_cutoff_minutes:
        raise ValueError("Ungültige Schlusszeiten")
    if min(c.opening_minutes, c.rvol_days, c.max_positions, c.max_per_sector) < 1:
        raise ValueError("Ungültige Anzahl")
    if not np.isfinite(c.minimum_rvol) or c.minimum_rvol <= 0:
        raise ValueError("Ungültiger Volumenfaktor")

def size_position(equity, free_cash, existing_risk, entry, c):
    eq, cash, risk, price = map(D, (equity, free_cash, existing_risk, entry))
    if eq <= 0 or price <= 0 or cash < 0 or risk < 0:
        raise ValueError("Ungültige Größenberechnung")
    capital = min(eq * D(c.max_position_capital), cash)
    budget = min(eq * D(c.max_trade_risk), max(D(0), eq * D(c.max_total_risk) - risk))
    distance = price * D(c.trailing)
    qty = int(min(capital / price, budget / distance).to_integral_value(rounding=ROUND_FLOOR))
    return dict(qty=qty, notional=qty * price, planned_risk=qty * distance,
                risk_budget=budget)

def entry_gate(start_equity, current_equity, confirmed_net_cashflows,
               occupied_sectors, candidate_sector, day_locked, c):
    start, current, flows = map(D, (start_equity, current_equity, confirmed_net_cashflows))
    if start <= 0 or current <= 0:
        raise ValueError("Ungültiges Eigenkapital")
    pnl = current - start - flows
    locked = bool(day_locked) or pnl <= -start * D(c.daily_loss)
    reasons = []
    if locked:
        reasons.append("Tagesverlustsperre")
    if len(occupied_sectors) >= c.max_positions:
        reasons.append("Positionsplätze belegt/reserviert")
    if not candidate_sector or any(not s for s in occupied_sectors):
        reasons.append("Sektorzuordnung fehlt")
    elif occupied_sectors.count(candidate_sector) >= c.max_per_sector:
        reasons.append("Sektorlimit erreicht")
    return dict(allowed=not reasons, day_locked=locked, day_pnl=pnl, reasons=reasons)

def simulate_exit(entry_price, entry_time, ticks, session_close, c):
    price, trail = D(entry_price), D(c.trailing)
    if price <= 0 or not 0 < trail < 1:
        raise ValueError("Ungültiger Trailing-Test")
    start, close = utc(entry_time), utc(session_close)
    cutoff = close - pd.Timedelta(minutes=c.close_before_minutes)
    if start >= cutoff:
        raise ValueError("Einstieg liegt nach Schließungsgrenze")
    high, stop, last_time = price, price * (1 - trail), start
    rows = [dict(time=start, price=price, high=high, stop=stop, event="START")]
    for stamp, raw_price in ticks:
        stamp, price = utc(stamp), D(raw_price)
        if stamp <= last_time or price <= 0:
            raise ValueError("Ungültige Kursfolge")
        if stamp >= cutoff:
            event = "TAGESABSCHLUSS"
        elif price <= stop:
            event = "TRAILING_STOP"
        else:
            high = max(high, price)
            stop = max(stop, high * (1 - trail))
            event = "HALTEN"
        rows.append(dict(time=stamp, price=price, high=high, stop=stop, event=event))
        last_time = stamp
        if event in ("TRAILING_STOP", "TAGESABSCHLUSS"):
            break
    return rows

def self_tests():
    """Feste Referenzparameter, getrennt von späteren Nutzer-Varianten."""
    c = Config()
    results, stop_rows = [], []
    for label, cash, risk, qty, notional, planned in [
        ("Normalfall", 10000, 0, 13, "2990", "29.90"),
        ("Wenig Cash", 500, 0, 2, "460", "4.60"),
        ("10 USD Risikospielraum", 10000, 290, 4, "920", "9.20"),
        ("Risiko ausgeschöpft", 10000, 300, 0, "0", "0"),
    ]:
        r = size_position(10000, cash, risk, 230, c)
        passed = r["qty"] == qty and r["notional"] == D(notional) and r["planned_risk"] == D(planned)
        results.append(dict(Bereich="Positionsgröße", Test=label, Bestanden=passed))
    for label, eq, flow, sectors, locked, expected in [
        ("Normalfall", 10000, 0, [], False, True),
        ("3 Plätze belegt", 10000, 0, ["Tech", "Industrie", "Gesundheit"], False, False),
        ("2 Tech-Positionen", 10000, 0, ["Tech", "Tech"], False, False),
        ("Genau 200 USD Verlust", 9800, 0, [], False, False),
        ("Einzahlung", 11000, 1000, [], False, True),
        ("Entnahme", 9000, -1000, [], False, True),
        ("Einzahlung plus Verlust", 10800, 1000, [], False, False),
        ("Tag bleibt gesperrt", 10000, 0, [], True, False),
    ]:
        r = entry_gate(10000, eq, flow, sectors, "Tech", locked, c)
        passed = r["allowed"] == expected
        if label in ("Einzahlung", "Entnahme"):
            passed = passed and r["day_pnl"] == 0
        results.append(dict(Bereich="Einstiegssperren", Test=label, Bestanden=passed))
    def at(hm):
        return pd.Timestamp(f"2026-09-22 {hm}", tz=NY)
    for label, start, ticks, close, reason, final_stop in [
        ("Steigt und fällt", "10:00", [("10:01", 101), ("10:02", 102), ("10:03", 101.5), ("10:04", 100.98)], "16:00", "TRAILING_STOP", "100.98"),
        ("Kurssprung", "10:00", [("10:01", 97)], "16:00", "TRAILING_STOP", "99"),
        ("Regulärer Abschluss", "15:28", [("15:29", 100.5), ("15:30", 100.4)], "16:00", "TAGESABSCHLUSS", "99.495"),
        ("Synthetischer Kurzhandelstag", "12:28", [("12:29", 100.5), ("12:30", 100.4)], "13:00", "TAGESABSCHLUSS", "99.495"),
    ]:
        rows = simulate_exit(100, at(start), [(at(t), p) for t, p in ticks], at(close), c)
        passed = rows[-1]["event"] == reason and rows[-1]["stop"] == D(final_stop)
        passed = passed and all(b["stop"] >= a["stop"] for a, b in zip(rows, rows[1:]))
        if label == "Steigt und fällt":
            passed = passed and [r["stop"] for r in rows] == list(map(D, ["99", "99.99", "100.98", "100.98", "100.98"]))
        results.append(dict(Bereich="Ausstieg", Test=label, Bestanden=passed))
        stop_rows.extend(dict(Test=label, **row) for row in rows)
    return pd.DataFrame(results), pd.DataFrame(stop_rows)

def normalize_bars(frame):
    df = frame.reset_index() if "timestamp" not in frame.columns else frame.copy()
    columns = ["symbol", "timestamp", "open", "high", "low", "close", "volume"]
    if not set(columns).issubset(df.columns):
        raise ValueError("Benötigte Kursdatenspalten fehlen")
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    numeric = ["open", "high", "low", "close", "volume"]
    df[numeric] = df[numeric].apply(pd.to_numeric, errors="raise")
    df["feed"] = "sip"
    return df.sort_values(["symbol", "timestamp"]).reset_index(drop=True)

def checked_window(df, symbol, start, end):
    window = df[(df.symbol == symbol) & (df.timestamp >= start) & (df.timestamp < end)].sort_values("timestamp")
    expected = pd.date_range(start, end, freq="min", inclusive="left")
    if not pd.DatetimeIndex(window.timestamp).equals(expected):
        raise ValueError(f"{symbol}: Minuten fehlen, sind doppelt oder zeitlich falsch")
    cols = ["open", "high", "low", "close", "volume"]
    if not np.isfinite(window[cols].to_numpy()).all():
        raise ValueError(f"{symbol}: Ungültige Zahlen")
    if not ((window[["open", "high", "low", "close"]] > 0).all(axis=1)
            & (window.volume >= 0)
            & (window.high >= window[["open", "close", "low"]].max(axis=1))
            & (window.low <= window[["open", "close", "high"]].min(axis=1))).all():
        raise ValueError(f"{symbol}: Widersprüchliche Kurswerte")
    return window

def scan_day(df, opening, closing, c):
    or_end = opening + pd.Timedelta(minutes=c.opening_minutes)
    cutoff = closing - pd.Timedelta(minutes=c.entry_cutoff_minutes)
    if or_end >= cutoff:
        raise ValueError("Handelssitzung ist für diese Zeitregeln zu kurz")
    decisions, ranges = [], []
    for symbol in c.symbols:
        stock = checked_window(df, symbol, opening, closing)
        initial = stock[stock.timestamp < or_end]
        high, low = D(initial.high.max()), D(initial.low.min())
        threshold = high * (1 + D(c.breakout_buffer))
        cap = high * (1 + D(c.entry_cap))
        ranges.append(dict(Aktie=symbol, OR_Hoch=float(high), OR_Tief=float(low),
                           Ausbruchsschwelle=float(threshold), Preisobergrenze=float(cap),
                           OR_Volumen=float(initial.volume.sum())))
        previous = D(initial.close.iloc[-1])
        for bar in stock[stock.timestamp >= or_end].itertuples():
            decision = bar.timestamp + pd.Timedelta(minutes=1)
            if decision >= cutoff:
                break
            close = D(bar.close)
            crossed = previous <= threshold and close > threshold
            if not crossed:
                result = "NO_CROSSING"
            elif close < D(c.minimum_price):
                result = "BELOW_MINIMUM_PRICE"
            elif close > cap:
                result = "TOO_EXTENDED"
            else:
                result = "PRICE_CANDIDATE"
            decisions.append(dict(symbol=symbol, bar_time_utc=bar.timestamp.isoformat(),
                decision_time_utc=decision.isoformat(), decision_time_ny=decision.tz_convert(NY).isoformat(),
                or_high=float(high), threshold=float(threshold), price_cap=float(cap),
                previous_close=float(previous), close=float(close), result=result, feed="sip"))
            previous = close
    return pd.DataFrame(ranges), pd.DataFrame(decisions)

RVOL_COLUMNS = ["Aktie", "Signalzeit_NY", "decision_time_utc", "Volumen_bis_Signal",
    "Durchschnitt_20_Tage", "RVOL", "Mindest_RVOL", "Vollständige_Vergleichstage", "Ergebnis"]

def check_rvol(candidates, current, history, past_sessions, current_open, c):
    outputs, references = [], []
    for candidate in candidates.itertuples():
        decision = utc(candidate.decision_time_utc)
        elapsed = decision - current_open
        actual = float(checked_window(current, candidate.symbol, current_open, decision).volume.sum())
        volumes = []
        for session in past_sessions:
            start, close = session["open"], session["close"]
            end = start + elapsed
            volume, status = None, "OK"
            if end > close:
                status = "Handelstag für Vergleich zu kurz"
            else:
                try:
                    volume = float(checked_window(history, candidate.symbol, start, end).volume.sum())
                except ValueError:
                    status = "Minuten fehlen oder sind ungültig"
            references.append(dict(symbol=candidate.symbol, signal_time_utc=decision.isoformat(),
                reference_date=session["date"], volume=volume, status=status))
            if volume is not None:
                volumes.append(volume)
        avg = float(np.mean(volumes)) if len(volumes) == c.rvol_days else None
        rvol = actual / avg if avg is not None and avg > 0 else None
        result = "NOT_CHECKABLE" if rvol is None else ("VOLUME_PASS" if rvol >= c.minimum_rvol else "VOLUME_REJECT")
        outputs.append(dict(Aktie=candidate.symbol, Signalzeit_NY=decision.tz_convert(NY).strftime("%H:%M"),
            decision_time_utc=decision.isoformat(), Volumen_bis_Signal=actual, Durchschnitt_20_Tage=avg,
            RVOL=rvol, Mindest_RVOL=c.minimum_rvol, Vollständige_Vergleichstage=len(volumes), Ergebnis=result))
    return pd.DataFrame(outputs, columns=RVOL_COLUMNS), pd.DataFrame(references,
        columns=["symbol", "signal_time_utc", "reference_date", "volume", "status"])

def json_write(path, content):
    with Path(path).open("x", encoding="utf-8") as f:
        json.dump(content, f, ensure_ascii=False, indent=2, default=str, allow_nan=False)

def csv_write(path, frame):
    frame.to_csv(path, index=False, mode="x")

def calendar_rows(items):
    rows = [dict(date=str(s.date), open=utc(s.open), close=utc(s.close)) for s in items]
    rows.sort(key=lambda s: s["date"])
    if len({r["date"] for r in rows}) != len(rows):
        raise ValueError("Börsenkalender enthält doppelte Tage")
    return rows

def run_lab(broker, data_client, project_root, c):
    """Nur lesende Brokerzugriffe; alle Ergebnisse in einem eigenen Laufordner."""
    from alpaca.trading.requests import GetCalendarRequest, GetOrdersRequest
    from alpaca.trading.enums import QueryOrderStatus
    from alpaca.data.requests import StockBarsRequest
    from alpaca.data.enums import DataFeed, Adjustment
    from alpaca.data.timeframe import TimeFrame
    validate_config(c)
    root = Path(project_root)
    if not root.is_dir():
        raise ValueError("Projektordner fehlt; Google Drive prüfen")
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
    out = root / "v0_2_runs" / run_id
    out.mkdir(parents=True, exist_ok=False)
    source = Path(__file__).read_text(encoding="utf-8")
    (out / "source.py").write_text(source, encoding="utf-8")
    package_versions = {}
    for package in ("alpaca-py", "pandas", "numpy"):
        package_versions[package] = importlib.metadata.version(package)
    json_write(out / "manifest.json", dict(version=VERSION, run_id=run_id,
        created_at_utc=datetime.now(timezone.utc).isoformat(), mode="historical_read_only",
        config=asdict(c), feed="sip", code_sha256=hashlib.sha256(source.encode()).hexdigest(),
        packages=package_versions, python=platform.python_version(),
        scope="3-Symbol-Testlabor; keine Orders, Ausführungen oder Performance-Simulation",
        daily_loss_definition="Equity jetzt minus Equity Tagesstart minus bestätigte Nettoeinzahlungen; Grenze 2% Tagesstart, Sperre bis nächste Sitzung (nur Logiktest)",
        assumptions="Strategieparameter Entwurf; keine Optimierung oder Profitabilitätsaussage", pending=PENDING))
    try:
        print("1/4 Prüfe die 16 bekannten Rechen- und Ausstiegstests …")
        tests, stops = self_tests()
        csv_write(out / "self_tests.csv", tests)
        csv_write(out / "synthetic_stop_paths.csv", stops)
        if not tests.Bestanden.all():
            raise ValueError("Mindestens ein Referenztest ist fehlgeschlagen")
        print("✅ 16/16 Referenztests bestanden")

        print("2/4 Lese Paper-Konto und Börsenkalender …")
        account = broker.get_account()
        clock = broker.get_clock()
        positions = broker.get_all_positions()
        orders = broker.get_orders(filter=GetOrdersRequest(status=QueryOrderStatus.OPEN, limit=500))
        account_status = getattr(account.status, "value", str(account.status))
        snapshot = dict(read_at_utc=datetime.now(timezone.utc).isoformat(), status=account_status,
            currency=account.currency, equity=str(account.equity), cash=str(account.cash),
            account_blocked=account.account_blocked, trading_blocked=account.trading_blocked,
            positions_count=len(positions), open_orders_returned=len(orders),
            orders_list_may_be_truncated=len(orders) >= 500, market_open=clock.is_open)
        json_write(out / "paper_account_read_only.json", snapshot)
        if account.currency != "USD":
            raise ValueError("Testprojekt erwartet ein USD-Konto")
        day = datetime.strptime(c.test_date, "%Y-%m-%d").date()
        sessions = calendar_rows(broker.get_calendar(GetCalendarRequest(start=day, end=day)))
        if len(sessions) != 1:
            raise ValueError("Gewähltes Datum ist kein eindeutiger Handelstag")
        session = sessions[0]
        if session["close"] > utc(clock.timestamp) - pd.Timedelta(minutes=15):
            raise ValueError("Bitte einen vollständig vergangenen Handelstag wählen (mindestens 15 Minuten nach Schluss)")
        json_write(out / "session_calendar.json", session)

        print(f"3/4 Lade SIP-Daten für {c.test_date}: {', '.join(c.symbols)} …")
        def fetch(symbols, start, end):
            bars = data_client.get_stock_bars(StockBarsRequest(
                symbol_or_symbols=list(symbols), timeframe=TimeFrame.Minute,
                start=start.to_pydatetime(), end=end.to_pydatetime(),
                feed=DataFeed.SIP, adjustment=Adjustment.RAW))
            if not any(bars.data.values()):
                raise ValueError("Keine SIP-Kursdaten geliefert")
            return normalize_bars(bars.df)
        current = fetch(c.symbols, session["open"], session["close"])
        current = current[(current.timestamp >= session["open"]) & (current.timestamp < session["close"])].copy()
        csv_write(out / "session_bars_sip.csv", current)
        ranges, decisions = scan_day(current, session["open"], session["close"], c)
        csv_write(out / "opening_ranges.csv", ranges)
        csv_write(out / "minute_decisions.csv", decisions)
        candidates = decisions[decisions.result == "PRICE_CANDIDATE"].copy()

        print(f"4/4 Prüfe Volumen für {len(candidates)} Preiskandidat(en) …")
        if not candidates.empty:
            past = calendar_rows(broker.get_calendar(GetCalendarRequest(
                start=day - timedelta(days=max(60, c.rvol_days * 3)), end=day - timedelta(days=1))))[-c.rvol_days:]
            if len(past) != c.rvol_days:
                raise ValueError("Nicht genug historische Vergleichstage")
            json_write(out / "reference_calendar.json", past)
            history = fetch(sorted(candidates.symbol.unique()), past[0]["open"], past[-1]["close"])
            csv_write(out / "reference_bars_sip.csv", history)
            volumes, reference_checks = check_rvol(candidates, current, history, past, session["open"], c)
        else:
            volumes = pd.DataFrame(columns=RVOL_COLUMNS)
            reference_checks = pd.DataFrame(columns=["symbol", "signal_time_utc", "reference_date", "volume", "status"])
        csv_write(out / "volume_candidates.csv", volumes)
        csv_write(out / "volume_reference_checks.csv", reference_checks)
        summary_rows = []
        for symbol in c.symbols:
            signal_rows = candidates[candidates.symbol == symbol]
            volume_rows = volumes[volumes.Aktie == symbol]
            pass_count = int((volume_rows.Ergebnis == "VOLUME_PASS").sum())
            unknown = int((volume_rows.Ergebnis == "NOT_CHECKABLE").sum())
            summary_rows.append(dict(Aktie=symbol, Preiskandidaten=len(signal_rows),
                Volumen_bestanden=pass_count, Volumen_nicht_prüfbar=unknown,
                Ergebnis=("Weitere Filter offen" if pass_count else
                          "Volumendaten unzureichend" if unknown else
                          "Volumenfilter nicht bestanden" if len(signal_rows) else "Kein Preiskandidat")))
        summary = pd.DataFrame(summary_rows)
        csv_write(out / "summary.csv", summary)
        json_write(out / "completed.json", dict(status="completed", reference_tests_passed=16,
            price_candidates=len(candidates), volume_pass=int((volumes.Ergebnis == "VOLUME_PASS").sum()),
            order_methods_called=0, executions_simulated=False,
            finished_at_utc=datetime.now(timezone.utc).isoformat()))
        return dict(folder=out, account=snapshot, tests=tests, ranges=ranges,
                    volumes=volumes, summary=summary)
    except Exception as exc:
        # Keine API-Antworten, Zugangsdaten oder vollständigen Accountobjekte protokollieren.
        status = getattr(exc, "status_code", None)
        detail = str(exc) if type(exc) is ValueError else "API-/Laufzeitfehler; HTTP-Status und Fehlertyp prüfen"
        json_write(out / "failed.json", dict(status="failed", type=type(exc).__name__,
            http_status=status, detail=detail, time_utc=datetime.now(timezone.utc).isoformat()))
        raise RuntimeError(f"Testlauf gestoppt: {detail}. Typ: {type(exc).__name__}; HTTP: {status}. Protokoll: {out}") from None


In [ ]:
%%writefile /content/us_orb_scanner_v03.py
"""V0.3 historical scanner. Read-only; no order submission or fill simulation."""
from dataclasses import dataclass, asdict, replace
from datetime import datetime, timedelta, timezone
from pathlib import Path
from types import SimpleNamespace
import hashlib
import importlib.metadata
import json
import time
import urllib.request
import uuid
import numpy as np
import pandas as pd
import us_orb_test_v02 as base

VERSION = '0.3-scanner-draft'
MEMBERS_URL = 'https://api.nasdaq.com/api/quote/list-type/nasdaq100'
SECTORS_URL = 'https://api.nasdaq.com/api/screener/stocks?tableonly=true&limit=10000&download=true'
EMPTY_BARS = ['symbol', 'timestamp', 'open', 'high', 'low', 'close', 'volume', 'vwap']

@dataclass(frozen=True)
class ScannerConfig:
    minimum_average_dollar_volume: float = 50_000_000
    maximum_spread: float = 0.001  # 0.10%, divided by midpoint
    maximum_quote_age_seconds: float = 2.0
    quote_search_seconds: int = 10
    watchlist_size: int = 5
    signal_ttl_seconds: int = 60
    weight_rvol: float = 40
    weight_relative_strength: float = 25
    weight_breakout: float = 20
    weight_spread: float = 15
    rvol_score_ceiling: float = 3.0
    relative_strength_score_ceiling: float = 0.01  # 1 percentage point
    batch_size: int = 20
    request_spacing_seconds: float = 0.4


def validate_scanner(c, s):
    base.validate_config(c)
    for name, value in asdict(s).items():
        if not np.isfinite(value) or value <= 0:
            raise ValueError(f'Ungültiger Scannerparameter: {name}')
    if s.watchlist_size > 5 or s.rvol_score_ceiling <= c.minimum_rvol:
        raise ValueError('Ungültige Watchlist oder RVOL-Skalierung')
    if s.quote_search_seconds < s.maximum_quote_age_seconds:
        raise ValueError('Quote-Suchfenster ist zu kurz')
    if s.signal_ttl_seconds > 60:
        raise ValueError('Diese Version unterstützt höchstens 60 Sekunden Signalgültigkeit')
    if float(c.entry_cap) <= float(c.breakout_buffer):
        raise ValueError('Ranking benötigt einen positiven Preisabstandsbereich')
    if not np.isclose(sum([s.weight_rvol, s.weight_relative_strength,
                           s.weight_breakout, s.weight_spread]), 100):
        raise ValueError('Rankinggewichte müssen zusammen 100 ergeben')


def fetch_json(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0', 'Accept': 'application/json'})
    with urllib.request.urlopen(req, timeout=20) as response:
        return json.load(response)


def assemble_universe(members, sectors):
    rows = members['data']['data']['rows']
    sector_rows = sectors['data']['rows']
    mapping = {}
    for row in sector_rows:
        symbol = row['symbol'].strip().upper()
        mapping.setdefault(symbol, set()).add((row.get('sector') or '').strip())
    output = []
    for row in rows:
        symbol = row['symbol'].strip().upper()
        choices = mapping.get(symbol, set()) - {''}
        output.append(dict(symbol=symbol, company=row.get('companyName', ''),
                           sector=next(iter(choices)) if len(choices) == 1 else ''))
    frame = pd.DataFrame(output)
    if len(frame) < 90 or len(frame) > 120 or frame.symbol.duplicated().any():
        raise ValueError('Nasdaq-Mitgliederliste ist unplausibel')
    return frame.sort_values('symbol').reset_index(drop=True)


def load_universe(out, embedded):
    # This is a diagnostic snapshot, NEVER a claimed historical point-in-time universe.
    try:
        members = fetch_json(MEMBERS_URL)
        sectors = fetch_json(SECTORS_URL)
        frame = assemble_universe(members, sectors)
    except Exception as exc:
        frame = pd.DataFrame(embedded['rows'])
        metadata = {k: v for k, v in embedded.items() if k != 'rows'}
        metadata.update(source_mode='embedded_snapshot', refresh_failure_type=type(exc).__name__)
        print('Nasdaq-Aktualisierung nicht verfügbar; verwende gekennzeichneten Snapshot vom',
              metadata['retrieved_at_utc'])
    else:
        base.json_write(out / 'nasdaq_members_source.json', members)
        base.json_write(out / 'nasdaq_sectors_source.json', sectors)
        metadata = dict(source_mode='fresh_current_snapshot',
                        retrieved_at_utc=datetime.now(timezone.utc).isoformat(),
                        quote_date_label=members['data'].get('date'),
                        member_source=MEMBERS_URL, sector_source=SECTORS_URL)
    if frame.empty or frame.symbol.duplicated().any() or not {'symbol', 'sector'}.issubset(frame.columns):
        raise ValueError('Universum fehlt oder ist uneindeutig')
    metadata.update(historical_membership_verified=False, historical_sectors_verified=False,
                    scope='Retrospektiver Funktionstest mit einem datierten Mitglieder-/Sektorsnapshot')
    base.csv_write(out / 'universe_snapshot.csv', frame)
    base.json_write(out / 'universe_provenance.json', metadata)
    return frame, metadata


def liquidity_check(daily, symbol, dates, s):
    rows = daily[daily.symbol == symbol].copy()
    if not rows.empty:
        rows['session_date'] = rows.timestamp.dt.tz_convert(base.NY).dt.strftime('%Y-%m-%d')
        rows = rows[rows.session_date.isin(dates)].sort_values('session_date')
    if len(rows) != len(dates) or rows.empty or set(rows.session_date) != set(dates):
        return dict(average_dollar_volume=None, liquidity_result='DAILY_HISTORY_INCOMPLETE')
    if rows.session_date.duplicated().any() or 'vwap' not in rows:
        return dict(average_dollar_volume=None, liquidity_result='DAILY_HISTORY_INVALID')
    values = rows[['vwap', 'volume']].to_numpy(dtype=float)
    if not np.isfinite(values).all() or (values <= 0).any():
        return dict(average_dollar_volume=None, liquidity_result='DAILY_HISTORY_INVALID')
    average = float((rows.vwap * rows.volume).mean())
    return dict(average_dollar_volume=average,
                liquidity_result='PASS' if average >= s.minimum_average_dollar_volume else 'ILLIQUID')


def scan_prefix(current, symbol, opening, closing, c):
    """Only a complete prefix can generate a signal; later gaps do not erase earlier signals."""
    or_end = opening + pd.Timedelta(minutes=c.opening_minutes)
    cutoff = closing - pd.Timedelta(minutes=c.entry_cutoff_minutes)
    if or_end >= cutoff:
        raise ValueError('Handelssitzung ist zu kurz')
    current = current[current.symbol == symbol]
    initial = base.checked_window(current, symbol, opening, or_end)
    high, low = base.D(initial.high.max()), base.D(initial.low.min())
    threshold = high * (1 + base.D(c.breakout_buffer))
    cap = high * (1 + base.D(c.entry_cap))
    previous = base.D(initial.close.iloc[-1])
    prefix_valid, decisions = True, []
    for stamp in pd.date_range(or_end, cutoff - pd.Timedelta(minutes=1), freq='min', inclusive='left'):
        decision = stamp + pd.Timedelta(minutes=1)
        close = None
        if prefix_valid:
            try:
                one = base.checked_window(current, symbol, stamp, decision)
                close = base.D(one.close.iloc[0])
            except ValueError:
                prefix_valid = False
        if not prefix_valid:
            reason = 'INCOMPLETE_DATA_PREFIX'
        elif not (previous <= threshold and close > threshold):
            reason = 'NO_CROSSING'
        elif close < base.D(c.minimum_price):
            reason = 'BELOW_MINIMUM_PRICE'
        elif close > cap:
            reason = 'TOO_EXTENDED'
        else:
            reason = 'PRICE_CANDIDATE'
        decisions.append(dict(symbol=symbol, bar_time_utc=stamp.isoformat(),
            decision_time_utc=decision.isoformat(), decision_time_ny=decision.tz_convert(base.NY).isoformat(),
            or_high=float(high), threshold=float(threshold), price_cap=float(cap),
            close=float(close) if close is not None else None, result=reason))
        if close is not None:
            previous = close
    opening_range = dict(symbol=symbol, or_high=float(high), or_low=float(low),
                         or_open=float(initial.open.iloc[0]), or_volume=float(initial.volume.sum()))
    return opening_range, pd.DataFrame(decisions)


def quote_check(quote, decision, threshold, cap, s, minimum_price=20.0):
    result = dict(quote_result='QUOTE_MISSING', quote_time_utc=None,
                  bid=None, ask=None, bid_size=None, ask_size=None,
                  quote_age_seconds=None, spread=None)
    if quote is None:
        return result
    try:
        stamp = base.utc(quote.timestamp)
        bid, ask, bsize, asize = map(float, [quote.bid_price, quote.ask_price, quote.bid_size, quote.ask_size])
        age = (base.utc(decision) - stamp).total_seconds()
        result.update(quote_time_utc=stamp.isoformat(), bid=bid, ask=ask,
                      bid_size=bsize, ask_size=asize, quote_age_seconds=age)
        if not np.isfinite([bid, ask, bsize, asize, age]).all() or min(bid, ask, bsize, asize) <= 0 or bid > ask:
            result['quote_result'] = 'QUOTE_INVALID'
        elif age < 0:
            result['quote_result'] = 'QUOTE_FROM_FUTURE'
        elif age > s.maximum_quote_age_seconds:
            result['quote_result'] = 'QUOTE_STALE'
        else:
            spread = (ask - bid) / ((ask + bid) / 2)
            result['spread'] = spread
            result['quote_result'] = ('ASK_BELOW_MINIMUM_PRICE' if ask < minimum_price else
                                     'SPREAD_TOO_WIDE' if spread > s.maximum_spread else
                                     'ASK_OUTSIDE_ENTRY_BAND' if not float(threshold) < ask <= float(cap) else 'PASS')
    except (AttributeError, TypeError, ValueError):
        result['quote_result'] = 'QUOTE_INVALID'
    return result


def timing_status(decision, started_at, now, cutoff, reconciled, already_seen, s):
    decision, started, now, cutoff = map(base.utc, [decision, started_at, now, cutoff])
    if decision < started:
        return 'HISTORICAL_ONLY'
    if decision > now:
        return 'FUTURE_SIGNAL'
    if decision >= cutoff or now >= cutoff:
        return 'ENTRY_CUTOFF'
    if (now - decision).total_seconds() >= s.signal_ttl_seconds:
        return 'EXPIRED'
    if already_seen:
        return 'DUPLICATE_SIGNAL'
    if not reconciled:
        return 'BROKER_RECONCILIATION_REQUIRED'
    return 'CURRENT_SIGNAL_REQUIRES_ORDER_ENGINE'  # No trade authorization here.


def rank_candidates(frame, c, s):
    if frame.empty:
        return pd.DataFrame(columns=list(frame.columns) + ['score', 'rank', 'watchlist', 'ranking_result'])
    if frame.duplicated(['decision_time_utc', 'symbol']).any():
        raise ValueError('Doppeltes Signal in Rangfolge')
    ranked = frame.copy()
    ranked['rvol_component'] = ((ranked.rvol - c.minimum_rvol) /
                                (s.rvol_score_ceiling - c.minimum_rvol)).clip(0, 1) * s.weight_rvol
    ranked['strength_component'] = (ranked.relative_strength / s.relative_strength_score_ceiling).clip(0, 1) * s.weight_relative_strength
    ranked['breakout_component'] = ((ranked.breakout - float(c.breakout_buffer)) /
                                    (float(c.entry_cap) - float(c.breakout_buffer))).clip(0, 1) * s.weight_breakout
    ranked['spread_component'] = (1 - ranked.spread / s.maximum_spread).clip(0, 1) * s.weight_spread
    ranked['score'] = ranked[['rvol_component', 'strength_component', 'breakout_component', 'spread_component']].sum(axis=1)
    ranked = ranked.sort_values(['decision_time_utc', 'score', 'rvol', 'spread', 'symbol'],
                                 ascending=[True, False, False, True, True], kind='stable')
    ranked['rank'] = ranked.groupby('decision_time_utc').cumcount() + 1
    ranked['watchlist'] = ranked['rank'] <= s.watchlist_size
    ranked['ranking_result'] = np.where(ranked.watchlist, 'WATCHLIST_ONLY', 'OUTSIDE_TOP_FIVE')
    return ranked.reset_index(drop=True)


def reservation_test_model(ranked, portfolio, c):
    """Pure synthetic reservation model; not used to imply historical trades."""
    sectors = list(portfolio['occupied_sectors'])
    held = set(portfolio['occupied_symbols'])
    cash, risk = base.D(portfolio['free_cash']), base.D(portfolio['existing_risk'])
    output = []
    for row in ranked[ranked.watchlist].itertuples():
        qty, reason = 0, 'RESERVED_IN_SYNTHETIC_TEST'
        gate = base.entry_gate(portfolio['start_equity'], portfolio['equity'],
            portfolio['confirmed_flows'], sectors, row.sector, portfolio['day_locked'], c)
        if not portfolio['reconciled']:
            reason = 'RECONCILIATION_REQUIRED'
        elif row.symbol in held:
            reason = 'SYMBOL_ALREADY_OCCUPIED'
        elif not gate['allowed']:
            reason = '; '.join(gate['reasons'])
        else:
            size = base.size_position(portfolio['equity'], cash, risk, row.ask, c)
            qty = size['qty']
            if qty == 0:
                reason = 'NO_CAPITAL_OR_RISK_BUDGET'
            else:
                cash -= size['notional']
                risk += size['planned_risk']
                sectors.append(row.sector)
                held.add(row.symbol)
        output.append(dict(symbol=row.symbol, qty=qty, reason=reason,
                           remaining_cash=float(cash), reserved_risk=float(risk)))
    return pd.DataFrame(output)


def scanner_tests():
    c, s = base.Config(), ScannerConfig()
    stamp = pd.Timestamp('2026-09-22 14:00:00Z')
    tests = []
    def check(name, condition):
        tests.append(dict(Bereich='Scanner', Test=name, Bestanden=bool(condition)))
    def quote(age=0, bid=100, ask=100.05):
        return SimpleNamespace(timestamp=stamp-pd.Timedelta(seconds=age), bid_price=bid,
                               ask_price=ask, bid_size=10, ask_size=10)
    check('Frischer enger Spread', quote_check(quote(), stamp, 100.01, 100.3, s)['quote_result'] == 'PASS')
    check('Veraltete Quote', quote_check(quote(age=3), stamp, 100.01, 100.3, s)['quote_result'] == 'QUOTE_STALE')
    check('Quote aus Zukunft', quote_check(quote(age=-1), stamp, 100.01, 100.3, s)['quote_result'] == 'QUOTE_FROM_FUTURE')
    check('Breiter Spread', quote_check(quote(ask=100.2), stamp, 100.01, 100.3, s)['quote_result'] == 'SPREAD_TOO_WIDE')
    check('Briefkurs ueber Preislimit', quote_check(quote(bid=100.4, ask=100.41), stamp, 100.01, 100.3, s)['quote_result'] == 'ASK_OUTSIDE_ENTRY_BAND')
    cutoff = stamp + pd.Timedelta(hours=4)
    check('Spaetstart kauft kein altes Signal', timing_status(stamp, stamp+pd.Timedelta(seconds=1), stamp+pd.Timedelta(seconds=1), cutoff, True, False, s) == 'HISTORICAL_ONLY')
    check('Neues Signal braucht Brokerabgleich', timing_status(stamp, stamp, stamp, cutoff, False, False, s) == 'BROKER_RECONCILIATION_REQUIRED')
    check('Signal bei 60 Sekunden abgelaufen', timing_status(stamp, stamp, stamp+pd.Timedelta(seconds=60), cutoff, True, False, s) == 'EXPIRED')
    check('Kein doppeltes Signal', timing_status(stamp, stamp, stamp, cutoff, True, True, s) == 'DUPLICATE_SIGNAL')
    check('Frisches Signal bleibt ohne Orderfreigabe', timing_status(stamp, stamp, stamp, cutoff, True, False, s) == 'CURRENT_SIGNAL_REQUIRES_ORDER_ENGINE')
    rows = [dict(symbol=symbol, sector='Tech' if i < 3 else 'Other', decision_time_utc=stamp.isoformat(),
                 rvol=2, relative_strength=.005, breakout=.002, spread=.0005, ask=100.05)
            for i, symbol in enumerate('ABCDEF')]
    ranked = rank_candidates(pd.DataFrame(rows[::-1]), c, s)
    check('Sechs gleiche Signale deterministisch / Top 5', list(ranked.symbol) == list('ABCDEF') and ranked.watchlist.sum() == 5)
    portfolio = dict(occupied_sectors=[], occupied_symbols=[], free_cash=10000, existing_risk=0,
                     start_equity=10000, equity=10000, confirmed_flows=0, day_locked=False, reconciled=True)
    reservations = reservation_test_model(ranked, portfolio, c)
    check('Maximal 3 Plaetze und 2 je Sektor', list(reservations[reservations.qty > 0].symbol) == ['A', 'B', 'D'])
    poor = reservation_test_model(ranked, dict(portfolio, free_cash=150), c)
    check('Cash wird zwischen Reservierungen abgezogen', poor.qty.sum() == 1 and poor.remaining_cash.min() >= 0)
    locked = reservation_test_model(ranked, dict(portfolio, day_locked=True), c)
    check('Tagessperre blockiert alle Reservierungen', locked.qty.sum() == 0)
    other = pd.DataFrame([dict(rows[0], symbol='Z', decision_time_utc=(stamp+pd.Timedelta(minutes=1)).isoformat(), rvol=3)])
    mixed = rank_candidates(pd.concat([pd.DataFrame(rows), other]), c, s)
    check('Spaeteres Signal aendert frueheres Ranking nicht', list(mixed[mixed.decision_time_utc == stamp.isoformat()].symbol) == list('ABCDEF'))
    return pd.DataFrame(tests), reservations


def run_scanner(broker, data_client, project_root, c, s, embedded):
    from alpaca.trading.requests import GetCalendarRequest, GetAssetsRequest, GetOrdersRequest
    from alpaca.trading.enums import AssetStatus, AssetClass, QueryOrderStatus
    from alpaca.data.requests import StockBarsRequest, StockQuotesRequest
    from alpaca.data.enums import DataFeed, Adjustment
    from alpaca.data.timeframe import TimeFrame
    from alpaca.common.enums import Sort
    validate_scanner(c, s)
    started = pd.Timestamp.now(tz='UTC')
    out = Path(project_root) / 'v0_3_runs' / (started.strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8])
    out.mkdir(parents=True, exist_ok=False)
    print('Ergebnisordner:', out)
    sources = {}
    for name, path in [('scanner_source.py', __file__), ('base_source.py', base.__file__)]:
        content = Path(path).read_text(encoding='utf-8')
        (out / name).write_text(content, encoding='utf-8')
        sources[name] = hashlib.sha256(content.encode()).hexdigest()
    base.json_write(out / 'manifest.json', dict(version=VERSION, created_at_utc=started.isoformat(),
        mode='historical_read_only_snapshot_universe', base_config=asdict(c), scanner_config=asdict(s),
        code_hashes=sources, packages={p: importlib.metadata.version(p) for p in ['alpaca-py', 'pandas', 'numpy']},
        membership_is_point_in_time=False, orders_enabled=False,
        assumptions=['Alle neuen Scannerparameter sind unvalidierte Entwurfswerte.',
                     'Aktuelle Indexmitglieder, Sektoren und Broker-Handelbarkeit; keine historischen Stammdaten.',
                     'RAW-Daten; Kapitalmassnahmen und spaetere Datenkorrekturen nicht rekonstruiert.',
                     'Keine Ausfuehrungssimulation oder Performance; keine Portfoliopositionen aus Rankings.']))
    last_request = [0.0]
    def call(method, *args, **kwargs):
        wait = s.request_spacing_seconds - (time.monotonic() - last_request[0])
        if wait > 0:
            time.sleep(wait)
        last_request[0] = time.monotonic()
        return method(*args, **kwargs)
    def fetch_bars(symbols, start, end, timeframe):
        frames = []
        for offset in range(0, len(symbols), s.batch_size):
            result = call(data_client.get_stock_bars, StockBarsRequest(
                symbol_or_symbols=list(symbols[offset:offset+s.batch_size]), timeframe=timeframe,
                start=start.to_pydatetime(), end=end.to_pydatetime(), feed=DataFeed.SIP,
                adjustment=Adjustment.RAW, asof=c.test_date))
            if any(result.data.values()):
                frames.append(base.normalize_bars(result.df))
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=EMPTY_BARS)
    try:
        print('1/6 Referenztests und Scanner-Grenzfaelle …')
        old_tests, stops = base.self_tests()
        new_tests, reservations = scanner_tests()
        tests = pd.concat([old_tests, new_tests], ignore_index=True)
        base.csv_write(out / 'self_tests.csv', tests)
        base.csv_write(out / 'synthetic_stop_paths.csv', stops)
        base.csv_write(out / 'synthetic_reservations.csv', reservations)
        if not tests.Bestanden.all():
            raise ValueError('Referenztest fehlgeschlagen')
        print(f'✅ {len(tests)}/{len(tests)} Referenztests bestanden')
        account = call(broker.get_account)
        clock = call(broker.get_clock)
        positions = call(broker.get_all_positions)
        orders = call(broker.get_orders, filter=GetOrdersRequest(status=QueryOrderStatus.OPEN, limit=500))
        base.json_write(out / 'paper_account_snapshot.json', dict(
            read_at_utc=pd.Timestamp.now(tz='UTC').isoformat(), equity=str(account.equity), cash=str(account.cash),
            currency=account.currency, account_blocked=account.account_blocked,
            trading_blocked=account.trading_blocked, positions_count=len(positions),
            open_orders_returned=len(orders), orders_may_be_truncated=len(orders) >= 500,
            scope='Nur Bestandsaufnahme; kein operativer Brokerabgleich oder Backtest-Portfolio'))
        day = datetime.strptime(c.test_date, '%Y-%m-%d').date()
        calendar = base.calendar_rows(call(broker.get_calendar, GetCalendarRequest(
            start=day-timedelta(days=max(70, c.rvol_days*4)), end=day)))
        session_rows = [r for r in calendar if r['date'] == c.test_date]
        past = [r for r in calendar if r['date'] < c.test_date][-c.rvol_days:]
        if len(session_rows) != 1 or len(past) != c.rvol_days:
            raise ValueError('Handelstag oder 20 Vergleichssitzungen fehlen')
        session = session_rows[0]
        if session['close'] > base.utc(clock.timestamp)-pd.Timedelta(minutes=15):
            raise ValueError('Bitte einen abgeschlossenen Handelstag mit mindestens 15 Minuten Abstand waehlen')
        base.json_write(out / 'calendar.json', dict(session=session, references=past))
        print('2/6 Nasdaq-Mitglieder, Sektoren und aktuelle Alpaca-Handelbarkeit …')
        universe, provenance = load_universe(out, embedded)
        assets = call(broker.get_all_assets, filter=GetAssetsRequest(status=AssetStatus.ACTIVE, asset_class=AssetClass.US_EQUITY))
        tradeable = {a.symbol for a in assets if a.tradable}
        universe['tradable_now'] = universe.symbol.isin(tradeable)
        print(f'✅ {len(universe)} Symbole; Mitgliederstand: {provenance["retrieved_at_utc"]}')
        print('3/6 Liquiditaet aus 20 vorherigen Tageskerzen (VWAP × Volumen) …')
        daily = fetch_bars(list(universe.symbol), past[0]['open'].normalize(), session['open'].normalize(), TimeFrame.Day)
        base.csv_write(out / 'liquidity_daily_sip.csv', daily)
        audit = []
        dates = [r['date'] for r in past]
        for row in universe.itertuples():
            liquidity = liquidity_check(daily, row.symbol, dates, s)
            reason = ('SECTOR_UNKNOWN' if not row.sector else 'NOT_TRADABLE_NOW' if not row.tradable_now else liquidity['liquidity_result'])
            audit.append(dict(symbol=row.symbol, sector=row.sector, tradable_now=row.tradable_now,
                              **liquidity, universe_result=reason))
        audit = pd.DataFrame(audit)
        base.csv_write(out / 'universe_filters.csv', audit)
        eligible = list(audit.loc[audit.universe_result == 'PASS', 'symbol'])
        print(f'✅ {len(eligible)}/{len(universe)} Symbole bestehen den Vorfilter')
        print('4/6 Minutenkurse, Opening Ranges und Ausbrueche …')
        current = fetch_bars(sorted(set(eligible + ['QQQ'])), session['open'], session['close'], TimeFrame.Minute)
        base.csv_write(out / 'session_bars_sip.csv', current)
        range_rows, decision_frames, scan_status = [], [], []
        for symbol_number, symbol in enumerate(eligible, 1):
            if symbol_number == 1 or symbol_number % 10 == 0:
                print(f'  Opening Range und Minutenentscheidungen: {symbol_number}/{len(eligible)}', flush=True)
            try:
                opening_range, decisions = scan_prefix(current, symbol, session['open'], session['close'], c)
            except ValueError:
                scan_status.append(dict(symbol=symbol, scan_result='OPENING_RANGE_DATA_INVALID', price_candidates=0))
                continue
            range_rows.append(opening_range)
            decision_frames.append(decisions)
            scan_status.append(dict(symbol=symbol, scan_result='PREFIX_DATA_GAP' if (decisions.result == 'INCOMPLETE_DATA_PREFIX').any() else 'CHECKED',
                                    price_candidates=int((decisions.result == 'PRICE_CANDIDATE').sum())))
        decisions = pd.concat(decision_frames, ignore_index=True) if decision_frames else pd.DataFrame(columns=['symbol', 'result', 'decision_time_utc'])
        ranges = pd.DataFrame(range_rows, columns=['symbol', 'or_high', 'or_low', 'or_open', 'or_volume'])
        base.csv_write(out / 'opening_ranges.csv', ranges)
        base.csv_write(out / 'minute_decisions.csv', decisions)
        base.csv_write(out / 'scan_status.csv', pd.DataFrame(scan_status, columns=['symbol', 'scan_result', 'price_candidates']))
        candidates = decisions[decisions.result == 'PRICE_CANDIDATE'].copy()
        print(f'5/6 RVOL fuer {len(candidates)} Preisereignisse; Referenzdaten tageweise …')
        history_symbols = sorted(candidates.symbol.unique())
        history_frames = []
        history_dir = out / 'rvol_reference_bars'
        history_dir.mkdir()
        if history_symbols:
            for number, reference in enumerate(past, 1):
                frame = fetch_bars(history_symbols, reference['open'], reference['close'], TimeFrame.Minute)
                base.csv_write(history_dir / (reference['date'] + '.csv'), frame)
                history_frames.append(frame)
                print(f'  RVOL-Referenztag {number}/{len(past)} gespeichert', flush=True)
        history = pd.concat(history_frames, ignore_index=True) if history_frames else pd.DataFrame(columns=EMPTY_BARS)
        volume_frames, reference_frames = [], []
        for symbol, events in candidates.groupby('symbol', sort=True):
            symbol_volumes, symbol_references = base.check_rvol(events,
                current[current.symbol == symbol], history[history.symbol == symbol], past, session['open'], c)
            volume_frames.append(symbol_volumes)
            reference_frames.append(symbol_references)
            print(f'  RVOL geprueft: {symbol} ({len(events)} Preisereignisse)', flush=True)
        volumes = pd.concat(volume_frames, ignore_index=True) if volume_frames else pd.DataFrame(columns=base.RVOL_COLUMNS)
        references = pd.concat(reference_frames, ignore_index=True) if reference_frames else pd.DataFrame(columns=['symbol', 'signal_time_utc', 'reference_date', 'volume', 'status'])
        base.csv_write(out / 'volume_candidates.csv', volumes)
        base.csv_write(out / 'volume_reference_checks.csv', references)
        print('6/6 Relative Staerke vs. QQQ, historische SIP-Quotes und Rangfolge …')
        volume_lookup = {(r.Aktie, r.decision_time_utc): r for r in volumes.itertuples()}
        sectors = dict(zip(universe.symbol, universe.sector))
        filtered = []
        for candidate in candidates.itertuples():
            decision = base.utc(candidate.decision_time_utc)
            row = candidate._asdict()
            row.update(sector=sectors[candidate.symbol],
                       signal_id=f'{c.test_date}|{candidate.symbol}|{decision.isoformat()}|ORB_LONG',
                       timing_status=timing_status(decision, started, started,
                           session['close']-pd.Timedelta(minutes=c.entry_cutoff_minutes), False, False, s),
                       rvol=None, relative_strength=None, breakout=candidate.close/candidate.or_high-1,
                       filter_result='PENDING', quote_result='NOT_REQUESTED', spread=None, ask=None)
            volume = volume_lookup[(candidate.symbol, candidate.decision_time_utc)]
            row['rvol'] = volume.RVOL
            if volume.Ergebnis != 'VOLUME_PASS':
                row['filter_result'] = volume.Ergebnis
            else:
                try:
                    qqq = base.checked_window(current, 'QQQ', session['open'], decision)
                    stock = base.checked_window(current, candidate.symbol, session['open'], decision)
                    row['relative_strength'] = float(stock.close.iloc[-1]/stock.open.iloc[0] - qqq.close.iloc[-1]/qqq.open.iloc[0])
                except ValueError:
                    row['filter_result'] = 'BENCHMARK_OR_STOCK_PREFIX_INVALID'
                else:
                    quotes = call(data_client.get_stock_quotes, StockQuotesRequest(
                        symbol_or_symbols=candidate.symbol, start=(decision-pd.Timedelta(seconds=s.quote_search_seconds)).to_pydatetime(),
                        end=decision.to_pydatetime(), sort=Sort.DESC, limit=1, feed=DataFeed.SIP, asof=c.test_date))
                    items = quotes.data.get(candidate.symbol, [])
                    quote = max(items, key=lambda q: base.utc(q.timestamp)) if items else None
                    row.update(quote_check(quote, decision, candidate.threshold, candidate.price_cap, s, c.minimum_price))
                    row['filter_result'] = row['quote_result']
            filtered.append(row)
        filters = pd.DataFrame(filtered) if filtered else pd.DataFrame(columns=['symbol', 'decision_time_utc', 'sector', 'rvol', 'relative_strength', 'breakout', 'spread', 'ask', 'filter_result'])
        base.csv_write(out / 'candidate_filters.csv', filters)
        ranked = rank_candidates(filters[filters.filter_result == 'PASS'], c, s)
        base.csv_write(out / 'ranked_candidates.csv', ranked)
        watchlist = ranked[ranked.watchlist == True].copy()
        base.csv_write(out / 'watchlist_by_minute.csv', watchlist)
        summary = audit.merge(pd.DataFrame(scan_status, columns=['symbol', 'scan_result', 'price_candidates']), on='symbol', how='left')
        counts = filters.loc[filters.filter_result == 'PASS'].groupby('symbol').size()
        summary['scanner_pass_count'] = summary.symbol.map(counts).fillna(0).astype(int)
        summary['price_candidates'] = summary.price_candidates.fillna(0).astype(int)
        base.csv_write(out / 'summary.csv', summary)
        base.json_write(out / 'completed.json', dict(status='completed', tests_passed=len(tests),
            universe_symbols=len(universe), prefilter_pass=len(eligible), price_events=len(candidates),
            scanner_pass=len(ranked), watchlist_events=len(watchlist), orders_sent=0,
            portfolio_simulated=False, finished_at_utc=pd.Timestamp.now(tz='UTC').isoformat()))
        return dict(folder=out, tests=tests, summary=summary, filters=filters, ranked=ranked,
                    watchlist=watchlist, provenance=provenance)
    except Exception as exc:
        status = getattr(exc, 'status_code', None)
        detail = str(exc) if type(exc) is ValueError else 'API-/Laufzeitfehler; siehe Typ und HTTP-Status'
        base.json_write(out / 'failed.json', dict(status='failed', error_type=type(exc).__name__,
            http_status=status, detail=detail, orders_sent=0))
        raise RuntimeError(f'Scanner gestoppt: {detail}; Typ {type(exc).__name__}; HTTP {status}. Protokoll: {out}') from None


In [ ]:
%%writefile /content/bot_mode_v16.py
"""V1.6 operating mode: exactly one main mode, and who may send orders.

    OBSERVE   BOT 2 only: SIP 16 min delayed, SIMULATED            (V1.5 SIP_ONLY)
    DRY_RUN   BOT 1: IEX real time, internal simulation, no orders (V1.5 IEX_ONLY, no approval)
    PAPER     BOT 1: IEX real time + Alpaca PAPER orders           (V1.5 IEX_ONLY + approval)
    BACKTEST  historical SIP replay, simulated execution, never orders

Optional for DRY_RUN and PAPER: the SIP comparison bot in parallel (V1.5 DUAL_MODE).
It only observes and compares; it never receives an order client.

The mode is held here, in one place. The order layer (paper_orders_v16) and the
simulation loop (orb_sim_v16) ask `assert_orders_allowed()` before any order path,
so BACKTEST, OBSERVE and DRY_RUN block orders technically, not only in the UI.
"""
from dataclasses import dataclass

MODES = ('OBSERVE', 'DRY_RUN', 'PAPER', 'BACKTEST')
LABELS = {'OBSERVE': 'OBSERVE', 'DRY_RUN': 'DRY RUN', 'PAPER': 'PAPER', 'BACKTEST': 'BACKTEST'}
BANNERS = {
    'OBSERVE': ('OBSERVE', 'SIP 16 Min. verzögert · nur beobachten · SIMULIERT · keine Orders'),
    'DRY_RUN': ('DRY RUN', 'IEX Echtzeit · interne Simulation · keine Orders'),
    'PAPER': ('PAPER TRADING', 'Alpaca Paper Account · Orders nur mit gültiger Tagesfreigabe'),
    'BACKTEST': ('BACKTEST', 'Historical Data · NO LIVE ORDERS'),
}
DEFAULT_MODE = 'OBSERVE'

_state = {'mode': None}


class OrdersBlocked(PermissionError):
    """An order path was reached in a mode that must never send orders."""


@dataclass(frozen=True)
class ModePlan:
    mode: str
    kind: str                 # 'live' (dual_bot_v16.run_session) or 'backtest'
    dual_mode: str = None     # V1.5 orchestration mode for live runs
    approval: str = None      # only PAPER passes the approval text on
    sip_compare: bool = False
    orders_possible: bool = False


def normalize(mode):
    value = str(mode or '').strip().upper().replace(' ', '_').replace('-', '_')
    if value not in MODES:
        raise ValueError(f'Unbekannter Modus {mode!r}; zulässig: {", ".join(MODES)}.')
    return value


def plan(mode, sip_compare=False, approval=None):
    """Map the one main mode (+ optional SIP comparison bot) to the V1.5 orchestration."""
    mode = normalize(mode)
    if sip_compare and mode not in ('DRY_RUN', 'PAPER'):
        raise ValueError('Der SIP-Vergleichsbot ist nur bei DRY RUN und PAPER wählbar.')
    if mode == 'OBSERVE':
        return ModePlan(mode, 'live', dual_mode='SIP_ONLY')
    if mode == 'BACKTEST':
        return ModePlan(mode, 'backtest')
    dual = 'DUAL_MODE' if sip_compare else 'IEX_ONLY'
    if mode == 'DRY_RUN':
        return ModePlan(mode, 'live', dual_mode=dual, sip_compare=bool(sip_compare))
    text = (approval or '').strip()
    if not text:
        raise ValueError('PAPER braucht die Tagesfreigabe „PAPER-ORDERS-OK <Paper-Kontonummer> '
                         '<YYYY-MM-DD>“. Ohne Freigabe bitte DRY RUN wählen.')
    return ModePlan(mode, 'live', dual_mode=dual, approval=text, sip_compare=bool(sip_compare),
                    orders_possible=True)


def set_mode(mode):
    _state['mode'] = normalize(mode)
    return _state['mode']


def current():
    return _state['mode']


def orders_allowed():
    return _state['mode'] == 'PAPER'


def assert_orders_allowed():
    """Raise unless the active main mode is PAPER (unset counts as 'not allowed')."""
    if not orders_allowed():
        raise OrdersBlocked(f'Orders gesperrt: aktiver Modus ist {_state["mode"] or "nicht gesetzt"}, '
                            'Orders gibt es nur im Modus PAPER.')


In [ ]:
%%writefile /content/orb_portfolio_v15.py
"""Only virtual long positions on one data feed (SIP or IEX); no Alpaca trading client or order methods."""
from collections import Counter
from dataclasses import replace
from datetime import timezone
import math

import numpy as np
import pandas as pd
import us_orb_test_v02 as base

BAR_COLUMNS = ['open', 'high', 'low', 'close', 'volume']


def stamp(value):
    return pd.Timestamp(value).tz_convert('UTC')


def clean_bars(current, symbol, start, end):
    """Valid bars of one symbol in [start, end) and the minutes of invalid ones.

    Duplicated or contradictory bars are data errors: only that minute is
    dropped, the symbol stays usable. numpy instead of many small pandas
    operations: this runs for every symbol in every minute."""
    start, end = stamp(start), stamp(end)
    rows = current[(current.symbol == symbol) & (current.timestamp >= start) &
                   (current.timestamp < end)]
    if rows.empty:
        return rows.set_index('timestamp')[BAR_COLUMNS], set()
    o, h, l, c, v = (pd.to_numeric(rows[column], errors='coerce').to_numpy(dtype=float)
                     for column in BAR_COLUMNS)
    with np.errstate(invalid='ignore'):
        valid = (np.isfinite(o) & np.isfinite(h) & np.isfinite(l) & np.isfinite(c) &
                 np.isfinite(v) & (o > 0) & (h > 0) & (l > 0) & (c > 0) & (v >= 0) &
                 (h >= np.maximum(np.maximum(o, c), l)) & (l <= np.minimum(np.minimum(o, c), h)) &
                 ~rows.timestamp.duplicated(keep=False).to_numpy())
    stamps = pd.DatetimeIndex(rows.timestamp)
    good = pd.DataFrame(dict(open=o[valid], high=h[valid], low=l[valid], close=c[valid],
                             volume=v[valid]), index=stamps[valid]).sort_index()
    return good, {stamp(value) for value in stamps[~valid]}


def fresh_bid(quote, when, max_age_seconds=2):
    """Bid of a checked exit quote if it is at most `max_age_seconds` old."""
    if not quote or quote.get('quote_result') != 'PASS' or not quote.get('bid'):
        return None
    age = (stamp(when) - stamp(quote['quote_time_utc'])).total_seconds()
    return float(quote['bid']) if 0 <= age <= max_age_seconds else None


STOP_REASONS = ('TRAILING_STOP_PROXY', 'STOP_GAP_OPEN_PROXY', 'STOP_BID_QUOTE_PROXY')


class ShadowPortfolio:
    def __init__(self, config, opening, planned_close, initial_cash=10000,
                 settle_seconds=120, max_gap_minutes=3, feed='sip', wall_clock=None,
                 trail_overrides=None, max_reentries=0, reentry_cooldown_minutes=15):
        self.c = config
        # Strategy variants (defaults = V1.5 rules): per-symbol trailing distance
        # (e.g. from ATR) and re-entries after a stop exit.
        self.trail_overrides = dict(trail_overrides or {})
        self.max_reentries = int(max_reentries)
        self.reentry_cooldown = pd.Timedelta(minutes=reentry_cooldown_minutes)
        self.entry_counts, self.last_exit = Counter(), {}
        self.feed = feed
        # market_event_time = `at_utc` (simulated/market time); processing_time = real time
        self.wall_clock = wall_clock or (lambda: pd.Timestamp.now(tz='UTC'))
        self.settle = pd.Timedelta(seconds=settle_seconds)
        self.stop_observation_grace = pd.Timedelta(seconds=15)
        self.max_gap_minutes = max_gap_minutes
        self.opening = stamp(opening)
        self.planned_close = stamp(planned_close)
        self.start_cash = float(initial_cash)
        self.cash = float(initial_cash)
        self.positions = {}
        self.closed = []
        self.events = []
        self.used_symbols = set()
        self.last_marks = {}
        self.last_decision = None
        self.locked = False
        self.data_degraded = False
        self.peak_equity = self.start_cash
        self.max_drawdown = 0.0
        self.last_equity = self.start_cash

    def log(self, kind, when, **fields):
        event = dict(kind=kind, at_utc=stamp(when).isoformat(),
                     processing_utc=self.wall_clock().isoformat(), **fields)
        self.events.append(event)
        return event

    def reject(self, when, symbol, reason):
        self.log('ENTRY_REJECTED', when, symbol=symbol, reason=reason)
        return reason

    def _mark(self, decision):
        if self.positions:
            stale = [s for s in self.positions if self.last_marks.get(s, (None, None))[1] is None]
            if stale:
                self.data_degraded = True
                return None
        equity = self.cash + sum(pos['qty'] * self.last_marks[s][1]
                                 for s, pos in self.positions.items())
        self.last_equity = equity
        self.peak_equity = max(self.peak_equity, equity)
        self.max_drawdown = max(self.max_drawdown, self.peak_equity - equity)
        if equity <= self.start_cash * (1 - float(self.c.daily_loss)):
            if not self.locked:
                self.log('DAILY_LOSS_LOCK', decision, equity=equity)
            self.locked = True
        return equity

    def update(self, decision, bars, now=None):
        """Process completed SIP bars; stop active from the next full minute.

        `last_bar` is the last *settled* minute. A minute without a bar stays
        pending (it may still be delivered) until a later bar exists or it is
        older than `settle`; then it counts as no observed SIP bar: the stop
        is unchanged and the minute is not stop-observed."""
        decision = stamp(decision)
        now = decision if now is None else stamp(now)
        if not bars.empty and ('feed' not in bars or not (bars.feed == self.feed).all()):
            raise ValueError(f'Virtuelles {self.feed.upper()}-Portfolio akzeptiert nur '
                             f'{self.feed.upper()}-Minutenkerzen.')
        if self.last_decision is not None and decision <= self.last_decision:
            raise ValueError('Virtuelle Entscheidungen dürfen nicht rückwärts laufen.')
        if self.last_decision is not None and decision - self.last_decision > pd.Timedelta(minutes=1):
            self.data_degraded = True
            self.log('MISSED_DECISIONS', decision,
                     count=int((decision-self.last_decision)/pd.Timedelta(minutes=1))-1)
        self.last_decision = decision
        minute = pd.Timedelta(minutes=1)
        for symbol, pos in list(self.positions.items()):
            if pos['force_gap_exit']:
                continue  # Await a fresh bid; do not use older candle prices.
            first = pos['last_bar'] + minute
            good, invalid = clean_bars(bars, symbol, first, decision)
            latest = good.index.max() if len(good) else None
            pos['pending_minutes'] = 0
            for when in pd.date_range(first, decision, freq='min', inclusive='left'):
                if when in good.index:
                    row = good.loc[when]
                    if now > when + minute + self.stop_observation_grace:
                        # A bar that surfaces after its decision cannot supply
                        # an executable stop fill at its historical OHLC.
                        self.data_degraded = True
                        pos['stop_coverage_unreliable'] = True
                        pos['needs_gap_quote'] = True
                        pos['unobserved_minutes'] += 1
                        pos['gap_run'] += 1
                        self.log('LATE_HELD_BAR_STOP_NOT_BACKDATED', now,
                                 symbol=symbol, bar_minute_utc=when.isoformat(),
                                 stop_before_gap=pos['stop'],
                                 reported_low=float(row.low))
                        pos['last_bar'] = when
                        if pos['gap_run'] > self.max_gap_minutes:
                            pos['force_gap_exit'] = True
                            break
                        continue
                    old_stop = pos['stop']
                    if float(row.open) <= old_stop:
                        fill, reason = float(row.open), 'STOP_GAP_OPEN_PROXY'
                    elif float(row.low) <= old_stop:
                        fill, reason = old_stop, 'TRAILING_STOP_PROXY'
                    else:
                        fill, reason = None, None
                    if fill is not None:
                        self.close(symbol, now, fill, reason,
                                   trigger_bar_minute=when)
                        break
                    pos['high_water'] = max(pos['high_water'], float(row.high))
                    pos['low_water'] = min(pos['low_water'], float(row.low))
                    pos['stop'] = max(old_stop, pos['high_water']*(1-pos['trail']))
                    pos['gap_run'] = 0
                    self.last_marks[symbol] = (when, float(row.close))
                    self.log('BAR_REVIEWED', when, symbol=symbol,
                             high_water=pos['high_water'], stop_next_bar=pos['stop'],
                             close_proxy=float(row.close))
                elif when in invalid:
                    self.data_degraded = True
                    pos['stop_coverage_unreliable'] = True
                    pos['needs_gap_quote'] = True
                    pos['unobserved_minutes'] += 1
                    pos['gap_run'] += 1
                    self.log('INVALID_HELD_BAR', when, symbol=symbol)
                elif (latest is not None and when < latest) or now >= when + minute + self.settle:
                    self.data_degraded = True
                    pos['stop_coverage_unreliable'] = True
                    pos['needs_gap_quote'] = True
                    pos['unobserved_minutes'] += 1
                    pos['gap_run'] += 1
                    self.log('NO_SIP_BAR_WHILE_HELD', when, symbol=symbol,
                             consecutive=pos['gap_run'])
                else:
                    pos['pending_minutes'] = int((decision - when) / minute)
                    break
                pos['last_bar'] = when
                if pos['gap_run'] > self.max_gap_minutes:
                    pos['force_gap_exit'] = True
                    break
        self._mark(decision)

    def gap_symbols(self):
        """Held symbols whose latest settled minute had no usable SIP bar."""
        return sorted(s for s, pos in self.positions.items()
                      if pos['gap_run'] > 0 or pos['needs_gap_quote'] or pos['force_gap_exit'])

    def entry_data_blocked(self):
        """Only unresolved stop observations block additional virtual entries.

        `data_degraded` remains the permanent audit flag, not a permanent
        entry lock after a fresh bid or a subsequent good minute resolves a gap.
        """
        return any(pos['pending_minutes'] > 0 or pos['gap_run'] > 0 or
                   pos['needs_gap_quote'] or pos['force_gap_exit']
                   for pos in self.positions.values())

    def check_gap_exits(self, when, bids):
        """Without a SIP bar the stop is checked against a fresh SIP bid; after
        more than `max_gap_minutes` unobserved minutes the position is left."""
        when = stamp(when)
        for symbol in self.gap_symbols():
            pos = self.positions[symbol]
            bid = fresh_bid(bids.get(symbol), when)
            if bid is not None and bid <= pos['stop']:
                self.close(symbol, when, bid, 'STOP_BID_QUOTE_PROXY')
            elif pos['force_gap_exit'] or pos['gap_run'] > self.max_gap_minutes:
                if bid is not None:
                    self.close(symbol, when, bid, 'EXIT_DATA_GAP_BID_PROXY')
                else:
                    self.data_degraded = True
                    self.log('DATA_GAP_EXIT_PRICE_UNAVAILABLE', when, symbol=symbol,
                             consecutive=pos['gap_run'])
            elif bid is not None:
                pos['needs_gap_quote'] = False

    def enter(self, when, signal, quote, signal_ttl_seconds):
        """Use a newly fetched SIP quote at this actual calculation time."""
        when = stamp(when)
        symbol = signal['symbol']
        cutoff = self.planned_close-pd.Timedelta(minutes=
            self.c.entry_cutoff_minutes-self.c.close_before_minutes)
        if when >= cutoff:
            return self.reject(when, symbol, 'ENTRY_CUTOFF')
        if (when-stamp(signal['decision_time_utc'])).total_seconds() >= signal_ttl_seconds:
            return self.reject(when, symbol, 'SIGNAL_EXPIRED')
        if symbol in self.used_symbols:
            refusal = self.reentry_refusal(symbol, when)
            if refusal:
                return self.reject(when, symbol, refusal)
        if self.locked:
            return self.reject(when, symbol, 'DAILY_LOSS_LOCK')
        if self.entry_data_blocked():
            return self.reject(when, symbol, 'HELD_POSITION_STOP_OBSERVATION_UNRESOLVED')
        if quote is None or quote.get('quote_result') != 'PASS':
            return self.reject(when, symbol, 'EXECUTION_QUOTE_' + str(
                (quote or {}).get('quote_result', 'MISSING')))
        qtime = stamp(quote['quote_time_utc'])
        if qtime > when or (when-qtime).total_seconds() > 2:
            return self.reject(when, symbol, 'EXECUTION_QUOTE_NOT_FRESH')
        ask = float(quote['ask'])
        if not math.isfinite(ask) or ask <= 0:
            return self.reject(when, symbol, 'INVALID_ENTRY_PRICE')
        if self._mark(when) is None:
            return self.reject(when, symbol, 'PORTFOLIO_MARK_MISSING')
        sector = signal['sector']
        occupied = [pos['sector'] for pos in self.positions.values()]
        gate = base.entry_gate(self.start_cash, self.last_equity, 0,
                               occupied, sector, self.locked, self.c)
        self.locked = self.locked or gate['day_locked']
        if not gate['allowed']:
            return self.reject(when, symbol, ';'.join(gate['reasons']))
        existing_risk = sum(p['qty']*max(0, p['entry']-p['stop'])
                            for p in self.positions.values())
        trail = self.trail(symbol)
        sizing = base.size_position(self.last_equity, self.cash, existing_risk, ask,
                                    self.c if trail == float(self.c.trailing)
                                    else replace(self.c, trailing=str(trail)))
        qty = sizing['qty']
        if qty <= 0:
            return self.reject(when, symbol, 'NO_CAPITAL_OR_RISK_BUDGET')
        cost = qty*ask
        self.cash -= cost
        self.used_symbols.add(symbol)
        self.entry_counts[symbol] += 1
        entry_minute = when.floor('min')
        pos = dict(symbol=symbol, sector=sector, qty=qty, entry=ask,
                   entry_at_utc=when.isoformat(), entry_quote_at_utc=qtime.isoformat(),
                   signal_at_utc=signal['decision_time_utc'], rank=signal['rank'],
                   score=signal['score'], rvol=signal['rvol'],
                   trail=trail, entry_number=self.entry_counts[symbol],
                   stop=ask*(1-trail), high_water=ask, low_water=ask,
                   last_bar=entry_minute, unobserved_minutes=0,
                   gap_run=0, pending_minutes=0, stop_coverage_unreliable=False,
                   needs_gap_quote=False, force_gap_exit=False,
                   entry_minute_not_observed=True)
        self.positions[symbol] = pos
        self.last_marks[symbol] = (entry_minute, ask)
        self.log('VIRTUAL_ENTRY', when, symbol=symbol, sector=sector,
                 qty=qty, ask_proxy=ask, notional=cost, initial_stop=pos['stop'],
                 initial_planned_risk=float(sizing['planned_risk']),
                 signal_rank=pos['rank'], signal_score=pos['score'],
                 quote_at_utc=qtime.isoformat())
        return 'VIRTUAL_ENTRY'

    def trail(self, symbol):
        return float(self.trail_overrides.get(symbol, self.c.trailing))

    def reentry_refusal(self, symbol, when):
        """None if a further entry is allowed, else the rejection reason.

        Default (max_reentries=0): one entry per symbol and day, as before."""
        if self.max_reentries <= 0:
            return 'SYMBOL_ALREADY_USED_TODAY'
        last = self.last_exit.get(symbol)
        if symbol in self.positions or last is None:
            return 'SYMBOL_ALREADY_HELD'
        if self.entry_counts[symbol] > self.max_reentries:
            return 'REENTRY_LIMIT_REACHED'
        if last['reason'] not in STOP_REASONS:
            return 'REENTRY_ONLY_AFTER_STOP'
        if when < last['at'] + self.reentry_cooldown:
            return 'REENTRY_COOLDOWN'
        return None

    def close(self, symbol, when, price, reason, trigger_bar_minute=None):
        when = stamp(when)
        pos = self.positions.pop(symbol)
        self.last_exit[symbol] = dict(at=when, reason=reason)
        price = float(price)
        if not math.isfinite(price) or price <= 0:
            raise ValueError('Ungültiger virtueller Ausstiegspreis')
        proceeds = pos['qty']*price
        self.cash += proceeds
        pnl = pos['qty']*(price-pos['entry'])
        # Scenario: adverse 5 bp on both sides, separate from the bid/ask proxy.
        stress = pnl-.0005*pos['qty']*(price+pos['entry'])
        item = dict(**{k: v for k, v in pos.items()
                       if k not in ('last_bar', 'gap_run', 'pending_minutes',
                                    'needs_gap_quote', 'force_gap_exit')},
                    exit_at_utc=when.isoformat(), exit_price_proxy=price,
                    exit_reason=reason, pnl_usd=pnl, pnl_10bp_stress_usd=stress,
                    # excursions over the observed bars while held (entry minute excluded)
                    mfe_usd=pos['qty']*(pos['high_water']-pos['entry']),
                    mae_usd=pos['qty']*(min(pos['low_water'], price)-pos['entry']),
                    exit_trigger_bar_minute_utc=(stamp(trigger_bar_minute).isoformat()
                        if trigger_bar_minute is not None else None),
                    outcome_unreliable=pos['stop_coverage_unreliable'])
        self.closed.append(item)
        self.last_marks.pop(symbol, None)
        self.log('VIRTUAL_EXIT', when, symbol=symbol, qty=pos['qty'],
                 exit_price_proxy=price, reason=reason, pnl_usd=pnl,
                 pnl_10bp_stress_usd=stress,
                 trigger_bar_minute_utc=item['exit_trigger_bar_minute_utc'],
                 outcome_unreliable=item['outcome_unreliable'])
        self._mark(when)
        return item

    def finish(self, when, bids):
        """Close at fresh bid; otherwise a visibly flagged last-bar proxy."""
        when = stamp(when)
        for symbol, pos in list(self.positions.items()):
            bid = fresh_bid(bids.get(symbol), when)
            if bid is not None:
                self.close(symbol, when, bid, 'PLANNED_CLOSE_BID_PROXY')
                continue
            mark = self.last_marks.get(symbol)
            if mark and mark[0] > stamp(pos['entry_at_utc']).floor('min') and \
                    0 <= (when-mark[0]).total_seconds() <= 180:
                self.data_degraded = True
                self.close(symbol, when, mark[1], 'PLANNED_CLOSE_LAST_BAR_PROXY')
            else:
                self.data_degraded = True
                self.log('CLOSE_PRICE_UNAVAILABLE', when, symbol=symbol)
        self._mark(when)

    def report(self):
        rejected = Counter(event['reason'] for event in self.events
                           if event['kind'] == 'ENTRY_REJECTED')
        return dict(mode='VIRTUAL_ONLY_NO_BROKER_ORDERS', orders_sent=0, feed=self.feed,
                    starting_cash_usd=self.start_cash, ending_cash_usd=self.cash,
                    ending_equity_proxy_usd=self.last_equity,
                    realized_pnl_usd=sum(p['pnl_usd'] for p in self.closed),
                    pnl_proxy_from_unreliable_trades_usd=sum(
                        p['pnl_usd'] for p in self.closed if p['outcome_unreliable']),
                    realized_pnl_10bp_stress_usd=sum(p['pnl_10bp_stress_usd']
                                                      for p in self.closed),
                    max_drawdown_proxy_usd=self.max_drawdown,
                    virtual_trades=len(self.closed), open_symbols=sorted(self.positions),
                    unreliable_virtual_trades=sum(p['outcome_unreliable'] for p in self.closed),
                    entry_data_blocked=self.entry_data_blocked(),
                    rejected_signals=dict(rejected), data_degraded=self.data_degraded,
                    assumptions=['Long only; SIP quote ask/bid are hypothetical fills, no fills or partial fills verified.',
                                 'Stop uses the preceding bar stop; opening gap exits at next bar open; trailing high applies from next bar.',
                                 'The partial entry minute and minutes without SIP bars are not stop-observed; '
                                 'such a minute is modeled as no observed SIP trade once a later bar exists or it is settled.',
                                 'Without a SIP bar the stop is checked against a fresh SIP bid; after too many '
                                 'consecutive unobserved minutes the position is left at the fresh bid.',
                                 'A SIP bar first observed over 15 seconds after its decision minute never triggers a backdated stop; affected trades are outcome_unreliable.',
                                 'Separate 10 bp total adverse execution scenario; no commissions, latency or market impact modeled.'])


In [ ]:
%%writefile /content/orb_sim_v16.py
"""Read-only ORB simulation (SIP or IEX), live, delayed or replayed. Contains no order methods."""
from dataclasses import asdict, dataclass
from datetime import timedelta
from pathlib import Path
import hashlib
import json
import math
import sys
import time
import uuid

import numpy as np
import pandas as pd
import us_orb_test_v02 as base
import us_orb_scanner_v03 as scanner
import orb_portfolio_v15 as shadow

VERSION = '1.6.1-orb-simulation-sip-iex-no-orders'
FEEDS = ('sip', 'iex')
NY = 'America/New_York'


@dataclass(frozen=True)
class DryRunSettings:
    poll_seconds: int = 5
    feed: str = 'sip'                     # intraday bars, quotes and RVOL references: 'sip' or 'iex'
    initial_cash: float = 10000.0         # virtual start equity of the day
    batch_size: int = 20
    historical_control_date: str = '2026-09-22'
    # A missing SIP minute is modeled as no price print; an outage cannot be ruled out.
    min_opening_bars: int = 12            # of 15 opening-range minutes; first one required
    min_coverage: float = 0.85            # observed share of minutes since the open
    max_previous_age_minutes: int = 5     # previous real SIP close for a crossing
    benchmark_max_age_minutes: int = 2    # last real QQQ close before the decision
    bar_settle_seconds: int = 120         # a held minute stays pending this long
    max_held_gap_minutes: int = 3         # consecutive unobserved held minutes
    # RVOL: average over the valid reference sessions if at least this many of the
    # `rvol_days` (20) are valid; V1.4.3 required all 20 (user decision 2026-09-24).
    min_rvol_reference_days: int = 18
    # Strategy variants for the replay comparison; the defaults are the V1.5 rules.
    max_reentries: int = 0                # further entries per symbol after a stop exit
    reentry_cooldown_minutes: int = 15
    atr_trailing_multiple: float = 0.0    # > 0: trailing = multiple x 20-day range %, clamped
    atr_trailing_min: float = 0.0075
    atr_trailing_max: float = 0.03
    print_every_minute: bool = True       # replay prints only a daily summary
    save_minute_snapshots: bool = True    # session_bars_sip.csv after every minute
    # V1.6.1 persistence only (no decision depends on them): the backtest buffers the
    # per-minute files of a day and writes them at the planned close / end of the day;
    # RVOL reference minutes stay in memory instead of a CSV copy per run; no console text.
    persist_every_minute: bool = True
    reference_cache_on_disk: bool = True
    quiet: bool = False


class SystemClock:
    """Wall clock of the live run. The replay swaps in a simulated clock."""

    def now(self):
        return pd.Timestamp.now(tz='UTC')

    def sleep(self, seconds):
        time.sleep(seconds)


SYSTEM_CLOCK = SystemClock()


def utc(value):
    return pd.Timestamp(value).tz_convert('UTC')


def save_json(path, content):
    encoded = json.dumps(content, ensure_ascii=False, indent=2, allow_nan=False, default=str)
    temporary = path.with_name(path.name + '.incomplete')
    temporary.write_text(encoded + '\n', encoding='utf-8')
    temporary.replace(path)


class RunWriter:
    """Run-folder files: written at once (live) or buffered until flush() (backtest).

    A buffered JSON file keeps the encoded text of the moment it was 'written', so
    reading it back behaves exactly like reading the file."""

    def __init__(self, immediate=True):
        self.immediate = immediate
        self.pending = {}
        self.writes = 0

    def json(self, path, content):
        if self.immediate:
            save_json(path, content)
            self.writes += 1
        else:
            self.pending[Path(path)] = ('json', json.dumps(content, ensure_ascii=False, indent=2,
                                                           allow_nan=False, default=str))

    def frame(self, path, frame):
        if self.immediate:
            save_frame(path, frame)
            self.writes += 1
        else:
            self.pending[Path(path)] = ('csv', frame.copy())

    def exists(self, path):
        return Path(path) in self.pending or Path(path).exists()

    def read_json(self, path):
        kind, content = self.pending.get(Path(path), (None, None))
        return json.loads(content) if kind == 'json' else json.loads(Path(path).read_text())

    BUNDLED = ('signals', 'reference_checks')

    def flush(self):
        """Write buffered files. Per-signal JSON/CSV files are bundled per day
        (signals_bundle.json, reference_checks_bundle.csv): hundreds of small Drive
        writes become two. research_export_v16.signal_records() reads both forms."""
        bundles = {}
        for path, entry in list(self.pending.items()):
            if path.parent.name in self.BUNDLED:
                bundles.setdefault((path.parent.parent, path.parent.name), {})[path.name] = entry
                del self.pending[path]
        for (out, kind), entries in bundles.items():
            if kind == 'signals':
                target = out / 'signals_bundle.json'
                merged = json.loads(target.read_text()) if target.exists() else {}
                merged.update({name: json.loads(content) for name, (_, content) in entries.items()})
                save_json(target, merged)
            else:
                target = out / 'reference_checks_bundle.csv'
                parts = [pd.read_csv(target)] if target.exists() else []
                parts += [frame.assign(source_file=name) for name, (_, frame) in sorted(entries.items())]
                save_frame(target, pd.concat(parts, ignore_index=True))
            self.writes += 1
        for path, (kind, content) in sorted(self.pending.items()):
            path.parent.mkdir(parents=True, exist_ok=True)
            if kind == 'json':
                temporary = path.with_name(path.name + '.incomplete')
                temporary.write_text(content + '\n', encoding='utf-8')
                temporary.replace(path)
            else:
                save_frame(path, content)
            self.writes += 1
        self.pending.clear()


class ReferenceHistory:
    """RVOL reference minutes of one symbol, also split by session (same rows).

    checked_volume() answers the common clean case from prepared numpy arrays and
    hands every unusual window (duplicates, invalid bars, empty, > 40 gaps) to the
    unchanged checked_reference_volume(), so results and error texts stay identical."""

    def __init__(self, frame, past):
        self.frame = frame
        self.by_day = {s['date']: frame[(frame.timestamp >= s['open']) & (frame.timestamp < s['close'])]
                       for s in past}
        self.arrays = {}

    def _prepared(self, day, symbol):
        if day not in self.arrays:
            part = self.by_day[day]
            prepared = None
            if len(part) and (part.symbol == symbol).all():
                stamps = pd.DatetimeIndex(part.timestamp).as_unit('ns').asi8
                order = np.argsort(stamps, kind='stable')
                ts = stamps[order]
                o, h, l, c, v = (pd.to_numeric(part[k], errors='coerce').to_numpy(dtype=float)[order]
                                 for k in ('open', 'high', 'low', 'close', 'volume'))
                with np.errstate(invalid='ignore'):
                    row_ok = (~np.isnan(o) & ~np.isnan(h) & ~np.isnan(l) & ~np.isnan(c) & ~np.isnan(v) &
                              (v >= 0) & (o > 0) & (h > 0) & (l > 0) & (c > 0) &
                              (h >= np.maximum(np.maximum(o, c), l)) & (l <= np.minimum(np.minimum(o, c), h)))
                    first_finite = bool(np.isfinite([o[0], h[0], l[0], c[0], v[0]]).all())
                first_window = int(np.searchsorted(ts, ts[0] + MINUTE_NS, 'left'))
                duplicate = np.concatenate([[False], ts[1:] == ts[:-1]])
                prepared = dict(ts=ts, volume=v, bad_prefix=np.cumsum(~row_ok | duplicate),
                                first_ok=first_window == 1 and first_finite and bool(row_ok[0]))
            self.arrays[day] = prepared
        return self.arrays[day]

    def checked_volume(self, data, session, symbol, start, end, out, feed='sip'):
        prepared = self._prepared(session['date'], symbol)
        start_ns, end_ns = utc(start).value, utc(end).value
        if prepared is not None and prepared['first_ok'] and prepared['ts'][0] >= start_ns:
            n = int(np.searchsorted(prepared['ts'], end_ns, 'left'))
            if n and not prepared['bad_prefix'][n - 1]:
                size = max(0, -(-(end_ns - start_ns) // MINUTE_NS))
                grid = start_ns + np.arange(size, dtype=np.int64) * MINUTE_NS
                gaps = grid[~np.isin(grid, prepared['ts'][:n])]
                if len(gaps) <= 40:
                    extra = sum(missing_odd_lot_volume(data, symbol, pd.Timestamp(int(g), tz='UTC'), out, feed)
                                for g in gaps)
                    return float(prepared['volume'][:n].sum() + extra), len(gaps), float(extra)
        return checked_reference_volume(data, self.by_day[session['date']], symbol, start, end, out, feed)


def save_frame(path, frame):
    temporary = path.with_name(path.name + '.incomplete')
    frame.to_csv(temporary, index=False)
    temporary.replace(path)


def restore_frame(path):
    frame = pd.read_csv(path)
    frame['timestamp'] = pd.to_datetime(frame.timestamp, utc=True)
    return frame


def completed_decision(now, delay_seconds=5):
    now = utc(now)
    if now.second < delay_seconds:
        return now.floor('min') - pd.Timedelta(minutes=1)
    return now.floor('min')


def session_start_window(started, opening, closing, c):
    """Return a same-session start state; never invent another trading date."""
    started, opening, closing = map(utc, (started, opening, closing))
    first_decision = opening + pd.Timedelta(minutes=c.opening_minutes + 1)
    cutoff = closing - pd.Timedelta(minutes=c.entry_cutoff_minutes)
    planned_close = closing - pd.Timedelta(minutes=c.close_before_minutes)
    if not first_decision < cutoff < planned_close < closing:
        raise ValueError('Handelssitzung ist für die Strategie zu kurz.')
    if started >= planned_close:
        raise ValueError('Geplanter virtueller Tagesabschluss ist bereits vorbei; '
                         'heute kein sinnvoller Start mehr.')
    if started < opening:
        return 'WAITING_FOR_OPEN'
    if started < first_decision:
        return 'OPENING_RANGE_IN_PROGRESS'
    if started < cutoff:
        return 'LATE_START_PARTIAL_DAY'
    return 'ENTRY_CUTOFF_PASSED_OBSERVATION_ONLY'


def minute_key(decision, symbol):
    return f'{utc(decision):%Y%m%dT%H%M%SZ}_{symbol}.json'


def fetch_bars(data, symbols, start, end, feed, timeframe, batch_size=20):
    if feed not in FEEDS:
        raise ValueError(f'Unbekannter Datenfeed: {feed}')
    from alpaca.data.requests import StockBarsRequest
    from alpaca.data.enums import DataFeed, Adjustment
    from alpaca.data.timeframe import TimeFrame
    mode = data_feed(feed)
    scale = TimeFrame.Minute if timeframe == 'minute' else TimeFrame.Day
    frames = []
    for offset in range(0, len(symbols), batch_size):
        chunk = symbols[offset:offset + batch_size]
        answer = data.get_stock_bars(StockBarsRequest(
            symbol_or_symbols=chunk, timeframe=scale,
            start=utc(start).to_pydatetime(), end=utc(end).to_pydatetime(),
            adjustment=Adjustment.RAW, feed=mode))
        if any(answer.data.values()):
            # V1.6.1: the replay hands over bars already in normalize_bars() form
            # (sorted, numeric, UTC); live answers are normalized as before.
            frame = answer.df.assign(feed='sip') if getattr(answer, 'normalized', False) \
                else base.normalize_bars(answer.df)
            # Alpaca may include a bar exactly at the requested end boundary.
            # All requested windows in this notebook are [start, end).
            frame = frame[(frame.timestamp >= utc(start)) &
                          (frame.timestamp < utc(end))].copy()
            if frame.empty:
                continue
            frame['feed'] = feed  # base.normalize_bars labels the older test's SIP source.
            frames.append(frame)
    columns = ['symbol', 'timestamp', 'open', 'high', 'low', 'close', 'volume', 'vwap', 'feed']
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=columns)


def data_feed(feed):
    from alpaca.data.enums import DataFeed
    if feed not in FEEDS:
        raise ValueError(f'Unbekannter Datenfeed: {feed}')
    return DataFeed.SIP if feed == 'sip' else DataFeed.IEX


def check_realtime_sip_access(data, now):
    return check_realtime_access(data, now, 'sip')


def check_realtime_access(data, now, feed='sip'):
    """Explicit recent requests on `feed`; a successful old query proves nothing."""
    now = utc(now)
    start = now - pd.Timedelta(minutes=2)
    recent = fetch_bars(data, ['AAPL'], start, now, feed, 'minute', batch_size=1)
    from alpaca.data.requests import StockQuotesRequest
    from alpaca.data.enums import DataFeed
    from alpaca.common.enums import Sort
    quotes = data.get_stock_quotes(StockQuotesRequest(
        symbol_or_symbols='AAPL', start=start.to_pydatetime(),
        end=now.to_pydatetime(), sort=Sort.DESC, limit=1, feed=data_feed(feed)))
    if not isinstance(quotes.data, dict):
        raise ValueError(f'{feed.upper()}-Quote-Antwort ist ungültig.')
    return dict(result=f'RECENT_{feed.upper()}_BARS_AND_QUOTES_ACCEPTED',
                checked_at_utc=now.isoformat(),
                request_end_within_last_15_minutes=True,
                recent_aapl_bars=len(recent),
                recent_aapl_quotes=len(quotes.data.get('AAPL', [])),
                feed=feed, orders_sent=0,
                note='Außerhalb der Handelszeit können beide Abfragen korrekt null Datensätze liefern.')


def fresh_universe(broker):
    from alpaca.trading.requests import GetAssetsRequest
    from alpaca.trading.enums import AssetStatus, AssetClass
    members = scanner.fetch_json(scanner.MEMBERS_URL)
    sectors = scanner.fetch_json(scanner.SECTORS_URL)
    frame = scanner.assemble_universe(members, sectors)
    assets = broker.get_all_assets(filter=GetAssetsRequest(
        status=AssetStatus.ACTIVE, asset_class=AssetClass.US_EQUITY))
    tradeable = {asset.symbol for asset in assets if asset.tradable}
    frame['tradable_now'] = frame.symbol.isin(tradeable)
    frame['universe_result'] = frame.apply(lambda r:
        'SECTOR_UNKNOWN' if not r.sector else
        'NOT_TRADABLE' if not r.tradable_now else 'PASS', axis=1)
    return frame


def review_opening(symbol, current, opening, decision, closing, c):
    """One historical prefix; do not infer a missing SIP bar from a trade elsewhere."""
    if decision < opening + pd.Timedelta(minutes=c.opening_minutes + 1):
        return 'OPENING_RANGE_IN_PROGRESS', None
    try:
        opening_range, rows = scanner.scan_prefix(current, symbol, opening, closing, c)
    except ValueError:
        return 'OPENING_RANGE_DATA_INVALID', None
    match = rows[rows.decision_time_utc == decision.isoformat()]
    if match.empty:
        return 'DECISION_NOT_RECONSTRUCTED', None
    row = match.iloc[0].to_dict()
    row.update(opening_range)
    return row['result'], row


def minute_series(current, symbol, opening, decision):
    """One row per minute in [opening, decision); missing minutes stay NaN.

    `observed` marks a real SIP bar, `volume` counts a missing minute as 0,
    `last_close`/`last_seen` carry the latest real SIP print forward."""
    grid = pd.date_range(utc(opening), utc(decision), freq='min', inclusive='left')
    good, invalid = shadow.clean_bars(current, symbol, opening, decision)
    size = len(grid)
    columns = {name: np.full(size, np.nan) for name in ('open', 'high', 'low', 'close', 'volume')}
    observed = np.zeros(size, dtype=bool)
    positions = grid.get_indexer(good.index)
    on_grid = positions >= 0          # off-grid timestamps are ignored, as before
    for name in columns:
        columns[name][positions[on_grid]] = good[name].to_numpy(dtype=float)[on_grid]
    observed[positions[on_grid]] = True
    flagged = np.zeros(size, dtype=bool)
    bad = grid.get_indexer(pd.DatetimeIndex(sorted(invalid))) if invalid else np.array([], dtype=int)
    flagged[bad[bad >= 0]] = True
    seen = np.maximum.accumulate(np.where(observed, np.arange(size), -1)) if size else np.array([], dtype=int)
    series = pd.DataFrame(columns, index=grid)
    series['observed'] = observed
    series['invalid'] = flagged
    series['volume'] = np.nan_to_num(columns['volume'], nan=0.0)
    series['last_close'] = np.where(seen >= 0, columns['close'][np.maximum(seen, 0)], np.nan)
    series['last_seen'] = grid[np.maximum(seen, 0)].where(seen >= 0) if size else grid
    series['coverage'] = np.cumsum(observed) / np.arange(1, size + 1)
    return series


def opening_range(series, opening, c, settings):
    """Opening range from the observed bars; None if coverage is too thin."""
    end_or = utc(opening) + pd.Timedelta(minutes=c.opening_minutes)
    part = series[series.index < end_or]
    observed = part[part.observed]
    if len(part) < c.opening_minutes or not part.observed.iloc[0] or \
            len(observed) < settings.min_opening_bars:
        return None
    high, low = base.D(observed.high.max()), base.D(observed.low.min())
    return dict(high=high, low=low, threshold=high * (1 + base.D(c.breakout_buffer)),
                cap=high * (1 + base.D(c.entry_cap)), bars=len(observed),
                last_close=base.D(observed.close.iloc[-1]), last_seen=observed.index[-1])


def atr_trailing(daily, settings):
    """Per-symbol trailing distance = multiple x mean daily range (high-low)/close
    of the prior sessions, clamped to [atr_trailing_min, atr_trailing_max]."""
    trails = {}
    for symbol, rows in daily.groupby('symbol'):
        spread = ((rows.high - rows.low) / rows.close).dropna()
        if len(spread):
            value = settings.atr_trailing_multiple * float(spread.mean())
            trails[symbol] = min(max(value, settings.atr_trailing_min), settings.atr_trailing_max)
    return trails


def split_by_symbol(frame):
    """Per-symbol views of a multi-symbol bar frame; avoids rescanning it per symbol."""
    empty = frame.iloc[0:0]
    parts = {symbol: part for symbol, part in frame.groupby('symbol', sort=False)}
    return lambda symbol: parts.get(symbol, empty)


MINUTE_NS = 60_000_000_000


def symbol_arrays(current, symbol, start, end):
    """numpy form of clean_bars() for one symbol in [start, end): the same validity
    rules (finite, positive, OHLC consistent, duplicate minutes invalid)."""
    start_ns, end_ns = utc(start).value, utc(end).value
    if current.empty:
        return None
    stamps = pd.DatetimeIndex(current['timestamp']).as_unit('ns').asi8
    mask = (current['symbol'].to_numpy() == symbol) & (stamps >= start_ns) & (stamps < end_ns)
    if not mask.any():
        return None
    ts = stamps[mask]
    o, h, l, c, v = (pd.to_numeric(current[column].to_numpy()[mask], errors='coerce').astype(float)
                     for column in shadow.BAR_COLUMNS)
    unique, inverse, counts = np.unique(ts, return_inverse=True, return_counts=True)
    duplicated = counts[inverse] > 1
    with np.errstate(invalid='ignore'):
        valid = (np.isfinite(o) & np.isfinite(h) & np.isfinite(l) & np.isfinite(c) &
                 np.isfinite(v) & (o > 0) & (h > 0) & (l > 0) & (c > 0) & (v >= 0) &
                 (h >= np.maximum(np.maximum(o, c), l)) & (l <= np.minimum(np.minimum(o, c), h)) &
                 ~duplicated)
    order = np.argsort(ts[valid], kind='stable')
    return dict(ts=ts[valid][order], high=h[valid][order], low=l[valid][order],
                close=c[valid][order], invalid_ts=ts[~valid])


def price_candidates_until(current, symbol, opening, decision, c, settings=DryRunSettings(),
                           rejected=None):
    """Price crossings on the SIP minute grid; missing minutes carry the last print.

    A crossing needs a real bar in the signal minute, a previous real close no
    older than `max_previous_age_minutes` and enough coverage since the open.
    Returns (candidates, opening_invalid, later_data_issue).

    V1.6.1: the same rules as price_candidates_until_slow() on numpy arrays instead of
    a pandas minute frame (runs for every symbol in every minute); pinned by tests.
    If `rejected` is a list, crossings that fail the price band or the data rules
    are appended to it (research log only; the result is unchanged)."""
    opening, decision = utc(opening), utc(decision)
    end_or = opening + pd.Timedelta(minutes=c.opening_minutes)
    if decision <= end_or:
        return [], False, False
    start_ns, end_ns, or_ns = opening.value, decision.value, end_or.value
    size = max(0, -(-(end_ns - start_ns) // MINUTE_NS))
    observed = np.zeros(size, dtype=bool)
    close = np.full(size, np.nan)
    high = np.full(size, np.nan)
    low = np.full(size, np.nan)
    flagged = np.zeros(size, dtype=bool)
    bars = symbol_arrays(current, symbol, opening, decision)
    if bars is not None:
        offset = bars['ts'] - start_ns
        on_grid = offset % MINUTE_NS == 0
        positions = (offset[on_grid] // MINUTE_NS).astype(np.int64)
        observed[positions] = True
        close[positions] = bars['close'][on_grid]
        high[positions] = bars['high'][on_grid]
        low[positions] = bars['low'][on_grid]
        bad = bars['invalid_ts'] - start_ns
        bad = bad[bad % MINUTE_NS == 0] // MINUTE_NS
        flagged[bad[(bad >= 0) & (bad < size)].astype(np.int64)] = True
    # opening range (same rules as opening_range())
    n_or = min(size, max(0, -(-(or_ns - start_ns) // MINUTE_NS)))
    part = observed[:n_or]
    if n_or < c.opening_minutes or not part[0] or int(part.sum()) < settings.min_opening_bars:
        return [], True, False
    or_index = np.flatnonzero(part)
    or_high, or_low = base.D(np.max(high[:n_or][part])), base.D(np.min(low[:n_or][part]))
    threshold_d = or_high * (1 + base.D(c.breakout_buffer))
    cap = or_high * (1 + base.D(c.entry_cap))
    first_close = base.D(close[or_index[-1]])
    first_seen = start_ns + int(or_index[-1]) * MINUTE_NS
    later_issue = bool(flagged.any())
    max_age = pd.Timedelta(minutes=settings.max_previous_age_minutes).value
    coverage = np.cumsum(observed) / np.arange(1, size + 1)
    later_index = np.flatnonzero(observed[n_or:]) + n_or
    moments = start_ns + later_index.astype(np.int64) * MINUTE_NS
    closes = close[later_index]
    coverages = coverage[later_index]
    previous_closes = np.concatenate([[float(first_close)], closes[:-1]])
    threshold = float(threshold_d)
    margin = abs(threshold) * 1e-9
    candidates = []
    for k in np.flatnonzero((closes > threshold - margin) & (previous_closes <= threshold + margin)):
        moment = int(moments[k])
        close_d = base.D(closes[k])
        previous = first_close if k == 0 else base.D(closes[k - 1])
        previous_seen = first_seen if k == 0 else int(moments[k - 1])
        if not (previous <= threshold_d and close_d > threshold_d):
            continue
        details = dict(symbol=symbol,
            decision_time_utc=pd.Timestamp(moment + MINUTE_NS, tz='UTC').isoformat(),
            or_high=float(or_high), or_low=float(or_low),
            threshold=float(threshold_d), price_cap=float(cap),
            close=float(close_d), opening_bars=int(part.sum()),
            coverage=float(coverages[k]),
            previous_close_age_minutes=int((moment - previous_seen) / MINUTE_NS))
        if close_d >= base.D(c.minimum_price) and close_d <= cap:
            if moment - previous_seen > max_age or coverages[k] < settings.min_coverage:
                later_issue = True
                if rejected is not None:
                    rejected.append(dict(details, status='DATA_INCOMPLETE',
                        detail='PREVIOUS_CLOSE_TOO_OLD' if moment - previous_seen > max_age
                               else 'COVERAGE_BELOW_MINIMUM'))
            else:
                candidates.append(details)
        elif rejected is not None:
            rejected.append(dict(details, status='BELOW_MINIMUM_PRICE'
                                 if close_d < base.D(c.minimum_price) else 'TOO_EXTENDED'))
    return candidates, False, later_issue


def price_candidates_until_slow(current, symbol, opening, decision, c, settings=DryRunSettings(),
                           rejected=None):
    """REFERENCE implementation (V1.6.0) kept for the equivalence tests of the fast path.

    Price crossings on the SIP minute grid; missing minutes carry the last print.

    A crossing needs a real bar in the signal minute, a previous real close no
    older than `max_previous_age_minutes` and enough coverage since the open.
    Returns (candidates, opening_invalid, later_data_issue).

    V1.6: if `rejected` is a list, crossings that fail the price band or the data
    rules are appended to it (research log only; the result is unchanged)."""
    opening, decision = utc(opening), utc(decision)
    end_or = opening + pd.Timedelta(minutes=c.opening_minutes)
    if decision <= end_or:
        return [], False, False
    series = minute_series(current, symbol, opening, decision)
    rng = opening_range(series, opening, c, settings)
    if rng is None:
        return [], True, False
    later_issue = bool(series.invalid.any())
    max_age = pd.Timedelta(minutes=settings.max_previous_age_minutes)
    later = series[series.index >= end_or]
    observed = later.observed.to_numpy()
    moments = later.index[observed]
    closes = later.close.to_numpy()[observed]
    coverages = later.coverage.to_numpy()[observed]
    previous_closes = np.concatenate([[float(rng['last_close'])], closes[:-1]])
    # Exact Decimal rule below; this float pre-filter only skips minutes that are
    # clearly no crossing (margin far above float rounding).
    threshold = float(rng['threshold'])
    margin = abs(threshold) * 1e-9
    candidates = []
    for k in np.flatnonzero((closes > threshold - margin) & (previous_closes <= threshold + margin)):
        moment = moments[k]
        close = base.D(closes[k])
        previous = rng['last_close'] if k == 0 else base.D(closes[k - 1])
        previous_seen = rng['last_seen'] if k == 0 else moments[k - 1]
        if not (previous <= rng['threshold'] and close > rng['threshold']):
            continue
        details = dict(symbol=symbol,
            decision_time_utc=(moment + pd.Timedelta(minutes=1)).isoformat(),
            or_high=float(rng['high']), or_low=float(rng['low']),
            threshold=float(rng['threshold']), price_cap=float(rng['cap']),
            close=float(close), opening_bars=rng['bars'],
            coverage=float(coverages[k]),
            previous_close_age_minutes=int((moment - previous_seen) / pd.Timedelta(minutes=1)))
        if close >= base.D(c.minimum_price) and close <= rng['cap']:
            if moment - previous_seen > max_age or coverages[k] < settings.min_coverage:
                later_issue = True
                if rejected is not None:
                    rejected.append(dict(details, status='DATA_INCOMPLETE',
                        detail='PREVIOUS_CLOSE_TOO_OLD' if moment - previous_seen > max_age
                               else 'COVERAGE_BELOW_MINIMUM'))
            else:
                candidates.append(details)
        elif rejected is not None:
            rejected.append(dict(details, status='BELOW_MINIMUM_PRICE'
                                 if close < base.D(c.minimum_price) else 'TOO_EXTENDED'))
    return candidates, False, later_issue


def exit_quotes(data, symbols, clock=SYSTEM_CLOCK, feed='sip'):
    """Fresh SIP bid per symbol, each checked at its own request time."""
    bids = {}
    for symbol in sorted(symbols):
        quote_now = clock.now()
        try:
            bids[symbol] = exit_quote_quality(fresh_quote(data, symbol, quote_now, feed=feed), quote_now)
        except Exception as exc:
            bids[symbol] = dict(quote_result='ERROR', error_type=type(exc).__name__)
    return bids


def check_gap_exits(data, portfolio, clock=SYSTEM_CLOCK, feed='sip'):
    """Held symbols whose last minute had no SIP bar: stop via the fresh bid,
    or leave after too many consecutive unobserved minutes."""
    symbols = portfolio.gap_symbols()
    if symbols:
        bids = exit_quotes(data, symbols, clock, feed)
        portfolio.check_gap_exits(clock.now(), bids)


def account_snapshot(broker, clock=SYSTEM_CLOCK):
    from alpaca.trading.requests import GetOrdersRequest
    from alpaca.trading.enums import QueryOrderStatus
    account = broker.get_account()
    positions = broker.get_all_positions()
    orders = broker.get_orders(filter=GetOrdersRequest(status=QueryOrderStatus.OPEN, limit=500))
    if len(orders) == 500:
        raise ValueError('Offene Orderliste könnte abgeschnitten sein.')
    return dict(equity=float(account.equity), cash=float(account.cash),
        currency=account.currency, account_blocked=bool(account.account_blocked),
        trading_blocked=bool(account.trading_blocked),
        position_symbols=[p.symbol for p in positions],
        open_order_ids=[str(o.id) for o in orders],
        checked_at_utc=clock.now().isoformat())


def fresh_quote(data, symbol, at, search_seconds=10, feed='sip'):
    from alpaca.data.requests import StockQuotesRequest
    from alpaca.data.enums import DataFeed
    from alpaca.common.enums import Sort
    at = utc(at)
    response = data.get_stock_quotes(StockQuotesRequest(
        symbol_or_symbols=symbol,
        start=(at-pd.Timedelta(seconds=search_seconds)).to_pydatetime(),
        end=at.to_pydatetime(), sort=Sort.DESC, limit=1, feed=data_feed(feed)))
    quotes = response.data.get(symbol, [])
    return max(quotes, key=lambda q: utc(q.timestamp)) if quotes else None


def exit_quote_quality(quote, at):
    if quote is None:
        return dict(quote_result='MISSING')
    try:
        bid, ask = float(quote.bid_price), float(quote.ask_price)
        bid_size = float(quote.bid_size)
        at = utc(at)
        qtime = utc(quote.timestamp)
        if not all(math.isfinite(x) for x in (bid, ask, bid_size)) or \
                min(bid, ask, bid_size) <= 0 or bid > ask or \
                qtime > at or (at-qtime).total_seconds() > 2:
            return dict(quote_result='INVALID_OR_STALE')
        return dict(quote_result='PASS', bid=bid, ask=ask,
                    quote_time_utc=qtime.isoformat())
    except (AttributeError, TypeError, ValueError):
        return dict(quote_result='INVALID_OR_STALE')


def prepare_reference(data, symbol, past, out, settings):
    """Fetch the 20-day SIP history through the data client, then cache each session."""
    if not settings.reference_cache_on_disk:
        # Same request and the same consistency checks, without the per-run CSV copy.
        frame = fetch_bars(data, [symbol], past[0]['open'], past[-1]['close'],
                           settings.feed, 'minute', settings.batch_size)
        if not frame.empty and frame.timestamp.duplicated().any():
            raise ValueError('Doppelte historische SIP-Kerzen.')
        frames = [frame[(frame.timestamp >= s['open']) & (frame.timestamp < s['close'])]
                  for s in past]
        return pd.concat(frames, ignore_index=True)
    folder = out / f'{settings.feed}_reference_minutes' / symbol
    folder.mkdir(parents=True, exist_ok=True)
    dates = {session['date'] for session in past}
    cached = {path.stem for path in folder.glob('*.csv')}
    if cached != dates:
        # A partial cache is never treated as complete. alpaca-py pages the
        # request itself, so live and replay use the same data client.
        frame = fetch_bars(data, [symbol], past[0]['open'], past[-1]['close'],
                           settings.feed, 'minute', settings.batch_size)
        if not frame.empty and frame.timestamp.duplicated().any():
            raise ValueError('Doppelte historische SIP-Kerzen.')
        prepared = {}
        for session in past:
            day = frame[(frame.timestamp >= session['open']) &
                        (frame.timestamp < session['close'])].copy()
            prepared[session['date']] = day
        for session in past:
            save_frame(folder / (session['date'] + '.csv'), prepared[session['date']])
        save_json(out / f'{settings.feed}_reference_minutes' / f'{symbol}_download.json',
                  dict(symbol=symbol, date_first=past[0]['date'],
                       date_last=past[-1]['date'], feed=settings.feed,
                       returned_bars=len(frame), cached_regular_bars=sum(map(len, prepared.values()))))
    frames = []
    for session in past:
        frame = restore_frame(folder / (session['date'] + '.csv'))
        if not frame.empty and (set(frame.symbol) != {symbol} or
                (frame.timestamp < session['open']).any() or
                (frame.timestamp >= session['close']).any() or
                frame.timestamp.duplicated().any()):
            raise ValueError('Gespeicherter SIP-Vergleichstag ist widersprüchlich.')
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


def missing_odd_lot_volume(data, symbol, minute, out, feed='sip'):
    """Fetch one absent SIP bar's trade volume; never fabricate OHLC."""
    from alpaca.data.requests import StockTradesRequest
    from alpaca.data.enums import DataFeed
    folder = out / f'odd_lot_reference_minutes_{feed}' / symbol
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f'{utc(minute):%Y%m%dT%H%M%SZ}.json'
    if path.exists():
        saved = json.loads(path.read_text())
        if saved['symbol'] != symbol or saved['minute_utc'] != utc(minute).isoformat():
            raise ValueError('Odd-Lot-Cache gehört zu anderem Symbol/Minute.')
        return saved['volume']
    start, end = utc(minute), utc(minute) + pd.Timedelta(minutes=1)
    response = data.get_stock_trades(StockTradesRequest(
        symbol_or_symbols=symbol, start=start.to_pydatetime(),
        end=end.to_pydatetime(), feed=data_feed(feed)))
    trades = response.data.get(symbol, [])
    volume = 0
    for trade in trades:
        stamp = utc(trade.timestamp)
        size = float(trade.size)
        if not start <= stamp < end or not size > 0 or size != int(size):
            raise ValueError(f'Trade außerhalb Minute oder ungültige Stückzahl: fehlende Minute '
                             f'{start:%Y-%m-%d %H:%M} UTC, Trade {stamp.isoformat()} Stück {size}.')
        if set(trade.conditions or []) != {'@', 'I'}:
            raise ValueError(f'Fehlende Kerze {start:%Y-%m-%d %H:%M} UTC nicht aus Odd-Lots erklärbar: '
                             f'Trade {stamp.isoformat()} Stück {size:g} Bedingungen '
                             f'{sorted(trade.conditions or [])} (erwartet nur @,I).')
        volume += int(size)
    save_json(path, dict(symbol=symbol, minute_utc=start.isoformat(),
                         volume=volume, trade_count=len(trades),
                         method=f'{feed.upper()} odd-lot trade sizes for absent reference bar'))
    return volume


def checked_reference_volume(data, frame, symbol, start, end, out, feed='sip'):
    expected = pd.date_range(start, end, freq='min', inclusive='left')
    observed = frame[(frame.symbol == symbol) & (frame.timestamp >= start) &
                     (frame.timestamp < end)].sort_values('timestamp')
    if observed.timestamp.duplicated().any():
        twice = observed.timestamp[observed.timestamp.duplicated()].iloc[0]
        raise ValueError(f'Doppelte Referenzkerzen, z. B. {twice:%Y-%m-%d %H:%M} UTC.')
    if not observed.empty:
        base.checked_window(observed, symbol, observed.timestamp.iloc[0],
                            observed.timestamp.iloc[0] + pd.Timedelta(minutes=1))
        values = observed[['open', 'high', 'low', 'close', 'volume']]
        valid = (values.notna().all(axis=1) & (values.volume >= 0) &
                 (values[['open','high','low','close']] > 0).all(axis=1) &
                 (values.high >= values[['open','close','low']].max(axis=1)) &
                 (values.low <= values[['open','close','high']].min(axis=1)))
        if not valid.all():
            bad = observed[~valid].iloc[0]
            raise ValueError(f'Ungültige Referenzkerze {bad.timestamp:%Y-%m-%d %H:%M} UTC: '
                             f'O {bad.open} H {bad.high} L {bad.low} C {bad.close} V {bad.volume} '
                             f'({int((~valid).sum())} ungültige Kerzen im Fenster).')
    gaps = expected.difference(pd.DatetimeIndex(observed.timestamp))
    if len(gaps) > 40:
        raise ValueError('Mehr als 40 Referenzlücken; keine automatische Rekonstruktion.')
    extra = sum(missing_odd_lot_volume(data, symbol, minute, out, feed)
                for minute in gaps)
    return float(observed.volume.sum() + extra), len(gaps), float(extra)


def check_sip_rvol(data, symbol, current, history, past, opening, decision,
                   config, out, feed='sip', min_reference_days=None):
    """RVOL against references of the same feed (named for SIP, used for both)."""
    elapsed = decision - opening
    # Model an absent SIP bar as zero observed volume; distinguish outages separately.
    actual = float(minute_series(current, symbol, opening, decision).volume.sum())
    rows, volumes = [], []
    for session in past:
        start, end = session['open'], session['open'] + elapsed
        value, gaps, extra, status, detail = None, None, None, 'OK', None
        if end > session['close']:
            status = 'REFERENCE_SESSION_TOO_SHORT'
        else:
            try:
                if isinstance(history, ReferenceHistory):
                    value, gaps, extra = history.checked_volume(data, session, symbol, start, end, out, feed)
                else:
                    value, gaps, extra = checked_reference_volume(
                        data, history, symbol, start, end, out, feed)
                if gaps:
                    status = 'ODD_LOT_VOLUME_RECOVERED'
            except Exception as exc:
                status = 'REFERENCE_UNVERIFIED_' + type(exc).__name__
                detail = str(exc)[:500]
        rows.append(dict(symbol=symbol, signal_time_utc=decision.isoformat(),
                         reference_date=session['date'], volume=value,
                         recovered_gap_minutes=gaps, recovered_volume=extra,
                         status=status, error_detail=detail))
        if value is not None:
            volumes.append(value)
    needed = config.rvol_days if min_reference_days is None else min(min_reference_days, config.rvol_days)
    average = sum(volumes) / len(volumes) if len(volumes) >= needed and volumes else None
    rvol = actual / average if average is not None and average > 0 else None
    verdict = ('NOT_CHECKABLE' if rvol is None else
               'VOLUME_PASS' if rvol >= config.minimum_rvol else 'VOLUME_REJECT')
    return dict(Ergebnis=verdict, RVOL=rvol, Vollständige_Vergleichstage=len(volumes),
                Volumen_bis_Signal=actual, Durchschnitt_20_Tage=average), pd.DataFrame(rows)


def series_endpoints(current, symbol, opening, decision, max_age_minutes):
    """Opening open and latest real close; the close may be up to
    `max_age_minutes` old when the last SIP minutes had no trade."""
    series = minute_series(current, symbol, opening, decision)
    if series.empty or not series.observed.iloc[0]:
        raise ValueError(f'{symbol}: erste SIP-Kerze fehlt oder ist ungültig.')
    # A bar starting at t closes at t + 1 min; the age is measured from that close.
    last_closed = series.last_seen.iloc[-1] + pd.Timedelta(minutes=1)
    if utc(decision) - last_closed > pd.Timedelta(minutes=max_age_minutes):
        raise ValueError(f'{symbol}: letzter SIP-Kurs ist älter als {max_age_minutes} Minuten.')
    return (float(series.open.iloc[0]), float(series.last_close.iloc[-1]),
            int((~series.observed).sum()))


def checked_qqq_endpoints(current, opening, decision, max_age_minutes=2):
    """Relative strength uses only the opening open and last real close."""
    return series_endpoints(current, 'QQQ', opening, decision, max_age_minutes)


def prefix_diagnostic(current, symbol, opening, decision):
    """Persist the data actually seen at a decision; no later backfill."""
    window = current[(current.symbol == symbol) &
                     (current.timestamp >= opening) &
                     (current.timestamp < decision)].sort_values('timestamp')
    expected = pd.date_range(opening, decision, freq='min', inclusive='left')
    missing = expected.difference(pd.DatetimeIndex(window.timestamp))
    return dict(symbol=symbol, bars_seen=len(window), expected_bars=len(expected),
                missing_minutes_utc=[stamp.isoformat() for stamp in missing[:20]],
                total_missing=len(missing),
                duplicate_timestamps=int(window.timestamp.duplicated().sum()),
                first_time_utc=window.timestamp.iloc[0].isoformat() if len(window) else None,
                last_time_utc=window.timestamp.iloc[-1].isoformat() if len(window) else None)


def quote_or_ranking_expired(item, decision, ttl, stage, clock=SYSTEM_CLOCK):
    age = (clock.now() - decision).total_seconds()
    item['signal_age_seconds'] = age
    if age >= ttl:
        item['status'] = f'EXPIRED_{stage}'
        return True
    return False


def inspect_signal(data, symbol, candidate, current, past, opening,
                   out, c, s, settings, refs, last_live_decision, clock=SYSTEM_CLOCK, writer=None):
    writer = writer or RunWriter()
    decision = utc(candidate['decision_time_utc'])
    item = dict(symbol=symbol, decision_time_utc=decision.isoformat(),
                decision_time_ny=decision.tz_convert(NY).isoformat(),
                feed=settings.feed, decision_data_complete=True,
                processing_utc=pd.Timestamp.now(tz='UTC').isoformat(),
                or_high=candidate['or_high'], or_low=candidate['or_low'],
                threshold=candidate['threshold'], price_cap=candidate['price_cap'],
                close=candidate['close'], status='PENDING', rvol=None,
                relative_strength=None, spread=None, bid=None, ask=None,
                score=None, planned_qty=None, orders_sent=0)
    if decision <= last_live_decision:
        item['status'] = 'HISTORICAL_ONLY_STARTUP_OR_RESTART'
        return item
    now = clock.now()
    if now - decision >= pd.Timedelta(seconds=s.signal_ttl_seconds):
        item['status'] = 'EXPIRED_LIVE_DECISION'
        return item
    item.update(opening_bars=candidate.get('opening_bars'),
                coverage=candidate.get('coverage'),
                previous_close_age_minutes=candidate.get('previous_close_age_minutes'))
    try:
        stock_open, stock_close, stock_missing = series_endpoints(
            current, symbol, opening, decision, 0)  # signal bar must be real
        qqq_open, qqq_close, qqq_missing = checked_qqq_endpoints(
            current, opening, decision, settings.benchmark_max_age_minutes)
        item['stock_missing_minutes'] = stock_missing
        item['benchmark_missing_internal_minutes'] = qqq_missing
    except ValueError as exc:
        item['status'] = 'BENCHMARK_OR_STOCK_PREFIX_INVALID'
        item['decision_data_complete'] = False
        item['data_error'] = str(exc)
        item['data_diagnostic'] = {name: prefix_diagnostic(current, name, opening, decision)
                                   for name in (symbol, 'QQQ')}
        snapshot = current[(current.symbol.isin([symbol, 'QQQ'])) &
                           (current.timestamp >= opening) & (current.timestamp < decision)]
        writer.frame(out / 'signal_market_snapshots' /
                     minute_key(decision, symbol).replace('.json', '.csv'), snapshot)
        return item
    try:
        if symbol not in refs:
            refs[symbol] = ReferenceHistory(prepare_reference(data, symbol, past, out, settings), past)
        result, ref_check = check_sip_rvol(data, symbol, current,
            refs[symbol], past, opening, decision, c, out, settings.feed,
            settings.min_rvol_reference_days)
    except Exception as exc:
        item.update(status='RVOL_DATA_ERROR', error_type=type(exc).__name__)
        return item
    item.update(rvol_status=result['Ergebnis'], rvol=float(result['RVOL']) if pd.notna(result['RVOL']) else None,
                complete_reference_days=int(result['Vollständige_Vergleichstage']),
                current_volume=float(result['Volumen_bis_Signal']),
                reference_average=float(result['Durchschnitt_20_Tage']) if pd.notna(result['Durchschnitt_20_Tage']) else None)
    writer.frame(out / 'reference_checks' / minute_key(decision, symbol).replace('.json', '.csv'), ref_check)
    if result['Ergebnis'] != 'VOLUME_PASS':
        item['status'] = result['Ergebnis']
        return item
    item['relative_strength'] = float(stock_close / stock_open - qqq_close / qqq_open)
    if quote_or_ranking_expired(item, decision, s.signal_ttl_seconds, 'DURING_RVOL_CHECK', clock):
        return item
    try:
        from alpaca.data.requests import StockQuotesRequest
        from alpaca.data.enums import DataFeed
        from alpaca.common.enums import Sort
        response = data.get_stock_quotes(StockQuotesRequest(
            symbol_or_symbols=symbol,
            start=(decision - pd.Timedelta(seconds=s.quote_search_seconds)).to_pydatetime(),
            end=decision.to_pydatetime(), sort=Sort.DESC, limit=1, feed=data_feed(settings.feed)))
        quotes = response.data.get(symbol, [])
        quote = max(quotes, key=lambda q: base.utc(q.timestamp)) if quotes else None
        quality = scanner.quote_check(quote, decision, item['threshold'],
                                      item['price_cap'], s, c.minimum_price)
    except Exception as exc:
        item.update(status='QUOTE_DATA_ERROR', error_type=type(exc).__name__)
        return item
    item.update(quality)
    if quote_or_ranking_expired(item, decision, s.signal_ttl_seconds, 'DURING_QUOTE_CHECK', clock):
        return item
    item['status'] = quality['quote_result']
    if quality['quote_result'] == 'PASS':
        item['breakout'] = item['close'] / item['or_high'] - 1
        item['status'] = 'SCANNER_PASS_NO_ORDER'
    return item


def portfolio_context(portfolio, sector=None):
    """Portfolio state at a decision, logged next to a signal (research only)."""
    sectors = [pos['sector'] for pos in portfolio.positions.values()]
    return dict(ctx_open_positions=len(portfolio.positions),
                ctx_same_sector_positions=sectors.count(sector) if sector else None,
                ctx_equity_proxy=float(portfolio.last_equity), ctx_cash=float(portfolio.cash),
                ctx_existing_risk=float(sum(p['qty'] * max(0, p['entry'] - p['stop'])
                                            for p in portfolio.positions.values())),
                ctx_daily_loss_locked=bool(portfolio.locked))


def equity_mark(portfolio, at):
    """Cash plus positions at their last observed close (the portfolio's own marks)."""
    marks = [portfolio.last_marks.get(s, (None, None))[1] for s in portfolio.positions]
    complete = all(m is not None for m in marks)
    value = float(sum(pos['qty'] * m for pos, m in zip(portfolio.positions.values(), marks)
                      if m is not None))
    return dict(timestamp=utc(at).isoformat(), equity=float(portfolio.cash) + value if complete
                else float(portfolio.last_equity), cash=float(portfolio.cash), market_value=value,
                open_positions=len(portfolio.positions), mark_complete=complete,
                symbols=sorted(portfolio.positions))


def run_dryrun(broker, data, project_root, c, s, settings=DryRunSettings(),
               clock=SYSTEM_CLOCK, universe_loader=fresh_universe, replay=None,
               cockpit=None, order_hook=None, output_folder=None, cockpit_mode=None):
    """One trading day on `settings.feed`. Live: wall clock and real-time data.
    Replay/delayed: `clock`, `broker` and `data` come from orb_replay_v16 and
    serve only data already published at the simulated time."""
    feed = settings.feed
    emit = (lambda *args, **kwargs: None) if settings.quiet else print
    if feed not in FEEDS:
        raise ValueError(f'Unbekannter Datenfeed: {feed}')
    if not 1 <= settings.min_rvol_reference_days <= c.rvol_days:
        raise ValueError('min_rvol_reference_days muss zwischen 1 und rvol_days liegen.')
    if order_hook is not None and replay:
        # Replayed or delayed decisions must never become broker orders.
        raise ValueError('Broker-Orders sind nur im Echtzeitbetrieb zulässig.')
    if order_hook is not None:
        import bot_mode_v16
        bot_mode_v16.assert_orders_allowed()   # V1.6: only the main mode PAPER
    from alpaca.trading.requests import GetCalendarRequest
    if not 1 <= settings.poll_seconds <= 15:
        raise ValueError('Ungültige Abfragefrequenz.')
    scanner.validate_scanner(c, s)
    started = clock.now()
    broker_clock = broker.get_clock()
    broker_time = utc(broker_clock.timestamp)
    if abs((broker_time - started).total_seconds()) > 120:
        raise ValueError('Broker-Uhr und Colab-Uhr weichen zu stark ab.')
    date_ny = broker_time.tz_convert(NY).date()
    calendar = base.calendar_rows(broker.get_calendar(GetCalendarRequest(
        start=date_ny-timedelta(days=70), end=date_ny)))
    dates = [r for r in calendar if r['date'] == str(date_ny)]
    past = [r for r in calendar if r['date'] < str(date_ny)][-c.rvol_days:]
    if len(dates) != 1 or len(past) != c.rvol_days:
        raise ValueError('Handelstag oder 20 vorherige Sitzungen fehlen.')
    session = dates[0]
    opening, closing = session['open'], session['close']
    cutoff = closing - pd.Timedelta(minutes=c.entry_cutoff_minutes)
    start_window = session_start_window(started, opening, closing, c)

    out = Path(project_root) / (output_folder or (replay or {}).get('output_folder', 'shadow_replay_v1_6')
                                if (output_folder or replay) else 'shadow_sim_v1_6') / session['date'] / (
        started.strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8])
    out.mkdir(parents=True, exist_ok=False)
    (out / 'signals').mkdir()
    (out / 'reference_checks').mkdir()
    (out / 'signal_market_snapshots').mkdir()
    portfolio = shadow.ShadowPortfolio(c, opening,
        closing - pd.Timedelta(minutes=c.close_before_minutes), initial_cash=settings.initial_cash,
        feed=feed,
        settle_seconds=settings.bar_settle_seconds,
        max_gap_minutes=settings.max_held_gap_minutes,
        max_reentries=settings.max_reentries,
        reentry_cooldown_minutes=settings.reentry_cooldown_minutes)
    broker_start = None
    eligible_count = 0
    cockpit_error = None
    last_price_bar = None
    def paint(stage, decision=None, signals=()):
        nonlocal cockpit_error
        if cockpit is None or cockpit_error is not None:
            return
        try:
            cockpit.update(out, portfolio, clock.now(), decision=decision,
                stage=stage, mode=cockpit_mode or (replay.get('cockpit_mode', 'replay') if replay else 'live'),
                clock_lag_seconds=getattr(clock, 'lag_seconds', 0),
                eligible_count=eligible_count, last_price_bar=last_price_bar,
                signals=signals, broker_start=broker_start)
        except Exception as exc:
            cockpit_error = type(exc).__name__
            portfolio.log('COCKPIT_ERROR', clock.now(), error_type=cockpit_error)
            emit('Cockpit konnte nicht aktualisiert werden:', cockpit_error)
    portfolio_closed = False
    # V1.6.1: buffered run files in the backtest (flushed at the planned close and at the end)
    writer = RunWriter(immediate=settings.persist_every_minute)
    minute_summaries = []
    # V1.6 research logs (never read by the strategy): rejected price crossings,
    # symbols without a valid opening range, equity after every decision minute.
    rejections, rejection_keys, equity_marks = [], set(), []
    def log_rejections(symbol, decision, opening_invalid, found):
        if opening_invalid and (symbol, 'OR') not in rejection_keys:
            rejection_keys.add((symbol, 'OR'))
            rejections.append(dict(symbol=symbol, decision_time_utc=utc(decision).isoformat(),
                                   status='OR_INCOMPLETE', feed=feed))
        for item in found:
            key = (symbol, item['decision_time_utc'])
            if key not in rejection_keys:
                rejection_keys.add(key)
                rejections.append(dict(item, feed=feed, first_seen_decision_utc=utc(decision).isoformat()))
    emit('Simulations-Ergebnisordner:', out)
    status = 'INCOMPLETE'
    session_scope = 'NOT_READY'
    ready_at = None
    try:
        save_json(out / 'manifest.json', dict(version=VERSION, created_at_utc=started.isoformat(),
            mode=(replay.get('mode', 'REPLAY_VIRTUAL_PORTFOLIO') if replay
                  else 'FLEXIBLE_START_VIRTUAL_PORTFOLIO'),
            replay=replay, orders_enabled=False, orders_sent=0,
            start_window=start_window, session_open_utc=opening.isoformat(),
            session_close_utc=closing.isoformat(),
            live_feed=feed, reference_volume_feed=feed,
            prior_day_liquidity_feed='sip_delayed_historical_only',
            strategy=asdict(c), scanner=asdict(s), settings=asdict(settings),
            code_sha256={Path(module.__file__).name: hashlib.sha256(
                Path(module.__file__).read_bytes()).hexdigest()
                for module in ((base, scanner, shadow, sys.modules[__name__]) +
                    tuple(module for module in (sys.modules.get('orb_cockpit_v16'),)
                          if module is not None))},
            virtual_start_cash_usd=settings.initial_cash,
            important='Virtuelle Transaktionen: keine Brokerorders oder echten Ausführungen.'))
        snapshot = account_snapshot(broker, clock)
        broker_start = snapshot
        save_json(out / 'broker_snapshot_start.json', snapshot)
        emit('Paper-Kontoequity:', snapshot['equity'], 'USD; Cash:', snapshot['cash'], 'USD')
        if snapshot['currency'] != 'USD' or snapshot['account_blocked'] or snapshot['trading_blocked']:
            raise ValueError('Paper-Konto ist gesperrt oder nicht in USD.')
        if order_hook is not None:
            order_hook.on_start(out, clock)   # own reconciliation; may disable itself
        elif snapshot['position_symbols'] or snapshot['open_order_ids']:
            raise ValueError('Paper-Konto hat bestehende Positionen/Orders; Dry-Run setzt einen leeren Brokerstatus voraus.')
        if replay:
            # Historical SIP is served by the replay store; the real-time check and
            # the fixed control day are live-only (the control day may lie in the
            # simulated future).
            save_json(out / f'{feed}_access_check.json', dict(
                result=replay.get('access_result', 'REPLAY_HISTORICAL_SIP'), feed=feed, orders_sent=0))
        else:
            try:
                sip_access = check_realtime_access(data, clock.now(), feed)
            except Exception as exc:
                status = f'{feed.upper()}_ACCESS_CHECK_FAILED_NO_ORDERS'
                save_json(out / f'{feed}_access_check.json', dict(
                    result=f'RECENT_{feed.upper()}_UNAVAILABLE_OR_API_ERROR',
                    error_type=type(exc).__name__, feed=feed, orders_sent=0,
                    note='Kein Rückgriff auf einen anderen oder verzögerten Feed. Datenberechtigung und Verbindung prüfen.'))
                raise RuntimeError(f'Echtzeit-{feed.upper()} für aktuelle Minutenkerzen und Quotes '
                    'nicht bestätigt. Simulation sicher gestoppt; keine Brokerorders.') from exc
            save_json(out / f'{feed}_access_check.json', sip_access)
            emit(f'Aktuelle {feed.upper()}-Minutenkerzen und Quotes: Zugriff bestätigt.')
            control_day = pd.Timestamp(settings.historical_control_date + ' 09:30', tz=NY).tz_convert('UTC')
            control = fetch_bars(data, ['AAPL', 'MSFT', 'NVDA'], control_day,
                control_day + pd.Timedelta(minutes=15), feed, 'minute', settings.batch_size)
            control_count = {stock: int((control.symbol == stock).sum()) for stock in ('AAPL', 'MSFT', 'NVDA')}
            save_json(out / f'{feed}_control.json', dict(date=settings.historical_control_date,
                opening_minutes=15, bars_per_symbol=control_count, feed=feed))
            emit(f'{feed.upper()}-Kontrollfenster 15 Min:', control_count)
            if any(count != 15 for count in control_count.values()):
                raise ValueError('15-Minuten-Kontrollfenster ist nicht exakt vollständig.')
        universe = universe_loader(broker)  # Live: no undated or cached index-member fallback.
        save_frame(out / 'universe_snapshot.csv', universe)
        emit('Nasdaq-100-Kandidaten:', len(universe))
        last_past_close = past[-1]['close'] + pd.Timedelta(minutes=1)
        first_past_start = past[0]['open'].normalize()
        daily = fetch_bars(data, list(universe.symbol), first_past_start,
                           last_past_close, 'sip', 'day', settings.batch_size)
        # Prior-day liquidity prefilter stays on historical SIP for every feed:
        # both bots start from the same stock universe.
        save_frame(out / 'liquidity_daily_historical_sip.csv', daily)
        eligible = []
        reasons = []
        for stock in universe.itertuples():
            liquid = scanner.liquidity_check(daily, stock.symbol, [p['date'] for p in past], s)
            verdict = stock.universe_result if stock.universe_result != 'PASS' else liquid['liquidity_result']
            reasons.append(dict(symbol=stock.symbol, sector=stock.sector,
                                average_dollar_volume=liquid['average_dollar_volume'], result=verdict))
            if verdict == 'PASS':
                eligible.append(stock.symbol)
        save_frame(out / 'universe_filter_results.csv', pd.DataFrame(reasons))
        if settings.atr_trailing_multiple > 0:
            portfolio.trail_overrides = atr_trailing(daily, settings)
            save_json(out / 'atr_trailing.json', portfolio.trail_overrides)
        emit(f'{len(eligible)} Aktien passieren Universums- und historischen Liquiditätsfilter')
        if not eligible:
            raise ValueError('Keine zulässigen Aktien; Dry-Run stoppt sicher.')
        eligible_count = len(eligible)
        symbols = sorted(set(eligible + ['QQQ']))
        refs = {}
        seen_decisions = set()
        sector_of = dict(zip(universe.symbol, universe.sector))
        ready_at = clock.now()
        if ready_at >= portfolio.planned_close:
            session_scope = 'NO_SIMULATION_WINDOW_AFTER_STARTUP'
            raise ValueError('Startprüfungen endeten nach dem geplanten virtuellen '
                             'Tagesabschluss; keine Handelssimulation mehr möglich.')
        first_decision = opening + pd.Timedelta(minutes=c.opening_minutes + 1)
        session_scope = ('FULL_SIGNAL_WINDOW' if ready_at < first_decision else
                         'PARTIAL_DAY_LATE_START' if ready_at < cutoff else
                         'PARTIAL_DAY_AFTER_ENTRY_CUTOFF')
        # Decisions before readiness are historical. The current decision
        # may still be processed while its 60-second TTL remains valid.
        startup_decision = ready_at.floor('min') - pd.Timedelta(minutes=1)
        first_expected_decision = max(first_decision,
            completed_decision(ready_at + pd.Timedelta(seconds=5)))
        save_json(out / 'startup.json', dict(started_at_utc=started.isoformat(),
            market_data_ready_at_utc=ready_at.isoformat(),
            market_open_utc=opening.isoformat(), first_signal_utc=first_decision.isoformat(),
            entry_cutoff_utc=cutoff.isoformat(), planned_close_utc=portfolio.planned_close.isoformat(),
            session_close_utc=closing.isoformat(), start_window=start_window,
            session_scope=session_scope, orders_sent=0))
        last_decision = None
        missed_decision_minutes = 0
        data_error_minutes = 0
        after_cutoff_logged = False
        planned_close_checked = False
        planned_close_error = None
        planned_close_processed = None
        end_at = closing
        emit('Startstatus:', start_window, '| Auswertung:', session_scope)
        emit('Virtuelle Simulation; reguläres Ende:', end_at.tz_convert(NY),
              '(NY). Keine Brokerorders möglich.')
        paint('WARTE_AUF_SIGNALFENSTER')
        historical_price_events = 0
        last_current = None
        next_wait_message = clock.now()
        while clock.now() < end_at:
            now = clock.now()
            if now < first_decision:
                if now >= next_wait_message and settings.print_every_minute:
                    stage = 'Börsenöffnung' if now < opening else 'erste Opening Range'
                    target = opening if now < opening else first_decision
                    emit('Warte auf', stage, 'bis', target.tz_convert(NY),
                          '(NY); geplanter Börsenschluss:', closing.tz_convert(NY), '(NY)')
                    next_wait_message = now + pd.Timedelta(minutes=10)
                paint('WARTE_AUF_SIGNALFENSTER')   # the cockpit repaints at most once a minute
                clock.sleep(settings.poll_seconds)
                continue
            decision = completed_decision(now)
            if decision >= cutoff and last_decision is None:
                if ready_at < cutoff:
                    session_scope = 'PARTIAL_DAY_RUNTIME_INTERRUPTED'
                    portfolio.data_degraded = True
                    portfolio.log('MISSED_ENTRY_WINDOW_DURING_RUNTIME', now,
                                  ready_at_utc=ready_at.isoformat())
                # No past signal becomes a live virtual entry.
                try:
                    history = fetch_bars(data, symbols, opening, cutoff,
                        feed, 'minute', settings.batch_size)
                    save_frame(out / f'historical_only_bars_{feed}.csv', history)
                    invalid_openings = invalid_later = 0
                    history_of = split_by_symbol(history)
                    for symbol in eligible:
                        matches, opening_bad, later_bad = price_candidates_until(
                            history_of(symbol), symbol, opening, cutoff, c, settings)
                        invalid_openings += int(opening_bad)
                        invalid_later += int(later_bad)
                        for details in matches:
                            stamp = utc(details['decision_time_utc'])
                            if stamp >= cutoff:
                                continue
                            seen_decisions.add((symbol, stamp.isoformat()))
                            writer.json(out / 'signals' / minute_key(stamp, symbol),
                                dict(symbol=symbol, decision_time_utc=stamp.isoformat(),
                                     status='HISTORICAL_ONLY_STARTUP_OR_RESTART',
                                     feed=feed, or_high=details['or_high'],
                                     processing_utc=pd.Timestamp.now(tz='UTC').isoformat(),
                                     or_low=details['or_low'], orders_sent=0))
                            historical_price_events += 1
                    if invalid_openings or invalid_later:
                        portfolio.data_degraded = True
                    save_json(out / 'historical_backfill_quality.json',
                        dict(eligible_stocks=len(eligible),
                             opening_invalid_stocks=invalid_openings,
                             later_prefix_invalid_stocks=invalid_later,
                             historical_price_events=historical_price_events))
                    emit('Nach Einstiegsschluss gestartet:', historical_price_events,
                          'frühere Kursereignisse nur historisch dokumentiert.')
                except Exception as exc:
                    data_error_minutes += 1
                    portfolio.data_degraded = True
                    portfolio.log('HISTORICAL_BACKFILL_ERROR', clock.now(),
                                  error_type=type(exc).__name__)
                    save_json(out / 'historical_backfill_error.json',
                        dict(error_type=type(exc).__name__, orders_sent=0))
            if decision >= cutoff:
                if not after_cutoff_logged:
                    after_cutoff_logged = True
                    emit('15:15 NY: Keine neuen Einstiege; virtuelle Stops laufen bis 15:30 NY weiter.')
                if decision != last_decision and decision <= portfolio.planned_close:
                    held_bars = pd.DataFrame(columns=['symbol','timestamp','open','high',
                        'low','close','volume','vwap','feed'])
                    requested_at = clock.now()
                    if portfolio.positions:
                        try:
                            held_bars = fetch_bars(data, sorted(portfolio.positions), opening,
                                decision+pd.Timedelta(seconds=1), feed, 'minute', settings.batch_size)
                        except Exception as exc:
                            # A failed request is no evidence of "no SIP trade":
                            # keep the held minutes pending and retry next minute.
                            held_bars = None
                            data_error_minutes += 1
                            portfolio.data_degraded = True
                            portfolio.log('HELD_BAR_FETCH_ERROR', decision,
                                          error_type=type(exc).__name__)
                    if held_bars is not None:
                        # A bar returned by this request was visible when it started;
                        # a slow response must not make timely bars look late.
                        portfolio.update(decision, held_bars, requested_at)
                        check_gap_exits(data, portfolio, clock, feed)
                        if not held_bars.empty:
                            last_price_bar = held_bars.timestamp.max()
                    if order_hook is not None:
                        order_hook.on_minute(decision, clock.now())
                    last_decision = decision
                    equity_marks.append(equity_mark(portfolio, decision))
                    if writer.immediate:
                        save_json(out / 'shadow_state.json', portfolio.report())
                        save_json(out / 'shadow_events.json', portfolio.events)
                    paint('POSITIONEN_UEBERWACHEN', decision)
                if now >= portfolio.planned_close + pd.Timedelta(seconds=5) and not portfolio_closed:
                    planned_close_processed = clock.now()
                    bids = exit_quotes(data, portfolio.positions, clock, feed)
                    portfolio.finish(clock.now(), bids)
                    portfolio_closed = True
                    equity_marks.append(equity_mark(portfolio, clock.now()))
                    writer.flush()
                    if order_hook is not None:
                        order_hook.on_planned_close(clock.now())
                    save_json(out / 'shadow_events.json', portfolio.events)
                    save_json(out / 'shadow_trades.json', portfolio.closed)
                    save_json(out / 'shadow_state.json', portfolio.report())
                    planned_close_checked = True
                    try:
                        save_json(out / 'broker_snapshot_planned_close.json', account_snapshot(broker, clock))
                    except Exception as exc:
                        planned_close_error = type(exc).__name__
                        save_json(out / 'broker_snapshot_planned_close_error.json',
                                  dict(status='CHECK_FAILED', error_type=planned_close_error))
                    emit('15:30 NY: Virtuelle Positionen geschlossen oder als ungelöst markiert;',
                          len(portfolio.closed), 'virtuelle Trades; keine Brokerorders.')
                    paint('GEPLANTER_TAGESABSCHLUSS', decision)
                elif portfolio_closed:
                    paint('GEPLANTER_TAGESABSCHLUSS', last_decision)   # keep "Stand" moving until 16:00
                clock.sleep(settings.poll_seconds)
                continue
            if decision == last_decision or decision < first_decision:
                clock.sleep(settings.poll_seconds)
                continue
            if last_decision is None and decision > first_expected_decision:
                missed = int((decision-first_expected_decision) / pd.Timedelta(minutes=1))
                missed_decision_minutes += missed
                portfolio.data_degraded = True
                writer.json(out / 'signals' / f'{decision:%Y%m%dT%H%M%SZ}_START_SKIPPED.json',
                    dict(status='MISSED_DECISION_MINUTES_AFTER_READY', count=missed,
                         first_expected_utc=first_expected_decision.isoformat(),
                         first_actual_utc=decision.isoformat(), orders_sent=0))
            if last_decision is not None and decision - last_decision > pd.Timedelta(minutes=1):
                missed = int((decision - last_decision) / pd.Timedelta(minutes=1)) - 1
                missed_decision_minutes += missed
                writer.json(out / 'signals' / f'{decision:%Y%m%dT%H%M%SZ}_SKIPPED.json',
                          dict(status='MISSED_DECISION_MINUTES', count=missed,
                               after_utc=last_decision.isoformat(),
                               before_utc=decision.isoformat(), orders_sent=0))
            last_decision = decision
            if decision - now > pd.Timedelta(seconds=2):
                raise ValueError('Lokale Uhrzeit weicht von berechneter Entscheidung ab.')
            requested_at = clock.now()
            try:
                current = fetch_bars(data, symbols, opening,
                    decision + pd.Timedelta(seconds=1), feed, 'minute', settings.batch_size)
            except Exception as exc:
                data_error_minutes += 1
                portfolio.data_degraded = True
                portfolio.log('MARKET_DATA_FETCH_ERROR', decision,
                              error_type=type(exc).__name__)
                writer.json(out / 'signals' / f'{decision:%Y%m%dT%H%M%SZ}_DATA_ERROR.json',
                    dict(decision_time_utc=decision.isoformat(), status='DATA_ERROR_NO_SIGNAL',
                         error_type=type(exc).__name__, orders_sent=0))
                paint('DATENFEHLER', decision)
                continue
            if settings.save_minute_snapshots:
                save_frame(out / f'session_bars_{feed}.csv', current)
            last_current = current
            if not current.empty:
                last_price_bar = current.timestamp.max()
            portfolio.update(decision, current, requested_at)   # see held-bar comment above
            check_gap_exits(data, portfolio, clock, feed)
            if order_hook is not None:
                order_hook.on_minute(decision, clock.now())
            events = []
            invalid_or = 0
            invalid_later = 0
            current_of = split_by_symbol(current)
            for symbol in eligible:
                # Detect already elapsed opportunities, but never turn them into new signals.
                found = []
                matches, opening_invalid, later_invalid = price_candidates_until(
                    current_of(symbol), symbol, opening, decision, c, settings, rejected=found)
                log_rejections(symbol, decision, opening_invalid, found)
                if opening_invalid:
                    invalid_or += 1
                    continue
                if later_invalid:
                    invalid_later += 1
                for details in matches:
                    stamp = utc(details['decision_time_utc'])
                    pair = (symbol, stamp.isoformat())
                    if pair in seen_decisions:
                        continue
                    seen_decisions.add(pair)
                    filename = out / 'signals' / minute_key(stamp, symbol)
                    if writer.exists(filename):
                        continue
                    if stamp != decision or stamp <= startup_decision:
                        result = dict(symbol=symbol, decision_time_utc=stamp.isoformat(),
                            status='HISTORICAL_ONLY_STARTUP_OR_RESTART', feed=feed,
                            processing_utc=pd.Timestamp.now(tz='UTC').isoformat(),
                            or_high=details['or_high'], or_low=details['or_low'],
                            orders_sent=0)
                        historical_price_events += 1
                    else:
                        result = inspect_signal(data, symbol, details, current, past,
                            opening, out, c, s, settings, refs, startup_decision, clock, writer)
                    result.update(sector=sector_of.get(symbol),
                                  **portfolio_context(portfolio, sector_of.get(symbol)))
                    writer.json(filename, result)
                    events.append(result)
            for item in events:
                if item['status'] == 'SCANNER_PASS_NO_ORDER' and quote_or_ranking_expired(
                        item, utc(item['decision_time_utc']), s.signal_ttl_seconds,
                        'BEFORE_RANKING', clock):
                    writer.json(out / 'signals' / minute_key(decision, item['symbol']), item)
            live_passes = [item for item in events if item['status'] == 'SCANNER_PASS_NO_ORDER']
            if live_passes:
                frame = pd.DataFrame([dict(symbol=item['symbol'],
                    decision_time_utc=item['decision_time_utc'],
                    sector=str(universe.set_index('symbol').at[item['symbol'], 'sector']),
                    rvol=item['rvol'], relative_strength=item['relative_strength'],
                    breakout=item['breakout'], spread=item['spread'], ask=item['ask'])
                    for item in live_passes])
                ranked = scanner.rank_candidates(frame, c, s)
                for record in ranked.itertuples():
                    path = out / 'signals' / minute_key(decision, record.symbol)
                    item = writer.read_json(path)
                    item.update(rank=int(record.rank), score=float(record.score),
                        previous_status=item['status'],
                        ranked_at_utc=clock.now().isoformat(),
                        ranking_result=record.ranking_result,
                        rvol_component=float(record.rvol_component),
                        strength_component=float(record.strength_component),
                        breakout_component=float(record.breakout_component),
                        spread_component=float(record.spread_component),
                        status='WATCHLIST_NO_ORDER' if record.watchlist else 'OUTSIDE_TOP_FIVE_NO_ORDER')
                    if record.watchlist:
                        now_entry = clock.now()
                        if (now_entry-decision).total_seconds() >= s.signal_ttl_seconds:
                            quality = dict(quote_result='SIGNAL_EXPIRED')
                        else:
                            try:
                                latest = fresh_quote(data, record.symbol, now_entry,
                                                    s.quote_search_seconds, feed)
                                quality = scanner.quote_check(latest, now_entry,
                                    item['threshold'], item['price_cap'], s, c.minimum_price)
                            except Exception as exc:
                                quality = dict(quote_result='ERROR',
                                               error_type=type(exc).__name__)
                        item['virtual_entry_quote'] = quality
                        item['sector'] = record.sector
                        item['entry_attempt_utc'] = clock.now().isoformat()
                        item.update({'entry_' + k: v for k, v in
                                     portfolio_context(portfolio, record.sector).items()})
                        item['shadow_status'] = portfolio.enter(pd.Timestamp(item['entry_attempt_utc']),
                            dict(symbol=record.symbol, sector=record.sector,
                                 rank=int(record.rank), score=float(record.score),
                                 rvol=float(record.rvol), decision_time_utc=item['decision_time_utc']),
                            quality, s.signal_ttl_seconds)
                        if order_hook is not None:
                            item['paper_order'] = order_hook.on_signal(
                                decision, clock.now(), dict(
                                    symbol=record.symbol, sector=record.sector,
                                    rank=int(record.rank), score=float(record.score),
                                    rvol=float(record.rvol), threshold=item['threshold'],
                                    price_cap=item['price_cap'],
                                    decision_time_utc=item['decision_time_utc']), quality)
                    writer.json(path, item)
                    for event in events:   # the cockpit shows these in-memory items
                        if event['symbol'] == record.symbol and \
                                event['decision_time_utc'] == item['decision_time_utc']:
                            event.update(status=item['status'],
                                         shadow_status=item.get('shadow_status'))
            if writer.immediate:
                save_json(out / 'shadow_events.json', portfolio.events)
                save_json(out / 'shadow_trades.json', portfolio.closed)
                save_json(out / 'shadow_state.json', portfolio.report())
            minute_summary = (out / 'signals' / f'{decision:%Y%m%dT%H%M%SZ}_SUMMARY.json',
                dict(decision_time_utc=decision.isoformat(), feed=feed,
                     eligible_stocks=len(eligible), opening_invalid_stocks=invalid_or,
                     later_prefix_invalid_stocks=invalid_later,
                     new_price_events=len(events), live_scanner_passes=len(live_passes),
                     missed_decision_minutes=missed_decision_minutes,
                     data_error_minutes=data_error_minutes,
                     virtual_open_positions=len(portfolio.positions),
                     virtual_closed_trades=len(portfolio.closed), orders_sent=0))
            if writer.immediate:
                save_json(*minute_summary)
            else:
                minute_summaries.append(minute_summary[1])   # one file per day instead of one per minute
            if settings.print_every_minute:
                emit(decision.tz_convert(NY).strftime('%H:%M'), 'NY:',
                      len(events), 'Preisereignisse,', len(live_passes), 'Scanner-Treffer;',
                      invalid_or, 'Opening Ranges unvollständig;',
                      invalid_later, 'spätere Kursreihen unvollständig')
            equity_marks.append(equity_mark(portfolio, decision))
            paint('SIGNALPRUEFUNG', decision, events)
            clock.sleep(settings.poll_seconds)
        status = ('SHADOW_SIM_INCOMPLETE_OR_DATA_DEGRADED_NO_ORDERS'
                  if missed_decision_minutes or data_error_minutes or planned_close_error or
                  not planned_close_checked or portfolio.positions or portfolio.data_degraded
                  else 'SHADOW_SIM_COMPLETED_NO_ORDERS' if session_scope == 'FULL_SIGNAL_WINDOW'
                  else 'SHADOW_SIM_PARTIAL_DAY_COMPLETED_NO_ORDERS')
        save_json(out / 'coverage.json', dict(missed_decision_minutes=missed_decision_minutes,
                    data_error_minutes=data_error_minutes,
                    cockpit_error=cockpit_error,
                    session_scope=session_scope, historical_price_events_total=historical_price_events,
                    started_at_utc=started.isoformat(),
                    market_data_ready_at_utc=ready_at.isoformat(),
                    planned_close_checked=planned_close_checked,
                    planned_close_processed_utc=(planned_close_processed.isoformat()
                                                 if planned_close_processed is not None else None),
                    planned_close_error=planned_close_error,
                    session_close_utc=closing.isoformat(),
                    note='Ab 15:15 NY nur bestehende virtuelle Positionen überwacht; keine Brokerorders.'))
        if last_current is not None and not settings.save_minute_snapshots:
            save_frame(out / f'session_bars_{feed}.csv', last_current)
    except Exception as exc:
        # Persist why the run stopped; the exception still propagates.
        save_json(out / 'failed.json', dict(status='failed', error_type=type(exc).__name__,
            detail=str(exc)[:2000], http_status=getattr(exc, 'status_code', None),
            failed_at_utc=clock.now().isoformat(), orders_sent=0))
        raise
    finally:
        if order_hook is not None:
            try:   # stopped early or finished: never leave unmanaged paper positions
                order_hook.on_stop(clock.now())
            except Exception as exc:
                status = 'PAPER_FLATTEN_FAILED_MANUAL_REVIEW_REQUIRED'
                emit('ACHTUNG: Paper-Glattstellung fehlgeschlagen:', type(exc).__name__)
        if out.exists():
            writer.flush()
            if minute_summaries:
                save_json(out / 'minute_summaries.json', minute_summaries)
            save_json(out / 'candidate_rejections.json', rejections)
            save_json(out / 'equity_curve.json', equity_marks)
            save_json(out / 'shadow_events.json', portfolio.events)
            save_json(out / 'shadow_trades.json', portfolio.closed)
            save_json(out / 'shadow_state.json', portfolio.report())
            try:
                ending = account_snapshot(broker, clock)
                save_json(out / 'broker_snapshot_end.json', ending)
                if ending['position_symbols'] or ending['open_order_ids']:
                    status = 'BROKER_HAS_OPEN_ITEMS_MANUAL_REVIEW_REQUIRED'
                    emit('ACHTUNG: Broker meldet offene Positionen/Orders. Im Paper-Dashboard manuell prüfen.')
            except Exception as exc:
                status = 'BROKER_END_SNAPSHOT_FAILED_MANUAL_REVIEW_REQUIRED'
                emit('Abschlussabgleich fehlgeschlagen:', type(exc).__name__)
            if order_hook is not None and order_hook.orders_sent and status.endswith('_NO_ORDERS'):
                status = status[:-len('_NO_ORDERS')] + '_WITH_PAPER_ORDERS'
            save_json(out / 'status.json', dict(status=status, finished_at_utc=
                clock.now().isoformat(),
                orders_sent=order_hook.orders_sent if order_hook is not None else 0,
                session_scope=session_scope,
                warning='Virtuelle Ergebnisse sind Modellwerte, keine Brokerfills.'))
            paint('FAILED' if status in ('INCOMPLETE',
                    'BROKER_HAS_OPEN_ITEMS_MANUAL_REVIEW_REQUIRED',
                    'BROKER_END_SNAPSHOT_FAILED_MANUAL_REVIEW_REQUIRED',
                    f'{feed.upper()}_ACCESS_CHECK_FAILED_NO_ORDERS') else 'FINISHED', last_decision if
                    'last_decision' in locals() else None)
    emit('Fertig:', out, '– virtuelle Trades:', len(portfolio.closed),
          '; keine Brokerorders gesendet.')
    return out


In [ ]:
%%writefile /content/orb_cockpit_v16.py
"""Read-only Colab cockpit for virtual ORB positions, backed by run-folder snapshots."""
from collections import deque
from html import escape
from pathlib import Path
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from types import SimpleNamespace
import json
import math
import threading

import pandas as pd


NY = 'America/New_York'
HISTORICAL_MODES = ('replay', 'backtest')   # simulated clock, no account


def utc(value):
    return pd.Timestamp(value).tz_convert('UTC')


def safe(value):
    return escape(str(value if value is not None else '—'), quote=True)


def dollars(value):
    return f'{float(value):,.2f} USD' if value is not None and math.isfinite(float(value)) else '—'


def ny_time(value):
    return utc(value).tz_convert(NY).strftime('%H:%M:%S') if value is not None else '—'


def table(headings, rows, empty):
    header = ''.join(f'<th>{safe(h)}</th>' for h in headings)
    body = ''.join('<tr>' + ''.join(f'<td>{cell}</td>' for cell in row) + '</tr>'
                   for row in rows)
    if not body:
        body = f'<tr><td colspan="{len(headings)}" class="muted">{safe(empty)}</td></tr>'
    return f'<div class="scroll"><table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>'


def snapshot(portfolio, at, decision, stage, mode, eligible_count,
             last_price_bar=None, signals=(), broker_start=None, clock_lag_seconds=0):
    """Use state already computed by the scanner; never request market data."""
    report = portfolio.report()
    positions = []
    for symbol, pos in sorted(portfolio.positions.items()):
        mark_time, mark = portfolio.last_marks.get(symbol, (None, None))
        entry, stop, qty = float(pos['entry']), float(pos['stop']), int(pos['qty'])
        entry_mark = (mark_time is not None and
                      mark_time <= utc(pos['entry_at_utc']).floor('min'))
        positions.append(dict(symbol=symbol, qty=qty,
            entry_at_utc=pos['entry_at_utc'], entry=entry, last_close_proxy=mark,
            last_close_minute_utc=mark_time.isoformat() if mark_time is not None else None,
            mark_is_entry_proxy=entry_mark,
            stop=stop, stop_distance_usd=stop-entry,
            stop_distance_pct=100*(stop/entry-1),
            pnl_open_proxy_usd=qty*(mark-entry) if mark is not None else None,
            planned_risk_usd=qty*max(0, entry-stop),
            stop_coverage_unreliable=bool(pos.get('stop_coverage_unreliable')),
            unobserved_minutes=pos.get('unobserved_minutes', 0),
            pending_minutes=pos.get('pending_minutes', 0)))
    closed = [dict(symbol=p['symbol'], qty=p['qty'],
                   entry_at_utc=p['entry_at_utc'], exit_at_utc=p['exit_at_utc'],
                   entry=p['entry'], exit_price_proxy=p['exit_price_proxy'],
                   pnl_usd=p['pnl_usd'], reason=p['exit_reason'],
                   outcome_unreliable=bool(p.get('outcome_unreliable')))
              for p in portfolio.closed[-12:]]
    return dict(version='1.6', mode=mode, stage=stage, clock_lag_seconds=clock_lag_seconds,
        updated_at_utc=utc(at).isoformat(),
        checked_decision_utc=utc(decision).isoformat() if decision is not None else None,
        last_price_bar_utc=utc(last_price_bar).isoformat() if last_price_bar is not None else None,
        broker_start_equity_usd=(broker_start or {}).get('equity'),
        eligible_symbols=eligible_count,
        closed_count=len(portfolio.closed),
        wins=sum(1 for p in portfolio.closed if p['pnl_usd'] > 0),
        losses=sum(1 for p in portfolio.closed if p['pnl_usd'] < 0),
        start_equity_usd=report.get('starting_cash_usd'),
        orders_sent=0, equity_proxy_usd=report['ending_equity_proxy_usd'],
        cash_proxy_usd=report['ending_cash_usd'],
        invested_usd=sum(float(p['qty'])*float(p['entry']) for p in portfolio.positions.values()),
        planned_open_risk_usd=sum(p['planned_risk_usd'] for p in positions),
        realized_pnl_proxy_usd=report['realized_pnl_usd'],
        realized_pnl_10bp_stress_usd=report['realized_pnl_10bp_stress_usd'],
        open_pnl_proxy_usd=sum(p['pnl_open_proxy_usd'] or 0 for p in positions),
        data_degraded=report['data_degraded'],
        entry_data_blocked=report['entry_data_blocked'],
        unreliable_closed_trades=report['unreliable_virtual_trades'],
        open_positions=positions, closed_trades=closed,
        signals=[dict(symbol=x.get('symbol'), status=x.get('status'),
                      shadow_status=x.get('shadow_status'),
                      time=x.get('decision_time_utc')) for x in signals],
        # Routine per-minute bar reviews would push entries, exits and gaps out of view.
        recent_events=[dict(kind=e['kind'], time=e['at_utc'],
                            symbol=e.get('symbol'), reason=e.get('reason'))
                       for e in [e for e in portfolio.events if e['kind'] != 'BAR_REVIEWED'][-8:]])


def render_html(data):
    """A single self-contained HTML surface; all external data is escaped."""
    lag_minutes = round(data.get('clock_lag_seconds', 0) / 60)
    mode = ('BACKTEST · historische SIP-Daten · KEINE LIVE-ORDERS' if data['mode'] == 'backtest' else
            'Historischer Replay' if data['mode'] == 'replay' else
            f'Verzögerte Live-Beobachtung · SIP {lag_minutes} Min. versetzt' if data['mode'] == 'delayed'
            else f'BOT 2 – SIP-Delayed-Bot · SIMULIERT · SIP {lag_minutes} Min. versetzt'
            if data['mode'] == 'sip_shadow'
            else 'BOT 1 – IEX-Echtzeit-Bot · IEX-Simulation' if data['mode'] == 'iex_live'
            else 'Live-Beobachtung')
    status = ('Lauf mit Fehler beendet' if data['stage'] == 'FAILED' else
              'Datenqualität eingeschränkt' if data['data_degraded'] else
              'Beobachtung läuft' if data['stage'] not in ('FINISHED', 'FAILED') else
              'Lauf beendet')
    status_class = 'bad' if data['data_degraded'] or data['stage'] == 'FAILED' else 'ok'
    entry_status = ('Neue Einstiege: ausstehende Stop-Beobachtung' if
                    data['entry_data_blocked'] else 'Neue Einstiege: keine Datenlückensperre')
    pos_rows = []
    for p in data['open_positions']:
        distance = p['stop_distance_usd']
        tone = 'bad' if distance < 0 else 'ok' if distance > 0 else 'neutral'
        stop = (f'<span class="{tone}">{dollars(p["stop"])} '
                f'({distance:+.2f} / {p["stop_distance_pct"]:+.2f} %)</span>')
        quality = 'Stop-Verlauf unsicher' if p['stop_coverage_unreliable'] else 'beobachtet'
        if p['unobserved_minutes']:
            quality += f' · {p["unobserved_minutes"]} Min. ohne Kerze'
        if p['pending_minutes']:
            quality += f' · {p["pending_minutes"]} Min. ausstehend'
        last_label = ('Einstiegs-Ask · Folgekerze ausstehend' if p['mark_is_entry_proxy'] else
                      f'Kerze {ny_time(p["last_close_minute_utc"])} NY')
        last = (f'{dollars(p["last_close_proxy"])}<br><small>{safe(last_label)}</small>')
        pos_rows.append((safe(p['symbol']), safe(p['qty']),
            f'{dollars(p["entry"])}<br><small>{ny_time(p["entry_at_utc"])} NY</small>',
            last, stop, dollars(p['pnl_open_proxy_usd']),
            f'{dollars(p["planned_risk_usd"])}<br><small>{safe(quality)}</small>'))
    trade_rows = [(safe(t['symbol']),
                   f'{ny_time(t["entry_at_utc"])} → {ny_time(t["exit_at_utc"])} NY',
                   safe(t['qty']), dollars(t['entry']), dollars(t['exit_price_proxy']),
                   dollars(t['pnl_usd']), safe(t['reason'])+
                   (' · <span class="bad">unsicher</span>' if t['outcome_unreliable'] else ''))
                  for t in reversed(data['closed_trades'])]
    signal_rows = [(safe(s['symbol']), ny_time(s['time']), safe(s['status']),
                    safe(s['shadow_status'])) for s in reversed(data['signals'])]
    event_rows = [(ny_time(e['time']), safe(e['symbol']), safe(e['kind']),
                   safe(e['reason'])) for e in reversed(data['recent_events'])]
    bar_note = (' (nach 15:15 NY Kursabruf nur für offene Positionen)'
                if data['checked_decision_utc'] and data['last_price_bar_utc'] and
                utc(data['checked_decision_utc']) - utc(data['last_price_bar_utc']) > pd.Timedelta(minutes=2)
                and not data['open_positions'] else '')
    account_note = (f'Backtest/Replay: kein Alpaca-Konto abgefragt; virtuelles Startkapital des Tages '
                    f'{dollars(data.get("start_equity_usd"))}.'
                    if data['mode'] in HISTORICAL_MODES else
                    'SIMULIERT: eigenes virtuelles Konto von BOT 2; keinerlei Alpaca-Orders möglich.'
                    if data['mode'] == 'sip_shadow' else
                    f'Start-Snapshot des Alpaca-Paper-Kontos: {dollars(data["broker_start_equity_usd"])}; '
                    'niemals mit dem virtuellen Portfolio-Ergebnis verrechnen.')
    def tile(label, value):
        return f'<div class="tile"><span>{safe(label)}</span><strong>{value}</strong></div>'
    cards = ''.join((tile('Virtuelles Equity · Modell', dollars(data['equity_proxy_usd'])),
                     tile('Virtuelles Cash', dollars(data['cash_proxy_usd'])),
                     tile('In offenen Positionen', dollars(data['invested_usd'])),
                     tile('Geplantes offenes Risiko', dollars(data['planned_open_risk_usd'])),
                     tile('Offenes Ergebnis · letzter Kerzenschluss', dollars(data['open_pnl_proxy_usd'])),
                     tile('Realisierter Modell-PnL', dollars(data['realized_pnl_proxy_usd'])),
                     tile('Mit 10 bp Kostenstress', dollars(data['realized_pnl_10bp_stress_usd'])),
                     tile('Offen / geschlossen', f'{len(data["open_positions"])} / {data["closed_count"]}')))
    return f'''<!doctype html><html lang="de"><head><meta charset="utf-8"><style>
.orb-cockpit{{font:14px/1.42 system-ui,-apple-system,sans-serif;color:#182338;background:#f5f8fc;
padding:18px;border-radius:14px;max-width:1260px;margin:0 auto}}
.orb-cockpit h2{{margin:0 0 6px;font-size:22px}} .orb-cockpit h3{{font-size:16px;margin:18px 0 7px}}
.orb-cockpit .subtitle,.orb-cockpit small,.orb-cockpit .muted{{color:#58677a}}
.orb-cockpit .ok{{color:#177245;font-weight:700}} .orb-cockpit .bad{{color:#ba3042;font-weight:700}}
.orb-cockpit .neutral{{color:#344e75;font-weight:700}}
.orb-cockpit .tiles{{display:grid;grid-template-columns:repeat(auto-fit,minmax(190px,1fr));gap:9px;margin:14px 0}}
.orb-cockpit .tile{{background:white;border:1px solid #d9e1ed;border-radius:10px;padding:10px 12px}}
.orb-cockpit .tile span{{display:block;color:#58677a;font-size:12px}} .orb-cockpit .tile strong{{font-size:17px}}
.orb-cockpit .scroll{{overflow-x:auto}} .orb-cockpit table{{border-collapse:collapse;width:100%;background:white}}
.orb-cockpit th,.orb-cockpit td{{border-bottom:1px solid #e1e7f0;text-align:left;padding:8px 9px;vertical-align:top;white-space:nowrap}}
.orb-cockpit th{{background:#eaf0fa;font-size:12px}} .orb-cockpit .note{{margin-top:13px;font-size:12px;color:#47566d}}
</style></head><body><div class="orb-cockpit">
<h2>US-Aktien-Bot · virtuelles Cockpit</h2>
<div class="subtitle" id="orb-stand" data-updated-ms="{int(utc(data['updated_at_utc']).timestamp() * 1000)}" data-quiet="{'1' if data['stage'] in QUIET_STAGES else '0'}" data-replay="{'1' if data['mode'] in HISTORICAL_MODES else '0'}" data-lag-ms="{int(data.get('clock_lag_seconds', 0) * 1000)}">{safe(mode)} · {safe(data['stage'])} · Stand {ny_time(data['updated_at_utc'])} NY ·
letzte Strategieentscheidung {ny_time(data['checked_decision_utc'])} NY ·
letzte verfügbare Kursminute {ny_time(data['last_price_bar_utc'])} NY{bar_note}</div>
<p><span class="{status_class}">{safe(status)}</span> · SIP · {data['eligible_symbols']} Aktien im Filter ·
Alpaca-Orders: 0 · offene Trades sind ausschließlich virtuell.<br>
<span class="{'bad' if data['entry_data_blocked'] else 'neutral'}">{safe(entry_status)}</span></p>
<div class="tiles">{cards}</div>
<h3>Offene virtuelle Positionen</h3>
{table(('Aktie','Stück','Einstieg','Letzter Kerzenschluss','Trailing-Stop','Offener PnL','Risiko / Qualität'), pos_rows, 'Keine offene virtuelle Position.')}
<h3>Abgeschlossene virtuelle Deals · letzte 12</h3>
{table(('Aktie','Zeit','Stück','Kauf-Proxy','Verkauf-Proxy','PnL','Ausstieg / Qualität'), trade_rows, 'Noch kein virtueller Abschluss.')}
<h3>Letzte Signale</h3>
{table(('Aktie','Signal NY','Scannerstatus','Virtueller Einstieg'), signal_rows, 'Noch kein auswertbares Signal.')}
<h3>Letzte Portfolioereignisse</h3>
{table(('Zeit NY','Aktie','Ereignis','Grund'), event_rows, 'Noch keine Portfolioereignisse.')}
<div class="note">Kurse und PnL sind Modellwerte aus abgeschlossenen SIP-Minuten und SIP-Quotes, keine Alpaca-Fills.
Das offene Ergebnis ist eine Bewertung zum letzten beobachteten Kerzenschluss; bei Datenlücken kann sie veraltet sein.
Virtuelle Trades erscheinen nicht im Alpaca-Paper-Konto. Datum/Handelszeit America/New_York.
Scanner prüft abgeschlossene Minuten einmal pro Minute; neue Einstiege bis 15:15 NY, virtuelle Glattstellung 15:30 NY.
<br>{account_note} Nach simulierten Kosten positive Stops sind gesondert zu prüfen.</div>
</div></body></html>'''


BROWSER_REFRESH_SECONDS = 15
# No minute is computed in these stages (waiting for the signal window, after the
# planned close, finished): an old timestamp there is expected, not a stall.
QUIET_STAGES = ('WARTE_AUF_SIGNALFENSTER', 'GEPLANTER_TAGESABSCHLUSS', 'FINISHED', 'FAILED')


def browser_page(page, refresh_seconds=BROWSER_REFRESH_SECONDS):
    """Same view for a normal browser tab: reloads itself and shows its age."""
    head = (f'<meta charset="utf-8"><meta http-equiv="refresh" content="{int(refresh_seconds)}">'
            '<meta name="viewport" content="width=device-width,initial-scale=1">')
    age = ('<p class="muted" id="orb-age"></p><script>(function(){'
           'var s=document.getElementById("orb-stand");if(!s)return;'
           'var e=document.getElementById("orb-age");'
           'if(s.dataset.replay==="1"){e.textContent="Browseransicht lädt alle '
           f'{int(refresh_seconds)}'
           ' s neu · Replay: Uhrzeiten sind simuliert";return;}'
           'var m=Math.floor((Date.now()-Number(s.dataset.lagMs||0)-Number(s.dataset.updatedMs))/60000);if(!(m>=0))return;'
           f'e.textContent="Browseransicht lädt alle {int(refresh_seconds)} s neu · Stand ist "+m+" Min. alt";'
           'if(m>=3&&s.dataset.quiet!=="1"){e.className="bad";'
           'e.textContent+=" – Lauf läuft evtl. nicht mehr (Colab prüfen)";}})();</script>')
    return page.replace('<meta charset="utf-8">', head, 1).replace(
        '<div class="tiles">', age + '<div class="tiles">', 1)


WAITING_PAGE = browser_page('<!doctype html><html lang="de"><head><meta charset="utf-8"></head>'
    '<body style="font:15px system-ui,sans-serif;padding:24px">'
    '<h2>US-Aktien-Bot · virtuelles Cockpit</h2><p>Warte auf den ersten Cockpit-Stand …</p>'
    '<div class="tiles"></div></body></html>')


def write_atomic(path, payload):
    temp = path.with_name(path.name + '.incomplete')
    temp.write_text(payload, encoding='utf-8')
    temp.replace(path)


class Cockpit:
    def __init__(self, display_enabled=True, latest_path=None, latest_json_path=None):
        """`latest_path`: one fixed, self-refreshing HTML file (e.g. on Drive)
        that always shows the newest view, across replay days."""
        self.display_enabled = display_enabled
        self.latest_path = Path(latest_path) if latest_path is not None else None
        self.latest_json_path = Path(latest_json_path) if latest_json_path is not None else None
        self.latest_browser_page = WAITING_PAGE
        self.latest_snapshot_json = '{}'
        self.write_errors = 0
        self.handle = None
        self.last_signals = deque(maxlen=8)
        self.signal_total = 0
        self.closed_count = 0
        self.last_render_key = None
        self.last_run_folder = None

    def update(self, folder, portfolio, at, decision=None, stage='RUNNING',
               mode='live', eligible_count=0, last_price_bar=None,
               signals=(), broker_start=None, clock_lag_seconds=0):
        if str(folder) != self.last_run_folder:
            self.last_run_folder = str(folder)
            self.last_signals.clear()
            self.signal_total = 0
            self.closed_count = 0
            self.last_render_key = None
        for item in signals:
            self.last_signals.append(item)
        self.signal_total += len(signals)
        now = utc(at)
        changed = self.closed_count != len(portfolio.closed)
        # Replay paints every 5 simulated minutes, and on position/signal changes.
        if (mode in HISTORICAL_MODES and stage in ('SIGNALPRUEFUNG', 'POSITIONEN_UEBERWACHEN',
                                           'WARTE_AUF_SIGNALFENSTER', 'GEPLANTER_TAGESABSCHLUSS')
                and not changed and not signals
                and now.minute % 5 != 0):
            return None
        key = (str(folder), now.floor('min'), stage, len(portfolio.closed))
        if key == self.last_render_key and not changed and not signals:
            return None
        self.last_render_key = key
        self.closed_count = len(portfolio.closed)
        data = snapshot(portfolio, now, decision, stage, mode, eligible_count,
                        last_price_bar, self.last_signals, broker_start, clock_lag_seconds)
        data['signal_total'] = self.signal_total
        page = render_html(data)
        snapshot_json = json.dumps(data, ensure_ascii=False, indent=2, default=str)
        self.latest_browser_page = browser_page(page)   # served by the live link
        self.latest_snapshot_json = snapshot_json
        folder = Path(folder)
        targets = [(folder / 'cockpit_snapshot.json', snapshot_json), (folder / 'cockpit.html', page)]
        if self.latest_path is not None:
            targets.append((self.latest_path, self.latest_browser_page))
        if self.latest_json_path is not None:
            targets.append((self.latest_json_path, snapshot_json))
        for path, payload in targets:
            try:
                path.parent.mkdir(parents=True, exist_ok=True)
                write_atomic(path, payload)
            except OSError as exc:
                # A Drive hiccup must not switch the cockpit off for the rest of the day;
                # the next update writes the file again.
                self.write_errors += 1
                print('Cockpit-Datei nicht geschrieben:', path.name, type(exc).__name__)
        if self.display_enabled:
            from IPython.display import HTML, display
            if self.handle is None:
                self.handle = display(HTML(page), display_id=True)
            else:
                self.handle.update(HTML(page))
        return data


def refresh_saved_cockpit(folder, cockpit, ui_hash, mode='replay'):
    """Refresh a cached day's final view without querying or recomputing trades."""
    if cockpit is None:
        return
    folder = Path(folder)
    marker = folder / 'cockpit_render.json'
    if marker.exists() and (folder / 'cockpit.html').exists() and \
            json.loads(marker.read_text()).get('cockpit_sha256') == ui_hash:
        return
    state = json.loads((folder / 'shadow_state.json').read_text())
    if state.get('open_symbols'):
        # No intraday position state is available after a runtime restart.
        return
    closed = json.loads((folder / 'shadow_trades.json').read_text())
    events = json.loads((folder / 'shadow_events.json').read_text())
    previous = json.loads((folder / 'cockpit_snapshot.json').read_text()) if \
        (folder / 'cockpit_snapshot.json').exists() else {}
    status = json.loads((folder / 'status.json').read_text())
    view = SimpleNamespace(positions={}, closed=closed, events=events,
                           last_marks={}, report=lambda: state)
    signals = [dict(symbol=s.get('symbol'), status=s.get('status'),
                    shadow_status=s.get('shadow_status'),
                    decision_time_utc=s.get('time'))
               for s in previous.get('signals', [])]
    cockpit.update(folder, view, utc(status['finished_at_utc']),
        decision=previous.get('checked_decision_utc'),
        stage='FINISHED', mode=mode,
        eligible_count=previous.get('eligible_symbols', 0),
        last_price_bar=previous.get('last_price_bar_utc'),
        signals=signals, broker_start={'equity': state.get('starting_cash_usd')})
    temporary = marker.with_name(marker.name + '.incomplete')
    temporary.write_text(json.dumps(dict(cockpit_sha256=ui_hash,
        rendered_from_saved_state=True, strategy_replayed=False), indent=2))
    temporary.replace(marker)


class _Handler(BaseHTTPRequestHandler):
    cockpit = None

    def do_GET(self):
        path = self.path.split('?', 1)[0]
        if path in ('/', '/index.html'):
            body, kind = self.cockpit.latest_browser_page, 'text/html; charset=utf-8'
        elif path == '/snapshot.json':
            body, kind = self.cockpit.latest_snapshot_json, 'application/json; charset=utf-8'
        else:
            self.send_error(404)
            return
        payload = body.encode('utf-8')
        self.send_response(200)
        self.send_header('Content-Type', kind)
        self.send_header('Cache-Control', 'no-store')
        self.send_header('Content-Length', str(len(payload)))
        self.end_headers()
        self.wfile.write(payload)

    def log_message(self, *args):   # keep the notebook output clean
        pass


def serve_cockpit(cockpit, port=8765, attempts=20):
    """Serve the newest cockpit view over HTTP from a daemon thread.

    Read-only: only the page/snapshot already rendered by Cockpit.update are
    returned; no market data, no orders, no file system access."""
    handler = type('CockpitHandler', (_Handler,), dict(cockpit=cockpit))
    for candidate in range(port, port + attempts):
        try:
            server = ThreadingHTTPServer(('', candidate), handler)
        except OSError:
            continue   # e.g. a previous run of the cell still holds the port
        server.daemon_threads = True
        threading.Thread(target=server.serve_forever, daemon=True,
                         name='orb-cockpit-server').start()
        return server
    raise OSError('Kein freier Port für die Cockpit-Browseransicht.')


def open_browser_view(cockpit, port=8765):
    """Start the server and, inside Colab, show a link that opens it in a browser tab."""
    server = serve_cockpit(cockpit, port)
    port = server.server_address[1]
    try:
        from google.colab import output
    except ImportError:
        print(f'Cockpit im Browser: http://localhost:{port}/')
        return server
    output.serve_kernel_port_as_window(port, path='/', anchor_text='Cockpit im Browser öffnen')
    print('Der Link funktioniert, solange diese Colab-Sitzung läuft; '
          f'die Seite lädt sich alle {BROWSER_REFRESH_SECONDS} s selbst neu.')
    return server


def page_parts(page):
    """(style, body) of a rendered cockpit page, to compose several sections."""
    style = page.split('<style>', 1)[1].split('</style>', 1)[0]
    body = page.split('<body>', 1)[1].rsplit('</body>', 1)[0]
    return style, body


In [ ]:
%%writefile /content/orb_replay_v16.py
"""Replay historical SIP sessions through the unchanged live code path. No orders.

The live loop (sip_shadow_sim_v14.run_dryrun) runs against a simulated clock.
ReplayData serves no future timestamps: a final historical minute bar becomes
visible from `bar_delay_seconds` after its minute, quotes/trades up to now.
The actual arrival of later trades and bar revisions is not in the archive.
Every data request advances the clock by `request_latency_seconds`.
Historical SIP data is fetched once from Alpaca and cached on disk (Drive).
"""
from dataclasses import asdict, dataclass, replace
from datetime import date, datetime, timedelta
from pathlib import Path
from types import SimpleNamespace
import hashlib
import json
import math
import sys
import time
import uuid

import numpy as np
import pandas as pd
import us_orb_test_v02 as base
import us_orb_scanner_v03 as scanner
import orb_portfolio_v15 as shadow
import orb_sim_v16 as sim
import orb_cockpit_v16 as cockpit_module

VERSION = '1.6.1-sip-replay-no-orders'
REPLAY_FOLDER = 'shadow_replay_v1_6'
NY = 'America/New_York'
BAR_COLUMNS = ['symbol', 'timestamp', 'open', 'high', 'low', 'close', 'volume', 'vwap']


@dataclass(frozen=True)
class ReplaySettings:
    start_before_open_minutes: int = 10
    bar_delay_seconds: float = 2.0         # assumed availability, not actual historical arrival time
    request_latency_seconds: float = 0.3   # simulated time consumed by each data request
    source_spacing_seconds: float = 0.35   # real pause between Alpaca requests (rate limit)
    source_batch_size: int = 50


def as_utc(value):
    """alpaca-py stores request datetimes as naive UTC; never read them as NY."""
    stamp = pd.Timestamp(value)
    return stamp.tz_localize('UTC') if stamp.tzinfo is None else stamp.tz_convert('UTC')


def ny_day(value):
    return as_utc(value).tz_convert(NY).strftime('%Y-%m-%d')


def request_key(*parts):
    return hashlib.sha256(json.dumps(parts, default=str).encode()).hexdigest()[:24]


def write_gz(path, frame):
    temporary = path.with_name(path.name + '.incomplete')
    frame.to_csv(temporary, index=False, compression='gzip')
    temporary.replace(path)


def read_gz(path):
    frame = pd.read_csv(path, compression='gzip')
    frame['timestamp'] = pd.to_datetime(frame.timestamp, utc=True)
    return frame


def bars_frame(answer):
    """Alpaca BarSet (or a replay answer) -> flat frame with BAR_COLUMNS."""
    if not any(answer.data.values()):
        return pd.DataFrame(columns=BAR_COLUMNS)
    frame = base.normalize_bars(answer.df)
    if 'vwap' not in frame:
        frame['vwap'] = float('nan')
    return frame[BAR_COLUMNS]


class ReplayClock:
    def __init__(self, start):
        self.current = as_utc(start)

    def now(self):
        return self.current

    def sleep(self, seconds):
        self.advance(seconds)

    def advance(self, seconds):
        self.current += pd.Timedelta(seconds=seconds)


class ReplayStore:
    """Disk cache of historical SIP data; `source_data`/`source_broker` fill gaps.

    Without a source every missing item raises: a replay never silently runs
    on incomplete data."""

    def __init__(self, folder, source_data=None, source_broker=None,
                 spacing_seconds=0.35, batch_size=50):
        self.folder = Path(folder)
        self.folder.mkdir(parents=True, exist_ok=True)
        self.source_data, self.source_broker = source_data, source_broker
        self.spacing, self.batch_size = spacing_seconds, batch_size
        self.last_call = 0.0
        self.minutes = {}       # NY date -> frame of all cached symbols
        self.calendars = {}     # year -> list of calendar dicts
        self.source_calls = 0

    def _call(self, client, method, *args, **kwargs):
        if client is None:
            raise ValueError(f'Replay-Daten fehlen im Cache ({method}) und keine Datenquelle angegeben.')
        wait = self.spacing - (time.monotonic() - self.last_call)
        if wait > 0:
            time.sleep(wait)
        self.last_call = time.monotonic()
        self.source_calls += 1
        return getattr(client, method)(*args, **kwargs)

    # ---- calendar ----
    def _calendar_year(self, year):
        if year not in self.calendars:
            path = self.folder / 'calendar' / f'{year}.json'
            if not path.exists():
                from alpaca.trading.requests import GetCalendarRequest
                rows = self._call(self.source_broker, 'get_calendar', GetCalendarRequest(
                    start=date(year, 1, 1), end=date(year, 12, 31)))
                path.parent.mkdir(parents=True, exist_ok=True)
                sim.save_json(path, [dict(date=str(r.date), open=str(r.open), close=str(r.close))
                                     for r in rows])
            self.calendars[year] = json.loads(path.read_text())
        return self.calendars[year]

    def calendar(self, start, end):
        """Alpaca-shaped calendar entries (naive NY open/close) in [start, end]."""
        start, end = str(start)[:10], str(end)[:10]
        rows = []
        for year in range(int(start[:4]), int(end[:4]) + 1):
            rows.extend(r for r in self._calendar_year(year) if start <= r['date'] <= end)
        return [SimpleNamespace(date=date.fromisoformat(r['date']),
                                open=datetime.fromisoformat(r['open']),
                                close=datetime.fromisoformat(r['close'])) for r in rows]

    def sessions(self, start, end):
        return base.calendar_rows(self.calendar(start, end))

    # ---- universe ----
    def universe(self):
        """One dated Nasdaq-100 snapshot for all replay days (survivorship bias!)."""
        path = self.folder / 'universe_snapshot.csv'
        if not path.exists():
            frame = sim.fresh_universe(self.source_broker)
            self.set_universe(frame, dict(source='nasdaq_api_current_members'))
        return pd.read_csv(path, keep_default_na=False).assign(
            tradable_now=lambda f: f.tradable_now.astype(str) == 'True')

    def set_universe(self, frame, meta):
        sim.save_frame(self.folder / 'universe_snapshot.csv', frame)
        sim.save_json(self.folder / 'universe_provenance.json', dict(
            meta, retrieved_at_utc=pd.Timestamp.now(tz='UTC').isoformat(),
            historical_membership_verified=False,
            note='Aktuelle Indexmitglieder für alle Replay-Tage: Survivorship-Bias.'))

    def universe_meta(self):
        path = self.folder / 'universe_provenance.json'
        return json.loads(path.read_text()) if path.exists() else None

    # ---- minute bars ----
    def day_minutes(self, symbols, day):
        """Regular-hours SIP minute bars of one NY date, fetched once per symbol."""
        folder = self.folder / 'minute'
        path, index_path = folder / f'{day}.csv.gz', folder / f'{day}.symbols.json'
        if day not in self.minutes:
            if path.exists() and index_path.exists():
                self.minutes[day] = (read_gz(path), set(json.loads(index_path.read_text())))
            else:
                self.minutes[day] = (pd.DataFrame(columns=BAR_COLUMNS), set())
        frame, known = self.minutes[day]
        missing = sorted(set(symbols) - known)
        if missing:
            from alpaca.data.requests import StockBarsRequest
            from alpaca.data.enums import DataFeed, Adjustment
            from alpaca.data.timeframe import TimeFrame
            start = pd.Timestamp(f'{day} 09:30', tz=NY).tz_convert('UTC')
            end = pd.Timestamp(f'{day} 16:00', tz=NY).tz_convert('UTC')
            parts = [frame] if len(frame) else []
            for offset in range(0, len(missing), self.batch_size):
                answer = self._call(self.source_data, 'get_stock_bars', StockBarsRequest(
                    symbol_or_symbols=missing[offset:offset + self.batch_size],
                    timeframe=TimeFrame.Minute, start=start.to_pydatetime(),
                    end=end.to_pydatetime(), feed=DataFeed.SIP, adjustment=Adjustment.RAW))
                fetched = bars_frame(answer)
                parts.append(fetched[(fetched.timestamp >= start) & (fetched.timestamp < end)])
            parts = [part for part in parts if len(part)]
            frame = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=BAR_COLUMNS)
            known = known | set(missing)
            folder.mkdir(parents=True, exist_ok=True)
            write_gz(path, frame)
            sim.save_json(index_path, sorted(known))
            self.minutes[day] = (frame, known)
        return frame[frame.symbol.isin(symbols)]

    def keep_days(self, days):
        """Drop in-memory minute data outside `days` (disk cache stays)."""
        for day in list(self.minutes):
            if day not in days:
                del self.minutes[day]

    # ---- other requests, cached per request ----
    def _cached(self, kind, key, fetch):
        path = self.folder / kind / f'{key}.json'
        if not path.exists():
            path.parent.mkdir(parents=True, exist_ok=True)
            sim.save_json(path, fetch())
        return json.loads(path.read_text())

    def daily(self, symbols, start, end):
        def fetch():
            from alpaca.data.requests import StockBarsRequest
            from alpaca.data.enums import DataFeed, Adjustment
            from alpaca.data.timeframe import TimeFrame
            frames = []
            for offset in range(0, len(symbols), self.batch_size):
                answer = self._call(self.source_data, 'get_stock_bars', StockBarsRequest(
                    symbol_or_symbols=symbols[offset:offset + self.batch_size],
                    timeframe=TimeFrame.Day, start=start.to_pydatetime(),
                    end=end.to_pydatetime(), feed=DataFeed.SIP, adjustment=Adjustment.RAW))
                frames.append(bars_frame(answer))
            frames = [f for f in frames if len(f)]
            frame = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=BAR_COLUMNS)
            return json.loads(frame.to_json(orient='records', date_format='iso'))
        rows = self._cached('daily', request_key('daily', sorted(symbols), start, end), fetch)
        frame = pd.DataFrame(rows, columns=BAR_COLUMNS)
        frame['timestamp'] = pd.to_datetime(frame.timestamp, utc=True)
        return frame

    def quotes(self, symbol, start, end, limit, sort):
        def fetch():
            from alpaca.data.requests import StockQuotesRequest
            from alpaca.data.enums import DataFeed
            answer = self._call(self.source_data, 'get_stock_quotes', StockQuotesRequest(
                symbol_or_symbols=symbol, start=start.to_pydatetime(), end=end.to_pydatetime(),
                limit=limit, sort=sort, feed=DataFeed.SIP))
            return [dict(timestamp=as_utc(q.timestamp).isoformat(), bid_price=q.bid_price,
                         ask_price=q.ask_price, bid_size=q.bid_size, ask_size=q.ask_size)
                    for q in answer.data.get(symbol, [])]
        key = request_key('quotes', symbol, start, end, limit, str(sort))
        return [SimpleNamespace(**dict(q, timestamp=pd.Timestamp(q['timestamp'])))
                for q in self._cached(f'quotes/{ny_day(start)}', key, fetch)]

    def trades(self, symbol, start, end):
        def fetch():
            from alpaca.data.requests import StockTradesRequest
            from alpaca.data.enums import DataFeed
            answer = self._call(self.source_data, 'get_stock_trades', StockTradesRequest(
                symbol_or_symbols=symbol, start=start.to_pydatetime(),
                end=end.to_pydatetime(), feed=DataFeed.SIP))
            return [dict(timestamp=as_utc(t.timestamp).isoformat(), size=t.size,
                         conditions=list(t.conditions or []))
                    for t in answer.data.get(symbol, [])]
        key = request_key('trades', symbol, start, end)
        return [SimpleNamespace(**dict(t, timestamp=pd.Timestamp(t['timestamp'])))
                for t in self._cached(f'trades/{ny_day(start)}', key, fetch)]


class ReplayData:
    """Never returns future timestamps; final historical bars may contain revisions."""

    def __init__(self, store, clock, settings=ReplaySettings()):
        self.store, self.clock, self.settings = store, clock, settings
        self._prepared_key, self._prepared, self._normalized = None, None, False

    def _symbols(self, request):
        feed = getattr(request, 'feed', None)
        if feed is not None and getattr(feed, 'value', feed) != 'sip':
            raise ValueError('Der Replay-Speicher enthält nur SIP-Daten.')
        value = request.symbol_or_symbols
        return [value] if isinstance(value, str) else list(value)

    def _done(self, answer):
        self.clock.advance(self.settings.request_latency_seconds)
        return answer

    def published_until(self):
        """Latest minute-bar timestamp that is already published."""
        return self.clock.now() - pd.Timedelta(seconds=60 + self.settings.bar_delay_seconds)

    def get_stock_bars(self, request):
        symbols = self._symbols(request)
        start, end = as_utc(request.start), as_utc(request.end)
        now = self.clock.now()
        if getattr(request.timeframe, 'value', str(request.timeframe)) == '1Day':
            frame = self.store.daily(symbols, start, end)
            today = now.tz_convert(NY).strftime('%Y-%m-%d')
            frame = frame[frame.timestamp.dt.tz_convert(NY).dt.strftime('%Y-%m-%d') < today]
        else:
            # V1.6.1: the day's bars are prepared once (concatenated and sorted) and every
            # minute only cut by timestamp - the same rows in the same order as before.
            span = (ny_day(start), ny_day(min(end, now)))
            key = (tuple(symbols), span)
            if self._prepared_key != key:
                days = [s['date'] for s in self.store.sessions(*span)]
                parts = [part for part in (self.store.day_minutes(symbols, day) for day in days) if len(part)]
                prepared = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=BAR_COLUMNS)
                prepared = prepared.sort_values(['symbol', 'timestamp']).reset_index(drop=True)
                # normalize_bars() form (as orb_sim.fetch_bars would produce it) when the
                # columns allow it; otherwise the loop normalizes as before.
                self._normalized = list(prepared.columns) == BAR_COLUMNS and all(
                    str(prepared[k].dtype) == 'float64' for k in BAR_COLUMNS[2:]) and \
                    isinstance(prepared.timestamp.dtype, pd.DatetimeTZDtype)
                stamps = pd.DatetimeIndex(prepared.timestamp).as_unit('ns').asi8 if len(prepared) \
                    else np.array([], dtype=np.int64)
                self._prepared_key, self._prepared = key, (prepared, stamps)
            prepared, stamps = self._prepared
            mask = ((stamps <= self.published_until().value) & (stamps >= start.value) &
                    (stamps <= end.value))
            frame = prepared[mask].reset_index(drop=True)
            counts = frame.symbol.value_counts(sort=False)
            data = {symbol: [True] * int(counts[symbol]) for symbol in frame.symbol.unique()}
            return self._done(SimpleNamespace(data=data, df=frame, normalized=self._normalized))
        frame = frame[(frame.timestamp >= start) & (frame.timestamp <= end)]
        frame = frame.sort_values(['symbol', 'timestamp']).reset_index(drop=True)
        data = {symbol: [True] * int((frame.symbol == symbol).sum())
                for symbol in frame.symbol.unique()}
        return self._done(SimpleNamespace(data=data, df=frame))

    def _clamped(self, request):
        start, end = as_utc(request.start), min(as_utc(request.end), self.clock.now())
        return start, end

    def get_stock_quotes(self, request):
        (symbol,) = self._symbols(request)
        start, end = self._clamped(request)
        quotes = [] if end < start else self.store.quotes(
            symbol, start, end, request.limit, request.sort)
        quotes = [q for q in quotes if q.timestamp <= self.clock.now()]
        return self._done(SimpleNamespace(data={symbol: quotes} if quotes else {}))

    def get_stock_trades(self, request):
        (symbol,) = self._symbols(request)
        start, end = self._clamped(request)
        trades = [] if end < start else self.store.trades(symbol, start, end)
        return self._done(SimpleNamespace(data={symbol: trades} if trades else {}))


class ReplayBroker:
    """Read-only broker view of the replay: calendar and an empty virtual account.

    There is no order method here; V1.6 passes the day's virtual start equity."""

    def __init__(self, store, clock, equity=10000.0):
        self.store, self.clock, self.equity = store, clock, float(equity)

    def get_clock(self):
        return SimpleNamespace(timestamp=self.clock.now().to_pydatetime(), is_open=None)

    def get_calendar(self, request):
        return self.store.calendar(request.start, request.end)

    def get_account(self):
        return SimpleNamespace(equity=str(self.equity), cash=str(self.equity), currency='USD',
                               account_blocked=False, trading_blocked=False)

    def get_all_positions(self):
        return []

    def get_orders(self, filter=None):
        return []


def replay_settings_default():
    return replace(sim.DryRunSettings(), print_every_minute=False, save_minute_snapshots=False)


def code_hashes(include_cockpit=False):
    """Replay identity follows trading logic; preserve UI provenance separately."""
    modules = (base, scanner, shadow, sim, sys.modules[__name__])
    if include_cockpit:
        modules += (cockpit_module,)
    return {Path(m.__file__).name: hashlib.sha256(Path(m.__file__).read_bytes()).hexdigest()
            for m in modules}


def completed_run(project_root, day, fingerprint, folder_name=REPLAY_FOLDER):
    """An earlier replay of `day` with identical code, settings and start equity, if any."""
    for manifest in sorted((Path(project_root) / folder_name / day).glob('*/manifest.json')):
        folder = manifest.parent
        status = folder / 'status.json'
        coverage = folder / 'coverage.json'
        if not status.exists() or not coverage.exists() or (folder / 'failed.json').exists():
            continue
        state = json.loads(status.read_text())
        quality = json.loads(coverage.read_text())
        if state.get('status') not in ('SHADOW_SIM_COMPLETED_NO_ORDERS',
                                       'SHADOW_SIM_INCOMPLETE_OR_DATA_DEGRADED_NO_ORDERS') or \
                not quality.get('planned_close_checked') or \
                quality.get('data_error_minutes', 0) or quality.get('missed_decision_minutes', 0):
            continue
        replay = json.loads(manifest.read_text()).get('replay') or {}
        if replay.get('fingerprint') == fingerprint:
            return folder
    return None


def replay_day(store, day, project_root, c, s, settings, rs=ReplaySettings(),
               fingerprint=None, cockpit=None, start_equity=None, cockpit_mode='replay',
               mode_label='REPLAY_VIRTUAL_PORTFOLIO', folder_name=REPLAY_FOLDER):
    """Replay one NY trading day; returns the run folder.

    V1.6: `start_equity` is the virtual equity at the day's start (carry-forward)."""
    (session,) = store.sessions(day, day)
    past = store.sessions(date.fromisoformat(day) - timedelta(days=70), day)[:-1][-c.rvol_days:]
    universe = store.universe()
    symbols = sorted(set(universe.symbol) | {'QQQ'})
    for reference in past + [session]:   # warm the cache in real time, not simulated time
        store.day_minutes(symbols, reference['date'])
    store.keep_days({r['date'] for r in past} | {day})
    if start_equity is not None:
        settings = replace(settings, initial_cash=float(start_equity))
    clock = ReplayClock(session['open'] - pd.Timedelta(minutes=rs.start_before_open_minutes))
    info = dict(version=VERSION, fingerprint=fingerprint, replay_settings=asdict(rs),
                universe=store.universe_meta(), cache_folder=str(store.folder),
                data_store=getattr(store, 'identity', 'replay_cache_v1'),
                output_folder=folder_name, cockpit_mode=cockpit_mode, mode=mode_label,
                cockpit_sha256=code_hashes(include_cockpit=True).get(
                    Path(cockpit_module.__file__).name),
                note='Keine späteren Minuten-Zeitstempel. Historische SIP-Kerzen können '
                     'später eingearbeitete Trades/Korrekturen enthalten; reale '
                     'Veröffentlichungszeitpunkte sind nicht rekonstruierbar.')
    return sim.run_dryrun(ReplayBroker(store, clock, settings.initial_cash),
                          ReplayData(store, clock, rs), project_root,
                          replace(c, test_date=day), s, settings, clock=clock,
                          universe_loader=lambda broker: universe, replay=info,
                          cockpit=cockpit)


def refresh_saved_cockpit(folder, cockpit, mode='replay'):
    """Delegate presentation-only rendering to the separately hashed UI."""
    if cockpit is not None:
        ui_hash = code_hashes(include_cockpit=True)[Path(cockpit_module.__file__).name]
        cockpit_module.refresh_saved_cockpit(folder, cockpit, ui_hash, mode)


def replay_fingerprint(store, c, s, settings, rs):
    """Identity of a replay: trading code, settings and the data store kind."""
    return request_key(code_hashes(), asdict(c), asdict(s), asdict(settings), asdict(rs),
                       getattr(store, 'identity', 'replay_cache_v1'))


def day_fingerprint(fingerprint, start_equity):
    return request_key(fingerprint, round(float(start_equity), 2))


def day_result(folder):
    state = json.loads((Path(folder) / 'shadow_state.json').read_text())
    return state, bool(state.get('open_symbols'))


def replay_days(store, project_root, c, s, days, settings=None, rs=ReplaySettings(),
                reuse_completed=True, cockpit=None, initial_equity=None, carry_forward=False,
                stop_on_failure=False, before_day=None, after_day=None,
                cockpit_mode='replay', mode_label='REPLAY_VIRTUAL_PORTFOLIO',
                folder_name=REPLAY_FOLDER):
    """Replay `days` in order; one record per day. The only replay day loop.

    carry_forward: day N starts with the equity after day N-1 (realized P&L);
    a day with an unresolved position ends the chain (its equity is unknown).
    before_day(day, equity) may raise to stop before a day is simulated;
    after_day(record) is called after every day (progress, cockpit); returning False stops."""
    settings = settings or replay_settings_default()
    fingerprint = replay_fingerprint(store, c, s, settings, rs)
    equity = float(settings.initial_cash if initial_equity is None else initial_equity)
    records = []
    for number, day in enumerate(days, 1):
        record = dict(date=day, number=number, days=len(days), start_equity=equity,
                      folder=None, status=None, error_type=None, detail=None)
        records.append(record)
        try:
            if before_day is not None:
                before_day(day, equity)
            day_fp = day_fingerprint(fingerprint, equity)
            folder = completed_run(project_root, day, day_fp, folder_name) if reuse_completed else None
            record['status'] = 'REUSED' if folder is not None else 'SIMULATED'
            if folder is None:
                folder = replay_day(store, day, project_root, c, s, settings, rs, day_fp, cockpit,
                                    start_equity=equity, cockpit_mode=cockpit_mode,
                                    mode_label=mode_label, folder_name=folder_name)
            elif cockpit is not None:
                try:
                    refresh_saved_cockpit(folder, cockpit, cockpit_mode)
                except Exception as exc:
                    print(f'{day}: Cockpit-Ansicht konnte nicht erneuert werden: {type(exc).__name__}')
        except Exception as exc:
            record.update(status=getattr(exc, 'day_status', 'FAILED'), error_type=type(exc).__name__,
                          detail=str(exc)[:500], end_equity=None)
            print(f'{day}: {record["status"]} {type(exc).__name__}: {str(exc)[:200]}')
            stop = after_day(record) is False if after_day is not None else False
            if stop or stop_on_failure or getattr(exc, 'stop_run', False):
                break
            continue
        record['folder'] = folder
        state, unresolved = day_result(folder)
        record.update(pnl=float(state['realized_pnl_usd']), trades=int(state['virtual_trades']),
                      end_equity=equity + float(state['realized_pnl_usd']) if not unresolved else None,
                      unresolved_positions=sorted(state.get('open_symbols') or []))
        if carry_forward and unresolved:
            record['status'] = 'UNRESOLVED_POSITION'      # set before after_day sees the record
        if after_day is not None and after_day(record) is False:
            break
        if carry_forward:
            if unresolved:
                if stop_on_failure:
                    break
                continue   # without carry-forward stop, keep the previous equity
            equity = record['end_equity']
    return records


def run_replay(store, project_root, c, s, start, end, settings=None, rs=ReplaySettings(),
               reuse_completed=True, cockpit=None):
    """Replay all trading days in [start, end]; resumable, then summarize (V1.5 behaviour:
    every day starts with settings.initial_cash)."""
    settings = settings or replay_settings_default()
    days = [r['date'] for r in store.sessions(start, end)]
    records = replay_days(store, project_root, c, s, days, settings, rs, reuse_completed, cockpit)
    folders = [r['folder'] for r in records if r['folder'] is not None]
    failures = [dict(date=r['date'], error_type=r['error_type'], detail=r['detail'])
                for r in records if r['folder'] is None]
    for r in records:
        if r['folder'] is not None:
            print(f"Replay {r['number']}/{r['days']} {r['date']}: {r['trades']} Trades, "
                  f"PnL {r['pnl']:.2f} USD")
    return summarize(folders, Path(project_root) / REPLAY_FOLDER / 'summaries' / (
        pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]),
        failures, dict(start=str(start), end=str(end),
                       fingerprint=replay_fingerprint(store, c, s, settings, rs),
                       replay_settings=asdict(rs), version=VERSION))


def summarize(folders, out, failures=(), meta=None):
    """Aggregate day folders into days.csv, trades.csv and summary.json."""
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    days, trades = [], []
    for folder in folders:
        folder = Path(folder)
        state = json.loads((folder / 'shadow_state.json').read_text())
        status = json.loads((folder / 'status.json').read_text())
        day = folder.parent.name
        days.append(dict(date=day, status=status['status'], trades=state['virtual_trades'],
                         pnl_usd=state['realized_pnl_usd'],
                         pnl_10bp_stress_usd=state['realized_pnl_10bp_stress_usd'],
                         max_drawdown_proxy_usd=state['max_drawdown_proxy_usd'],
                         open_symbols=len(state['open_symbols']),
                         data_degraded=state['data_degraded'],
                         unreliable_trades=state['unreliable_virtual_trades'],
                         unreliable_pnl_proxy_usd=state['pnl_proxy_from_unreliable_trades_usd'],
                         folder=str(folder)))
        for trade in json.loads((folder / 'shadow_trades.json').read_text()):
            notional = trade['qty'] * trade['entry']
            trades.append(dict(date=day, **{k: trade[k] for k in (
                'symbol', 'sector', 'qty', 'entry', 'exit_price_proxy', 'exit_reason',
                'entry_at_utc', 'exit_at_utc', 'pnl_usd', 'pnl_10bp_stress_usd',
                'rank', 'score', 'rvol', 'unobserved_minutes',
                'outcome_unreliable', 'exit_trigger_bar_minute_utc')},
                return_pct=100 * trade['pnl_usd'] / notional,
                return_10bp_stress_pct=100 * trade['pnl_10bp_stress_usd'] / notional))
    day_frame = pd.DataFrame(days)
    trade_frame = pd.DataFrame(trades)
    sim.save_frame(out / 'days.csv', day_frame)
    sim.save_frame(out / 'trades.csv', trade_frame)
    summary = dict(meta or {}, days_replayed=len(days), days_failed=len(failures),
                   failures=list(failures), orders_sent=0)
    if len(day_frame):
        cumulative = day_frame.pnl_usd.cumsum()
        summary.update(
            days_with_trades=int((day_frame.trades > 0).sum()),
            days_data_degraded=int(day_frame.data_degraded.sum()),
            unreliable_trades=int(day_frame.unreliable_trades.sum()),
            pnl_proxy_from_unreliable_trades_usd=float(day_frame.unreliable_pnl_proxy_usd.sum()),
            days_with_unresolved_positions=int((day_frame.open_symbols > 0).sum()),
            total_pnl_usd=float(day_frame.pnl_usd.sum()),
            total_pnl_10bp_stress_usd=float(day_frame.pnl_10bp_stress_usd.sum()),
            max_drawdown_of_daily_pnl_usd=float((cumulative.cummax().clip(lower=0) - cumulative).max()),
            best_day_usd=float(day_frame.pnl_usd.max()), worst_day_usd=float(day_frame.pnl_usd.min()))
    if len(trade_frame):
        wins = trade_frame[trade_frame.pnl_usd > 0].pnl_usd.sum()
        losses = -trade_frame[trade_frame.pnl_usd < 0].pnl_usd.sum()
        summary.update(
            trades=len(trade_frame),
            win_rate=float((trade_frame.pnl_usd > 0).mean()),
            average_trade_usd=float(trade_frame.pnl_usd.mean()),
            average_return_pct=float(trade_frame.return_pct.mean()),
            average_return_10bp_stress_pct=float(trade_frame.return_10bp_stress_pct.mean()),
            profit_factor=float(wins / losses) if losses > 0 else None,
            exit_reasons=trade_frame.exit_reason.value_counts().to_dict())
    summary['warning'] = ('Modellergebnis mit historischen SIP-Daten, aktuellem Index-Universum '
                          '(Survivorship-Bias) und virtuellen Ausführungen; historische '
                          'Kerzen können spätere Korrekturen enthalten; kein Rentabilitätsnachweis.')
    sim.save_json(out / 'summary.json', {k: (None if isinstance(v, float) and not math.isfinite(v) else v)
                                         for k, v in summary.items()})
    print('Replay-Auswertung:', out)
    return out


# ---------------------------------------------------------------------------
# Delayed live run: the live loop on today's SIP data, `lag_minutes` behind
# real time, for accounts without real-time SIP (free plan: SIP >= 15 min old).
# ---------------------------------------------------------------------------

class DelayedClock:
    """Real time minus `lag_minutes`. Sleeps and requests take real time."""

    def __init__(self, lag_minutes=16, base=None):
        if lag_minutes < 16:
            raise ValueError('Mindestens 16 Minuten Versatz: SIP ohne Abo erst nach 15 Minuten.')
        self.base = base or sim.SYSTEM_CLOCK
        self.lag = pd.Timedelta(minutes=lag_minutes)

    @property
    def lag_seconds(self):
        return self.lag.total_seconds()

    def now(self):
        return self.base.now() - self.lag

    def sleep(self, seconds):
        self.base.sleep(seconds)

    def advance(self, seconds):
        pass   # real requests already take real time


class DelayedData:
    """Alpaca data client limited to what was published at the delayed time.

    Every request end is cut to the delayed clock, so nothing younger than the
    lag is ever requested; minute bars appear `bar_delay_seconds` after their
    minute, quotes and trades up to the delayed now."""

    def __init__(self, source, clock, bar_delay_seconds=2.0):
        self.source, self.clock, self.bar_delay_seconds = source, clock, bar_delay_seconds

    def _clamped(self, request):
        start, end = as_utc(request.start), min(as_utc(request.end), self.clock.now())
        return request.model_copy(update={'end': end.to_pydatetime()}), start, end

    def get_stock_bars(self, request):
        clamped, start, end = self._clamped(request)
        now = self.clock.now()
        frame = pd.DataFrame(columns=BAR_COLUMNS)
        if end > start:
            frame = bars_frame(self.source.get_stock_bars(clamped))
        if getattr(request.timeframe, 'value', str(request.timeframe)) == '1Day':
            today = now.tz_convert(NY).strftime('%Y-%m-%d')
            frame = frame[frame.timestamp.dt.tz_convert(NY).dt.strftime('%Y-%m-%d') < today]
        else:
            published = now - pd.Timedelta(seconds=60 + self.bar_delay_seconds)
            frame = frame[frame.timestamp <= published]
        frame = frame[(frame.timestamp >= start) & (frame.timestamp <= end)]
        frame = frame.sort_values(['symbol', 'timestamp']).reset_index(drop=True)
        data = {symbol: [True] * int((frame.symbol == symbol).sum())
                for symbol in frame.symbol.unique()}
        return SimpleNamespace(data=data, df=frame)

    def _items(self, method, request):
        clamped, start, end = self._clamped(request)
        (symbol,) = [request.symbol_or_symbols] if isinstance(request.symbol_or_symbols, str) \
            else list(request.symbol_or_symbols)
        if end < start:
            return SimpleNamespace(data={})
        items = [item for item in getattr(self.source, method)(clamped).data.get(symbol, [])
                 if as_utc(item.timestamp) <= self.clock.now()]
        return SimpleNamespace(data={symbol: items} if items else {})

    def get_stock_quotes(self, request):
        return self._items('get_stock_quotes', request)

    def get_stock_trades(self, request):
        return self._items('get_stock_trades', request)


class DelayedBroker:
    """Read-only view of the real paper account; the clock is the delayed one.

    Only read methods exist here - there is nothing to place an order with."""

    def __init__(self, broker, clock):
        self._broker, self.clock = broker, clock

    def get_clock(self):
        return SimpleNamespace(timestamp=self.clock.now().to_pydatetime(), is_open=None)

    def get_calendar(self, request):
        return self._broker.get_calendar(request)

    def get_account(self):
        return self._broker.get_account()

    def get_all_positions(self):
        return self._broker.get_all_positions()

    def get_orders(self, filter=None):
        return self._broker.get_orders(filter=filter)

    def get_all_assets(self, filter=None):
        return self._broker.get_all_assets(filter=filter)


def run_delayed(broker, data, project_root, c, s, settings=None, cockpit=None,
                lag_minutes=16, rs=ReplaySettings(), base_clock=None,
                universe_loader=sim.fresh_universe):
    """Today's session through the unchanged live loop, `lag_minutes` behind."""
    clock = DelayedClock(lag_minutes, base_clock)
    settings = settings or replace(sim.DryRunSettings(), print_every_minute=False)
    info = dict(kind='delayed_sip', version=VERSION, lag_minutes=lag_minutes,
                mode='DELAYED_SIP_VIRTUAL_PORTFOLIO', output_folder='shadow_delayed_v1_6',
                cockpit_mode='delayed', access_result='DELAYED_HISTORICAL_SIP',
                bar_delay_seconds=rs.bar_delay_seconds,
                note=f'Heutige SIP-Daten, {lag_minutes} Minuten hinter Echtzeit; '
                     'gleiche Live-Schleife, keine Echtzeit-Lieferung getestet.')
    print(f'Verzögerter Lauf: Bot-Uhr {lag_minutes} Min. hinter Echtzeit '
          f'(jetzt {clock.now().tz_convert(NY):%H:%M} NY).')
    return sim.run_dryrun(DelayedBroker(broker, clock), DelayedData(data, clock, rs.bar_delay_seconds),
                          project_root, c, s, settings, clock=clock,
                          universe_loader=universe_loader, replay=info, cockpit=cockpit)


In [ ]:
%%writefile /content/hist_data_v16.py
"""Historical data layer (V1.6): Alpaca SIP data, cache-first, Parquet, manifest.

`HistoricalDataStore` has the interface of `orb_replay_v16.ReplayStore`
(sessions, calendar, universe, day_minutes, daily, quotes, trades), so
ReplayData, ReplayBroker and the unchanged strategy loop run on it as before.

Layout below `MyDrive/US_Aktien_Bot/data/historical/`:
    historical_data_manifest.json         one entry per partition (fast cache checks)
    calendar/<YYYY>.json                  Alpaca exchange calendar
    universe/universe_snapshot.csv        one dated Nasdaq-100 snapshot (survivorship bias!)
    sip/bars_1min/<SYMBOL>/<YYYY-MM>.parquet    regular-session 1-min bars, RAW
    sip/bars_1day/<SYMBOL>/<YYYY>.parquet       daily bars, RAW (liquidity filter)
    sip/quotes/<SYMBOL>/<YYYY-MM>.parquet       quote windows the strategy asked for
    sip/trades/<SYMBOL>/<YYYY-MM>.parquet       trade windows (odd-lot references, research)
    logs/download_log.jsonl               every source request (no credentials)

Rules:
- One feed per store (SIP). There is no IEX fallback anywhere in this module.
- A session counts as cached only after it is complete (close + 15 min) and was
  downloaded without error. A SIP minute without a bar is "no trade", not a gap
  in the cache; missing minutes are reported, never interpolated.
- Quotes and trades are cached per requested time window: only what the strategy
  asks for (the last quote before a decision, the trades of a missing reference
  minute). Full quote/trade streams are never loaded unless explicitly requested.
- Partitions are checked on first use (file present, checksum, row count);
  anything inconsistent is treated as not cached and downloaded again.
"""
from dataclasses import dataclass
from datetime import date, datetime
from pathlib import Path
from types import SimpleNamespace
import hashlib
import json
import math
import time

import numpy as np
import pandas as pd

NY = 'America/New_York'
IDENTITY_VERSION = 'historical_data_v16'
BAR_COLUMNS = ['symbol', 'timestamp', 'open', 'high', 'low', 'close', 'volume', 'vwap']
QUOTE_FIELDS = ['bid_price', 'ask_price', 'bid_size', 'ask_size', 'bid_exchange', 'ask_exchange',
                'conditions', 'tape']
TRADE_FIELDS = ['price', 'size', 'exchange', 'conditions', 'id', 'tape']
SETTLE = pd.Timedelta(minutes=15)       # historical SIP is complete/fetchable after 15 min
SOURCE = 'alpaca_market_data_v2_historical'
FEEDS = ('sip',)


# ------------------------------------------------------------------ errors
class HistoricalDataError(RuntimeError):
    """A data problem the backtest must report; never replaced silently."""
    kind = 'DATA_ERROR'
    day_status = 'DATA_ERROR'
    stop_run = False

    def __init__(self, message, status_code=None):
        super().__init__(message)
        self.status_code = status_code


class DataAccessDenied(HistoricalDataError):
    kind, day_status, stop_run = 'ACCESS_DENIED', 'SIP_UNAVAILABLE', True


class RateLimited(HistoricalDataError):
    kind, day_status, stop_run = 'RATE_LIMITED', 'INCOMPLETE_DATA', True


class InvalidRequest(HistoricalDataError):
    kind, day_status, stop_run = 'INVALID_REQUEST', 'INCOMPLETE_DATA', True


class DataUnavailable(HistoricalDataError):
    kind, day_status, stop_run = 'DATA_UNAVAILABLE', 'INCOMPLETE_DATA', True


def http_status(exc):
    """HTTP status of an alpaca-py APIError (or a test double), else None."""
    for candidate in (getattr(exc, 'status_code', None),
                      getattr(getattr(exc, 'response', None), 'status_code', None),
                      getattr(getattr(getattr(exc, 'http_error', None), 'response', None),
                              'status_code', None)):
        if isinstance(candidate, int):
            return candidate
    return None


def warning_text(feed, reason):
    return ('HISTORICAL DATA WARNING\n\n'
            f'Requested Feed: {feed.upper()}\n'
            f'{feed.upper()} unavailable: {reason}\n\n'
            'No automatic IEX fallback performed.')


# ------------------------------------------------------------------ helpers
def as_utc(value):
    stamp = pd.Timestamp(value)
    return stamp.tz_localize('UTC') if stamp.tzinfo is None else stamp.tz_convert('UTC')


def ny_day(value):
    return as_utc(value).tz_convert(NY).strftime('%Y-%m-%d')


def month_of(day):
    return str(day)[:7]


def safe_symbol(symbol):
    return str(symbol).replace('/', '_')


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()


def write_json(path, content):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.incomplete')
    temporary.write_text(json.dumps(content, ensure_ascii=False, indent=1, default=str,
                                    allow_nan=False) + '\n', encoding='utf-8')
    temporary.replace(path)


def write_parquet(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.incomplete')
    frame.to_parquet(temporary, index=False)
    temporary.replace(path)


def bars_frame(answer):
    """Alpaca BarSet (or a test answer) -> flat frame with BAR_COLUMNS, UTC timestamps."""
    if not answer or not any(getattr(answer, 'data', {}).values()):
        return pd.DataFrame(columns=BAR_COLUMNS)
    frame = answer.df
    frame = frame.reset_index() if 'timestamp' not in frame.columns else frame.copy()
    if 'vwap' not in frame:
        frame['vwap'] = float('nan')
    frame = frame[BAR_COLUMNS].copy()
    frame['timestamp'] = pd.to_datetime(frame.timestamp, utc=True)
    for column in ('open', 'high', 'low', 'close', 'volume', 'vwap'):
        frame[column] = pd.to_numeric(frame[column], errors='coerce').astype(float)
    frame['symbol'] = frame.symbol.astype(str)
    return frame


def validate_bars(frame, now=None):
    """Counts of problems in a bar frame. Invalid rows stay (the strategy flags
    them per minute exactly as before); future rows are a hard error."""
    if frame.empty:
        return dict(rows=0, duplicates=0, ohlc_invalid=0, nonfinite=0, future_rows=0,
                    unsorted=False, timezone_aware=True)
    o, h, l, c, v = (frame[k].to_numpy(dtype=float) for k in ('open', 'high', 'low', 'close', 'volume'))
    with np.errstate(invalid='ignore'):
        finite = np.isfinite(o) & np.isfinite(h) & np.isfinite(l) & np.isfinite(c) & np.isfinite(v)
        ohlc = finite & (h >= o) & (h >= c) & (l <= o) & (l <= c) & (h >= l) & (o > 0) & (l > 0) & (v >= 0)
    stamps = pd.DatetimeIndex(frame.timestamp)
    future = int((stamps > as_utc(now)).sum()) if now is not None else 0
    ordered = frame.sort_values(['symbol', 'timestamp'])
    return dict(rows=len(frame),
                duplicates=int(frame.duplicated(['symbol', 'timestamp']).sum()),
                ohlc_invalid=int((finite & ~ohlc).sum()), nonfinite=int((~finite).sum()),
                future_rows=future, timezone_aware=stamps.tz is not None,
                unsorted=not ordered.index.equals(frame.index))


def missing_minutes(bars, symbol, opening, closing):
    """Regular-session minutes in [opening, closing) without a SIP bar."""
    grid = pd.date_range(as_utc(opening), as_utc(closing), freq='min', inclusive='left')
    have = pd.DatetimeIndex(bars.loc[bars.symbol == symbol, 'timestamp']) if len(bars) else pd.DatetimeIndex([])
    return grid.difference(have)


# ------------------------------------------------------------------ read-only broker
class ReadOnlyBroker:
    """Only calendar and asset lookups reach the trading client; everything else
    (orders, positions, account changes) raises. Used for every backtest."""
    ALLOWED = ('get_calendar', 'get_all_assets')

    def __init__(self, client):
        object.__setattr__(self, '_client', client)

    def __getattr__(self, name):
        if name in ReadOnlyBroker.ALLOWED:
            return getattr(object.__getattribute__(self, '_client'), name)
        raise PermissionError(f'Im Backtest gesperrt: {name} (nur Kalender und Aktienliste lesbar).')

    def __setattr__(self, name, value):
        raise PermissionError('ReadOnlyBroker ist unveränderlich.')


@dataclass(frozen=True)
class RetryPolicy:
    max_attempts: int = 5
    backoff_seconds: tuple = (2, 5, 15, 30)
    spacing_seconds: float = 0.35      # <= ~170 requests/min (free plan: 200/min)
    batch_size: int = 50               # symbols per multi-symbol bar request


# ------------------------------------------------------------------ the store
class HistoricalDataStore:
    """Cache-first historical SIP data; drop-in for ReplayStore."""

    def __init__(self, root, source_data=None, source_broker=None, feed='sip',
                 retry=RetryPolicy(), now=None, sleep=time.sleep, progress=None):
        feed = str(feed).lower()
        if feed not in FEEDS:
            raise ValueError(f'Historischer Feed {feed!r} wird nicht unterstützt; Backtests nutzen SIP. '
                             'Kein automatischer IEX-Fallback.')
        self.folder = Path(root)
        self.folder.mkdir(parents=True, exist_ok=True)
        self.feed = feed
        self.identity = f'{IDENTITY_VERSION}:{feed}'
        self.source_data = source_data
        self.source_broker = (source_broker if source_broker is None or
                              isinstance(source_broker, ReadOnlyBroker) else ReadOnlyBroker(source_broker))
        self.retry, self.sleep, self.progress = retry, sleep, progress
        self.now = now or (lambda: pd.Timestamp.now(tz='UTC'))
        self.last_call = 0.0
        self.source_calls = 0
        self.retries = 0
        self.calendars = {}
        self.partitions = {}       # key -> DataFrame (loaded)
        self.dirty = set()
        self.verified = set()
        self.day_frames = {}       # day -> (frame, known symbols) for fast ReplayData access
        self.manifest_path = self.folder / 'historical_data_manifest.json'
        self.manifest = self._load_manifest()
        self.stats = dict(bars_sessions_cached=0, bars_sessions_downloaded=0,
                          quote_windows_cached=0, quote_windows_downloaded=0,
                          trade_windows_cached=0, trade_windows_downloaded=0,
                          partitions_rejected=0, empty_results=0)

    # ---------------- manifest ----------------
    def _load_manifest(self):
        if self.manifest_path.exists():
            try:
                data = json.loads(self.manifest_path.read_text(encoding='utf-8'))
                if isinstance(data, dict) and isinstance(data.get('partitions'), dict):
                    return data
            except ValueError:
                pass
            broken = self.manifest_path.with_name(self.manifest_path.name + '.unreadable')
            self.manifest_path.replace(broken)
            self.log('MANIFEST_UNREADABLE', moved_to=broken.name)
        return dict(schema_version='1.0', feed=self.feed, source=SOURCE, timezone='UTC',
                    created_at=self.now().isoformat(), partitions={}, migrations={})

    def save_manifest(self):
        self.manifest['last_updated'] = self.now().isoformat()
        write_json(self.manifest_path, self.manifest)

    def log(self, kind, **fields):
        path = self.folder / 'logs' / 'download_log.jsonl'
        path.parent.mkdir(parents=True, exist_ok=True)
        record = dict(kind=kind, at_utc=pd.Timestamp.now(tz='UTC').isoformat(), feed=self.feed, **fields)
        with path.open('a', encoding='utf-8') as handle:
            handle.write(json.dumps(record, ensure_ascii=False, default=str) + '\n')

    def _key(self, dtype, symbol, period):
        return f'{self.feed}/{dtype}/{safe_symbol(symbol)}/{period}'

    def _path(self, key):
        return self.folder / (key + '.parquet')

    def _empty(self, dtype):
        if dtype in ('bars_1min', 'bars_1day'):
            return pd.DataFrame({k: pd.Series(dtype='float64') for k in BAR_COLUMNS}).assign(
                symbol=pd.Series(dtype='object'),
                timestamp=pd.Series(dtype='datetime64[ns, UTC]'))[BAR_COLUMNS]
        fields = QUOTE_FIELDS if dtype == 'quotes' else TRADE_FIELDS
        frame = pd.DataFrame({k: pd.Series(dtype='object') for k in ['window_id'] + fields})
        frame['timestamp'] = pd.Series(dtype='datetime64[ns, UTC]')
        return frame

    def _partition(self, dtype, symbol, period):
        """Loaded partition frame; an inconsistent file is dropped from the cache."""
        key = self._key(dtype, symbol, period)
        if key in self.partitions:
            return key, self.partitions[key]
        entry = self.manifest['partitions'].get(key)
        path = self._path(key)
        frame = self._empty(dtype)
        if entry is not None:
            problem = None
            if not path.exists():
                problem = 'FILE_MISSING'
            elif entry.get('checksum') and sha256_file(path) != entry['checksum']:
                problem = 'CHECKSUM_MISMATCH'
            else:
                try:
                    frame = pd.read_parquet(path)
                    frame['timestamp'] = pd.to_datetime(frame.timestamp, utc=True)
                    if len(frame) != entry.get('rows'):
                        problem = 'ROW_COUNT_MISMATCH'
                except Exception as exc:   # damaged file
                    problem = 'UNREADABLE_' + type(exc).__name__
            if problem:
                self.stats['partitions_rejected'] += 1
                self.log('PARTITION_REJECTED', partition=key, reason=problem)
                del self.manifest['partitions'][key]
                frame = self._empty(dtype)
                self.dirty.add(key)
        elif path.exists():
            # a file without manifest entry (e.g. interrupted write): never trusted
            self.stats['partitions_rejected'] += 1
            self.log('PARTITION_REJECTED', partition=key, reason='NOT_IN_MANIFEST')
        self.partitions[key] = frame
        return key, frame

    def _entry(self, key, dtype, symbol):
        return self.manifest['partitions'].setdefault(key, dict(
            symbol=symbol, feed=self.feed, data_type=dtype,
            timeframe={'bars_1min': '1Min', 'bars_1day': '1Day'}.get(dtype),
            file=key + '.parquet', timezone='UTC', source=SOURCE, file_version=1,
            sessions=[], empty_sessions=[], windows=[], rows=0))

    def flush(self):
        """Write changed partitions and the manifest (called after each download step)."""
        if not self.dirty:
            return
        for key in sorted(self.dirty):
            frame = self.partitions.get(key)
            entry = self.manifest['partitions'].get(key)
            path = self._path(key)
            if entry is None or frame is None:
                continue
            frame = frame.sort_values('timestamp', kind='stable').reset_index(drop=True)
            self.partitions[key] = frame
            write_parquet(path, frame)
            entry.update(rows=int(len(frame)), checksum=sha256_file(path),
                         start=frame.timestamp.min().isoformat() if len(frame) else None,
                         end=frame.timestamp.max().isoformat() if len(frame) else None,
                         last_updated=self.now().isoformat())
        self.dirty.clear()
        self.save_manifest()

    # ---------------- source calls ----------------
    def _call(self, client, method, request, what):
        if client is None:
            raise DataUnavailable(f'Historische Daten fehlen im Cache ({what}) und keine Datenquelle '
                                  'angegeben.')
        policy = self.retry
        for attempt in range(1, policy.max_attempts + 1):
            wait = policy.spacing_seconds - (time.monotonic() - self.last_call)
            if wait > 0:
                self.sleep(wait)
            self.last_call = time.monotonic()
            self.source_calls += 1
            started = time.monotonic()
            try:
                answer = getattr(client, method)(request)
            except Exception as exc:
                status = http_status(exc)
                self.log('REQUEST_FAILED', what=what, method=method, attempt=attempt,
                         http_status=status, error_type=type(exc).__name__)
                if status in (401, 403):
                    raise DataAccessDenied(warning_text(self.feed, f'HTTP {status} – keine Berechtigung '
                                                        f'für {what}'), status) from None
                if status in (400, 404, 422):
                    raise InvalidRequest(f'Ungültige Anfrage für {what} (HTTP {status}).', status) from None
                retryable = status == 429 or (status is not None and status >= 500) or (
                    status is None and isinstance(exc, (ConnectionError, TimeoutError, OSError)))
                if not retryable or attempt == policy.max_attempts:
                    error = RateLimited if status == 429 else DataUnavailable
                    raise error(f'{what}: Abruf nach {attempt} Versuch(en) fehlgeschlagen '
                                f'({type(exc).__name__}, HTTP {status}).', status) from None
                self.retries += 1
                self.sleep(policy.backoff_seconds[min(attempt - 1, len(policy.backoff_seconds) - 1)])
                continue
            self.log('REQUEST_OK', what=what, method=method, attempt=attempt,
                     seconds=round(time.monotonic() - started, 3))
            return answer
        raise AssertionError('unreachable')

    # ---------------- calendar & universe ----------------
    def _calendar_year(self, year):
        if year not in self.calendars:
            path = self.folder / 'calendar' / f'{year}.json'
            if not path.exists():
                from alpaca.trading.requests import GetCalendarRequest
                rows = self._call(self.source_broker, 'get_calendar', GetCalendarRequest(
                    start=date(year, 1, 1), end=date(year, 12, 31)), f'Kalender {year}')
                write_json(path, [dict(date=str(r.date), open=str(r.open), close=str(r.close))
                                  for r in rows])
            self.calendars[year] = json.loads(path.read_text())
        return self.calendars[year]

    def calendar(self, start, end):
        """Alpaca-shaped calendar entries (naive NY open/close) in [start, end]."""
        start, end = str(start)[:10], str(end)[:10]
        rows = []
        for year in range(int(start[:4]), int(end[:4]) + 1):
            rows.extend(r for r in self._calendar_year(year) if start <= r['date'] <= end)
        return [SimpleNamespace(date=date.fromisoformat(r['date']),
                                open=datetime.fromisoformat(r['open']),
                                close=datetime.fromisoformat(r['close'])) for r in rows]

    def sessions(self, start, end):
        import us_orb_test_v02 as base
        return base.calendar_rows(self.calendar(start, end))

    def session(self, day):
        rows = self.sessions(day, day)
        return rows[0] if rows else None

    def settled(self, session):
        return session['close'] + SETTLE <= self.now()

    def universe(self):
        path = self.folder / 'universe' / 'universe_snapshot.csv'
        if not path.exists():
            import orb_sim_v16 as sim
            frame = sim.fresh_universe(self.source_broker)
            self.set_universe(frame, dict(source='nasdaq_api_current_members'))
        return pd.read_csv(path, keep_default_na=False).assign(
            tradable_now=lambda f: f.tradable_now.astype(str) == 'True')

    def set_universe(self, frame, meta):
        folder = self.folder / 'universe'
        folder.mkdir(parents=True, exist_ok=True)
        temporary = folder / 'universe_snapshot.csv.incomplete'
        frame.to_csv(temporary, index=False)
        temporary.replace(folder / 'universe_snapshot.csv')
        write_json(folder / 'universe_provenance.json', dict(
            meta, retrieved_at_utc=pd.Timestamp.now(tz='UTC').isoformat(),
            historical_membership_verified=False,
            note='Aktuelle Indexmitglieder für alle Backtest-Tage: Survivorship-Bias.'))

    def universe_meta(self):
        path = self.folder / 'universe' / 'universe_provenance.json'
        return json.loads(path.read_text()) if path.exists() else None

    # ---------------- minute bars ----------------
    def bars_covered(self, symbol, day):
        key = self._key('bars_1min', symbol, month_of(day))
        entry = self.manifest['partitions'].get(key)
        if entry is None or day not in entry.get('sessions', ()):
            return False
        self._partition('bars_1min', symbol, month_of(day))    # verifies the file once
        entry = self.manifest['partitions'].get(key)
        return entry is not None and day in entry.get('sessions', ())

    def missing_bar_sessions(self, symbols, days):
        return {symbol: [d for d in days if not self.bars_covered(symbol, d)] for symbol in symbols}

    def ensure_minute_bars(self, symbols, days, label='Minutenkerzen'):
        """Download only (symbol, session) pairs that are not cached yet."""
        sessions = {d: self.session(d) for d in days}
        for d, session in sessions.items():
            if session is None:
                raise InvalidRequest(f'{d} ist laut Börsenkalender kein Handelstag.')
            if not self.settled(session):
                raise DataUnavailable(f'{d}: Sitzung ist noch nicht abgeschlossen (Schluss + 15 Min.); '
                                      'unvollständige Tage werden nie gecacht.')
        symbols = sorted(set(symbols))
        missing = self.missing_bar_sessions(symbols, list(days))
        total = sum(len(v) for v in missing.values())
        cached = len(symbols) * len(days) - total
        self.stats['bars_sessions_cached'] += cached
        if not total:
            return dict(cached=cached, downloaded=0)
        # group symbols with the same missing sessions -> few multi-symbol requests
        groups = {}
        for symbol, needed in missing.items():
            if needed:
                groups.setdefault(tuple(needed), []).append(symbol)
        done = 0
        finished_symbols = len([s for s, v in missing.items() if not v])
        for needed, group in groups.items():
            runs = contiguous_runs(list(needed), [r['date'] for r in self.sessions(needed[0], needed[-1])])
            for offset in range(0, len(group), self.retry.batch_size):
                batch = group[offset:offset + self.retry.batch_size]
                for run in runs:
                    self._download_bars(batch, [sessions[d] for d in run])
                    done += len(batch) * len(run)
                    self._report(label, symbols_done=finished_symbols, symbols_total=len(symbols),
                                 progress=(cached + done) / (cached + total),
                                 cached_share=cached / (cached + total),
                                 downloaded_share=done / (cached + total),
                                 period=f'{days[0]} – {days[-1]}')
                finished_symbols += len(batch)
                self.flush()
        self.stats['bars_sessions_downloaded'] += total
        self._report(label, symbols_done=len(symbols), symbols_total=len(symbols), progress=1.0,
                     cached_share=cached / (cached + total), downloaded_share=total / (cached + total),
                     period=f'{days[0]} – {days[-1]}')
        return dict(cached=cached, downloaded=total)

    def _download_bars(self, symbols, sessions):
        from alpaca.data.requests import StockBarsRequest
        from alpaca.data.enums import DataFeed, Adjustment
        from alpaca.data.timeframe import TimeFrame
        start, end = sessions[0]['open'], sessions[-1]['close']
        what = f'SIP-Minutenkerzen {len(symbols)} Symbole {sessions[0]["date"]}–{sessions[-1]["date"]}'
        answer = self._call(self.source_data, 'get_stock_bars', StockBarsRequest(
            symbol_or_symbols=list(symbols), timeframe=TimeFrame.Minute,
            start=start.to_pydatetime(), end=end.to_pydatetime(),
            feed=DataFeed.SIP, adjustment=Adjustment.RAW), what)
        frame = bars_frame(answer)
        check = validate_bars(frame, self.now())
        if check['future_rows'] or not check['timezone_aware']:
            raise DataUnavailable(f'{what}: Zeitstempel aus der Zukunft oder ohne Zeitzone – verworfen.')
        frame = frame[frame.symbol.isin(symbols)]
        # V1.6.1: label each row with its session once, keep regular hours, then store
        # every symbol's sessions in one merge per partition (not one per session).
        labels = frame.timestamp.dt.tz_convert(NY).dt.strftime('%Y-%m-%d') if len(frame) \
            else pd.Series([], dtype=object)
        bounds = {s['date']: (s['open'], s['close']) for s in sessions}
        opens = labels.map(lambda d: bounds.get(d, (None, None))[0])
        closes = labels.map(lambda d: bounds.get(d, (None, None))[1])
        known = opens.notna()
        regular = frame[known & (frame.timestamp >= pd.to_datetime(opens.where(known), utc=True)) &
                        (frame.timestamp < pd.to_datetime(closes.where(known), utc=True))]
        regular_labels = labels[regular.index]
        empty_sessions = []
        for session in sessions:
            if not (regular_labels == session['date']).any() and len(symbols) > 1:
                # A whole batch without a single bar on a trading day is not "no trades".
                self.stats['empty_results'] += 1
                self.log('EMPTY_BATCH_SESSION', what=what, session=session['date'], symbols=len(symbols))
                empty_sessions.append(session['date'])
        good_days = [s['date'] for s in sessions if s['date'] not in empty_sessions]
        by_symbol = dict(tuple(regular.assign(_day=regular_labels).groupby('symbol', sort=False)))
        for symbol in symbols:
            rows = by_symbol.get(symbol, regular.iloc[0:0].assign(_day=pd.Series([], dtype=object)))
            self._store_many('bars_1min', symbol, good_days, rows, month_of)
        if empty_sessions:
            raise DataUnavailable(f'{empty_sessions[0]}: SIP lieferte für {len(symbols)} Symbole keine '
                                  'einzige Kerze (leeres Ergebnis, keine Berechtigungsaussage).')

    def _store_many(self, dtype, symbol, days, rows, period_of):
        """Store several complete sessions of one symbol (rows carry a `_day` label)."""
        for period in sorted({period_of(d) for d in days}):
            period_days = [d for d in days if period_of(d) == period]
            key, frame = self._partition(dtype, symbol, period)
            entry = self._entry(key, dtype, symbol)
            if len(frame):
                existing = frame.timestamp.dt.tz_convert(NY).dt.strftime('%Y-%m-%d')
                frame = frame[~existing.isin(period_days)]
            new = rows[rows._day.isin(period_days)]
            parts = [part for part in (frame, new[BAR_COLUMNS]) if len(part)]
            self.partitions[key] = pd.concat(parts, ignore_index=True) if parts else frame
            entry['sessions'] = sorted(set(entry['sessions']) | set(period_days))
            present = set(new._day)
            empty = [d for d in period_days if d not in present]
            if empty:
                entry['empty_sessions'] = sorted(set(entry.get('empty_sessions', [])) | set(empty))
            if len(new):
                issues = validate_bars(new)
                if issues['duplicates'] or issues['ohlc_invalid'] or issues['nonfinite']:
                    for day, part in new.groupby('_day'):
                        found = validate_bars(part)
                        if found['duplicates'] or found['ohlc_invalid'] or found['nonfinite']:
                            entry.setdefault('validation', {})[day] = {
                                k: found[k] for k in ('duplicates', 'ohlc_invalid', 'nonfinite')}
            stamp = self.now().isoformat()
            entry.setdefault('downloaded_at', {}).update({d: stamp for d in period_days})
            self.dirty.add(key)

    def _store_bars(self, dtype, symbol, day, period, rows):
        key, frame = self._partition(dtype, symbol, period)
        entry = self._entry(key, dtype, symbol)
        stamps = frame.timestamp.dt.tz_convert(NY).dt.strftime('%Y-%m-%d') if len(frame) else pd.Series([], dtype=str)
        keep = frame[stamps != day] if len(frame) else frame
        merged = pd.concat([keep, rows[BAR_COLUMNS]], ignore_index=True) if len(rows) else keep
        self.partitions[key] = merged
        entry['sessions'] = sorted(set(entry['sessions']) | {day})
        if rows.empty:
            entry['empty_sessions'] = sorted(set(entry.get('empty_sessions', [])) | {day})
        issues = validate_bars(rows)
        if issues['duplicates'] or issues['ohlc_invalid'] or issues['nonfinite']:
            entry.setdefault('validation', {})[day] = {k: issues[k] for k in
                                                        ('duplicates', 'ohlc_invalid', 'nonfinite')}
        entry.setdefault('downloaded_at', {})[day] = self.now().isoformat()
        self.dirty.add(key)

    def day_minutes(self, symbols, day):
        """Regular-hours SIP minute bars of one NY session (cache first)."""
        frame, known = self.day_frames.get(day, (None, set()))
        wanted = set(symbols)
        if frame is None or not wanted <= known:
            session = self.session(day)
            if session is None:
                return pd.DataFrame(columns=BAR_COLUMNS)
            self.ensure_minute_bars(sorted(wanted - known), [day])
            parts = [frame] if frame is not None and len(frame) else []
            for symbol in sorted(wanted - known):
                _, part = self._partition('bars_1min', symbol, month_of(day))
                if len(part):
                    part = part[(part.timestamp >= session['open']) & (part.timestamp < session['close'])]
                    parts.append(part[BAR_COLUMNS])
            frame = (pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=BAR_COLUMNS))
            known = known | wanted
            self.day_frames[day] = (frame, known)
        return frame[frame.symbol.isin(wanted)]

    def keep_days(self, days):
        """Drop in-memory data outside `days` (disk cache stays)."""
        months = {month_of(d) for d in days}
        for day in list(self.day_frames):
            if day not in days:
                del self.day_frames[day]
        for key in list(self.partitions):
            if key.split('/')[1] == 'bars_1min' and key.rsplit('/', 1)[1] not in months and key not in self.dirty:
                del self.partitions[key]

    # ---------------- daily bars ----------------
    def daily(self, symbols, start, end):
        """Daily RAW bars with a timestamp in [start, end], cached per symbol and year."""
        start, end = as_utc(start), as_utc(end)
        days = [r['date'] for r in self.sessions(ny_day(start - pd.Timedelta(days=1)), ny_day(end))
                if start <= pd.Timestamp(r['date'], tz=NY).tz_convert('UTC') <= end]
        days = [d for d in days if self.settled(self.session(d))]
        missing = {}
        for symbol in symbols:
            key = self._key('bars_1day', symbol, str(days[0])[:4]) if days else None
            need = []
            for d in days:
                key = self._key('bars_1day', symbol, d[:4])
                self._partition('bars_1day', symbol, d[:4])
                entry = self.manifest['partitions'].get(key)
                if entry is None or d not in entry.get('sessions', ()):
                    need.append(d)
            if need:
                missing.setdefault(tuple(need), []).append(symbol)
        total = sum(len(g) for g in missing.values())
        done = 0
        for need, group in missing.items():
            for offset in range(0, len(group), self.retry.batch_size):
                batch = group[offset:offset + self.retry.batch_size]
                self._download_daily(batch, list(need))
                done += len(batch)
                self._report('daily_bars', symbols_done=done, symbols_total=total,
                             progress=done / max(total, 1), period=f'{need[0]} – {need[-1]}')
        if missing:
            self.flush()
        parts = []
        for symbol in symbols:
            for year in sorted({d[:4] for d in days}):
                _, frame = self._partition('bars_1day', symbol, year)
                if len(frame):
                    parts.append(frame[(frame.timestamp >= start) & (frame.timestamp <= end)])
        frame = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=BAR_COLUMNS)
        frame['timestamp'] = pd.to_datetime(frame.timestamp, utc=True)
        return frame[BAR_COLUMNS].sort_values(['symbol', 'timestamp']).reset_index(drop=True)

    def _download_daily(self, symbols, days):
        from alpaca.data.requests import StockBarsRequest
        from alpaca.data.enums import DataFeed, Adjustment
        from alpaca.data.timeframe import TimeFrame
        start = pd.Timestamp(days[0], tz=NY).tz_convert('UTC')
        end = pd.Timestamp(days[-1], tz=NY).tz_convert('UTC') + pd.Timedelta(hours=12)
        answer = self._call(self.source_data, 'get_stock_bars', StockBarsRequest(
            symbol_or_symbols=list(symbols), timeframe=TimeFrame.Day,
            start=start.to_pydatetime(), end=end.to_pydatetime(),
            feed=DataFeed.SIP, adjustment=Adjustment.RAW),
            f'SIP-Tageskerzen {len(symbols)} Symbole {days[0]}–{days[-1]}')
        frame = bars_frame(answer)
        frame = frame[frame.symbol.isin(symbols)]
        dates = frame.timestamp.dt.tz_convert(NY).dt.strftime('%Y-%m-%d') if len(frame) \
            else pd.Series([], dtype=object)
        frame = frame.assign(_day=dates)
        by_symbol = dict(tuple(frame.groupby('symbol', sort=False)))
        for symbol in symbols:
            rows = by_symbol.get(symbol, frame.iloc[0:0])
            self._store_many('bars_1day', symbol, list(days), rows[rows._day.isin(days)],
                             lambda d: d[:4])

    # ---------------- quotes & trades (windows) ----------------
    def _windowed(self, dtype, symbol, start, end, fetch):
        start, end = as_utc(start), as_utc(end)
        period = month_of(ny_day(start))
        key, frame = self._partition(dtype, symbol, period)
        entry = self._entry(key, dtype, symbol)
        for window in entry.get('windows', []):
            if pd.Timestamp(window['start']) <= start and pd.Timestamp(window['end']) >= end:
                self.stats[f'{dtype[:-1]}_windows_cached'] += 1
                rows = frame[frame.window_id == window['id']]
                return rows[(rows.timestamp >= start) & (rows.timestamp <= end)]
        if end + SETTLE > self.now():
            raise DataUnavailable(f'{symbol} {dtype}: Zeitfenster noch nicht abgeschlossen; '
                                  'wird nicht aus dem Cache bedient.')
        rows = fetch(start, end)
        window_id = hashlib.sha256(f'{symbol}|{start.isoformat()}|{end.isoformat()}'.encode()).hexdigest()[:16]
        rows = rows.assign(window_id=window_id)
        self.partitions[key] = pd.concat([frame, rows], ignore_index=True) if len(rows) else frame
        entry['windows'] = entry.get('windows', []) + [dict(
            id=window_id, start=start.isoformat(), end=end.isoformat(), rows=int(len(rows)),
            downloaded_at=self.now().isoformat())]
        self.dirty.add(key)
        self.stats[f'{dtype[:-1]}_windows_downloaded'] += 1
        return rows[(rows.timestamp >= start) & (rows.timestamp <= end)] if len(rows) else rows

    def _items_frame(self, items, fields):
        rows = []
        for item in items:
            row = dict(timestamp=as_utc(item.timestamp))
            for field in fields:
                value = getattr(item, field, None)
                if isinstance(value, (list, tuple)):
                    value = ','.join(map(str, value))
                elif value is not None and not isinstance(value, (int, float, str)):
                    value = str(getattr(value, 'value', value))
                row[field] = value
            rows.append(row)
        frame = pd.DataFrame(rows, columns=['timestamp'] + fields)
        frame['timestamp'] = pd.to_datetime(frame.timestamp, utc=True)
        for field in fields:
            frame[field] = frame[field].astype(object)
        return frame

    def quotes(self, symbol, start, end, limit, sort):
        """Quotes in [start, end] (the whole window is cached, then sort/limit as the API)."""
        def fetch(a, b):
            from alpaca.data.requests import StockQuotesRequest
            from alpaca.data.enums import DataFeed
            answer = self._call(self.source_data, 'get_stock_quotes', StockQuotesRequest(
                symbol_or_symbols=symbol, start=a.to_pydatetime(), end=b.to_pydatetime(),
                feed=DataFeed.SIP), f'SIP-Quotes {symbol} {a.isoformat()}')
            return self._items_frame((answer.data or {}).get(symbol, []), QUOTE_FIELDS)
        rows = self._windowed('quotes', symbol, start, end, fetch)
        descending = 'desc' in str(getattr(sort, 'value', sort)).lower()
        rows = rows.sort_values('timestamp', ascending=not descending, kind='stable')
        if limit:
            rows = rows.head(int(limit))
        return [SimpleNamespace(timestamp=r.timestamp, **{f: _plain(getattr(r, f)) for f in QUOTE_FIELDS})
                for r in rows.itertuples()]

    def trades(self, symbol, start, end):
        def fetch(a, b):
            from alpaca.data.requests import StockTradesRequest
            from alpaca.data.enums import DataFeed
            answer = self._call(self.source_data, 'get_stock_trades', StockTradesRequest(
                symbol_or_symbols=symbol, start=a.to_pydatetime(), end=b.to_pydatetime(),
                feed=DataFeed.SIP), f'SIP-Trades {symbol} {a.isoformat()}')
            return self._items_frame((answer.data or {}).get(symbol, []), TRADE_FIELDS)
        rows = self._windowed('trades', symbol, start, end, fetch).sort_values('timestamp', kind='stable')
        return [SimpleNamespace(timestamp=r.timestamp, price=_plain(r.price), size=_plain(r.size),
                                exchange=_plain(r.exchange), id=_plain(r.id), tape=_plain(r.tape),
                                conditions=[c for c in str(r.conditions or '').split(',') if c])
                for r in rows.itertuples()]

    def ensure_session_trades(self, symbol, day):
        """FULL_MARKET_DATA option: all SIP trades of one symbol and session (large!)."""
        session = self.session(day)
        self.trades(symbol, session['open'], session['close'] - pd.Timedelta(microseconds=1))
        self.flush()

    # ---------------- quality & status ----------------
    def day_bar_coverage(self, symbols, day, opening_minutes=15):
        """Per symbol: bars, missing regular minutes, opening-range bars (no download)."""
        session = self.session(day)
        rows = []
        for symbol in symbols:
            covered = self.bars_covered(symbol, day)
            frame = pd.DataFrame(columns=BAR_COLUMNS)
            if covered:
                _, part = self._partition('bars_1min', symbol, month_of(day))
                frame = part[(part.timestamp >= session['open']) & (part.timestamp < session['close'])]
            gaps = missing_minutes(frame, symbol, session['open'], session['close']) if covered else []
            or_end = session['open'] + pd.Timedelta(minutes=opening_minutes)
            rows.append(dict(symbol=symbol, cached=covered, bars=int(len(frame)),
                             missing_minutes=int(len(gaps)) if covered else None,
                             opening_range_bars=int((frame.timestamp < or_end).sum()) if covered else None,
                             first_minute_present=bool(covered and (frame.timestamp == session['open']).any())))
        return pd.DataFrame(rows)

    def data_gaps(self, symbols, day, limit=None):
        """DATA GAP entries: symbol, date and missing minute (NY)."""
        session = self.session(day)
        gaps = []
        for symbol in symbols:
            if not self.bars_covered(symbol, day):
                continue
            _, part = self._partition('bars_1min', symbol, month_of(day))
            for minute in missing_minutes(part, symbol, session['open'], session['close']):
                gaps.append(dict(symbol=symbol, date=day, missing_minute_utc=minute.isoformat(),
                                 missing_minute_ny=minute.tz_convert(NY).strftime('%H:%M')))
                if limit and len(gaps) >= limit:
                    return gaps
        return gaps

    def status(self, symbols, days):
        """Compact cache status for the cockpit (no download)."""
        missing = self.missing_bar_sessions(symbols, days)
        missing_days = sorted({d for v in missing.values() for d in v})
        size = sum(p.stat().st_size for p in (self.folder / self.feed).rglob('*.parquet')) \
            if (self.folder / self.feed).exists() else 0
        sessions = sorted({d for e in self.manifest['partitions'].values()
                           if e.get('data_type') == 'bars_1min' for d in e.get('sessions', [])})
        return dict(feed=self.feed.upper(), symbols=len(symbols), days=len(days),
                    bar_sessions_missing=sum(len(v) for v in missing.values()),
                    days_with_missing_bars=len(missing_days), missing_days=missing_days,
                    cached_range=(sessions[0], sessions[-1]) if sessions else None,
                    size_bytes=size, partitions=len(self.manifest['partitions']))

    def report(self):
        return dict(identity=self.identity, feed=self.feed, source=SOURCE, folder=str(self.folder),
                    source_calls=self.source_calls, retries=self.retries, **self.stats)

    def _report(self, label, **fields):
        if self.progress is not None:
            try:
                self.progress(dict(phase=label, **fields))
            except Exception:
                pass

    # ---------------- migration of the V1.4/V1.5 replay cache ----------------
    def migrate_replay_cache(self, old_folder):
        """Take over `replay_cache_sip/` once: calendar, universe, complete minute days.

        A day file is accepted only if it was written after the session was complete
        (file time >= close + 15 min), holds only regular-session bars and QQQ reaches
        the last minutes of the session. Everything else is downloaded again later.
        Quote/trade/daily files of the old cache are keyed by request hashes and are
        not migrated (small, re-fetched on demand)."""
        old = Path(old_folder)
        result = dict(days_accepted=0, days_rejected=0, days_already_done=0, rejected=[])
        if not old.exists():
            return result
        for path in sorted((old / 'calendar').glob('*.json')):
            target = self.folder / 'calendar' / path.name
            if not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(path.read_bytes())
        if (old / 'universe_snapshot.csv').exists() and not (self.folder / 'universe' / 'universe_snapshot.csv').exists():
            (self.folder / 'universe').mkdir(parents=True, exist_ok=True)
            (self.folder / 'universe' / 'universe_snapshot.csv').write_bytes(
                (old / 'universe_snapshot.csv').read_bytes())
            meta = json.loads((old / 'universe_provenance.json').read_text()) \
                if (old / 'universe_provenance.json').exists() else {}
            write_json(self.folder / 'universe' / 'universe_provenance.json',
                       dict(meta, migrated_from=str(old)))
        migrations = self.manifest.setdefault('migrations', {})
        files = sorted((old / 'minute').glob('*.csv.gz'))
        for number, path in enumerate(files, 1):
            day = path.name[:10]
            self._report('migration', days_done=number, days_total=len(files), progress=number / len(files))
            if day in migrations:
                result['days_already_done'] += 1
                continue
            reason = self._migrate_day(path, day)
            migrations[day] = dict(status='ACCEPTED' if reason is None else 'REJECTED', reason=reason,
                                   source_file=str(path), at_utc=self.now().isoformat())
            if reason is None:
                result['days_accepted'] += 1
            else:
                result['days_rejected'] += 1
                result['rejected'].append(dict(date=day, reason=reason))
            if number % 10 == 0:
                self.flush()          # V1.6.1: not after every day (hundreds of Drive writes)
        self.flush()
        self.save_manifest()
        return result

    def _migrate_day(self, path, day):
        index = path.with_name(day + '.symbols.json')
        if not index.exists():
            return 'SYMBOL_INDEX_MISSING'
        session = self.session(day)
        if session is None:
            return 'NOT_A_TRADING_DAY'
        written = pd.Timestamp(path.stat().st_mtime, unit='s', tz='UTC')
        if written < session['close'] + SETTLE:
            return 'WRITTEN_BEFORE_SESSION_COMPLETE'
        try:
            frame = pd.read_csv(path, compression='gzip')
            frame['timestamp'] = pd.to_datetime(frame.timestamp, utc=True)
            symbols = set(json.loads(index.read_text()))
        except Exception as exc:
            return 'UNREADABLE_' + type(exc).__name__
        if 'vwap' not in frame:
            frame['vwap'] = float('nan')
        frame = frame[BAR_COLUMNS]
        check = validate_bars(frame, self.now())
        if check['future_rows'] or not check['timezone_aware']:
            return 'INVALID_TIMESTAMPS'
        regular = frame[(frame.timestamp >= session['open']) & (frame.timestamp < session['close'])]
        if 'QQQ' in symbols:
            qqq = regular[regular.symbol == 'QQQ']
            if qqq.empty or qqq.timestamp.max() < session['close'] - pd.Timedelta(minutes=5) or \
                    qqq.timestamp.min() > session['open']:
                return 'INCOMPLETE_DAY_QQQ_CHECK'
        elif regular.empty:
            return 'EMPTY_DAY'
        for symbol in sorted(symbols):
            if not self.bars_covered(symbol, day):
                self._store_bars('bars_1min', symbol, day, month_of(day),
                                 regular[regular.symbol == symbol])
                self.manifest['partitions'][self._key('bars_1min', symbol, month_of(day))].setdefault(
                    'migrated_sessions', []).append(day)
        return None


def contiguous_runs(days, calendar_days):
    """Split sorted session dates into runs of consecutive trading sessions."""
    position = {d: i for i, d in enumerate(calendar_days)}
    runs, current = [], []
    for d in days:
        if current and position.get(d, -10) != position.get(current[-1], -20) + 1:
            runs.append(current)
            current = []
        current.append(d)
    if current:
        runs.append(current)
    return runs


def _plain(value):
    if value is None:
        return None
    if isinstance(value, float) and math.isnan(value):
        return None
    if isinstance(value, np.generic):
        return value.item()
    return value


In [ ]:
%%writefile /content/data_access_v16.py
"""DATENZUGANG PRÜFEN (V1.6): does this Alpaca account deliver historical SIP data?

Three separate probes for AAPL on a recent complete session: 1-min bars, quotes,
trades. Each result is one of

    AVAILABLE        rows returned
    EMPTY_RESULT     request accepted, no rows (NOT a statement about permissions)
    NOT_AVAILABLE    HTTP 403 (subscription / permission)
    AUTH_FAILED      HTTP 401 (keys)
    RATE_LIMITED     HTTP 429 (try again later)
    INVALID_REQUEST  HTTP 400 / 404 / 422
    PAGINATION_SUSPECT  rows stop well before the end of the requested window
    ERROR            anything else (type and HTTP status only)

Optionally the history is probed (earliest tested date that returns data).
No credentials are read, printed or logged here: the caller passes a ready
data client, and error texts are shortened and scrubbed of long tokens.
"""
from pathlib import Path
import json
import re

import pandas as pd

NY = 'America/New_York'
HISTORY_PROBES = ('2016-01-04', '2018-01-02', '2020-01-02', '2022-01-03', '2024-01-02')
LABELS = {'AVAILABLE': 'AVAILABLE', 'EMPTY_RESULT': 'EMPTY RESULT', 'NOT_AVAILABLE': 'NOT AVAILABLE',
          'AUTH_FAILED': 'AUTH FAILED', 'RATE_LIMITED': 'RATE LIMITED',
          'INVALID_REQUEST': 'INVALID REQUEST', 'PAGINATION_SUSPECT': 'PAGINATION SUSPECT',
          'ERROR': 'ERROR'}
EXPLAIN = {
    'AVAILABLE': 'Daten geliefert.',
    'EMPTY_RESULT': 'Anfrage angenommen, aber keine Datensätze – das ist keine Aussage über die Berechtigung.',
    'NOT_AVAILABLE': 'HTTP 403: Das Konto/Abo erlaubt diese Daten nicht.',
    'AUTH_FAILED': 'HTTP 401: Schlüssel ungültig oder nicht freigegeben (Colab-Secrets prüfen).',
    'RATE_LIMITED': 'HTTP 429: zu viele Anfragen – in einer Minute erneut prüfen.',
    'INVALID_REQUEST': 'Ungültige Anfrage (HTTP 400/404/422).',
    'PAGINATION_SUSPECT': 'Daten enden deutlich vor dem Ende des Zeitfensters – Seitenabruf prüfen.',
    'ERROR': 'Sonstiger Fehler (Verbindung o. Ä.).',
}


def http_status(exc):
    for candidate in (getattr(exc, 'status_code', None),
                      getattr(getattr(exc, 'response', None), 'status_code', None)):
        if isinstance(candidate, int):
            return candidate
    return None


def scrub(text, limit=160):
    """Short error text without anything that looks like a key or token."""
    text = re.sub(r'[A-Za-z0-9_\-]{20,}', '***', str(text or ''))
    return text[:limit]


def classify_error(exc):
    status = http_status(exc)
    result = ('NOT_AVAILABLE' if status == 403 else 'AUTH_FAILED' if status == 401 else
              'RATE_LIMITED' if status == 429 else
              'INVALID_REQUEST' if status in (400, 404, 422) else 'ERROR')
    return dict(result=result, http_status=status, error_type=type(exc).__name__, message=scrub(exc))


def last_complete_session(calendar_rows, now):
    """Latest session whose close is at least 20 minutes ago (plain calendar dicts)."""
    now = pd.Timestamp(now).tz_convert('UTC')
    done = [r for r in calendar_rows if r['close'] + pd.Timedelta(minutes=20) <= now]
    if not done:
        raise ValueError('Kein abgeschlossener Handelstag im Kalender gefunden.')
    return done[-1]


def _count(answer, symbol, kind):
    if kind == 'bars':
        frame = getattr(answer, 'df', None)
        if frame is None or not any((getattr(answer, 'data', None) or {}).values()):
            return 0, None
        frame = frame.reset_index() if 'timestamp' not in frame.columns else frame
        stamps = pd.to_datetime(frame.timestamp, utc=True)
        return len(frame), stamps.max()
    items = (getattr(answer, 'data', None) or {}).get(symbol, [])
    last = max((pd.Timestamp(i.timestamp) for i in items), default=None)
    return len(items), (last.tz_convert('UTC') if last is not None and last.tzinfo else
                        last.tz_localize('UTC') if last is not None else None)


def probe(data_client, kind, symbol, start, end, check_end=True):
    """One raw request (no retries of our own) -> classified result."""
    from alpaca.data.enums import DataFeed, Adjustment
    from alpaca.data.requests import StockBarsRequest, StockQuotesRequest, StockTradesRequest
    from alpaca.data.timeframe import TimeFrame
    start, end = pd.Timestamp(start).tz_convert('UTC'), pd.Timestamp(end).tz_convert('UTC')
    try:
        if kind == 'bars':
            answer = data_client.get_stock_bars(StockBarsRequest(
                symbol_or_symbols=[symbol], timeframe=TimeFrame.Minute, start=start.to_pydatetime(),
                end=end.to_pydatetime(), feed=DataFeed.SIP, adjustment=Adjustment.RAW))
        elif kind == 'quotes':
            answer = data_client.get_stock_quotes(StockQuotesRequest(
                symbol_or_symbols=symbol, start=start.to_pydatetime(), end=end.to_pydatetime(),
                feed=DataFeed.SIP))
        else:
            answer = data_client.get_stock_trades(StockTradesRequest(
                symbol_or_symbols=symbol, start=start.to_pydatetime(), end=end.to_pydatetime(),
                feed=DataFeed.SIP))
    except Exception as exc:
        return dict(kind=kind, start=start.isoformat(), end=end.isoformat(), rows=None, **classify_error(exc))
    rows, last = _count(answer, symbol, kind)
    result = 'AVAILABLE' if rows else 'EMPTY_RESULT'
    # Bars end with the last minute (<= end - 1 min); quotes/trades within seconds of the end.
    slack = pd.Timedelta(minutes=5) if kind == 'bars' else pd.Timedelta(seconds=20)
    if rows and check_end and last is not None and last < end - slack:
        result = 'PAGINATION_SUSPECT'
    return dict(kind=kind, start=start.isoformat(), end=end.isoformat(), rows=rows,
                last_timestamp=last.isoformat() if last is not None else None,
                result=result, http_status=None, error_type=None, message=None)


def check_data_access(data_client, calendar_rows, now=None, symbol='AAPL', history=True,
                      save_folder=None):
    """Probe SIP bars/quotes/trades for `symbol`; optionally the reachable history."""
    now = pd.Timestamp.now(tz='UTC') if now is None else pd.Timestamp(now).tz_convert('UTC')
    session = last_complete_session(calendar_rows, now)
    ten = session['open'] + pd.Timedelta(minutes=30)
    results = dict(
        bars=probe(data_client, 'bars', symbol, session['open'], session['open'] + pd.Timedelta(minutes=30)),
        quotes=probe(data_client, 'quotes', symbol, ten, ten + pd.Timedelta(seconds=60)),
        trades=probe(data_client, 'trades', symbol, ten, ten + pd.Timedelta(seconds=60)))
    reach = {}
    if history:
        for kind in ('bars', 'quotes', 'trades'):
            if results[kind]['result'] != 'AVAILABLE':
                reach[kind] = dict(earliest_available_tested=None, tested=[])
                continue
            tested = []
            for day in HISTORY_PROBES:
                at = pd.Timestamp(f'{day} 10:00', tz=NY).tz_convert('UTC')
                width = pd.Timedelta(minutes=5) if kind == 'bars' else pd.Timedelta(seconds=10)
                outcome = probe(data_client, kind, symbol, at, at + width, check_end=False)
                tested.append(dict(date=day, result=outcome['result'], http_status=outcome['http_status']))
                if outcome['result'] == 'AVAILABLE':
                    break
            first = next((t['date'] for t in tested if t['result'] == 'AVAILABLE'), None)
            reach[kind] = dict(earliest_available_tested=first, tested=tested)
    report = dict(checked_at_utc=now.isoformat(), symbol=symbol, feed='SIP',
                  session=session['date'], results=results, history=reach,
                  quality_possible=quality_possible(results),
                  note='Leeres Ergebnis ist keine Berechtigungsaussage. Kein IEX-Ersatz.')
    if save_folder is not None:
        folder = Path(save_folder)
        folder.mkdir(parents=True, exist_ok=True)
        path = folder / f'data_access_{now:%Y%m%dT%H%M%SZ}.json'
        path.write_text(json.dumps(report, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
        report['saved_to'] = str(path)
    return report


def quality_possible(results):
    ok = {k: results[k]['result'] == 'AVAILABLE' for k in ('bars', 'quotes', 'trades')}
    if ok['bars'] and ok['quotes'] and ok['trades']:
        return 'FULL_MARKET_DATA'
    if ok['bars'] and ok['quotes']:
        return 'BAR_PLUS_QUOTES'
    if ok['bars']:
        return 'BAR_ONLY'
    return None


def format_text(report):
    lines = ['HISTORICAL DATA CAPABILITIES', '']
    for kind, label in (('bars', 'SIP Bars:  '), ('quotes', 'SIP Quotes:'), ('trades', 'SIP Trades:')):
        r = report['results'][kind]
        extra = f' (HTTP {r["http_status"]})' if r.get('http_status') else ''
        rows = f' · {r["rows"]} Zeilen' if r.get('rows') else ''
        lines.append(f'{label}  {LABELS[r["result"]]}{extra}{rows}')
    lines.append('')
    lines.append(f'Testtag: {report["session"]} · Symbol {report["symbol"]}')
    for kind, info in (report.get('history') or {}).items():
        if info.get('earliest_available_tested'):
            lines.append(f'Historie {kind}: Daten ab mindestens {info["earliest_available_tested"]} '
                         '(frühester getesteter Tag)')
    level = report.get('quality_possible')
    lines.append(f'Möglicher Backtest-Level: {level or "keiner – SIP-Bars fehlen"}')
    return '\n'.join(lines)


def format_html(report):
    from html import escape
    rows = []
    for kind, label in (('bars', 'SIP Bars'), ('quotes', 'SIP Quotes'), ('trades', 'SIP Trades')):
        r = report['results'][kind]
        ok = r['result'] == 'AVAILABLE'
        tone = '#177245' if ok else '#ba3042' if r['result'] in ('NOT_AVAILABLE', 'AUTH_FAILED') else '#8a5a00'
        detail = EXPLAIN[r['result']] + (f' · {r["rows"]} Zeilen' if r.get('rows') else '')
        rows.append(f'<tr><td>{label}</td><td style="color:{tone};font-weight:700">'
                    f'{escape(LABELS[r["result"]])}</td><td>{escape(detail)}</td></tr>')
    history = ''.join(
        f'<li>{escape(kind)}: ab mindestens {escape(info["earliest_available_tested"])}</li>'
        for kind, info in (report.get('history') or {}).items() if info.get('earliest_available_tested'))
    level = report.get('quality_possible') or 'keiner – SIP-Bars fehlen'
    return (f'<div style="font:14px system-ui,sans-serif"><b>HISTORICAL DATA CAPABILITIES</b> · '
            f'Testtag {escape(report["session"])} · {escape(report["symbol"])}'
            f'<table style="border-collapse:collapse;margin:6px 0">{"".join(rows)}</table>'
            f'{"<ul>" + history + "</ul>" if history else ""}'
            f'<div>Möglicher Backtest-Level: <b>{escape(level)}</b>. {escape(report["note"])}</div></div>')


In [ ]:
%%writefile /content/research_export_v16.py
"""Research export (V1.6): one folder per backtest run, analysable without the bot code.

CAPTURE DECISIONS, NOT ONLY TRADES: every price crossing the strategy saw is a row
in `signals`, with the stage and reason where it stopped.

This module only READS finished run folders and the historical cache. No strategy,
risk or execution module imports it, and it runs only after the replay of the whole
period has ended. Counterfactual columns (future_*) are computed here from bars
after the decision time and are never visible to the strategy.

Files per run (backtests/<RUN_ID>/):
    run_manifest.json  summary.json  schema.json
    trades.parquet/.csv  signals.parquet/.csv  daily.parquet/.csv
    equity_curve.parquet/.csv  events.parquet  data_gaps.parquet
    <RUN_ID>_ANALYSIS.zip  (manifest, summary, schema, CSVs, README)
Index of all runs: backtests/backtest_runs.csv
"""
from dataclasses import asdict
from pathlib import Path
import hashlib
import importlib.metadata
import json
import math
import platform
import zipfile

import numpy as np
import pandas as pd

import us_orb_test_v02 as base

RESEARCH_SCHEMA_VERSION = '1.0'
BOT_VERSION = 'V1.6.1'
STRATEGY_NAME = 'ORB15_LONG_NASDAQ100'
NY = 'America/New_York'
STOP_REASONS = ('TRAILING_STOP_PROXY', 'STOP_GAP_OPEN_PROXY', 'STOP_BID_QUOTE_PROXY')
EOD_REASONS = ('PLANNED_CLOSE_BID_PROXY', 'PLANNED_CLOSE_LAST_BAR_PROXY')
GAP_REASONS = ('EXIT_DATA_GAP_BID_PROXY',)
HORIZONS = (1, 5, 15, 30, 60)
EXCURSION_HORIZONS = (15, 30)
SESSION_BUCKETS = (('OPENING', '09:30', '10:00'), ('MORNING', '10:00', '11:30'),
                   ('MIDDAY', '11:30', '14:00'), ('AFTERNOON', '14:00', '15:30'),
                   ('LATE_SESSION', '15:30', '16:00'))
FLAGS = ('failed_price_band', 'failed_data', 'failed_timing', 'failed_rvol', 'failed_quote',
         'failed_quote_freshness', 'failed_spread', 'failed_ask_band', 'failed_ranking',
         'failed_symbol_used', 'failed_daily_loss_lock', 'failed_entry_quote',
         'failed_position_limit', 'failed_sector_limit', 'failed_risk_limit')
HARD_DATA_ERRORS = ('DataAccessDenied', 'RateLimited', 'DataUnavailable', 'InvalidRequest')


# =========================================================================== schema
def col(name, dtype, unit, description, source):
    return dict(column=name, datatype=dtype, unit=unit, description=description, source=source)


TIME_FEATURES = [
    col('minutes_since_open', 'float', 'minutes', 'Minuten seit Börsenöffnung (Kalender) bis zum Zeitpunkt.', 'research'),
    col('minutes_since_or_end', 'float', 'minutes', 'Minuten seit Ende der Opening Range (Öffnung + opening_minutes).', 'research'),
    col('hour', 'int', 'hour NY', 'Stunde (America/New_York).', 'research'),
    col('minute', 'int', 'minute', 'Minute der Stunde (America/New_York).', 'research'),
    col('session_bucket', 'string', None, 'Research-Label der Tageszeit, Grenzen siehe definitions.session_buckets. Keine Handelsregel.', 'research'),
]
CONTEXT = [
    col('qqq_return_since_open', 'float', 'decimal', 'QQQ: letzter Schluss vor dem Zeitpunkt / Eröffnungskurs − 1.', 'research: SIP-Kerzen vor dem Zeitpunkt'),
    col('qqq_5m_return', 'float', 'decimal', 'QQQ: letzter Schluss vor dem Zeitpunkt / letzter Schluss 5 Min. früher − 1.', 'research: SIP-Kerzen vor dem Zeitpunkt'),
    col('spy_return_since_open', 'float', 'decimal', 'Wie qqq_return_since_open für SPY (nur Research-Daten, nie an die Strategie übergeben).', 'research'),
    col('spy_5m_return', 'float', 'decimal', 'Wie qqq_5m_return für SPY.', 'research'),
    col('market_direction', 'string', None, 'UP/DOWN/FLAT nach Vorzeichen von qqq_return_since_open. Reines Research-Label.', 'research'),
]
FUTURE = ([col(f'future_return_{h}m', 'float', 'decimal',
               f'COUNTERFACTUAL: letzter SIP-Schluss bis {h} Min. nach der Entscheidung / Signalpreis − 1. '
               'Nach dem Lauf berechnet, nie Teil der Entscheidung. Null, wenn der Horizont nach Börsenschluss liegt.',
               'research: SIP-Kerzen nach dem Zeitpunkt') for h in HORIZONS] +
          [c for h in EXCURSION_HORIZONS for c in (
              col(f'future_max_return_{h}m', 'float', 'decimal', f'COUNTERFACTUAL: höchstes Kerzenhoch in {h} Min. nach der Entscheidung / Signalpreis − 1.', 'research'),
              col(f'future_max_drawdown_{h}m', 'float', 'decimal', f'COUNTERFACTUAL: tiefstes Kerzentief in {h} Min. nach der Entscheidung / Signalpreis − 1 (≤ 0 bei Rückgang).', 'research'))] +
          [col('future_max_return_to_close', 'float', 'decimal', 'COUNTERFACTUAL: höchstes Hoch bis Börsenschluss / Signalpreis − 1.', 'research'),
           col('future_max_drawdown_to_close', 'float', 'decimal', 'COUNTERFACTUAL: tiefstes Tief bis Börsenschluss / Signalpreis − 1.', 'research')])

TRADES_SCHEMA = [
    col('run_id', 'string', None, 'Backtest-Lauf.', 'run'),
    col('trade_id', 'string', None, '<run_id>|<Datum>|<Symbol>|<Nr.>', 'run'),
    col('date', 'date', None, 'Handelstag (NY).', 'engine'),
    col('symbol', 'string', None, 'Aktie.', 'engine'),
    col('sector', 'string', None, 'Sektor laut Universum-Snapshot.', 'engine'),
    col('signal_time', 'timestamp', 'UTC', 'Entscheidungszeitpunkt des Signals (Ende der Ausbruchsminute).', 'engine'),
    col('entry_time', 'timestamp', 'UTC', 'Virtueller Einstieg (simulierte Verarbeitungszeit).', 'engine'),
    col('exit_time', 'timestamp', 'UTC', 'Virtueller Ausstieg.', 'engine'),
    col('holding_minutes', 'float', 'minutes', 'exit_time − entry_time.', 'research'),
    col('entry_price', 'float', 'USD', 'Einstieg zum frischen SIP-Ask (Modell).', 'engine'),
    col('exit_price', 'float', 'USD', 'Ausstiegskurs-Proxy (Stop, Gap-Open oder frische SIP-Bid).', 'engine'),
    col('shares', 'int', 'shares', 'Stückzahl.', 'engine'),
    col('position_value', 'float', 'USD', 'shares × entry_price.', 'research'),
    col('equity_before_entry', 'float', 'USD', 'Virtuelles Equity direkt vor dem Einstieg (Portfolio-Markierung).', 'engine log'),
    col('risk_dollars', 'float', 'USD', 'Geplantes Anfangsrisiko = shares × (entry − initialer Stop).', 'engine'),
    col('risk_pct', 'float', 'decimal', 'risk_dollars / equity_before_entry.', 'research'),
    col('or_high', 'float', 'USD', 'Opening-Range-Hoch.', 'engine'),
    col('or_low', 'float', 'USD', 'Opening-Range-Tief.', 'engine'),
    col('or_range_pct', 'float', 'decimal', '(or_high − or_low) / or_low.', 'research'),
    col('breakout_price', 'float', 'USD', 'Schlusskurs der Ausbruchsminute.', 'engine'),
    col('breakout_distance_pct', 'float', 'decimal', 'breakout_price / or_high − 1 (Abstand über dem OR-Hoch).', 'research'),
    col('rvol_at_signal', 'float', 'ratio', 'RVOL: Volumen seit Öffnung bis Signal / Ø der gültigen Vergleichstage bis zur gleichen Uhrzeit.', 'engine'),
    col('bid_at_signal', 'float', 'USD', 'SIP-Bid der Signalprüfung.', 'engine'),
    col('ask_at_signal', 'float', 'USD', 'SIP-Ask der Signalprüfung.', 'engine'),
    col('mid_at_signal', 'float', 'USD', '(bid + ask) / 2.', 'research'),
    col('spread_abs', 'float', 'USD', 'ask − bid der Signalprüfung.', 'research'),
    col('spread_bps', 'float', 'bps', '(ask − bid) / mid × 10 000.', 'research'),
    col('quote_age_ms', 'float', 'ms', 'Alter der Signal-Quote bei der Entscheidung.', 'engine'),
    col('entry_slippage_bps', 'float', 'bps', '(entry_price − ask_at_signal) / ask_at_signal × 10 000: Verschiebung Signal-Quote → Einstiegs-Quote.', 'research'),
    col('initial_trailing_stop', 'float', 'USD', 'Stop beim Einstieg.', 'engine'),
    col('max_trailing_stop', 'float', 'USD', 'Höchster (letzter) Trailing-Stop des Trades.', 'engine'),
    col('mfe_dollars', 'float', 'USD', 'MFE: shares × (höchstes beobachtetes Kerzenhoch − Einstieg), Einstiegsminute ausgenommen.', 'engine'),
    col('mfe_pct', 'float', 'decimal', 'mfe_dollars / position_value.', 'research'),
    col('mae_dollars', 'float', 'USD', 'MAE: shares × (min(tiefstes beobachtetes Tief, Ausstieg) − Einstieg), ≤ 0.', 'engine'),
    col('mae_pct', 'float', 'decimal', 'mae_dollars / position_value.', 'research'),
    col('gross_pnl', 'float', 'USD', 'shares × (exit − entry) mit Ask/Bid-Proxy.', 'engine'),
    col('commission', 'float', 'USD', 'Nicht modelliert (Alpaca provisionsfrei) → 0.0 als Modellannahme.', 'model assumption'),
    col('slippage_cost', 'float', 'USD', 'Kosten des 10-bp-Stressszenarios (5 bp je Seite) = gross_pnl − net_pnl_10bp_stress.', 'engine'),
    col('net_pnl', 'float', 'USD', 'gross_pnl − commission.', 'research'),
    col('net_pnl_10bp_stress', 'float', 'USD', 'Ergebnis mit 10-bp-Kostenstress.', 'engine'),
    col('return_pct', 'float', 'decimal', 'net_pnl / position_value.', 'research'),
    col('exit_reason', 'string', None, 'Ausstiegsgrund der Engine (TRAILING_STOP_PROXY, PLANNED_CLOSE_BID_PROXY …).', 'engine'),
    col('exit_type', 'string', None, 'TRAILING_STOP / END_OF_DAY / DATA_GAP / OTHER.', 'research'),
    col('outcome_unreliable', 'bool', None, 'Stop-Beobachtung lückenhaft (verspätete/fehlende Kerzen).', 'engine'),
    col('intrabar_ambiguous', 'bool', None, 'Stop-Kerze hatte auch ein neues Hoch: Reihenfolge Hoch/Tief in der Minute unbekannt.', 'research'),
    col('rank', 'int', None, 'Rang in der Minute (1 = bester Score).', 'engine'),
    col('score', 'float', 'points', 'Ranking-Score 0–100.', 'engine'),
] + TIME_FEATURES + CONTEXT + [
    col('market_data_feed', 'string', None, 'Historischer Feed des Laufs (sip).', 'run'),
    col('data_quality_level', 'string', None, 'BAR_ONLY / BAR_PLUS_QUOTES / FULL_MARKET_DATA.', 'run'),
]

SIGNALS_SCHEMA = [
    col('run_id', 'string', None, 'Backtest-Lauf.', 'run'),
    col('signal_id', 'string', None, '<Datum>|<Symbol>|<Entscheidungszeit UTC>.', 'research'),
    col('date', 'date', None, 'Handelstag (NY).', 'engine'),
    col('timestamp', 'timestamp', 'UTC', 'Entscheidungszeitpunkt (Ende der Ausbruchsminute).', 'engine'),
    col('symbol', 'string', None, 'Aktie.', 'engine'),
    col('sector', 'string', None, 'Sektor laut Universum-Snapshot.', 'engine'),
    col('record_source', 'string', None, 'SIGNAL_FILE (von der Engine geprüft) oder PRICE_REJECTION (Kurs-/Datenregel vor der Prüfung).', 'research'),
    col('price', 'float', 'USD', 'Schlusskurs der Ausbruchsminute.', 'engine'),
    col('or_high', 'float', 'USD', 'Opening-Range-Hoch.', 'engine'),
    col('or_low', 'float', 'USD', 'Opening-Range-Tief.', 'engine'),
    col('or_range_pct', 'float', 'decimal', '(or_high − or_low) / or_low.', 'research'),
    col('breakout_distance_pct', 'float', 'decimal', 'price / or_high − 1.', 'research'),
    col('volume', 'float', 'shares', 'Volumen seit Öffnung bis zur Entscheidung (nur wenn RVOL geprüft).', 'engine'),
    col('rvol', 'float', 'ratio', 'RVOL (siehe definitions.rvol).', 'engine'),
    col('rvol_reference_days', 'int', 'days', 'Gültige Vergleichstage (≥ 18 von 20 nötig).', 'engine'),
    col('relative_strength', 'float', 'decimal', 'Rendite seit Öffnung Aktie − QQQ.', 'engine'),
    col('bid', 'float', 'USD', 'SIP-Bid der Signalprüfung.', 'engine'),
    col('ask', 'float', 'USD', 'SIP-Ask der Signalprüfung.', 'engine'),
    col('mid', 'float', 'USD', '(bid + ask) / 2.', 'research'),
    col('spread_bps', 'float', 'bps', '(ask − bid) / mid × 10 000.', 'research'),
    col('quote_age_ms', 'float', 'ms', 'Alter der Signal-Quote bei der Entscheidung.', 'engine'),
    col('score', 'float', 'points', 'Ranking-Score (nur wenn gerankt).', 'engine'),
    col('rank', 'int', None, 'Rang in der Minute (nur wenn gerankt).', 'engine'),
    col('open_positions', 'int', None, 'Offene virtuelle Positionen zum Zeitpunkt der Prüfung/des Einstiegsversuchs.', 'engine log'),
    col('same_sector_positions', 'int', None, 'Davon im selben Sektor.', 'engine log'),
    col('equity', 'float', 'USD', 'Virtuelles Equity zum Zeitpunkt der Prüfung/des Einstiegsversuchs.', 'engine log'),
    col('cash', 'float', 'USD', 'Virtuelles Cash.', 'engine log'),
    col('candidate_position_size', 'int', 'shares', 'Tatsächliche Stückzahl (Einstieg) bzw. nach size_position berechnete Stückzahl, wenn eine Einstiegs-Ask vorlag; sonst null.', 'engine / research (size_position)'),
    col('candidate_risk_dollars', 'float', 'USD', 'Stückzahl × Ask × Trailing-Abstand.', 'research'),
    col('candidate_risk_pct', 'float', 'decimal', 'candidate_risk_dollars / equity.', 'research'),
    col('signal_status', 'string', None, 'Letzter Engine-Status des Signals (bestehende Codes).', 'engine'),
    col('shadow_status', 'string', None, 'Ergebnis des virtuellen Einstiegs (VIRTUAL_ENTRY oder Ablehnungsgrund).', 'engine'),
    col('entry_quote_result', 'string', None, 'Ergebnis der frischen Einstiegs-Quote.', 'engine'),
    col('accepted', 'bool', None, 'true = virtueller Einstieg erfolgt.', 'research'),
    col('reject_stage', 'string', None, 'PRICE / DATA / TIMING / RVOL / QUOTE / RANKING / ENTRY (null wenn angenommen).', 'research'),
    col('reject_reason', 'string', None, 'Bestehender Engine-Code des Ablehnungsgrunds (bei mehreren Portfolio-Gründen mit ; getrennt).', 'engine'),
    col('data_reject_code', 'string', None, 'DATA_INCOMPLETE / OR_INCOMPLETE / RVOL_REFERENCE_INCOMPLETE / QUOTE_DATA_MISSING / TRADE_DATA_MISSING, wenn Daten die Ursache waren.', 'research'),
] + [col(f, 'bool', None, 'true = dieser Filter hat abgelehnt; false = geprüft und bestanden; null = nicht geprüft (die Engine stoppt beim ersten Fehlschlag).', 'research') for f in FLAGS] + [
    col('trade_id', 'string', None, 'Zugehöriger Trade, falls angenommen.', 'research'),
] + TIME_FEATURES + CONTEXT + FUTURE + [
    col('market_data_feed', 'string', None, 'Historischer Feed des Laufs.', 'run'),
    col('data_quality_level', 'string', None, 'Qualitätsstufe des Laufs.', 'run'),
]

DAILY_SCHEMA = [
    col('run_id', 'string', None, 'Backtest-Lauf.', 'run'),
    col('date', 'date', None, 'Handelstag laut Börsenkalender (NY).', 'calendar'),
    col('day_status', 'string', None, 'SIMULATED / REUSED / INCOMPLETE_DATA / SIP_UNAVAILABLE / FAILED / UNRESOLVED_POSITION / NOT_RUN.', 'run'),
    col('simulated', 'bool', None, 'true = Tag wurde (neu oder wiederverwendet) simuliert.', 'run'),
    col('session_open', 'timestamp', 'UTC', 'Öffnung laut Kalender.', 'calendar'),
    col('session_close', 'timestamp', 'UTC', 'Schluss laut Kalender (verkürzte Tage berücksichtigt).', 'calendar'),
    col('start_equity', 'float', 'USD', 'Virtuelles Equity zu Tagesbeginn (fortgeschrieben).', 'run'),
    col('end_equity', 'float', 'USD', 'start_equity + realisierter PnL (null, wenn nicht simuliert/ungelöst).', 'run'),
    col('gross_pnl', 'float', 'USD', 'Summe gross_pnl der Trades.', 'engine'),
    col('net_pnl', 'float', 'USD', 'Summe net_pnl der Trades.', 'research'),
    col('net_pnl_10bp_stress', 'float', 'USD', 'Summe mit Kostenstress.', 'engine'),
    col('daily_return_pct', 'float', 'decimal', 'net_pnl / start_equity.', 'research'),
    col('trades', 'int', None, 'Anzahl Trades.', 'engine'),
    col('winners', 'int', None, 'Trades mit net_pnl > 0.', 'research'),
    col('losers', 'int', None, 'Trades mit net_pnl < 0.', 'research'),
    col('signals_total', 'int', None, 'Zeilen in signals für den Tag.', 'research'),
    col('signals_accepted', 'int', None, 'Davon angenommen.', 'research'),
    col('signals_rejected', 'int', None, 'Davon abgelehnt.', 'research'),
    col('gross_profit', 'float', 'USD', 'Summe der Gewinne.', 'research'),
    col('gross_loss', 'float', 'USD', 'Summe der Verluste (≤ 0).', 'research'),
    col('max_intraday_drawdown', 'float', 'USD', 'Größter Rückgang des Equity innerhalb des Tages (≤ 0), Minutenmarkierung.', 'research'),
    col('max_intraday_drawdown_pct', 'float', 'decimal', 'Derselbe Rückgang relativ zum vorherigen Hoch.', 'research'),
    col('capital_utilization_avg', 'float', 'decimal', 'Ø Marktwert offener Positionen / Equity über die Minuten.', 'research'),
    col('capital_utilization_max', 'float', 'decimal', 'Maximum davon.', 'research'),
    col('max_simultaneous_positions', 'int', None, 'Höchste Zahl gleichzeitig offener Positionen.', 'research'),
    col('daily_loss_lock_triggered', 'bool', None, 'Tagesverlustsperre ausgelöst.', 'engine'),
    col('trailing_stop_exits', 'int', None, 'Ausstiege über Stop (inkl. Gap-Open, Bid-Stop).', 'research'),
    col('eod_exits', 'int', None, 'Glattstellungen zum geplanten Tagesabschluss.', 'research'),
    col('data_gap_exits', 'int', None, 'Ausstiege wegen Datenlücke.', 'research'),
    col('expected_symbols', 'int', None, 'Aktien nach Universums- und Liquiditätsfilter des Tages.', 'engine'),
    col('symbols_with_valid_data', 'int', None, 'Davon mit Kerzen und gültiger Opening Range.', 'research'),
    col('symbols_excluded', 'int', None, 'expected_symbols − symbols_with_valid_data (für den Tag ausgeschlossen).', 'research'),
    col('data_coverage_pct', 'float', 'decimal', 'symbols_with_valid_data / expected_symbols (kein fester Grenzwert).', 'research'),
    col('missing_bar_count', 'int', 'minutes', 'Regulären Minuten ohne SIP-Kerze, summiert über expected_symbols (ohne Handel = keine Kerze).', 'cache'),
    col('missing_quote_count', 'int', None, 'Benötigte Quotes, die fehlten oder nicht abrufbar waren (Signal-, Einstiegs-, Ausstiegsprüfung).', 'engine'),
    col('missing_trade_count', 'int', None, 'RVOL-Vergleichstage, deren Lücken nicht über SIP-Trades geklärt werden konnten.', 'engine'),
    col('systemic_data_error', 'bool', None, 'Tag wegen systemischem Datenproblem nicht simuliert.', 'run'),
    col('historical_only_signals', 'int', None, 'Signale mit HISTORICAL_ONLY_STARTUP_OR_RESTART (verpasst, nie gekauft).', 'engine'),
    col('missed_decision_minutes', 'int', 'minutes', 'Übersprungene Entscheidungsminuten (simulierte Rechenzeit).', 'engine'),
    col('unreliable_trades', 'int', None, 'Trades mit outcome_unreliable.', 'engine'),
    col('data_quality_issues', 'int', None, 'symbols_excluded + missing_quote_count + missing_trade_count + unreliable_trades + missed_decision_minutes + data_error_minutes.', 'research'),
    col('qqq_daily_return', 'float', 'decimal', 'QQQ Schluss / Eröffnung − 1 (reguläre Sitzung).', 'research'),
    col('spy_daily_return', 'float', 'decimal', 'SPY Schluss / Eröffnung − 1.', 'research'),
    col('run_folder', 'string', None, 'Tages-Laufordner der Engine.', 'run'),
    col('detail', 'string', None, 'Fehler-/Statusdetail.', 'run'),
]

EQUITY_SCHEMA = [
    col('run_id', 'string', None, 'Backtest-Lauf.', 'run'),
    col('timestamp', 'timestamp', 'UTC', 'Markierungszeitpunkt (Tagesbeginn, jede Entscheidungsminute, Tagesabschluss).', 'engine log'),
    col('date', 'date', None, 'Handelstag (NY).', 'engine log'),
    col('equity', 'float', 'USD', 'Cash + Positionen zum letzten beobachteten Kerzenschluss.', 'engine log'),
    col('cash', 'float', 'USD', 'Virtuelles Cash.', 'engine log'),
    col('market_value', 'float', 'USD', 'Marktwert offener Positionen.', 'engine log'),
    col('open_positions', 'int', None, 'Offene Positionen.', 'engine log'),
    col('drawdown', 'float', 'USD', 'equity − bisheriges Hoch (≤ 0), über den ganzen Lauf.', 'research'),
    col('drawdown_pct', 'float', 'decimal', 'equity / bisheriges Hoch − 1 (≤ 0).', 'research'),
]

EVENTS_SCHEMA = [
    col('run_id', 'string', None, 'Backtest-Lauf.', 'run'),
    col('date', 'date', None, 'Handelstag.', 'engine'),
    col('timestamp', 'timestamp', 'UTC', 'Marktzeit des Ereignisses.', 'engine'),
    col('symbol', 'string', None, 'Aktie.', 'engine'),
    col('event_type', 'string', None, 'ORB_SIGNAL / SIGNAL_ACCEPT / SIGNAL_REJECT / ENTRY / TRAIL_UPDATE / EXIT / EOD_EXIT / RISK_LOCK / DATA_GAP.', 'research'),
    col('price', 'float', 'USD', 'Relevanter Kurs (Einstieg, Ausstieg, Schluss).', 'engine'),
    col('bid', 'float', 'USD', 'Bid, falls vorhanden.', 'engine'),
    col('ask', 'float', 'USD', 'Ask, falls vorhanden.', 'engine'),
    col('position_qty', 'int', 'shares', 'Stückzahl.', 'engine'),
    col('trailing_stop', 'float', 'USD', 'Stop nach dem Ereignis.', 'engine'),
    col('rvol', 'float', 'ratio', 'RVOL des Signals.', 'engine'),
    col('event_reason', 'string', None, 'Engine-Grund/Code.', 'engine'),
]

GAPS_SCHEMA = [
    col('run_id', 'string', None, 'Backtest-Lauf.', 'run'),
    col('date', 'date', None, 'Handelstag.', 'cache'),
    col('symbol', 'string', None, 'Aktie (nur Tages-Universum).', 'cache'),
    col('missing_minute_utc', 'timestamp', 'UTC', 'Minute ohne SIP-Kerze.', 'cache'),
    col('missing_minute_ny', 'string', 'HH:MM NY', 'Dieselbe Minute in New Yorker Zeit.', 'cache'),
]

SCHEMAS = dict(trades=TRADES_SCHEMA, signals=SIGNALS_SCHEMA, daily=DAILY_SCHEMA,
               equity_curve=EQUITY_SCHEMA, events=EVENTS_SCHEMA, data_gaps=GAPS_SCHEMA)

DEFINITIONS = {
    'rvol': 'Relatives Volumen = Volumen der Aktie von Öffnung bis Entscheidung / Durchschnitt desselben '
            'Zeitfensters an den gültigen der letzten 20 Handelstage (mindestens 18 gültig). Fehlende '
            'Minuten zählen 0; fehlende Vergleichsminuten werden nur über SIP-Odd-Lot-Trades geklärt.',
    'spread_bps': '(Ask − Bid) / ((Ask + Bid) / 2) × 10 000 aus der SIP-Quote der Signalprüfung.',
    'mfe': 'Maximum Favorable Excursion: Stück × (höchstes Kerzenhoch während der Haltedauer − Einstieg). '
           'Die angeschnittene Einstiegsminute zählt nicht.',
    'mae': 'Maximum Adverse Excursion: Stück × (min(tiefstes Kerzentief, Ausstieg) − Einstieg), ≤ 0.',
    'risk_pct': 'Geplantes Anfangsrisiko (Stück × Abstand Einstieg – Anfangsstop) / Equity vor dem Einstieg.',
    'breakout_distance': 'Signal-Schlusskurs / Opening-Range-Hoch − 1.',
    'future_return': 'COUNTERFACTUAL nach dem Lauf: letzter SIP-Schluss innerhalb von N Minuten nach der '
                     'Entscheidung / Signalpreis − 1; fehlende Minuten tragen den letzten Schluss fort.',
    'future_max_drawdown': 'COUNTERFACTUAL: tiefstes Tief innerhalb des Horizonts / Signalpreis − 1.',
    'percent_values': 'Alle Prozentwerte sind Dezimalzahlen: 0.042 = 4,2 %.',
    'timestamps': 'ISO-8601 mit Zeitzone; intern UTC. Handelszeiten America/New_York.',
    'nulls': 'Nicht verfügbare Werte sind null/NaN; es werden keine Werte erfunden.',
    'session_buckets': {name: f'{a}–{b} NY (Beginn inklusive)' for name, a, b in SESSION_BUCKETS},
    'quality_levels': {
        'BAR_ONLY': 'Nur Kerzen: Spread, Quote-Frische und Bid/Ask-Ausführung nicht historisch reproduzierbar.',
        'BAR_PLUS_QUOTES': 'Kerzen + SIP-Quotes an allen Entscheidungspunkten (Signal, Einstieg, Ausstieg).',
        'FULL_MARKET_DATA': 'Zusätzlich alle SIP-Trades der Signal-Aktien und -Tage (Microstructure-Research).'},
    'reject_stages': 'PRICE → DATA → TIMING → RVOL → QUOTE → RANKING → ENTRY (Reihenfolge der Engine).',
    'counterfactual_protection': 'future_* entsteht ausschließlich in research_export_v16 nach dem Lauf; '
                                 'kein Strategie-, Risiko- oder Ausführungsmodul importiert dieses Modul.',
}


# =========================================================================== helpers
def read_json(path, default=None):
    path = Path(path)
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else default


def ts(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return pd.NaT
    stamp = pd.Timestamp(value)
    return stamp.tz_localize('UTC') if stamp.tzinfo is None else stamp.tz_convert('UTC')


def num(value):
    try:
        value = float(value)
    except (TypeError, ValueError):
        return np.nan
    return value if math.isfinite(value) else np.nan


def ratio(a, b):
    a, b = num(a), num(b)
    return a / b if b and not math.isnan(a) and not math.isnan(b) else np.nan


def session_bucket(stamp):
    if pd.isna(stamp):
        return None
    hm = stamp.tz_convert(NY).strftime('%H:%M')
    for name, a, b in SESSION_BUCKETS:
        if a <= hm < b:
            return name
    return 'OUTSIDE_SESSION'


def time_features(stamp, session, opening_minutes):
    if pd.isna(stamp) or session is None:
        return dict(minutes_since_open=np.nan, minutes_since_or_end=np.nan, hour=None, minute=None,
                    session_bucket=None)
    local = stamp.tz_convert(NY)
    since = (stamp - session['open']).total_seconds() / 60
    return dict(minutes_since_open=since, minutes_since_or_end=since - opening_minutes,
                hour=int(local.hour), minute=int(local.minute), session_bucket=session_bucket(stamp))


class Bars:
    """Read-only access to cached regular-session minute bars (research only)."""

    def __init__(self, store):
        self.store, self.cache = store, {}

    def day(self, symbol, day):
        key = (symbol, day)
        if key not in self.cache:
            frame = pd.DataFrame(columns=['timestamp', 'open', 'high', 'low', 'close'])
            try:
                if self.store.bars_covered(symbol, day):
                    session = self.store.session(day)
                    _, part = self.store._partition('bars_1min', symbol, day[:7])
                    frame = part[(part.timestamp >= session['open']) & (part.timestamp < session['close'])]
                    frame = frame.sort_values('timestamp')
            except Exception:
                pass
            self.cache[key] = frame.reset_index(drop=True)
        return self.cache[key]


def context_features(bars, day, stamp):
    out = {}
    for symbol, prefix in (('QQQ', 'qqq'), ('SPY', 'spy')):
        frame = bars.day(symbol, day)
        past = frame[frame.timestamp + pd.Timedelta(minutes=1) <= stamp] if len(frame) and not pd.isna(stamp) else frame.iloc[0:0]
        if len(past):
            last = float(past.close.iloc[-1])
            out[f'{prefix}_return_since_open'] = last / float(frame.open.iloc[0]) - 1
            earlier = past[past.timestamp + pd.Timedelta(minutes=1) <= stamp - pd.Timedelta(minutes=5)]
            out[f'{prefix}_5m_return'] = last / float(earlier.close.iloc[-1]) - 1 if len(earlier) else np.nan
        else:
            out[f'{prefix}_return_since_open'] = np.nan
            out[f'{prefix}_5m_return'] = np.nan
    value = out['qqq_return_since_open']
    out['market_direction'] = (None if pd.isna(value) else 'UP' if value > 0 else 'DOWN' if value < 0 else 'FLAT')
    return out


def counterfactuals(bars, symbol, day, stamp, price, session):
    """COUNTERFACTUAL outcome after the decision (research only; never fed back)."""
    out = {c['column']: np.nan for c in FUTURE}
    price = num(price)
    if pd.isna(stamp) or math.isnan(price) or price <= 0 or session is None:
        return out
    frame = bars.day(symbol, day)
    if frame.empty:
        return out
    close = session['close']
    before = frame[frame.timestamp < stamp]
    for h in HORIZONS:
        horizon = stamp + pd.Timedelta(minutes=h)
        if horizon > close:
            continue
        seen = frame[frame.timestamp < horizon]
        if len(seen):
            out[f'future_return_{h}m'] = float(seen.close.iloc[-1]) / price - 1
        elif len(before):
            out[f'future_return_{h}m'] = float(before.close.iloc[-1]) / price - 1
    for h in EXCURSION_HORIZONS:
        horizon = min(stamp + pd.Timedelta(minutes=h), close)
        window = frame[(frame.timestamp >= stamp) & (frame.timestamp < horizon)]
        if len(window) and stamp + pd.Timedelta(minutes=h) <= close:
            out[f'future_max_return_{h}m'] = float(window.high.max()) / price - 1
            out[f'future_max_drawdown_{h}m'] = float(window.low.min()) / price - 1
    window = frame[frame.timestamp >= stamp]
    if len(window):
        out['future_max_return_to_close'] = float(window.high.max()) / price - 1
        out['future_max_drawdown_to_close'] = float(window.low.min()) / price - 1
    return out


# =========================================================================== classification
QUOTE_STAGE = {
    'QUOTE_MISSING': ('failed_quote', 'QUOTE_DATA_MISSING'),
    'QUOTE_DATA_ERROR': ('failed_quote', 'QUOTE_DATA_MISSING'),
    'QUOTE_INVALID': ('failed_quote', None),
    'QUOTE_FROM_FUTURE': ('failed_quote', None),
    'QUOTE_STALE': ('failed_quote_freshness', None),
    'ASK_BELOW_MINIMUM_PRICE': ('failed_price_band', None),
    'SPREAD_TOO_WIDE': ('failed_spread', None),
    'ASK_OUTSIDE_ENTRY_BAND': ('failed_ask_band', None),
}
QUOTE_ORDER = ['failed_quote', 'failed_quote_freshness', 'failed_price_band', 'failed_spread', 'failed_ask_band']
GATE_REASONS = {'Tagesverlustsperre': 'failed_daily_loss_lock',
                'Positionsplätze belegt/reserviert': 'failed_position_limit',
                'Sektorlimit erreicht': 'failed_sector_limit',
                'Sektorzuordnung fehlt': 'failed_data'}


def classify(item):
    """Stage, reason and filter flags of one signal record (existing engine codes)."""
    flags = {f: None for f in FLAGS}

    def passed(*names):
        for name in names:
            if flags[name] is None:
                flags[name] = False

    status = item.get('status')
    shadow = item.get('shadow_status')
    previous = item.get('previous_status')
    result = dict(reject_stage=None, reject_reason=None, data_reject_code=None, accepted=False)
    if item.get('_source') == 'PRICE_REJECTION':
        if status in ('TOO_EXTENDED', 'BELOW_MINIMUM_PRICE'):
            flags['failed_price_band'] = True
            result.update(reject_stage='PRICE', reject_reason=status)
        else:
            passed('failed_price_band')
            flags['failed_data'] = True
            result.update(reject_stage='DATA', reject_reason=status, data_reject_code=status)
        return dict(result, **flags)
    passed('failed_price_band')
    if status in ('HISTORICAL_ONLY_STARTUP_OR_RESTART', 'EXPIRED_LIVE_DECISION'):
        flags['failed_timing'] = True
        return dict(result, reject_stage='TIMING', reject_reason=status, **flags)
    passed('failed_timing')
    if status == 'BENCHMARK_OR_STOCK_PREFIX_INVALID':
        flags['failed_data'] = True
        return dict(result, reject_stage='DATA', reject_reason=status, data_reject_code='DATA_INCOMPLETE', **flags)
    passed('failed_data')
    if status in ('RVOL_DATA_ERROR', 'NOT_CHECKABLE'):
        return dict(result, reject_stage='RVOL', reject_reason=status,
                    data_reject_code='RVOL_REFERENCE_INCOMPLETE', **flags)
    if status == 'VOLUME_REJECT':
        flags['failed_rvol'] = True
        return dict(result, reject_stage='RVOL', reject_reason=status, **flags)
    passed('failed_rvol')
    if status == 'EXPIRED_DURING_RVOL_CHECK':
        flags['failed_timing'] = True
        return dict(result, reject_stage='TIMING', reject_reason=status, **flags)
    quote_result = item.get('quote_result')
    if status in QUOTE_STAGE or quote_result in QUOTE_STAGE:
        code = status if status in QUOTE_STAGE else quote_result
        flag, data_code = QUOTE_STAGE[code]
        for name in QUOTE_ORDER:
            if name == flag:
                break
            passed(name)
        flags[flag] = True
        if data_code is None and item.get('error_type') in HARD_DATA_ERRORS:
            data_code = 'QUOTE_DATA_MISSING'
        return dict(result, reject_stage='QUOTE', reject_reason=code, data_reject_code=data_code, **flags)
    passed(*QUOTE_ORDER)
    if status in ('EXPIRED_DURING_QUOTE_CHECK', 'EXPIRED_BEFORE_RANKING'):
        flags['failed_timing'] = True
        return dict(result, reject_stage='TIMING', reject_reason=status, **flags)
    if status == 'OUTSIDE_TOP_FIVE_NO_ORDER':
        flags['failed_ranking'] = True
        return dict(result, reject_stage='RANKING', reject_reason=status, **flags)
    if status != 'WATCHLIST_NO_ORDER':
        return dict(result, reject_stage='OTHER', reject_reason=status or previous, **flags)
    passed('failed_ranking')
    if shadow == 'VIRTUAL_ENTRY':
        passed(*FLAGS)
        return dict(result, accepted=True, **flags)
    # ENTRY stage in the order of ShadowPortfolio.enter
    reason = str(shadow or 'UNKNOWN')
    data_code = None
    if reason in ('ENTRY_CUTOFF', 'SIGNAL_EXPIRED'):
        flags['failed_timing'] = True
    elif reason in ('SYMBOL_ALREADY_USED_TODAY', 'SYMBOL_ALREADY_HELD', 'REENTRY_LIMIT_REACHED',
                    'REENTRY_ONLY_AFTER_STOP', 'REENTRY_COOLDOWN'):
        flags['failed_symbol_used'] = True
    elif reason == 'DAILY_LOSS_LOCK':
        passed('failed_symbol_used')
        flags['failed_daily_loss_lock'] = True
    elif reason == 'HELD_POSITION_STOP_OBSERVATION_UNRESOLVED':
        passed('failed_symbol_used', 'failed_daily_loss_lock')
        flags['failed_data'] = True
        data_code = 'DATA_INCOMPLETE'
    elif reason.startswith('EXECUTION_QUOTE_') or reason == 'INVALID_ENTRY_PRICE':
        passed('failed_symbol_used', 'failed_daily_loss_lock')
        flags['failed_entry_quote'] = True
        if reason.endswith(('MISSING', 'ERROR')):
            data_code = 'QUOTE_DATA_MISSING'
    elif reason == 'PORTFOLIO_MARK_MISSING':
        passed('failed_symbol_used', 'failed_daily_loss_lock', 'failed_entry_quote')
        flags['failed_data'] = True
        data_code = 'DATA_INCOMPLETE'
    elif reason == 'NO_CAPITAL_OR_RISK_BUDGET':
        passed('failed_symbol_used', 'failed_daily_loss_lock', 'failed_entry_quote',
               'failed_position_limit', 'failed_sector_limit')
        flags['failed_risk_limit'] = True
    else:
        parts = [p for p in reason.split(';') if p]
        if parts and all(p in GATE_REASONS for p in parts):
            passed('failed_symbol_used', 'failed_entry_quote', 'failed_daily_loss_lock',
                   'failed_position_limit', 'failed_sector_limit')
            for part in parts:
                flags[GATE_REASONS[part]] = True
            if 'Sektorzuordnung fehlt' in parts:
                data_code = 'DATA_INCOMPLETE'
    return dict(result, reject_stage='ENTRY', reject_reason=reason, data_reject_code=data_code, **flags)


# =========================================================================== tables
_SIGNAL_CACHE = {}


def raw_signal_items(folder):
    """Signal JSON of a finished day: signals/*.json and/or signals_bundle.json (V1.6.1).
    Cached per folder (finished day folders do not change)."""
    folder = Path(folder)
    key = str(folder)
    if key not in _SIGNAL_CACHE:
        items = {path.name: read_json(path) for path in sorted((folder / 'signals').glob('*.json'))}
        items.update(read_json(folder / 'signals_bundle.json', {}) or {})
        if len(_SIGNAL_CACHE) > 8:
            _SIGNAL_CACHE.clear()
        _SIGNAL_CACHE[key] = [items[name] for name in sorted(items)]
    return _SIGNAL_CACHE[key]


def reference_check_frames(folder):
    folder = Path(folder)
    frames = []
    for path in (folder / 'reference_checks').glob('*.csv'):
        try:
            frames.append(pd.read_csv(path))
        except Exception:
            pass
    bundle = folder / 'reference_checks_bundle.csv'
    if bundle.exists():
        frames.append(pd.read_csv(bundle))
    return frames


def signal_records(folder):
    items = []
    for item in raw_signal_items(folder):
        if isinstance(item, dict) and 'symbol' in item and item.get('decision_time_utc'):
            items.append(dict(item, _source='SIGNAL_FILE'))
    seen = {(i['symbol'], i['decision_time_utc']) for i in items}
    for item in read_json(Path(folder) / 'candidate_rejections.json', []) or []:
        if (item['symbol'], item['decision_time_utc']) not in seen:
            items.append(dict(item, _source='PRICE_REJECTION'))
    return items


def build_tables(run_id, day_records, store, c, s, quality_level, feed='sip'):
    bars = Bars(store)
    trades, signals, daily, equity, events, gaps = [], [], [], [], [], []
    for record in day_records:
        day = record['date']
        session = store.session(day)
        folder = Path(record['folder']) if record.get('folder') else None
        simulated = record.get('status') in ('SIMULATED', 'REUSED', 'UNRESOLVED_POSITION') and folder is not None
        day_row = {c_['column']: None for c_ in DAILY_SCHEMA}
        day_row.update(run_id=run_id, date=day, day_status=record.get('status') or 'NOT_RUN',
                       simulated=bool(simulated),
                       session_open=session['open'] if session else pd.NaT,
                       session_close=session['close'] if session else pd.NaT,
                       start_equity=num(record.get('start_equity')), end_equity=num(record.get('end_equity')),
                       systemic_data_error=record.get('status') in ('INCOMPLETE_DATA', 'SIP_UNAVAILABLE'),
                       run_folder=str(folder) if folder else None, detail=record.get('detail'))
        for symbol, prefix in (('QQQ', 'qqq'), ('SPY', 'spy')):
            frame = bars.day(symbol, day) if session else pd.DataFrame()
            day_row[f'{prefix}_daily_return'] = (float(frame.close.iloc[-1]) / float(frame.open.iloc[0]) - 1
                                                 if len(frame) else np.nan)
        if not simulated:
            daily.append(day_row)
            continue
        state = read_json(folder / 'shadow_state.json', {})
        coverage = read_json(folder / 'coverage.json', {})
        closed = read_json(folder / 'shadow_trades.json', [])
        engine_events = read_json(folder / 'shadow_events.json', [])
        marks = read_json(folder / 'equity_curve.json', [])
        entry_events = {(e['symbol'], e['at_utc']): e for e in engine_events if e['kind'] == 'VIRTUAL_ENTRY'}
        items = signal_records(folder)
        by_signal = {(i['symbol'], i['decision_time_utc']): i for i in items}
        # ---- trades
        day_trades = []
        for number, t in enumerate(closed):
            trade_id = f'{run_id}|{day}|{t["symbol"]}|{number}'
            sig = by_signal.get((t['symbol'], t.get('signal_at_utc')), {})
            entry_event = entry_events.get((t['symbol'], t['entry_at_utc']), {})
            qty, entry, exit_ = int(t['qty']), num(t['entry']), num(t['exit_price_proxy'])
            value = qty * entry
            bid, ask = num(sig.get('bid')), num(sig.get('ask'))
            mid = (bid + ask) / 2 if not (math.isnan(bid) or math.isnan(ask)) else np.nan
            equity_before = num(sig.get('entry_ctx_equity_proxy'))
            risk = num(entry_event.get('initial_planned_risk', qty * (entry - num(t.get('stop', np.nan)))))
            gross, stress = num(t['pnl_usd']), num(t['pnl_10bp_stress_usd'])
            signal_time, entry_time = ts(t.get('signal_at_utc')), ts(t['entry_at_utc'])
            ambiguous = False
            if t.get('exit_reason') == 'TRAILING_STOP_PROXY' and t.get('exit_trigger_bar_minute_utc'):
                frame = bars.day(t['symbol'], day)
                trigger = frame[frame.timestamp == ts(t['exit_trigger_bar_minute_utc'])]
                ambiguous = bool(len(trigger) and float(trigger.high.iloc[0]) > num(t.get('high_water')))
            row = dict(run_id=run_id, trade_id=trade_id, date=day, symbol=t['symbol'], sector=t.get('sector'),
                       signal_time=signal_time, entry_time=entry_time, exit_time=ts(t['exit_at_utc']),
                       holding_minutes=(ts(t['exit_at_utc']) - entry_time).total_seconds() / 60,
                       entry_price=entry, exit_price=exit_, shares=qty, position_value=value,
                       equity_before_entry=equity_before, risk_dollars=risk,
                       risk_pct=ratio(risk, equity_before),
                       or_high=num(sig.get('or_high')), or_low=num(sig.get('or_low')),
                       or_range_pct=ratio(num(sig.get('or_high')) - num(sig.get('or_low')), sig.get('or_low')),
                       breakout_price=num(sig.get('close')),
                       breakout_distance_pct=ratio(sig.get('close'), sig.get('or_high')) - 1,
                       rvol_at_signal=num(t.get('rvol')), bid_at_signal=bid, ask_at_signal=ask, mid_at_signal=mid,
                       spread_abs=ask - bid, spread_bps=ratio(ask - bid, mid) * 1e4,
                       quote_age_ms=num(sig.get('quote_age_seconds')) * 1000,
                       entry_slippage_bps=ratio(entry - ask, ask) * 1e4,
                       initial_trailing_stop=num(entry_event.get('initial_stop')),
                       max_trailing_stop=num(t.get('stop')),
                       mfe_dollars=num(t.get('mfe_usd')), mfe_pct=ratio(t.get('mfe_usd'), value),
                       mae_dollars=num(t.get('mae_usd')), mae_pct=ratio(t.get('mae_usd'), value),
                       gross_pnl=gross, commission=0.0, slippage_cost=gross - stress, net_pnl=gross,
                       net_pnl_10bp_stress=stress, return_pct=ratio(gross, value),
                       exit_reason=t.get('exit_reason'),
                       exit_type=('TRAILING_STOP' if t.get('exit_reason') in STOP_REASONS else
                                  'END_OF_DAY' if t.get('exit_reason') in EOD_REASONS else
                                  'DATA_GAP' if t.get('exit_reason') in GAP_REASONS else 'OTHER'),
                       outcome_unreliable=bool(t.get('outcome_unreliable')), intrabar_ambiguous=ambiguous,
                       rank=t.get('rank'), score=num(t.get('score')),
                       market_data_feed=feed, data_quality_level=quality_level)
            row.update(time_features(entry_time, session, c.opening_minutes))
            row.update(context_features(bars, day, signal_time))
            day_trades.append(row)
            sig['_trade_id'] = trade_id
        trades.extend(day_trades)
        # ---- signals
        day_signals = []
        for item in items:
            stamp = ts(item['decision_time_utc'])
            flags = classify(item)
            bid, ask = num(item.get('bid')), num(item.get('ask'))
            mid = (bid + ask) / 2 if not (math.isnan(bid) or math.isnan(ask)) else np.nan
            prefix = 'entry_ctx_' if 'entry_ctx_open_positions' in item else 'ctx_'
            equity_now, cash_now = num(item.get(prefix + 'equity_proxy')), num(item.get(prefix + 'cash'))
            size = risk_usd = np.nan
            if flags['accepted']:
                event = next((e for e in engine_events if e['kind'] == 'VIRTUAL_ENTRY' and
                              e['symbol'] == item['symbol'] and ts(e['at_utc']) >= stamp), None)
                if event:
                    size, risk_usd = int(event['qty']), num(event['initial_planned_risk'])
            else:
                quote = item.get('virtual_entry_quote') or {}
                entry_ask = num(quote.get('ask')) if quote.get('quote_result') == 'PASS' else np.nan
                if not math.isnan(entry_ask) and not math.isnan(equity_now) and equity_now > 0:
                    try:
                        existing = num(item.get(prefix + 'existing_risk'))
                        sizing = base.size_position(equity_now, max(cash_now, 0), 0.0 if math.isnan(existing) else existing,
                                                    entry_ask, c)
                        size = int(sizing['qty'])
                        risk_usd = float(sizing['planned_risk'])
                    except ValueError:
                        pass
            row = {c_['column']: None for c_ in SIGNALS_SCHEMA}
            row.update(run_id=run_id, signal_id=f'{day}|{item["symbol"]}|{stamp.isoformat()}', date=day,
                       timestamp=stamp, symbol=item['symbol'], sector=item.get('sector'),
                       record_source=item['_source'], price=num(item.get('close')),
                       or_high=num(item.get('or_high')), or_low=num(item.get('or_low')),
                       or_range_pct=ratio(num(item.get('or_high')) - num(item.get('or_low')), item.get('or_low')),
                       breakout_distance_pct=ratio(item.get('close'), item.get('or_high')) - 1,
                       volume=num(item.get('current_volume')), rvol=num(item.get('rvol')),
                       rvol_reference_days=item.get('complete_reference_days'),
                       relative_strength=num(item.get('relative_strength')),
                       bid=bid, ask=ask, mid=mid, spread_bps=ratio(ask - bid, mid) * 1e4,
                       quote_age_ms=num(item.get('quote_age_seconds')) * 1000,
                       score=num(item.get('score')), rank=item.get('rank'),
                       open_positions=item.get(prefix + 'open_positions'),
                       same_sector_positions=item.get(prefix + 'same_sector_positions'),
                       equity=equity_now, cash=cash_now, candidate_position_size=size,
                       candidate_risk_dollars=risk_usd, candidate_risk_pct=ratio(risk_usd, equity_now),
                       signal_status=item.get('status'), shadow_status=item.get('shadow_status'),
                       entry_quote_result=(item.get('virtual_entry_quote') or {}).get('quote_result'),
                       trade_id=by_signal.get((item['symbol'], item['decision_time_utc']), {}).get('_trade_id'),
                       market_data_feed=feed, data_quality_level=quality_level, **flags)
            row.update(time_features(stamp, session, c.opening_minutes))
            row.update(context_features(bars, day, stamp))
            row.update(counterfactuals(bars, item['symbol'], day, stamp, item.get('close'), session))
            day_signals.append(row)
        signals.extend(day_signals)
        # ---- equity curve (starts at the day's start equity)
        start_equity = num(record.get('start_equity'))
        day_marks = [dict(timestamp=session['open'].isoformat(), equity=start_equity, cash=start_equity,
                          market_value=0.0, open_positions=0)] + list(marks)
        for m in day_marks:
            equity.append(dict(run_id=run_id, timestamp=ts(m['timestamp']), date=day, equity=num(m['equity']),
                               cash=num(m['cash']), market_value=num(m['market_value']),
                               open_positions=int(m['open_positions'])))
        values = pd.Series([num(m['equity']) for m in day_marks], dtype=float)
        running = values.cummax()
        utilization = pd.Series([ratio(m['market_value'], m['equity']) for m in day_marks], dtype=float)
        # ---- events
        for i in items:
            stamp = ts(i['decision_time_utc'])
            cls = classify(i)
            events.append(dict(run_id=run_id, date=day, timestamp=stamp, symbol=i['symbol'], event_type='ORB_SIGNAL',
                               price=num(i.get('close')), bid=num(i.get('bid')), ask=num(i.get('ask')),
                               position_qty=None, trailing_stop=np.nan, rvol=num(i.get('rvol')),
                               event_reason=i.get('status')))
            events.append(dict(run_id=run_id, date=day, timestamp=stamp, symbol=i['symbol'],
                               event_type='SIGNAL_ACCEPT' if cls['accepted'] else 'SIGNAL_REJECT',
                               price=num(i.get('close')), bid=num(i.get('bid')), ask=num(i.get('ask')),
                               position_qty=None, trailing_stop=np.nan, rvol=num(i.get('rvol')),
                               event_reason=None if cls['accepted'] else cls['reject_reason']))
        last_stop = {}
        for e in engine_events:
            kind, stamp = e['kind'], ts(e['at_utc'])
            base_row = dict(run_id=run_id, date=day, timestamp=stamp, symbol=e.get('symbol'), price=np.nan,
                            bid=np.nan, ask=np.nan, position_qty=None, trailing_stop=np.nan, rvol=np.nan,
                            event_reason=e.get('reason'))
            if kind == 'VIRTUAL_ENTRY':
                last_stop[e['symbol']] = e['initial_stop']
                events.append(dict(base_row, event_type='ENTRY', price=num(e['ask_proxy']), ask=num(e['ask_proxy']),
                                   position_qty=int(e['qty']), trailing_stop=num(e['initial_stop'])))
            elif kind == 'BAR_REVIEWED' and num(e.get('stop_next_bar')) > num(last_stop.get(e['symbol'], np.inf)):
                last_stop[e['symbol']] = e['stop_next_bar']
                events.append(dict(base_row, event_type='TRAIL_UPDATE', price=num(e.get('close_proxy')),
                                   trailing_stop=num(e['stop_next_bar']), event_reason='NEW_HIGH'))
            elif kind == 'VIRTUAL_EXIT':
                events.append(dict(base_row, event_type='EOD_EXIT' if e.get('reason') in EOD_REASONS else 'EXIT',
                                   price=num(e['exit_price_proxy']), position_qty=int(e['qty']),
                                   trailing_stop=num(last_stop.get(e['symbol'], np.nan))))
            elif kind == 'DAILY_LOSS_LOCK':
                events.append(dict(base_row, event_type='RISK_LOCK', event_reason='DAILY_LOSS_LOCK'))
            elif kind in ('NO_SIP_BAR_WHILE_HELD', 'INVALID_HELD_BAR', 'LATE_HELD_BAR_STOP_NOT_BACKDATED',
                          'DATA_GAP_EXIT_PRICE_UNAVAILABLE', 'CLOSE_PRICE_UNAVAILABLE'):
                events.append(dict(base_row, event_type='DATA_GAP', event_reason=kind))
        # ---- data quality
        filters = folder / 'universe_filter_results.csv'
        eligible = (list(pd.read_csv(filters, keep_default_na=False).query('result == "PASS"').symbol)
                    if filters.exists() else [])
        or_bad = {i['symbol'] for i in items if i['_source'] == 'PRICE_REJECTION' and i['status'] == 'OR_INCOMPLETE'}
        coverage_rows = store.day_bar_coverage(eligible, day, c.opening_minutes) if eligible else pd.DataFrame()
        no_bars = set(coverage_rows.loc[coverage_rows.bars == 0, 'symbol']) if len(coverage_rows) else set()
        valid = [sym for sym in eligible if sym not in or_bad and sym not in no_bars]
        for gap in store.data_gaps(eligible, day):
            gaps.append(dict(run_id=run_id, date=day, symbol=gap['symbol'],
                             missing_minute_utc=ts(gap['missing_minute_utc']),
                             missing_minute_ny=gap['missing_minute_ny']))
        quote_missing = sum(1 for i in items if i.get('status') in ('QUOTE_MISSING', 'QUOTE_DATA_ERROR')) + \
            sum(1 for i in items if (i.get('virtual_entry_quote') or {}).get('quote_result') in ('QUOTE_MISSING', 'ERROR')) + \
            sum(1 for e in engine_events if e['kind'] in ('DATA_GAP_EXIT_PRICE_UNAVAILABLE', 'CLOSE_PRICE_UNAVAILABLE'))
        trade_missing = sum(int(refs.status.astype(str).str.startswith('REFERENCE_UNVERIFIED').sum())
                            for refs in reference_check_frames(folder) if 'status' in refs)
        net = [r['net_pnl'] for r in day_trades]
        excluded = len(eligible) - len(valid)
        day_row.update(
            gross_pnl=float(sum(r['gross_pnl'] for r in day_trades)), net_pnl=float(sum(net)),
            net_pnl_10bp_stress=float(sum(r['net_pnl_10bp_stress'] for r in day_trades)),
            daily_return_pct=ratio(sum(net), start_equity), trades=len(day_trades),
            winners=sum(1 for x in net if x > 0), losers=sum(1 for x in net if x < 0),
            signals_total=len(day_signals), signals_accepted=sum(1 for r in day_signals if r['accepted']),
            signals_rejected=sum(1 for r in day_signals if not r['accepted']),
            gross_profit=float(sum(x for x in net if x > 0)), gross_loss=float(sum(x for x in net if x < 0)),
            max_intraday_drawdown=float((values - running).min()) if len(values) else 0.0,
            max_intraday_drawdown_pct=float((values / running - 1).min()) if len(values) else 0.0,
            capital_utilization_avg=float(utilization.mean()) if len(utilization) else np.nan,
            capital_utilization_max=float(utilization.max()) if len(utilization) else np.nan,
            max_simultaneous_positions=int(max((m['open_positions'] for m in day_marks), default=0)),
            daily_loss_lock_triggered=any(e['kind'] == 'DAILY_LOSS_LOCK' for e in engine_events),
            trailing_stop_exits=sum(1 for r in day_trades if r['exit_type'] == 'TRAILING_STOP'),
            eod_exits=sum(1 for r in day_trades if r['exit_type'] == 'END_OF_DAY'),
            data_gap_exits=sum(1 for r in day_trades if r['exit_type'] == 'DATA_GAP'),
            expected_symbols=len(eligible), symbols_with_valid_data=len(valid), symbols_excluded=excluded,
            data_coverage_pct=ratio(len(valid), len(eligible)),
            missing_bar_count=int(coverage_rows.missing_minutes.fillna(0).sum()) if len(coverage_rows) else 0,
            missing_quote_count=int(quote_missing), missing_trade_count=int(trade_missing),
            historical_only_signals=sum(1 for i in items if i.get('status') == 'HISTORICAL_ONLY_STARTUP_OR_RESTART'),
            missed_decision_minutes=int(coverage.get('missed_decision_minutes') or 0),
            unreliable_trades=int(state.get('unreliable_virtual_trades') or 0))
        day_row['data_quality_issues'] = int(excluded + quote_missing + trade_missing + day_row['unreliable_trades'] +
                                             day_row['missed_decision_minutes'] +
                                             int(coverage.get('data_error_minutes') or 0))
        daily.append(day_row)
    tables = dict(trades=trades, signals=signals, daily=daily, equity_curve=equity, events=events, data_gaps=gaps)
    frames = {name: typed_frame(rows, SCHEMAS[name]) for name, rows in tables.items()}
    return finalize(frames)


def finalize(frames):
    """Run-level columns that need all days: drawdown over the whole equity curve."""
    curve = frames['equity_curve']
    if len(curve):
        peak = curve.equity.cummax()
        curve['drawdown'] = curve.equity - peak
        curve['drawdown_pct'] = curve.equity / peak - 1
    return frames


def retype(frame, schema):
    """Schema dtypes for a frame read back from Parquet or concatenated from parts."""
    return typed_frame(frame.to_dict('records'), schema) if len(frame) else typed_frame([], schema)


PART_TABLES = ('trades', 'signals', 'daily', 'equity_curve', 'events', 'data_gaps')


def export_day(run_folder, run_id, record, store, c, s, quality_level, feed='sip'):
    """V1.6.1 auto-save: the research tables of one finished day (parts/<date>/).

    The final export concatenates the parts, so a resumed run yields the same tables."""
    folder = Path(run_folder) / 'parts' / record['date']
    folder.mkdir(parents=True, exist_ok=True)
    frames = build_tables(run_id, [record], store, c, s, quality_level, feed)
    for name in PART_TABLES:
        frames[name].to_parquet(folder / f'{name}.parquet', index=False)
    write_json(folder / 'part.json', dict(date=record['date'], status=record.get('status'),
                                          run_folder=str(record.get('folder')), run_id=run_id))
    return frames


def load_part(run_folder, run_id, record):
    folder = Path(run_folder) / 'parts' / record['date']
    meta = read_json(folder / 'part.json')
    if not meta or meta.get('status') != record.get('status') or \
            meta.get('run_folder') != str(record.get('folder')) or meta.get('run_id') != run_id:
        return None
    try:
        return {name: pd.read_parquet(folder / f'{name}.parquet') for name in PART_TABLES}
    except Exception:
        return None


def typed_frame(rows, schema):
    columns = [c['column'] for c in schema]
    frame = pd.DataFrame(rows, columns=columns)
    for c in schema:
        name, dtype = c['column'], c['datatype']
        if dtype == 'timestamp':
            frame[name] = pd.to_datetime(frame[name], utc=True)
        elif dtype == 'float':
            frame[name] = pd.to_numeric(frame[name], errors='coerce').astype('float64')
        elif dtype == 'int':
            frame[name] = pd.to_numeric(frame[name], errors='coerce').round().astype('Int64')
        elif dtype == 'bool':
            frame[name] = frame[name].astype('boolean')
        else:
            frame[name] = frame[name].astype('string')
    return frame


def csv_frame(frame):
    """CSV view: ISO timestamps, true/false, empty for null."""
    out = frame.copy()
    for name in out.columns:
        if isinstance(out[name].dtype, pd.DatetimeTZDtype):
            out[name] = out[name].map(lambda v: v.isoformat() if not pd.isna(v) else None)
        elif str(out[name].dtype) == 'boolean':
            out[name] = out[name].map(lambda v: None if pd.isna(v) else ('true' if v else 'false'))
    return out


# =========================================================================== summary
def summarize(run_id, frames, initial_equity, status, meta):
    trades, signals, daily, curve = frames['trades'], frames['signals'], frames['daily'], frames['equity_curve']
    simulated = daily[daily.simulated.fillna(False)]
    net = trades.net_pnl.astype(float) if len(trades) else pd.Series(dtype=float)
    wins, losses = net[net > 0], net[net < 0]
    final_equity = float(curve.equity.iloc[-1]) if len(curve) else float(initial_equity)
    realized_final = float(initial_equity + net.sum())
    returns = simulated.daily_return_pct.astype(float).dropna()

    def f(value):
        return None if value is None or (isinstance(value, float) and not math.isfinite(value)) else float(value)
    sharpe = sortino = None
    if len(returns) >= 20 and returns.std(ddof=1) > 0:
        sharpe = float(returns.mean() / returns.std(ddof=1) * math.sqrt(252))
        downside = returns[returns < 0]
        if len(downside) >= 2 and downside.std(ddof=1) > 0:
            sortino = float(returns.mean() / downside.std(ddof=1) * math.sqrt(252))
    rejected = signals[~signals.accepted.fillna(False)]
    return dict(
        run_id=run_id, research_schema_version=RESEARCH_SCHEMA_VERSION, bot_version=BOT_VERSION,
        status=status, period=dict(start=meta['backtest_start'], end=meta['backtest_end'],
                                   warmup_start=meta.get('warmup_start')),
        performance=dict(
            trading_days=int(len(simulated)), days_in_period=int(len(daily)),
            days_not_simulated=int((~daily.simulated.fillna(False)).sum()),
            initial_equity=float(initial_equity), final_equity=realized_final,
            net_profit=float(net.sum()), return_pct=f(net.sum() / initial_equity) if initial_equity else None,
            net_profit_10bp_stress=float(trades.net_pnl_10bp_stress.sum()) if len(trades) else 0.0,
            gross_profit=float(wins.sum()), gross_loss=float(losses.sum()),
            max_drawdown_usd=f(curve.drawdown.min()) if len(curve) else 0.0,
            max_drawdown_pct=f(curve.drawdown_pct.min()) if len(curve) else 0.0,
            max_daily_drawdown_usd=f(simulated.max_intraday_drawdown.min()) if len(simulated) else None,
            max_daily_drawdown_pct=f(simulated.max_intraday_drawdown_pct.min()) if len(simulated) else None,
            best_day_usd=f(simulated.net_pnl.max()) if len(simulated) else None,
            worst_day_usd=f(simulated.net_pnl.min()) if len(simulated) else None,
            sharpe_ratio_daily_annualized=sharpe, sortino_ratio_daily_annualized=sortino,
            ratio_note='Tagesrenditen, risikofreier Zins 0, √252; nur ab 20 simulierten Tagen.',
            equity_curve_last_mark=f(final_equity)),
        trades=dict(
            total=int(len(trades)), winners=int(len(wins)), losers=int(len(losses)),
            win_rate=f(len(wins) / len(trades)) if len(trades) else None,
            average_winner=f(wins.mean()) if len(wins) else None,
            average_loser=f(losses.mean()) if len(losses) else None,
            profit_factor=f(wins.sum() / -losses.sum()) if len(losses) and losses.sum() < 0 else None,
            expectancy=f(net.mean()) if len(net) else None,
            average_holding_minutes=f(trades.holding_minutes.mean()) if len(trades) else None,
            average_mfe_dollars=f(trades.mfe_dollars.mean()) if len(trades) else None,
            average_mfe_pct=f(trades.mfe_pct.mean()) if len(trades) else None,
            average_mae_dollars=f(trades.mae_dollars.mean()) if len(trades) else None,
            average_mae_pct=f(trades.mae_pct.mean()) if len(trades) else None,
            unreliable=int(trades.outcome_unreliable.fillna(False).sum()) if len(trades) else 0,
            intrabar_ambiguous_events=int(trades.intrabar_ambiguous.fillna(False).sum()) if len(trades) else 0),
        signals=dict(
            total=int(len(signals)), accepted=int(signals.accepted.fillna(False).sum()),
            rejected=int(len(rejected)),
            by_stage={str(k): int(v) for k, v in rejected.reject_stage.value_counts().items()},
            by_reason={str(k): int(v) for k, v in rejected.reject_reason.value_counts().items()},
            data_reject_codes={str(k): int(v) for k, v in rejected.data_reject_code.dropna().value_counts().items()},
            historical_only=int((signals.signal_status == 'HISTORICAL_ONLY_STARTUP_OR_RESTART').sum())),
        exits=dict(
            trailing_stop=int((trades.exit_type == 'TRAILING_STOP').sum()) if len(trades) else 0,
            end_of_day=int((trades.exit_type == 'END_OF_DAY').sum()) if len(trades) else 0,
            data_gap=int((trades.exit_type == 'DATA_GAP').sum()) if len(trades) else 0,
            by_reason={str(k): int(v) for k, v in trades.exit_reason.value_counts().items()} if len(trades) else {}),
        risk=dict(
            max_simultaneous_positions=int(simulated.max_simultaneous_positions.max()) if len(simulated) else 0,
            capital_utilization_avg=f(simulated.capital_utilization_avg.mean()) if len(simulated) else None,
            capital_utilization_max=f(simulated.capital_utilization_max.max()) if len(simulated) else None,
            daily_loss_lock_days=int(simulated.daily_loss_lock_triggered.fillna(False).sum()) if len(simulated) else 0,
            average_risk_pct=f(trades.risk_pct.mean()) if len(trades) else None,
            max_risk_pct=f(trades.risk_pct.max()) if len(trades) else None),
        data_quality=dict(
            level=meta['data_quality_level'], feed=meta['historical_feed'],
            days_systemic_error=int(daily.systemic_data_error.fillna(False).sum()),
            data_coverage_avg=f(simulated.data_coverage_pct.mean()) if len(simulated) else None,
            data_coverage_min=f(simulated.data_coverage_pct.min()) if len(simulated) else None,
            symbols_excluded_total=int(simulated.symbols_excluded.fillna(0).sum()) if len(simulated) else 0,
            missing_bar_count=int(simulated.missing_bar_count.fillna(0).sum()) if len(simulated) else 0,
            missing_quote_count=int(simulated.missing_quote_count.fillna(0).sum()) if len(simulated) else 0,
            missing_trade_count=int(simulated.missing_trade_count.fillna(0).sum()) if len(simulated) else 0,
            unreliable_trades=int(simulated.unreliable_trades.fillna(0).sum()) if len(simulated) else 0,
            missed_decision_minutes=int(simulated.missed_decision_minutes.fillna(0).sum()) if len(simulated) else 0,
            cache=meta.get('cache'), warnings=meta.get('warnings', [])),
        warning=('Modellergebnis mit historischen SIP-Daten, aktuellem Index-Universum (Survivorship-Bias) '
                 'und virtuellen Ausführungen; kein Rentabilitätsnachweis.'))


def clean_json(value):
    if isinstance(value, dict):
        return {str(k): clean_json(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [clean_json(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return None if not math.isfinite(float(value)) else float(value)
    if isinstance(value, (pd.Timestamp,)):
        return value.isoformat()
    if value is pd.NA or value is pd.NaT:
        return None
    return value


def write_json(path, content):
    temporary = Path(path).with_name(Path(path).name + '.incomplete')
    temporary.write_text(json.dumps(clean_json(content), ensure_ascii=False, indent=2, allow_nan=False) + '\n',
                         encoding='utf-8')
    temporary.replace(path)


# =========================================================================== manifest
def run_manifest(run_id, meta, c, s, settings, rs, created_at):
    """Everything that can change a result, taken from the runtime objects."""
    or_start = '09:30'
    return dict(
        run_id=run_id, research_schema_version=RESEARCH_SCHEMA_VERSION, bot_version=BOT_VERSION,
        created_at=created_at, status=meta['status'],
        backtest_start=meta['backtest_start'], backtest_end=meta['backtest_end'],
        warmup_start=meta.get('warmup_start'), warmup_trading_days=meta.get('warmup_trading_days'),
        trading_days=meta.get('trading_days'), timezone=NY,
        initial_equity=meta['initial_equity'], equity_carry_forward=True,
        universe=meta.get('universe'),
        historical_source=meta.get('historical_source'), historical_feed=meta['historical_feed'],
        bar_timeframe='1Min', bar_adjustment='raw', data_quality_level=meta['data_quality_level'],
        data_requirements=meta.get('data_requirements'),
        strategy_name=STRATEGY_NAME,
        strategy_parameters=dict(
            opening_range_start_ny=or_start, opening_range_minutes=c.opening_minutes,
            opening_range_end_ny=f'09:{30 + c.opening_minutes:02d}' if c.opening_minutes < 30 else None,
            breakout_buffer=float(c.breakout_buffer), entry_cap_above_or_high=float(c.entry_cap),
            rvol_lookback_days=c.rvol_days, rvol_threshold=c.minimum_rvol,
            rvol_min_reference_days=getattr(settings, 'min_rvol_reference_days', None),
            minimum_price=c.minimum_price,
            first_decision='Öffnung + opening_minutes + 1 Min. (09:46 NY an normalen Tagen)',
            entry_cutoff_minutes_before_close=c.entry_cutoff_minutes,
            planned_close_minutes_before_close=c.close_before_minutes,
            trailing_stop=float(c.trailing), take_profit=None, overnight=False,
            signal_ttl_seconds=s.signal_ttl_seconds, watchlist_size=s.watchlist_size,
            ranking_weights=dict(rvol=s.weight_rvol, relative_strength=s.weight_relative_strength,
                                 breakout=s.weight_breakout, spread=s.weight_spread),
            max_reentries=getattr(settings, 'max_reentries', 0)),
        risk_parameters=dict(
            max_positions=c.max_positions, max_position_capital=float(c.max_position_capital),
            max_trade_risk=float(c.max_trade_risk), max_total_risk=float(c.max_total_risk),
            max_per_sector=c.max_per_sector, daily_loss_limit=float(c.daily_loss),
            note='Risiko = Stück × Trailing-Abstand; die 30-%-Kapitalgrenze begrenzt meist zuerst.'),
        execution_parameters=dict(
            entry='frische SIP-Ask (Quote ≤ 2 s)', exit='Stop-Kurs / Gap-Open / frische SIP-Bid',
            slippage_model='keins zusätzlich; 10-bp-Stressszenario (5 bp je Seite) separat',
            commission=0.0, commission_note='nicht modelliert (Alpaca provisionsfrei)',
            bar_delay_seconds=rs.bar_delay_seconds, request_latency_seconds=rs.request_latency_seconds,
            start_before_open_minutes=rs.start_before_open_minutes),
        active_filters=dict(
            liquidity_min_avg_dollar_volume=s.minimum_average_dollar_volume,
            spread_threshold=s.maximum_spread, quote_age_threshold_seconds=s.maximum_quote_age_seconds,
            quote_search_seconds=s.quote_search_seconds, rvol=True, relative_strength_ranking=True,
            sector_limit=True, daily_loss_lock=True, min_opening_bars=settings.min_opening_bars,
            min_coverage=settings.min_coverage, max_previous_age_minutes=settings.max_previous_age_minutes),
        scanner_config=asdict(s), strategy_config=asdict(c), dry_run_settings=asdict(settings),
        replay_settings=asdict(rs), code_sha256=meta.get('code_sha256'),
        config_source=meta.get('config_source'), days=meta.get('days'), warnings=meta.get('warnings', []),
        environment=dict(python=platform.python_version(),
                         packages={p: _version(p) for p in ('alpaca-py', 'pandas', 'numpy', 'pyarrow')}),
        cache=meta.get('cache'))


def _version(package):
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None


README = """BACKTEST ANALYSIS BUNDLE ({run_id})

run_manifest.json  vollständige Einstellungen (Strategie, Risiko, Ausführung, Daten, Code-Hashes)
summary.json       Kennzahlen: performance, trades, signals, exits, risk, data_quality
schema.json        jede Spalte mit Datentyp, Einheit, Beschreibung, Quelle; Definitionen
trades.csv         eine Zeile je virtuellem Trade
signals.csv        eine Zeile je Kandidatensignal (auch abgelehnte) mit Stufe, Grund, Filter-Flags
                   und COUNTERFACTUAL future_* (nach dem Lauf berechnet, nie Teil der Entscheidung)
daily.csv          eine Zeile je Handelstag inkl. Datenabdeckung
equity_curve.csv   Equity je Entscheidungsminute

Prozentwerte sind Dezimalzahlen (0.042 = 4,2 %). Zeitstempel ISO-8601 (UTC).
Leere Felder = nicht verfügbar. Modellergebnis, kein Rentabilitätsnachweis.
"""


def export_run(run_folder, run_id, day_records, store, c, s, settings, rs, meta):
    """Write all research files of a finished run and the analysis ZIP."""
    run_folder = Path(run_folder)
    unfinished = [r['date'] for r in day_records if r.get('folder') and
                  not (Path(r['folder']) / 'status.json').exists()]
    if unfinished:
        raise RuntimeError(f'Research-Export erst nach Abschluss aller Tage (offen: {unfinished}).')
    feed = meta['historical_feed'].lower()
    parts = []
    for record in day_records:
        part = load_part(run_folder, run_id, record)
        parts.append(part if part is not None else export_day(
            run_folder, run_id, record, store, c, s, meta['data_quality_level'], feed))
    frames = {}
    for name in PART_TABLES:
        pieces = [part[name] for part in parts if len(part[name])]
        joined = pd.concat(pieces, ignore_index=True) if pieces else pd.DataFrame()
        frames[name] = retype(joined, SCHEMAS[name])
    for name in ('trades', 'signals'):
        if len(frames[name]):
            frames[name]['data_quality_level'] = pd.Series(meta['data_quality_level'],
                                                           index=frames[name].index, dtype='string')
    frames = finalize(frames)
    for name, frame in frames.items():
        frame.to_parquet(run_folder / f'{name}.parquet', index=False)
        if name in ('trades', 'signals', 'daily', 'equity_curve'):
            csv_frame(frame).to_csv(run_folder / f'{name}.csv', index=False)
    created_at = meta.get('created_at') or pd.Timestamp.now(tz='UTC').isoformat()
    manifest = run_manifest(run_id, meta, c, s, settings, rs, created_at)
    summary = summarize(run_id, frames, meta['initial_equity'], meta['status'], meta)
    schema = dict(research_schema_version=RESEARCH_SCHEMA_VERSION, tables=SCHEMAS, definitions=DEFINITIONS,
                  reject_codes=dict(
                      signal_status='Engine-Status (z. B. VOLUME_REJECT, NOT_CHECKABLE, SPREAD_TOO_WIDE, QUOTE_STALE, '
                                    'ASK_OUTSIDE_ENTRY_BAND, OUTSIDE_TOP_FIVE_NO_ORDER, WATCHLIST_NO_ORDER).',
                      shadow_status='Portfolio (VIRTUAL_ENTRY, DAILY_LOSS_LOCK, NO_CAPITAL_OR_RISK_BUDGET, '
                                    'Positionsplätze belegt/reserviert, Sektorlimit erreicht, ENTRY_CUTOFF …).',
                      price_rejections='TOO_EXTENDED, BELOW_MINIMUM_PRICE, DATA_INCOMPLETE, OR_INCOMPLETE.',
                      data_reject_code='DATA_INCOMPLETE, OR_INCOMPLETE, RVOL_REFERENCE_INCOMPLETE, QUOTE_DATA_MISSING, '
                                       'TRADE_DATA_MISSING.'))
    write_json(run_folder / 'run_manifest.json', manifest)
    write_json(run_folder / 'summary.json', summary)
    write_json(run_folder / 'schema.json', schema)
    bundle = run_folder / f'{run_id}_ANALYSIS.zip'
    with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for name in ('run_manifest.json', 'summary.json', 'schema.json', 'trades.csv', 'signals.csv',
                     'daily.csv', 'equity_curve.csv'):
            archive.write(run_folder / name, name)
        archive.writestr('README.txt', README.format(run_id=run_id))
    update_index(run_folder.parent, manifest, summary, run_folder)
    return dict(folder=run_folder, bundle=bundle, summary=summary, frames=frames)


def update_index(root, manifest, summary, run_folder):
    """backtests/backtest_runs.csv: one row per run, never overwritten."""
    path = Path(root) / 'backtest_runs.csv'
    row = dict(run_id=manifest['run_id'], created_at=manifest['created_at'], bot_version=manifest['bot_version'],
               status=manifest['status'], start=manifest['backtest_start'], end=manifest['backtest_end'],
               initial_equity=manifest['initial_equity'], final_equity=summary['performance']['final_equity'],
               return_pct=summary['performance']['return_pct'], trades=summary['trades']['total'],
               signals=summary['signals']['total'], feed=manifest['historical_feed'],
               quality_level=manifest['data_quality_level'],
               parameters_sha256=hashlib.sha256(json.dumps(
                   [manifest['strategy_parameters'], manifest['risk_parameters'], manifest['active_filters']],
                   sort_keys=True, default=str).encode()).hexdigest()[:16],
               path=str(run_folder))
    frame = pd.read_csv(path) if path.exists() and path.stat().st_size > 1 else pd.DataFrame()
    if len(frame) and row['run_id'] in set(frame.run_id):
        frame = frame[frame.run_id != row['run_id']]
    frame = pd.concat([frame, pd.DataFrame([row])], ignore_index=True)
    temporary = path.with_name(path.name + '.incomplete')
    frame.to_csv(temporary, index=False)
    temporary.replace(path)
    return path


In [ ]:
%%writefile /content/backtest_v16.py
"""BACKTEST (V1.6): cockpit settings -> historical data -> replay -> research package.

    Check Historical Cache -> Determine Required Data -> Download Missing SIP Data
    -> Validate -> Persist Cache -> Run Replay (orb_replay_v16.replay_days, the
    unchanged strategy loop) -> Research Dataset -> Analysis Bundle

No strategy code lives here. The replay gets a ReadOnlyBroker (calendar and
assets only) and never an order hook; the main mode is set to BACKTEST, so the
order layer refuses any order path (bot_mode_v16).

Data-quality policy (per trading day of the calendar):
    a) no session (weekend, holiday)          -> not part of the period, no error
    b) single missing minutes                 -> existing OR/RVOL/quote rules reject the
                                                 affected signal (reason stored)
    c) one symbol without usable data         -> excluded that day by the existing rules
    d) systemic problem (download failed, QQQ missing, or less than
       `systemic_min_symbol_share` of the universe with any bar; quotes not
       accessible)                           -> day not simulated, backtest stops,
                                                 run status INCOMPLETE_DATA, all outputs kept
    e) every calendar day of the period appears in `daily`, simulated or not.
"""
from dataclasses import asdict, dataclass, replace
from datetime import date, timedelta
from pathlib import Path
import json
import threading
import time
import uuid

import pandas as pd

import bot_mode_v16 as bot_mode
import hist_data_v16 as hist
import orb_replay_v16 as replay
import research_export_v16 as research

NY = 'America/New_York'
BACKTEST_FOLDER = 'backtests'


@dataclass(frozen=True)
class BacktestSettings:
    start: str
    end: str
    initial_equity: float = 10000.0
    feed: str = 'SIP'                        # historical backtest feed; there is no IEX fallback
    full_trades: bool = False                # FULL_MARKET_DATA: all SIP trades of signal symbols/days
    # Rule d): a day is systemic when fewer universe symbols than this share have any
    # bar at all (or QQQ is missing). It separates "the download failed" from
    # "a few symbols had no data"; per-day coverage itself is only recorded.
    systemic_min_symbol_share: float = 0.5
    stop_on_day_failure: bool = True         # carry-forward: later days would start with a wrong equity
    research_symbols: tuple = ('SPY',)       # context only, never passed to the strategy


class SystemicDataError(RuntimeError):
    day_status = 'INCOMPLETE_DATA'
    stop_run = True


def validate(bt, today=None):
    """Plain input checks before any data access; returns a list of messages."""
    errors = []
    try:
        start, end = date.fromisoformat(str(bt.start)[:10]), date.fromisoformat(str(bt.end)[:10])
    except ValueError:
        return ['Start- und Enddatum als Datum wählen.']
    today = today or pd.Timestamp.now(tz=NY).date()
    if start > end:
        errors.append('Das Startdatum liegt nach dem Enddatum.')
    if end >= today:
        errors.append('Das Enddatum muss ein vergangener Tag sein (heute ist noch nicht abgeschlossen).')
    if start < date(2016, 1, 1):
        errors.append('Alpaca liefert SIP-Historie ab 2016; früheres Startdatum nicht möglich.')
    if not bt.initial_equity or bt.initial_equity <= 0:
        errors.append('Startkapital muss größer als 0 sein.')
    if str(bt.feed).upper() != 'SIP':
        errors.append(hist.warning_text(str(bt.feed), 'Backtests laufen nur mit SIP'))
    if not 0 < bt.systemic_min_symbol_share <= 1:
        errors.append('systemic_min_symbol_share muss zwischen 0 und 1 liegen.')
    return errors


def requirements(c, s, bt):
    """Data the active strategy needs (Data Requirement Resolver)."""
    return dict(
        bars_1min=dict(required=True, reason='Opening Range, Ausbruch, RVOL, Trailing Stop, MFE/MAE'),
        bars_1day=dict(required=s.minimum_average_dollar_volume > 0,
                       reason='Liquiditätsfilter über die Vortage'),
        quotes=dict(required=s.maximum_spread > 0 or s.maximum_quote_age_seconds > 0,
                    scope='decision_points',
                    reason='Spread, Quote-Frische, Ask-Band bei Signal, Einstieg und Ausstieg'),
        trades=dict(required=True, scope='full_sessions_of_signal_symbols' if bt.full_trades
                    else 'missing_rvol_reference_minutes',
                    reason='Odd-Lot-Klärung fehlender RVOL-Vergleichsminuten'
                           + ('; zusätzlich alle Trades der Signal-Aktien (Research)' if bt.full_trades else '')),
        order_flow=dict(required=False, reason='keine Order-Flow-Regel in der Strategie'),
        warmup_trading_days=warmup_days(c))


def warmup_days(c):
    """RVOL and the liquidity filter both look back `rvol_days` sessions."""
    return int(c.rvol_days)


def plan_days(store, bt, c):
    days = [r['date'] for r in store.sessions(bt.start, bt.end)]
    if not days:
        return dict(days=[], warmup=[], warmup_start=None)
    first = date.fromisoformat(days[0])
    before = [r['date'] for r in store.sessions(first - timedelta(days=warmup_days(c) * 3 + 20),
                                                  first - timedelta(days=1))]
    warm = before[-warmup_days(c):]
    return dict(days=days, warmup=warm, warmup_start=warm[0] if warm else None)


def new_run_id(now=None):
    now = now or pd.Timestamp.now(tz='UTC')
    return f'BT_{now:%Y%m%dT%H%M%SZ}_{uuid.uuid4().hex[:8]}'


RUN_STATES = ('CREATED', 'RUNNING', 'PAUSED', 'COMPLETED', 'FAILED', 'INCOMPLETE_DATA', 'ABORTED')
OPEN_STATES = ('CREATED', 'RUNNING', 'PAUSED')
PHASES = ('Cache prüfen', 'Historische Daten laden', 'Daten prüfen', 'Indikatoren vorbereiten (RVOL-Warmup)',
          'Replay läuft', 'Research-Output erzeugen')
HEARTBEAT_SECONDS = 30
STALE_SECONDS = 180


def backtest_settings_default():
    """The replay loop with day-wise persistence and without console text (V1.6.1)."""
    return replace(replay.replay_settings_default(), persist_every_minute=False,
                   reference_cache_on_disk=False, quiet=True)


class PauseRequested(RuntimeError):
    day_status = 'PAUSED'
    stop_run = True


class BacktestProgress:
    """Aggregated state of a running backtest (Progress Aggregator).

    The strategy loop calls update() like a cockpit, at most once per simulated
    minute; it only stores a few numbers (no rendering, no file access). The UI reads
    snapshot() on its own timer; the loop never waits for the browser."""

    def __init__(self):
        self.lock = threading.Lock()
        self.pause_requested = False
        self.state = dict(status='CREATED', phase=0, phase_name=None, message=None, run_id=None,
                          period=None, days_total=0, days_done=0, current_day=None, sim_time=None,
                          universe=0, signals=0, trades=0, equity=None, coverage=None,
                          started_wall=time.time(), last_update_wall=time.time(), day_lines=[],
                          download=None, finished=False, result=None, error=None)
        self._day = dict(signals=0, trades=0)
        self.ui_updates_requested = 0

    def set(self, **fields):
        with self.lock:
            self.state.update(fields)
            self.state['last_update_wall'] = time.time()

    def phase(self, number, message=None):
        self.set(phase=number, phase_name=PHASES[number - 1], message=message)

    # cockpit interface used by orb_sim_v16.run_dryrun (paint)
    def update(self, folder, portfolio, at, decision=None, stage=None, mode=None, eligible_count=0,
               last_price_bar=None, signals=(), broker_start=None, clock_lag_seconds=0):
        with self.lock:
            trades = len(portfolio.closed)
            if trades != self._day['trades']:
                self.ui_updates_requested += 1          # entry/exit: worth an early refresh
            self._day['trades'] = trades
            self._day['signals'] += len(signals)
            self.state.update(sim_time=pd.Timestamp(at).isoformat(), universe=eligible_count or self.state['universe'],
                              day_signals=self._day['signals'], day_trades=trades,
                              open_positions=len(portfolio.positions),
                              intraday_equity=float(portfolio.last_equity),
                              last_update_wall=time.time())

    def day_started(self, day):
        with self.lock:
            self._day = dict(signals=0, trades=0)
            self.state.update(current_day=day, day_signals=0, day_trades=0, sim_time=None,
                              last_update_wall=time.time())

    def day_finished(self, row, line):
        with self.lock:
            s = self.state
            s['days_done'] += 1
            s['signals'] += int(row.get('signals_total') or 0)
            s['trades'] += int(row.get('trades') or 0)
            if row.get('end_equity') is not None and not pd.isna(row.get('end_equity')):
                s['equity'] = float(row['end_equity'])
            coverages = s.setdefault('_coverages', [])
            if row.get('data_coverage_pct') is not None and not pd.isna(row.get('data_coverage_pct')):
                coverages.append(float(row['data_coverage_pct']))
                s['coverage'] = sum(coverages) / len(coverages)
            s['day_lines'] = (s['day_lines'] + [line])[-400:]
            s['last_update_wall'] = time.time()
            self.ui_updates_requested += 1

    def snapshot(self):
        with self.lock:
            data = dict(self.state)
            data['day_lines'] = list(self.state['day_lines'])
            data.pop('_coverages', None)
            return data


class _Tee:
    """Feed the aggregator and an optional HTML cockpit from the same paint call."""

    def __init__(self, *targets):
        self.targets = [t for t in targets if t is not None]

    def update(self, *args, **kwargs):
        for target in self.targets:
            target.update(*args, **kwargs)


def checkpoint_path(folder):
    return Path(folder) / 'checkpoint.json'


def read_checkpoint(folder):
    path = checkpoint_path(folder)
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else None


class Checkpoint:
    """checkpoint.json of one run: written after every day and by a heartbeat thread."""

    def __init__(self, folder, state):
        self.folder, self.state, self.lock = Path(folder), state, threading.Lock()
        self._stop = threading.Event()
        self._thread = None

    def save(self, **fields):
        with self.lock:
            self.state.update(fields)
            now = pd.Timestamp.now(tz='UTC').isoformat()
            self.state['updated_at'] = now
            self.state['heartbeat_at'] = now
            research.write_json(checkpoint_path(self.folder), self.state)

    def start_heartbeat(self, interval=HEARTBEAT_SECONDS):
        def beat():
            while not self._stop.wait(interval):
                try:
                    self.save()
                except Exception:
                    pass
        self._thread = threading.Thread(target=beat, daemon=True, name='backtest-heartbeat')
        self._thread.start()

    def stop_heartbeat(self):
        self._stop.set()


def find_open_runs(project_root, stale_seconds=STALE_SECONDS, now=None):
    """Runs that can be resumed: CREATED / RUNNING / PAUSED (newest first).

    RUNNING without a heartbeat for `stale_seconds` counts as interrupted (runtime lost)."""
    now = now or pd.Timestamp.now(tz='UTC')
    found = []
    for path in sorted((Path(project_root) / BACKTEST_FOLDER).glob('BT_*/checkpoint.json'), reverse=True):
        try:
            state = json.loads(path.read_text(encoding='utf-8'))
        except ValueError:
            continue
        if state.get('status') not in OPEN_STATES:
            continue
        beat = pd.Timestamp(state.get('heartbeat_at') or state.get('created_at'))
        age = (now - beat).total_seconds()
        found.append(dict(state, folder=str(path.parent), heartbeat_age_seconds=age,
                          interrupted=state.get('status') == 'RUNNING' and age > stale_seconds))
    return found


def discard_run(folder):
    """DISCARD: mark an open run ABORTED; its files stay for inspection."""
    state = read_checkpoint(folder)
    if state is None:
        raise FileNotFoundError('Kein Checkpoint im Laufordner.')
    state.update(status='ABORTED', status_reason='DISCARDED_BY_USER',
                 updated_at=pd.Timestamp.now(tz='UTC').isoformat())
    research.write_json(checkpoint_path(folder), state)
    return state


def record_for_state(record):
    return {k: (str(v) if isinstance(v, Path) else v) for k, v in record.items()}


def run_backtest(store, project_root, c, s, bt=None, settings=None, rs=None, cockpit=None,
                 progress=None, config_source=None, tracker=None, resume=None):
    """One backtest run (or the rest of an interrupted one with `resume=<run folder>`).

    Returns dict(run_id, folder, status, status_reason, bundle, summary, records, warnings).
    After every trading day: research parts, checkpoint and run manifest are saved, so an
    interruption loses at most the day that was running."""
    bot_mode.set_mode('BACKTEST')
    settings = settings or backtest_settings_default()
    rs = rs or replay.ReplaySettings()
    report = progress or (lambda info: None)
    tracker = tracker or BacktestProgress()
    project_root = Path(project_root)
    if not isinstance(store, hist.HistoricalDataStore) or store.feed != 'sip':
        raise ValueError('Backtests brauchen den historischen SIP-Datenspeicher (kein IEX-Fallback).')
    fingerprint = replay.replay_fingerprint(store, c, s, settings, rs)
    done_records = []
    if resume is not None:
        folder = Path(resume)
        state = read_checkpoint(folder)
        if state is None:
            raise FileNotFoundError(f'Kein Checkpoint in {folder}.')
        if state.get('fingerprint') != fingerprint:
            raise ValueError('Code, Strategie- oder Laufparameter haben sich seit dem Start geändert; '
                             'dieser Lauf kann nicht fortgesetzt werden. Bitte einen neuen Backtest starten.')
        bt = BacktestSettings(**{k: tuple(v) if k == 'research_symbols' else v
                                 for k, v in state['settings'].items()})
        run_id = state['run_id']
        done_records = [dict(r, folder=Path(r['folder']) if r.get('folder') else None)
                        for r in state.get('records', []) if r.get('status') in ('SIMULATED', 'REUSED')]
        created = state['created_at']
    else:
        errors = validate(bt, store.now().tz_convert(NY).date())
        if errors:
            raise ValueError(' '.join(errors))
        run_id = new_run_id()
        folder = project_root / BACKTEST_FOLDER / run_id
        folder.mkdir(parents=True, exist_ok=False)
        created = pd.Timestamp.now(tz='UTC').isoformat()
    warnings, paused = [], False
    plan = plan_days(store, bt, c)
    days = plan['days']
    done_days = {r['date'] for r in done_records}
    todo = [d for d in days if d not in done_days]
    equity = float(done_records[-1]['end_equity']) if done_records else float(bt.initial_equity)
    universe = store.universe()
    symbols = sorted(set(universe.symbol) | {'QQQ'})
    research_symbols = [x for x in bt.research_symbols if x not in symbols]
    quality = 'BAR_PLUS_QUOTES'
    meta = dict(status='RUNNING', created_at=created, backtest_start=str(bt.start)[:10],
                backtest_end=str(bt.end)[:10], warmup_start=plan['warmup_start'],
                warmup_trading_days=warmup_days(c), trading_days=len(days),
                initial_equity=float(bt.initial_equity), historical_feed='SIP',
                historical_source=f'{hist.SOURCE} (bars, quotes, trades; feed=sip, adjustment=raw)',
                data_quality_level=quality, data_requirements=requirements(c, s, bt),
                universe=dict(provenance=store.universe_meta(), symbols=len(universe),
                              list=sorted(universe.symbol),
                              survivorship_bias='aktueller Nasdaq-100-Stand für alle Tage'),
                code_sha256=replay.code_hashes(include_cockpit=True),
                config_source=config_source, warnings=warnings, backtest_settings=asdict(bt),
                resumed=resume is not None)
    checkpoint = Checkpoint(folder, dict(
        run_id=run_id, research_schema_version=research.RESEARCH_SCHEMA_VERSION,
        bot_version=research.BOT_VERSION, created_at=created, status='CREATED', status_reason=None,
        period=dict(start=meta['backtest_start'], end=meta['backtest_end']), days_total=len(days),
        settings=asdict(bt), fingerprint=fingerprint, data_feed='SIP', data_quality_level=quality,
        strategy_parameters=asdict(c), scanner_parameters=asdict(s), dry_run_settings=asdict(settings),
        records=[record_for_state(r) for r in done_records],
        last_completed_date=done_records[-1]['date'] if done_records else None,
        current_equity=equity, cash=equity, completed_trades=sum(int(r.get('trades') or 0) for r in done_records),
        daily_results=[], research_state=dict(parts=sorted(done_days)), resumed_count=0))
    if resume is not None:
        old = read_checkpoint(folder) or {}
        checkpoint.state.update(daily_results=old.get('daily_results', []),
                                resumed_count=int(old.get('resumed_count', 0)) + 1)
    checkpoint.save(status='RUNNING')
    checkpoint.start_heartbeat()
    tracker.set(status='RUNNING', run_id=run_id, period=(meta['backtest_start'], meta['backtest_end']),
                days_total=len(days), days_done=len(done_records), universe=len(universe), equity=equity,
                started_wall=time.time(), trades=checkpoint.state['completed_trades'],
                signals=sum(int(d.get('signals_total') or 0) for d in checkpoint.state['daily_results']),
                day_lines=[d.get('line') for d in checkpoint.state['daily_results'] if d.get('line')])
    previous_progress = store.progress

    def data_progress(info):
        tracker.set(download=info)
        report(dict(info))
    store.progress = data_progress
    report(dict(phase='start', run_id=run_id, days=len(days), warmup=len(plan['warmup']),
                resumed=resume is not None, remaining=len(todo)))
    records = list(done_records)
    status, reason = 'RUNNING', None

    def after(record):
        return after_day(record, report, warnings, folder, run_id, store, c, s, quality, tracker,
                         checkpoint, records)
    try:
        if not days:
            status, reason = 'COMPLETED', 'NO_TRADING_DAYS'
            warnings.append('Im gewählten Zeitraum liegt laut Börsenkalender kein Handelstag.')
        elif todo:
            todo_plan = plan_days(store, BacktestSettings(todo[0], todo[-1], bt.initial_equity), c)
            if len(todo_plan['warmup']) < warmup_days(c):
                raise hist.DataUnavailable(f'Zu wenige Vortage für RVOL-Warmup ({len(todo_plan["warmup"])} '
                                           f'von {warmup_days(c)}).')
            if not store.settled(store.session(days[-1])):
                raise ValueError('Der letzte Tag ist noch nicht abgeschlossen (Schluss + 15 Min.).')
            data_phase(store, dict(todo_plan, days=todo), symbols, research_symbols, universe, report,
                       warnings, tracker)
            tracker.phase(5, 'Replay läuft – Tag für Tag, intern Minute für Minute')

            def before(day, _equity):
                if tracker.pause_requested:
                    raise PauseRequested('Pause angefordert.')
                tracker.day_started(day)
                check_day(store, day, universe, symbols, bt, report)
            replay.replay_days(
                store, project_root, c, s, todo, settings, rs, reuse_completed=True,
                cockpit=_Tee(tracker, cockpit), initial_equity=equity, carry_forward=True,
                stop_on_failure=bt.stop_on_day_failure, before_day=before, after_day=after,
                cockpit_mode='backtest', mode_label='BACKTEST_VIRTUAL_PORTFOLIO')
            paused = any(r.get('status') == 'PAUSED' for r in records)
            records[:] = [r for r in records if r.get('status') != 'PAUSED']
            status, reason = ('PAUSED', 'PAUSED_BY_USER') if paused else run_status(records, days)
            if bt.full_trades and status == 'COMPLETED':
                if full_trades(store, records, report, warnings):
                    quality = meta['data_quality_level'] = 'FULL_MARKET_DATA'
        else:
            status, reason = run_status(records, days)
    except hist.DataAccessDenied as exc:
        status, reason = 'INCOMPLETE_DATA', 'SIP_UNAVAILABLE'
        warnings.append(str(exc))
    except hist.HistoricalDataError as exc:
        status, reason = 'INCOMPLETE_DATA', type(exc).__name__
        warnings.append(f'{type(exc).__name__}: {exc}')
    except Exception as exc:          # never leave a run RUNNING without a trace
        status, reason = 'FAILED', type(exc).__name__
        warnings.append(f'{type(exc).__name__}: {str(exc)[:500]}')
        tracker.set(error=f'{type(exc).__name__}: {str(exc)[:300]}')
    except BaseException as exc:      # KeyboardInterrupt ('Ausführung unterbrechen') and the like
        checkpoint.stop_heartbeat()
        checkpoint.save(status='PAUSED', status_reason=f'INTERRUPTED_{type(exc).__name__}',
                        warnings=warnings)
        tracker.set(status='PAUSED', finished=True, error='Unterbrochen – mit RESUME fortsetzbar.')
        raise
    finally:
        store.progress = previous_progress
        try:
            store.flush()
        except Exception as exc:
            warnings.append(f'Cache konnte nicht geschrieben werden: {type(exc).__name__}')
    # every calendar day of the period appears, simulated or not (rule e)
    done = {r['date'] for r in records}
    export_records = sorted(records + [
        dict(date=day, status='NOT_RUN', start_equity=None, end_equity=None, folder=None,
             detail='noch nicht simuliert (Lauf pausiert)' if status == 'PAUSED'
             else 'nicht simuliert (Lauf vorher beendet)') for day in days if day not in done],
        key=lambda r: r['date'])
    meta.update(status=status, status_reason=reason, data_quality_level=quality,
                days=[record_for_state(r) for r in export_records], cache=store.report())
    tracker.phase(6, 'Research-Output wird geschrieben …')
    report(dict(phase='research', run_id=run_id))
    exported = research.export_run(folder, run_id, export_records, store, c, s, settings, rs, meta)
    checkpoint.stop_heartbeat()
    checkpoint.save(status=status, status_reason=reason, data_quality_level=quality,
                    records=[record_for_state(r) for r in records], warnings=warnings,
                    bundle=str(exported['bundle']))
    tracker.set(status=status, finished=True, result=dict(status=status, reason=reason,
                                                          bundle=str(exported['bundle'])))
    report(dict(phase='done', run_id=run_id, status=status, reason=reason, bundle=str(exported['bundle'])))
    return dict(run_id=run_id, folder=folder, status=status, status_reason=reason, bundle=exported['bundle'],
                summary=exported['summary'], records=export_records, warnings=warnings)


def data_phase(store, plan, symbols, research_symbols, universe, report, warnings, tracker=None):
    """Cache-first: calendar, daily bars (liquidity), minute bars incl. warmup, one quote probe."""
    tracker = tracker or BacktestProgress()
    tracker.phase(1, 'Prüfe, welche Handelstage schon im Cache liegen …')
    status = store.status(symbols, plan['warmup'] + plan['days'])
    tracker.set(cache_status=status)
    report(dict(phase='check_cache', **status))
    tracker.phase(2, f'Lade fehlende Tageskerzen und Minutenkerzen ({status["days_with_missing_bars"]} '
                     'Handelstage fehlen) …' if status['days_with_missing_bars'] else 'Alle Kerzen im Cache.')
    first = store.session(plan['warmup'][0])
    last_past = store.session(plan['days'][-1])
    store.daily(list(universe.symbol), first['open'].normalize(), last_past['close'])
    try:
        store.ensure_minute_bars(symbols, plan['warmup'] + plan['days'], label='download')
    except hist.DataAccessDenied:
        raise
    except hist.HistoricalDataError as exc:
        # Days whose download failed are caught by check_day (systemic) when reached.
        warnings.append(f'Download unvollständig: {exc}')
    if research_symbols:
        try:
            store.ensure_minute_bars(research_symbols, plan['days'], label='research')
        except hist.HistoricalDataError as exc:
            warnings.append(f'Research-Kontext {research_symbols} nicht verfügbar: {exc}')
    tracker.phase(3, 'Prüfe Datenzugang für Quotes und schreibe den Cache …')
    # Quotes are required by the strategy (spread/freshness). One probe before the run:
    # without access every signal would fail silently as QUOTE_DATA_ERROR.
    session = store.session(plan['days'][0])
    probe_at = session['open'] + pd.Timedelta(minutes=20)
    store.quotes('QQQ', probe_at - pd.Timedelta(seconds=10), probe_at, 1, 'desc')
    store.flush()
    tracker.phase(4, 'RVOL-Vergleichstage und Liquiditätsfilter werden je Tag aus dem Cache vorbereitet.')


def check_day(store, day, universe, symbols, bt, report):
    """Rule d) before a day is simulated: stop on systemic data problems."""
    store.flush()                  # persist quote/trade windows of the previous day
    try:
        store.ensure_minute_bars(symbols, [day])
    except hist.DataAccessDenied:
        raise
    except hist.HistoricalDataError as exc:
        raise SystemicDataError(f'{day}: SIP-Daten für den Tag nicht verfügbar – {exc}') from None
    coverage = store.day_bar_coverage(sorted(universe.symbol) + ['QQQ'], day)
    have = coverage[coverage.bars > 0]
    universe_share = have.symbol.isin(set(universe.symbol)).sum() / max(len(universe), 1)
    if 'QQQ' not in set(have.symbol):
        raise SystemicDataError(f'{day}: keine QQQ-Kerzen (Benchmark für jedes Signal).')
    if universe_share < bt.systemic_min_symbol_share:
        raise SystemicDataError(f'{day}: nur {universe_share:.0%} der Aktien mit Kerzen '
                                f'(Grenze {bt.systemic_min_symbol_share:.0%}).')
    report(dict(phase='day_start', day=day, symbols_with_bars=int(len(have))))


def hard_errors(folder):
    """Quote/trade access errors inside the engine (caught there per signal)."""
    found = []
    for item in research.raw_signal_items(folder):
        if isinstance(item, dict):
            for value in (item.get('error_type'), (item.get('virtual_entry_quote') or {}).get('error_type')):
                if value in research.HARD_DATA_ERRORS:
                    found.append(value)
    return found


def day_line(row):
    """Compact day summary: 2026-01-26 | 101/101 Symbols | 18 Signals | 3 Trades | +84 USD | Equity 10,284 USD"""
    def number(value, fmt):
        return '—' if value is None or pd.isna(value) else format(value, fmt)
    return (f'{row["date"]} | {number(row.get("symbols_with_valid_data"), "d")}/'
            f'{number(row.get("expected_symbols"), "d")} Symbols | {number(row.get("signals_total"), "d")} Signals | '
            f'{number(row.get("trades"), "d")} Trades | {number(row.get("net_pnl"), "+,.0f")} USD | '
            f'Equity {number(row.get("end_equity"), ",.0f")} USD | {row.get("day_status")}')


def after_day(record, report, warnings, folder=None, run_id=None, store=None, c=None, s=None,
              quality='BAR_PLUS_QUOTES', tracker=None, checkpoint=None, records=None):
    """After each day: hard-error check, research parts, checkpoint (auto-save), progress."""
    stop = False
    if record.get('folder'):
        errors = hard_errors(record['folder'])
        if 'DataAccessDenied' in errors:
            record['status'] = 'SIP_UNAVAILABLE'
            warnings.append(hist.warning_text('sip', f'{record["date"]}: Quotes/Trades während des Laufs '
                                                     'verweigert (HTTP 403) – Lauf gestoppt'))
            stop = True
        elif errors:
            warnings.append(f'{record["date"]}: {len(errors)} Datenabrufe mit Fehler ({sorted(set(errors))}).')
    if records is not None:
        records.append(record)
    if record.get('status') != 'PAUSED' and folder is not None:
        frames = research.export_day(folder, run_id, record, store, c, s, quality, 'sip')
        row = frames['daily'].iloc[0].to_dict() if len(frames['daily']) else dict(date=record['date'])
        row['day_status'] = record.get('status')
        line = day_line(row)
        if tracker is not None:
            tracker.day_finished(row, line)
        if checkpoint is not None:
            completed = [r for r in records if r.get('status') in ('SIMULATED', 'REUSED')]
            results = checkpoint.state.get('daily_results', []) + [dict(
                date=record['date'], status=record.get('status'), start_equity=record.get('start_equity'),
                end_equity=record.get('end_equity'), trades=record.get('trades'),
                signals_total=research.clean_json(row.get('signals_total')),
                data_coverage_pct=research.clean_json(row.get('data_coverage_pct')), line=line)]
            equity = record['end_equity'] if record.get('end_equity') is not None else \
                checkpoint.state['current_equity']
            checkpoint.save(records=[record_for_state(r) for r in completed],
                            last_completed_date=completed[-1]['date'] if completed else None,
                            current_equity=equity, cash=equity, daily_results=results,
                            completed_trades=sum(int(r.get('trades') or 0) for r in completed),
                            research_state=dict(parts=[r['date'] for r in completed]))
    report(dict(phase='day_done', **{k: v for k, v in record.items() if k != 'folder'}))
    return False if stop else None


def run_status(records, days):
    """(status, reason) with status in RUN_STATES."""
    states = [r.get('status') for r in records]
    if 'SIP_UNAVAILABLE' in states:
        return 'INCOMPLETE_DATA', 'SIP_UNAVAILABLE'
    if 'INCOMPLETE_DATA' in states:
        return 'INCOMPLETE_DATA', 'SYSTEMIC_DATA_PROBLEM'
    if 'UNRESOLVED_POSITION' in states:
        return 'INCOMPLETE_DATA', 'UNRESOLVED_POSITION'
    if 'FAILED' in states:
        return 'FAILED', 'DAY_FAILED'
    if len({r['date'] for r in records}) < len(days):
        return 'FAILED', 'DAYS_MISSING'
    return 'COMPLETED', None


def full_trades(store, records, report, warnings):
    """FULL_MARKET_DATA: all SIP trades of the symbols that produced signals (on request)."""
    ok = True
    for record in records:
        folder = record.get('folder')
        if not folder:
            continue
        symbols = sorted({item.get('symbol') for item in research.raw_signal_items(folder)
                          if isinstance(item, dict)} - {None})
        for symbol in symbols:
            try:
                store.ensure_session_trades(symbol, record['date'])
            except hist.HistoricalDataError as exc:
                ok = False
                warnings.append(f'Trades {symbol} {record["date"]}: {exc}')
        report(dict(phase='trades', day=record['date'], symbols=len(symbols)))
    return ok


In [ ]:
%%writefile /content/bot_accounts_v15.py
"""Persistent per-bot virtual accounts and logs (V1.5). Pure bookkeeping, no network.

Each bot has its own folder `<project>/v1_5/<BOT_ID>/` with
- `account.json`: start capital and every booked trading day (booked once per day),
- `<BOT_ID>_trades.csv`, `<BOT_ID>_signals.csv`, `<BOT_ID>_daily_performance.csv`:
  derived views, rebuilt from the run folders of the booked days.
Run folders are never modified; a restarted day gets a new run folder and only
the first complete run of a day is booked (no double booking)."""
from pathlib import Path
import json

import pandas as pd

TRADE_FIELDS = ('symbol', 'sector', 'qty', 'entry', 'entry_at_utc', 'signal_at_utc',
                'entry_quote_at_utc', 'exit_price_proxy', 'exit_at_utc', 'exit_reason',
                'exit_trigger_bar_minute_utc', 'pnl_usd', 'pnl_10bp_stress_usd',
                'mfe_usd', 'mae_usd', 'rank', 'score', 'rvol', 'stop',
                'unobserved_minutes', 'outcome_unreliable')
SIGNAL_FIELDS = ('symbol', 'decision_time_utc', 'processing_utc', 'feed', 'status',
                 'previous_status', 'shadow_status', 'paper_order', 'or_high', 'or_low',
                 'threshold', 'price_cap', 'close', 'rvol', 'rvol_status',
                 'complete_reference_days', 'current_volume', 'reference_average',
                 'relative_strength', 'spread', 'bid', 'ask', 'score', 'rank',
                 'rvol_component', 'strength_component', 'breakout_component',
                 'spread_component', 'ranking_result', 'opening_bars', 'coverage',
                 'previous_close_age_minutes', 'error_type')


def read_json(path, default=None):
    path = Path(path)
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else default


def write_json(path, content):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.incomplete')
    temporary.write_text(json.dumps(content, ensure_ascii=False, indent=2, default=str) + '\n',
                         encoding='utf-8')
    temporary.replace(path)


def write_csv(path, frame):
    path = Path(path)
    temporary = path.with_name(path.name + '.incomplete')
    frame.to_csv(temporary, index=False)
    temporary.replace(path)


def run_is_complete(folder):
    """A run can be booked when the close was processed and nothing stayed open."""
    folder = Path(folder)
    state = read_json(folder / 'shadow_state.json')
    coverage = read_json(folder / 'coverage.json')
    if not state or not coverage or (folder / 'failed.json').exists():
        return False
    return bool(coverage.get('planned_close_checked')) and not state.get('open_symbols')


class BotLedger:
    """Virtual account of one bot across days (e.g. SIP_DELAYED_SHADOW)."""

    def __init__(self, root, bot_id, start_cash=10000.0):
        self.bot_id = bot_id
        self.folder = Path(root) / bot_id
        self.folder.mkdir(parents=True, exist_ok=True)
        self.path = self.folder / 'account.json'
        state = read_json(self.path)
        if state is None:
            state = dict(bot_id=bot_id, start_cash=float(start_cash),
                         created_utc=pd.Timestamp.now(tz='UTC').isoformat(),
                         days={}, rejected_bookings=[])
            write_json(self.path, state)
        if state['bot_id'] != bot_id:
            raise ValueError('Kontodatei gehört zu einem anderen Bot.')
        self.state = state

    def save(self):
        write_json(self.path, self.state)

    def booked_days(self):
        return sorted(self.state['days'])

    def is_booked(self, day):
        return day in self.state['days']

    def start_equity(self, day):
        """Start capital plus the P&L of all booked days before `day`."""
        return self.state['start_cash'] + sum(
            entry['pnl_usd'] for d, entry in self.state['days'].items() if d < day)

    def equity(self):
        return self.state['start_cash'] + sum(e['pnl_usd'] for e in self.state['days'].values())

    def book(self, day, folder):
        """Book a complete run of `day` once. Returns True if booked now."""
        folder = Path(folder)
        if self.is_booked(day):
            self.state['rejected_bookings'].append(dict(
                date=day, folder=str(folder), reason='DAY_ALREADY_BOOKED',
                at_utc=pd.Timestamp.now(tz='UTC').isoformat()))
            self.save()
            return False
        if not run_is_complete(folder):
            self.state['rejected_bookings'].append(dict(
                date=day, folder=str(folder), reason='RUN_NOT_COMPLETE',
                at_utc=pd.Timestamp.now(tz='UTC').isoformat()))
            self.save()
            return False
        state = read_json(folder / 'shadow_state.json')
        status = read_json(folder / 'status.json', {})
        start = self.start_equity(day)
        self.state['days'][day] = dict(
            run_folder=str(folder), start_equity_usd=start,
            pnl_usd=float(state['realized_pnl_usd']),
            pnl_10bp_stress_usd=float(state['realized_pnl_10bp_stress_usd']),
            end_equity_usd=start + float(state['realized_pnl_usd']),
            trades=int(state['virtual_trades']),
            unreliable_trades=int(state.get('unreliable_virtual_trades', 0)),
            max_drawdown_proxy_usd=float(state['max_drawdown_proxy_usd']),
            data_degraded=bool(state['data_degraded']),
            status=status.get('status'), session_scope=status.get('session_scope'),
            booked_utc=pd.Timestamp.now(tz='UTC').isoformat())
        self.save()
        return True

    # ---- derived CSV logs ----
    def export_logs(self):
        trades, signals, daily = [], [], []
        for day in self.booked_days():
            entry = self.state['days'][day]
            folder = Path(entry['run_folder'])
            daily.append(dict(bot_id=self.bot_id, date=day, **{k: v for k, v in entry.items()
                                                                if k != 'run_folder'},
                              run_folder=entry['run_folder'],
                              return_pct=100 * entry['pnl_usd'] / entry['start_equity_usd']))
            for number, trade in enumerate(read_json(folder / 'shadow_trades.json', [])):
                trades.append(dict(bot_id=self.bot_id, date=day,
                                   trade_id=f'{self.bot_id}|{day}|{trade["symbol"]}|{number}',
                                   **{k: trade.get(k) for k in TRADE_FIELDS}))
            signals.extend(dict(bot_id=self.bot_id, date=day, **row)
                           for row in signal_rows(folder, self.bot_id, day))
        write_csv(self.folder / f'{self.bot_id}_trades.csv',
                  pd.DataFrame(trades, columns=['bot_id', 'date', 'trade_id', *TRADE_FIELDS]))
        write_csv(self.folder / f'{self.bot_id}_signals.csv',
                  pd.DataFrame(signals, columns=['bot_id', 'date', 'signal_id', *SIGNAL_FIELDS]))
        write_csv(self.folder / f'{self.bot_id}_daily_performance.csv', pd.DataFrame(daily))
        return self.folder


def signal_rows(folder, bot_id, day):
    """All detected signals of a run (historical, rejected, ranked, entered)."""
    rows = []
    for path in sorted((Path(folder) / 'signals').glob('*.json')):
        item = read_json(path)
        if not isinstance(item, dict) or 'symbol' not in item:
            continue   # per-minute summaries and skip markers
        if isinstance(item.get('paper_order'), dict):
            item['paper_order'] = json.dumps(item['paper_order'], ensure_ascii=False)
        rows.append(dict(signal_id=f'{bot_id}|{day}|{item["symbol"]}|{item.get("decision_time_utc")}',
                         **{k: item.get(k) for k in SIGNAL_FIELDS}))
    return rows


In [ ]:
%%writefile /content/sip_shadow_bot_v16.py
"""BOT 2 – SIP-Delayed-Bot (SIP_DELAYED_SHADOW). Virtual only, by construction.

This module never imports `alpaca.trading` and never receives a trading client.
It gets a market-data client (SIP, historical: >= 16 minutes old), the exchange
calendar and the stock universe as plain data, and a VirtualBroker that has no
network access and no order methods. All transactions are booked internally.

A (re)start re-simulates the session chronologically from the open ("catch-up")
and then follows real time with the lag; the day is booked into the virtual
account only once, from the first complete run.
"""
from dataclasses import replace
from datetime import date, datetime
from pathlib import Path
from types import SimpleNamespace

import pandas as pd
import us_orb_test_v02 as base
import orb_sim_v16 as sim
import orb_replay_v16 as replay
import orb_cockpit_v16 as cockpit_ui
import bot_accounts_v15 as accounts

BOT_ID = 'SIP_DELAYED_SHADOW'
DISPLAY_NAME = 'BOT 2 – SIP-Delayed-Bot'
LAG_MINUTES = 16
NY = 'America/New_York'


def calendar_payload(rows):
    """Alpaca calendar entries -> plain dicts (picklable, no client needed)."""
    return [dict(date=str(r.date), open=str(r.open), close=str(r.close)) for r in rows]


def calendar_objects(payload, start=None, end=None):
    start, end = str(start or '0000')[:10], str(end or '9999')[:10]
    return [SimpleNamespace(date=date.fromisoformat(r['date']),
                            open=datetime.fromisoformat(r['open']),
                            close=datetime.fromisoformat(r['close']))
            for r in payload if start <= r['date'] <= end]


class VirtualBroker:
    """Calendar and the bot's own virtual account. No network, no order methods."""

    def __init__(self, calendar, clock, equity):
        self._calendar, self.clock, self._equity = list(calendar), clock, float(equity)

    def get_clock(self):
        return SimpleNamespace(timestamp=self.clock.now().to_pydatetime(), is_open=None)

    def get_calendar(self, request):
        return calendar_objects(self._calendar, request.start, request.end)

    def get_account(self):
        return SimpleNamespace(equity=str(self._equity), cash=str(self._equity), currency='USD',
                               account_blocked=False, trading_blocked=False)

    def get_all_positions(self):
        return []

    def get_orders(self, filter=None):
        return []


class CatchUpClock(replay.DelayedClock):
    """Delayed clock that first replays the elapsed session quickly.

    While `now()` is behind (real time - lag) it runs simulated (sleep advances
    instantly); once it has caught up it follows real time minus the lag."""

    def __init__(self, lag_minutes=LAG_MINUTES, start=None, base=None):
        super().__init__(lag_minutes, base)
        self.simulated = None
        if start is not None and replay.as_utc(start) < self.target():
            self.simulated = replay.as_utc(start)

    def target(self):
        return self.base.now() - self.lag

    def catching_up(self):
        return self.simulated is not None

    @property
    def lag_seconds(self):
        return (self.base.now() - self.now()).total_seconds()

    def now(self):
        return self.simulated if self.simulated is not None else self.target()

    def _step(self, seconds):
        self.simulated += pd.Timedelta(seconds=seconds)
        if self.simulated >= self.target():
            self.simulated = None      # caught up: from now on real time minus lag

    def sleep(self, seconds):
        if self.simulated is None:
            self.base.sleep(seconds)
        else:
            self._step(seconds)

    def advance(self, seconds):
        if self.simulated is not None:
            self._step(seconds)


def session_for(calendar, day):
    rows = base.calendar_rows(calendar_objects(calendar, day, day))
    if len(rows) != 1:
        raise ValueError(f'{day} ist laut Börsenkalender kein Handelstag.')
    return rows[0]


def run_sip_shadow_day(data_client, project_root, c, s, calendar, universe,
                       cockpit=None, lag_minutes=LAG_MINUTES, base_clock=None,
                       rs=replay.ReplaySettings(), start_cash=10000.0):
    """Today's session for BOT 2; returns the run folder (or the booked one)."""
    root = Path(project_root) / 'v1_5'
    ledger = accounts.BotLedger(root, BOT_ID, start_cash)
    probe = replay.DelayedClock(lag_minutes, base_clock)
    day = probe.now().tz_convert(NY).strftime('%Y-%m-%d')
    session = session_for(calendar, day)
    if ledger.is_booked(day):
        print(f'{DISPLAY_NAME}: {day} ist bereits vollständig gebucht; keine Doppelbuchung.')
        return Path(ledger.state['days'][day]['run_folder'])
    clock = CatchUpClock(lag_minutes, session['open'] - pd.Timedelta(minutes=10), base_clock)
    if clock.catching_up():
        print(f'{DISPLAY_NAME}: Neustart/Spätstart – der Tag wird ab Börsenöffnung '
              'chronologisch nachsimuliert, dann folgt der Bot mit '
              f'{lag_minutes} Min. Versatz.')
    equity = ledger.start_equity(day)
    settings = replace(sim.DryRunSettings(), feed='sip', initial_cash=equity,
                       print_every_minute=False)
    info = dict(kind='sip_delayed_shadow', bot_id=BOT_ID, display_name=DISPLAY_NAME,
                version=replay.VERSION, lag_minutes=lag_minutes,
                mode='SIP_DELAYED_SHADOW_VIRTUAL', output_folder=f'v1_5/{BOT_ID}/runs',
                cockpit_mode='sip_shadow', access_result='DELAYED_HISTORICAL_SIP',
                bar_delay_seconds=rs.bar_delay_seconds, catch_up_from_open=clock.catching_up(),
                note='SIMULIERT: historische SIP-Daten mit Versatz; keine Broker-Orders.')
    out = sim.run_dryrun(VirtualBroker(calendar, clock, equity),
                         replay.DelayedData(data_client, clock, rs.bar_delay_seconds),
                         project_root, replace(c, test_date=day), s, settings, clock=clock,
                         universe_loader=lambda broker: universe, replay=info, cockpit=cockpit)
    if ledger.book(day, out):
        print(f'{DISPLAY_NAME}: Tag {day} im virtuellen Konto gebucht.')
    ledger.export_logs()
    return out


def alpaca_market_data(api_key, secret_key):
    """Market-data client only (StockHistoricalDataClient cannot place orders)."""
    from alpaca.data.historical import StockHistoricalDataClient
    return StockHistoricalDataClient(api_key=api_key, secret_key=secret_key)


def child_main(data_factory, factory_kwargs, project_root, c, s, calendar, universe_records,
               cockpit_folder, lag_minutes=LAG_MINUTES, clock_factory=None):
    """Entry point of BOT 2 in its own process (DUAL_MODE); only picklable inputs.

    `data_factory` ('module:function') builds the market-data client;
    `clock_factory` (tests only) builds the base clock."""
    import importlib

    def load(spec):
        module, name = spec.split(':')
        return getattr(importlib.import_module(module), name)
    data_client = load(data_factory)(**factory_kwargs)
    base_clock = load(clock_factory)() if clock_factory else None
    cockpit = cockpit_ui.Cockpit(display_enabled=False,
                                 latest_path=Path(cockpit_folder) / 'cockpit_latest.html',
                                 latest_json_path=Path(cockpit_folder) / 'cockpit_latest.json')
    run_sip_shadow_day(data_client, project_root, c, s, calendar,
                       pd.DataFrame(universe_records), cockpit=cockpit, lag_minutes=lag_minutes,
                       base_clock=base_clock)


In [ ]:
%%writefile /content/paper_orders_v16.py
"""Alpaca PAPER order gateway for BOT 1 (IEX_REALTIME_PAPER). Only BOT 1 imports this.

V1.6: every order path first checks bot_mode_v16 (main mode must be PAPER).

Every order path runs through `_guard()`:
- the client must point at https://paper-api.alpaca.markets (never live),
- the account must be ACTIVE, USD, not blocked and match the approved account,
- the approval must be exactly `approval_token(account_number, day)` for today,
- positions/orders are reconciled with Alpaca right before each order.
Without a valid approval the gateway stays disabled and sends nothing.

Orders: IOC limit buy at the fresh ask (partial fills allowed), then a broker-side
DAY trailing stop for the filled quantity; flatten at the planned close and
confirm position 0. Client order ids are deterministic per day and symbol, so a
restart can never submit a second entry for the same signal. Missed signals are
never executed later.
"""
from pathlib import Path
import json
import time

import pandas as pd
import us_orb_test_v02 as base
import bot_mode_v16 as bot_mode

PAPER_URL = 'https://paper-api.alpaca.markets'
PREFIX = 'ORB15'
TERMINAL = {'filled', 'canceled', 'expired', 'rejected', 'done_for_day', 'stopped', 'suspended'}


def approval_token(account_number, day):
    """The exact text a user must set to allow paper orders on `day` for this account."""
    return f'PAPER-ORDERS-OK {account_number} {day}'


def client_id(day, symbol, kind):
    return f'{PREFIX}-{day}-{symbol}-{kind}'


def value(item):
    return getattr(item, 'value', item)


class PaperOrderGateway:
    """Order hook for sim.run_dryrun (BOT 1 only)."""

    def __init__(self, trading_client, approval, day, config, scanner_config, sectors,
                 clock, log_folder, fill_timeout_seconds=10, sleep=time.sleep):
        self.client, self.approval, self.day = trading_client, approval, day
        self.c, self.s, self.sectors, self.clock = config, scanner_config, dict(sectors), clock
        self.log_path = Path(log_folder) / 'IEX_REALTIME_PAPER_orders.jsonl'
        self.fill_timeout_seconds, self.sleep = fill_timeout_seconds, sleep
        self.enabled, self.disabled_reason = False, 'NOT_STARTED'
        self.orders_sent, self.start_equity, self.flat_confirmed = 0, None, None
        self.account_number = None

    # ---------- logging ----------
    def log(self, kind, **fields):
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        record = dict(kind=kind, processing_utc=pd.Timestamp.now(tz='UTC').isoformat(),
                      market_time_utc=self.clock.now().isoformat(), day=self.day, **fields)
        with self.log_path.open('a', encoding='utf-8') as handle:
            handle.write(json.dumps(record, ensure_ascii=False, default=str) + '\n')
        return record

    def disable(self, reason):
        self.enabled, self.disabled_reason = False, reason
        self.log('ORDERS_DISABLED', reason=reason)

    # ---------- safety ----------
    def _paper_endpoint(self):
        url = str(value(getattr(self.client, '_base_url', ''))).rstrip('/')
        return url == PAPER_URL and getattr(self.client, '_sandbox', False) is True

    def _guard(self):
        """Raise unless every precondition for a paper order holds right now."""
        bot_mode.assert_orders_allowed()   # V1.6: only the main mode PAPER may order
        if not self._paper_endpoint():
            raise PermissionError('Kein Alpaca-Paper-Endpunkt – Orders gesperrt.')
        account = self.client.get_account()
        if value(account.status) != 'ACTIVE' or account.account_blocked or \
                account.trading_blocked or account.currency != 'USD':
            raise PermissionError('Paper-Konto nicht aktiv/gesperrt/nicht USD.')
        if self.account_number is not None and account.account_number != self.account_number:
            raise PermissionError('Anderes Paper-Konto als freigegeben.')
        if self.approval != approval_token(account.account_number, self.day):
            raise PermissionError('Keine gültige Order-Freigabe für heute und dieses Konto.')
        return account

    def _own(self, order):
        return str(getattr(order, 'client_order_id', '') or '').startswith(f'{PREFIX}-{self.day}-')

    def reconcile(self):
        """Current Alpaca state; every own position must have an own open stop."""
        from alpaca.trading.requests import GetOrdersRequest
        from alpaca.trading.enums import QueryOrderStatus
        positions = {p.symbol: p for p in self.client.get_all_positions()}
        orders = self.client.get_orders(filter=GetOrdersRequest(status=QueryOrderStatus.OPEN, limit=500))
        if len(orders) >= 500:
            raise PermissionError('Offene Orderliste möglicherweise abgeschnitten.')
        return positions, orders

    # ---------- run_dryrun hook interface ----------
    def on_start(self, out, clock):
        self.clock = clock
        try:
            if not self.approval:
                raise PermissionError('Keine Order-Freigabe gesetzt.')
            account = self._guard()
            self.account_number = account.account_number
            positions, orders = self.reconcile()
            foreign = [p for p in positions if not any(
                self._own(o) and o.symbol == p for o in orders)]
            foreign_orders = [o for o in orders if not self._own(o)]
            if foreign or foreign_orders:
                raise PermissionError('Paper-Konto hat fremde Positionen/Orders; bitte zuerst bereinigen.')
            self.start_equity = float(account.equity)
            self.enabled, self.disabled_reason = True, None
            self.log('ORDERS_ENABLED', account_number=account.account_number,
                     equity=self.start_equity, cash=float(account.cash),
                     adopted_positions=sorted(positions), open_own_orders=len(orders))
            self.ensure_stops(positions, orders)
        except Exception as exc:
            self.disable(f'{type(exc).__name__}: {exc}')

    def on_signal(self, decision, now, signal, quality):
        """Place a paper entry for a fresh, ranked watchlist signal (or say why not)."""
        symbol = signal['symbol']
        if not self.enabled:
            return dict(status='NOT_SENT', reason=self.disabled_reason)
        try:
            if (pd.Timestamp(now) - pd.Timestamp(decision)).total_seconds() >= self.s.signal_ttl_seconds:
                return self._skip(symbol, 'SIGNAL_EXPIRED')
            if quality.get('quote_result') != 'PASS':
                return self._skip(symbol, 'QUOTE_' + str(quality.get('quote_result')))
            quote_age = (pd.Timestamp(now) - pd.Timestamp(quality['quote_time_utc'])).total_seconds()
            if not 0 <= quote_age <= 2:
                return self._skip(symbol, 'QUOTE_NOT_FRESH')
            account = self._guard()
            positions, orders = self.reconcile()
            entry_id = client_id(self.day, symbol, 'entry')
            if self._order_exists(entry_id):
                return self._skip(symbol, 'ENTRY_ALREADY_SENT_TODAY')
            if symbol in positions or any(o.symbol == symbol for o in orders):
                return self._skip(symbol, 'SYMBOL_ALREADY_HELD_OR_PENDING')
            sectors = [self.sectors.get(p) for p in positions]
            gate = base.entry_gate(self.start_equity, float(account.equity), 0, sectors,
                                   signal['sector'], False, self.c)
            if not gate['allowed']:
                return self._skip(symbol, ';'.join(gate['reasons']))
            risk = sum(abs(float(p.qty)) * float(p.current_price) * float(self.c.trailing)
                       for p in positions.values())
            ask = float(quality['ask'])
            size = base.size_position(float(account.equity), float(account.cash), risk, ask, self.c)
            if size['qty'] <= 0:
                return self._skip(symbol, 'NO_CAPITAL_OR_RISK_BUDGET')
            return self._enter(symbol, int(size['qty']), ask, entry_id, signal)
        except Exception as exc:
            self.log('ENTRY_ERROR', symbol=symbol, error=f'{type(exc).__name__}: {exc}')
            return dict(status='ERROR', reason=type(exc).__name__)

    def on_minute(self, decision, now):
        if not self.enabled:
            return
        try:
            positions, orders = self.reconcile()
            self.ensure_stops(positions, orders)
        except Exception as exc:
            self.log('RECONCILE_ERROR', error=f'{type(exc).__name__}: {exc}')

    def on_planned_close(self, now):
        if self.account_number is not None:
            self.flatten('PLANNED_CLOSE')

    def on_stop(self, now):
        """End of run (normal or abort): never leave own paper positions behind."""
        if self.account_number is None:
            return
        positions, _ = self.reconcile()
        if positions:
            self.flatten('RUN_STOPPED')
        elif self.flat_confirmed is None:
            self.flat_confirmed = True
            self.log('POSITION_ZERO_CONFIRMED', reason='RUN_STOPPED')

    # ---------- order mechanics ----------
    def _skip(self, symbol, reason):
        self.log('ENTRY_SKIPPED', symbol=symbol, reason=reason)
        return dict(status='NOT_SENT', reason=reason)

    def _order_exists(self, client_order_id):
        try:
            self.client.get_order_by_client_id(client_order_id)
            return True
        except Exception:
            return False

    def _wait_terminal(self, client_order_id):
        deadline = time.monotonic() + self.fill_timeout_seconds
        while True:
            order = self.client.get_order_by_client_id(client_order_id)
            if value(order.status) in TERMINAL or time.monotonic() >= deadline:
                return order
            self.sleep(0.5)

    def _enter(self, symbol, qty, ask, entry_id, signal):
        from alpaca.trading.requests import LimitOrderRequest
        from alpaca.trading.enums import OrderSide, TimeInForce
        self._guard()
        self.client.submit_order(LimitOrderRequest(
            symbol=symbol, qty=qty, side=OrderSide.BUY, time_in_force=TimeInForce.IOC,
            limit_price=round(ask, 2), client_order_id=entry_id))
        self.orders_sent += 1
        order = self._wait_terminal(entry_id)
        filled = int(float(order.filled_qty or 0))
        price = float(order.filled_avg_price) if filled else None
        self.log('ENTRY_ORDER', symbol=symbol, qty=qty, limit=round(ask, 2), client_order_id=entry_id,
                 status=value(order.status), filled_qty=filled, filled_avg_price=price,
                 signal=signal)
        if filled:
            self._place_stop(symbol, filled)
        return dict(status=value(order.status), filled_qty=filled, filled_avg_price=price,
                    client_order_id=entry_id)

    def _place_stop(self, symbol, qty):
        from alpaca.trading.requests import TrailingStopOrderRequest
        from alpaca.trading.enums import OrderSide, TimeInForce
        stop_id = client_id(self.day, symbol, 'stop')
        if self._order_exists(stop_id):
            stop_id = client_id(self.day, symbol, f'stop{int(time.time())}')
        self._guard()
        self.client.submit_order(TrailingStopOrderRequest(
            symbol=symbol, qty=qty, side=OrderSide.SELL, time_in_force=TimeInForce.DAY,
            trail_percent=float(self.c.trailing) * 100, client_order_id=stop_id))
        self.orders_sent += 1
        self.log('TRAILING_STOP_ORDER', symbol=symbol, qty=qty, client_order_id=stop_id,
                 trail_percent=float(self.c.trailing) * 100)

    def ensure_stops(self, positions, orders):
        """Every own long position needs a broker-side sell stop for its quantity."""
        for symbol, position in positions.items():
            covered = sum(float(o.qty) for o in orders if o.symbol == symbol and
                          value(o.side) == 'sell' and self._own(o))
            missing = int(float(position.qty) - covered)
            if missing > 0:
                self.log('STOP_MISSING', symbol=symbol, missing_qty=missing)
                self._place_stop(symbol, missing)

    def flatten(self, reason):
        """Cancel own orders, close all positions, confirm position 0."""
        self._guard()
        self.client.cancel_orders()
        self.client.close_all_positions(cancel_orders=True)
        self.orders_sent += 1
        deadline = time.monotonic() + max(self.fill_timeout_seconds, 30)
        while True:
            positions = self.client.get_all_positions()
            if not positions:
                self.flat_confirmed = True
                self.log('POSITION_ZERO_CONFIRMED', reason=reason)
                return True
            if time.monotonic() >= deadline:
                self.flat_confirmed = False
                self.log('FLATTEN_NOT_CONFIRMED', reason=reason,
                         remaining=[p.symbol for p in positions])
                raise RuntimeError('Paper-Positionen nach Glattstellung nicht 0.')
            self.sleep(1)


In [ ]:
%%writefile /content/iex_paper_bot_v16.py
"""BOT 1 – IEX-Echtzeit-Bot (IEX_REALTIME_PAPER).

Runs the live loop on real-time IEX data. Always: an internal IEX simulation
with the same execution/cost model as BOT 2 (performance series 2, own virtual
account IEX_REALTIME_PAPER_SIM). Only with a valid approval for today and the
paper account: real Alpaca PAPER orders through paper_orders_v16 (series 1).
Decisions are never taken from delayed or replayed data (run_dryrun refuses an
order hook there), and missed signals are never ordered later.
"""
from dataclasses import replace
from pathlib import Path
import json

import pandas as pd
import orb_sim_v16 as sim
import bot_accounts_v15 as accounts
import paper_orders_v16 as paper

BOT_ID = 'IEX_REALTIME_PAPER'
SIM_ID = 'IEX_REALTIME_PAPER_SIM'
DISPLAY_NAME = 'BOT 1 – IEX-Echtzeit-Bot'
NY = 'America/New_York'


def run_iex_day(trading_client, data_client, project_root, c, s, universe, approval=None,
                cockpit=None, clock=sim.SYSTEM_CLOCK, gateway_factory=paper.PaperOrderGateway,
                start_cash=10000.0):
    """Today's IEX session; returns (run folder, gateway or None)."""
    root = Path(project_root) / 'v1_5'
    sim_ledger = accounts.BotLedger(root, SIM_ID, start_cash)
    day = clock.now().tz_convert(NY).strftime('%Y-%m-%d')
    settings = replace(sim.DryRunSettings(), feed='iex', print_every_minute=False,
                       initial_cash=sim_ledger.start_equity(day))
    gateway = None
    if approval:   # without any approval text the order gateway is not even created
        gateway = gateway_factory(trading_client, approval, day, c, s,
                                  dict(zip(universe.symbol, universe.sector)), clock, root / BOT_ID)
    out = sim.run_dryrun(trading_client, data_client, project_root, replace(c, test_date=day), s,
                         settings, clock=clock, universe_loader=lambda broker: universe,
                         cockpit=cockpit, order_hook=gateway,
                         output_folder=f'v1_5/{BOT_ID}/runs', cockpit_mode='iex_live')
    sim_ledger.book(day, out)
    sim_ledger.export_logs()
    record_paper_day(trading_client, gateway, root / BOT_ID, day, out)
    return out, gateway


def record_paper_day(trading_client, gateway, folder, day, run_folder):
    """Series 1 (actual paper fills) and the BOT 1 signal log."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    signals = pd.DataFrame(accounts.signal_rows(run_folder, BOT_ID, day))
    previous = folder / f'{BOT_ID}_signals.csv'
    if previous.exists() and previous.stat().st_size > 1:
        old = pd.read_csv(previous)
        signals = pd.concat([old[old.date != day], signals.assign(date=day, bot_id=BOT_ID)],
                            ignore_index=True)
    else:
        signals = signals.assign(date=day, bot_id=BOT_ID)
    accounts.write_csv(previous, signals)
    days_path = folder / 'paper_days.json'
    days = accounts.read_json(days_path, {})
    if gateway is None or gateway.account_number is None:
        days.setdefault(day, dict(orders_enabled=False, reason=getattr(gateway, 'disabled_reason',
                                                                       'NO_APPROVAL'),
                                  run_folder=str(run_folder)))
    else:
        account = trading_client.get_account()
        days[day] = dict(orders_enabled=True, run_folder=str(run_folder),
                         start_equity_usd=gateway.start_equity, end_equity_usd=float(account.equity),
                         pnl_usd=float(account.equity) - gateway.start_equity,
                         orders_sent=gateway.orders_sent, flat_confirmed=gateway.flat_confirmed)
    accounts.write_json(days_path, days)
    rows = [dict(bot_id=BOT_ID, date=d, **{k: v for k, v in e.items()}) for d, e in sorted(days.items())]
    accounts.write_csv(folder / f'{BOT_ID}_daily_performance.csv', pd.DataFrame(rows))
    fills = []
    log = folder / f'{BOT_ID}_orders.jsonl'
    if log.exists():
        for line in log.read_text(encoding='utf-8').splitlines():
            record = json.loads(line)
            if record['kind'] in ('ENTRY_ORDER', 'TRAILING_STOP_ORDER', 'POSITION_ZERO_CONFIRMED',
                                  'FLATTEN_NOT_CONFIRMED', 'ENTRY_SKIPPED', 'ORDERS_DISABLED'):
                fills.append({k: (json.dumps(v, ensure_ascii=False) if isinstance(v, (dict, list)) else v)
                              for k, v in record.items()})
    if gateway is not None and gateway.account_number is not None:
        fills.extend(paper_fills(trading_client, day, gateway))
    accounts.write_csv(folder / f'{BOT_ID}_trades.csv', pd.DataFrame(fills))


def paper_fills(trading_client, day, gateway):
    """Actual Alpaca paper fills of today for the symbols BOT 1 traded (entries,
    trailing stops and the closing orders of close_all_positions)."""
    from alpaca.trading.requests import GetOrdersRequest
    from alpaca.trading.enums import QueryOrderStatus
    traded = {line['symbol'] for line in map(json.loads,
              gateway.log_path.read_text(encoding='utf-8').splitlines())
              if line['kind'] == 'ENTRY_ORDER' and line.get('filled_qty')} if gateway.log_path.exists() else set()
    if not traded:
        return []
    after = pd.Timestamp(f'{day} 09:00', tz=NY).tz_convert('UTC').to_pydatetime()
    orders = trading_client.get_orders(filter=GetOrdersRequest(
        status=QueryOrderStatus.CLOSED, after=after, limit=500, symbols=sorted(traded)))
    return [dict(kind='ALPACA_FILL', day=day, symbol=o.symbol, side=paper.value(o.side),
                 order_type=paper.value(getattr(o, 'order_type', getattr(o, 'type', None))),
                 filled_qty=float(o.filled_qty or 0),
                 filled_avg_price=float(o.filled_avg_price) if o.filled_avg_price else None,
                 filled_at=str(getattr(o, 'filled_at', None)), client_order_id=o.client_order_id,
                 status=paper.value(o.status))
            for o in orders if float(o.filled_qty or 0) > 0]


In [ ]:
%%writefile /content/dual_bot_v16.py
"""V1.6 (unchanged V1.5 orchestration) – Dual-Bot-System (IEX + SIP): modes, orchestration,
dual cockpit, comparison. V1.6 cockpit modes map onto these: OBSERVE = SIP_ONLY,
DRY RUN / PAPER = IEX_ONLY, with the SIP comparison bot = DUAL_MODE (bot_mode_v16).

Modes (delivery state: SIP_ONLY):
    SIP_ONLY   BOT 1 off,                    BOT 2 active (virtual)
    IEX_ONLY   BOT 1 active (orders only with a valid approval), BOT 2 off
    DUAL_MODE  both; BOT 2 runs in its own process so it can never block BOT 1

BOT 2 never receives a trading client: the main process passes it the exchange
calendar and the stock universe as plain data and the market-data keys only.
"""
from pathlib import Path
import json
import multiprocessing
import time

import pandas as pd
import orb_sim_v16 as sim
import orb_cockpit_v16 as cockpit_ui
import bot_accounts_v15 as accounts
import sip_shadow_bot_v16 as bot2

VERSION = 'V1.6.1 – Dual-Bot-System (IEX + SIP)'
BOT1_ID, BOT1_NAME = 'IEX_REALTIME_PAPER', 'BOT 1 – IEX-Echtzeit-Bot'
BOT2_ID, BOT2_NAME = bot2.BOT_ID, bot2.DISPLAY_NAME
MODES = {'SIP_ONLY': (False, True), 'IEX_ONLY': (True, False), 'DUAL_MODE': (True, True)}
DELIVERY_MODE = 'SIP_ONLY'
NY = 'America/New_York'


def resolve_mode(mode):
    """Unknown or missing modes fall back to the non-trading delivery state."""
    if mode not in MODES:
        print(f'Unbekannter Modus {mode!r} – verwende {DELIVERY_MODE}.')
        mode = DELIVERY_MODE
    bot1_active, bot2_active = MODES[mode]
    return dict(mode=mode, bot1_active=bot1_active, bot2_active=bot2_active)


def prepare_shared(trading_client, day):
    """Read-only preparation in the main process: calendar and one common universe."""
    from datetime import date, timedelta
    from alpaca.trading.requests import GetCalendarRequest
    today = date.fromisoformat(day)
    calendar = bot2.calendar_payload(trading_client.get_calendar(GetCalendarRequest(
        start=today - timedelta(days=90), end=today + timedelta(days=7))))
    universe = sim.fresh_universe(trading_client)
    return calendar, universe


# ---------------------------------------------------------------- dual cockpit
class DualCockpit:
    """One page with three areas: BOT 1, BOT 2 (SIMULIERT) and the comparison."""

    def __init__(self, root, mode_info, display=True, latest_path=None):
        self.root, self.info, self.display = Path(root), mode_info, display
        self.latest_path = Path(latest_path) if latest_path is not None else None
        self.cockpits = {bot: cockpit_ui.Cockpit(
            display_enabled=False, latest_json_path=self.root / bot / 'cockpit_latest.json')
            for bot in (BOT1_ID, BOT2_ID)}
        self.snapshots, self.gateway = {}, None
        self.latest_browser_page = cockpit_ui.WAITING_PAGE
        self.latest_snapshot_json = '{}'
        self.handle, self.write_errors = None, 0

    def view(self, bot_id):
        return BotView(self, bot_id)

    def read_other(self, bot_id):
        """Snapshot written by a bot in another process (DUAL_MODE)."""
        path = self.root / bot_id / 'cockpit_latest.json'
        try:
            data = json.loads(path.read_text(encoding='utf-8'))
            if data.get('updated_at_utc'):
                self.snapshots[bot_id] = data
        except (OSError, ValueError):
            pass

    def refresh(self, read_files=()):
        for bot_id in read_files:
            self.read_other(bot_id)
        page = self.render()
        self.latest_browser_page = cockpit_ui.browser_page(page)
        self.latest_snapshot_json = json.dumps(self.snapshots, ensure_ascii=False, default=str)
        if self.latest_path is not None:
            try:
                cockpit_ui.write_atomic(self.latest_path, self.latest_browser_page)
            except OSError:
                self.write_errors += 1
        if self.display:
            from IPython.display import HTML, display
            if self.handle is None:
                self.handle = display(HTML(page), display_id=True)
            else:
                self.handle.update(HTML(page))
        return page

    def render(self):
        from html import escape
        style, sections = None, []
        for bot_id, name, active, badge in (
                (BOT1_ID, BOT1_NAME, self.info['bot1_active'], 'IEX · Echtzeit'),
                (BOT2_ID, BOT2_NAME, self.info['bot2_active'], 'SIMULIERT · SIP verzögert')):
            data = self.snapshots.get(bot_id)
            head = (f'<h2 class="dual-head">{escape(name)} <small>({bot_id}) · {escape(badge)}'
                    f'</small></h2>')
            if not active:
                sections.append(f'<div class="orb-cockpit">{head}<p class="muted">Deaktiviert im '
                                f'Modus {escape(self.info["mode"])} · keine Alpaca-Orders.</p></div>')
                continue
            if data is None:
                sections.append(f'<div class="orb-cockpit">{head}<p class="muted">Warte auf den '
                                'ersten Stand …</p></div>')
                continue
            section_style, body = cockpit_ui.page_parts(cockpit_ui.render_html(data))
            style = style or section_style
            body = body.replace('id="orb-stand"', 'class="orb-stand-section"')
            body = body.replace('<h2>US-Aktien-Bot · virtuelles Cockpit</h2>', head, 1)
            if bot_id == BOT1_ID:
                body = body.replace('<div class="tiles">', self.order_status() + '<div class="tiles">', 1)
            sections.append(body)
        style = style or base_style()
        now = pd.Timestamp.now(tz='UTC')
        quiet = all(d.get('stage') in cockpit_ui.QUIET_STAGES for d in self.snapshots.values()) \
            if self.snapshots else True
        header = (f'<div class="orb-cockpit"><h2>US-Aktien-Bot {escape(VERSION)}</h2>'
                  f'<div class="subtitle" id="orb-stand" data-updated-ms="{int(now.timestamp()*1000)}" '
                  f'data-quiet="{"1" if quiet else "0"}" data-replay="0" data-lag-ms="0">Modus '
                  f'<strong>{escape(self.info["mode"])}</strong> · Stand {now.tz_convert(NY):%H:%M:%S} NY '
                  '(Verarbeitungszeit)</div><div class="tiles"></div></div>')
        return ('<!doctype html><html lang="de"><head><meta charset="utf-8"><style>' + style +
                '.dual-head small{font-size:13px;font-weight:400;color:#58677a}'
                '.orb-cockpit{margin-bottom:14px}</style></head><body>' + header +
                ''.join(sections) + self.comparison_section() + '</body></html>')

    def order_status(self):
        from html import escape
        gateway = self.gateway
        if gateway is None:
            text, tone = 'Alpaca-Paper-Orders: AUS (keine Freigabe) – nur IEX-Simulation', 'neutral'
        elif gateway.enabled:
            text = (f'Alpaca-Paper-Orders: FREIGEGEBEN · Konto {gateway.account_number} · '
                    f'Start-Equity {gateway.start_equity:,.2f} USD · gesendete Orders {gateway.orders_sent}')
            tone = 'bad'
        else:
            text, tone = f'Alpaca-Paper-Orders: AUS – {gateway.disabled_reason}', 'neutral'
        if gateway is not None and gateway.flat_confirmed is not None:
            text += ' · Glattstellung: ' + ('Position 0 bestätigt' if gateway.flat_confirmed
                                           else 'NICHT bestätigt – manuell prüfen')
        return f'<p><span class="{tone}">{escape(text)}</span></p>'

    def comparison_section(self):
        from html import escape
        rows, decisions = [], []
        for bot_id, source in ((BOT1_ID, 'IEX Echtzeit (Simulation, Reihe 2)'),
                               (BOT2_ID, 'SIP verzögert (SIMULIERT, Reihe 3)')):
            data = self.snapshots.get(bot_id)
            ledger_id = BOT1_ID + '_SIM' if bot_id == BOT1_ID else BOT2_ID
            account = accounts.read_json(self.root / ledger_id / 'account.json')
            total = (sum(d['pnl_usd'] for d in account['days'].values()) if account else 0.0)
            if data is None:
                rows.append((escape(source), '—', '—', '—', '—', '—', '—', f'{total:,.2f} USD'))
                continue
            start = data.get('start_equity_usd') or 0
            day_pnl = (data.get('realized_pnl_proxy_usd') or 0) + (data.get('open_pnl_proxy_usd') or 0)
            if data.get('checked_decision_utc'):
                decisions.append(pd.Timestamp(data['checked_decision_utc']))
            rows.append((escape(source), f'{data.get("equity_proxy_usd", 0):,.2f} USD',
                         str(data.get('signal_total', 0)), str(data.get('closed_count', 0)),
                         f'{data.get("wins", 0)} / {data.get("losses", 0)}',
                         f'{day_pnl:,.2f} USD', f'{100 * day_pnl / start:+.2f} %' if start else '—',
                         f'{total:,.2f} USD'))
        both = self.info['bot1_active'] and self.info['bot2_active']
        common = min(decisions) if both and len(decisions) == 2 else None
        if not both:
            note = 'Vergleich erst im DUAL_MODE; hier nur der aktive Bot.'
        elif common is None:
            note = 'Noch kein gemeinsam verarbeiteter Marktzeitpunkt.'
        else:
            gap = (max(decisions) - min(decisions)).total_seconds() / 60
            note = (f'Zuletzt gemeinsam vollständig verarbeiteter Marktzeitpunkt: '
                    f'{common.tz_convert(NY):%H:%M} NY. ' +
                    (f'VORLÄUFIG: SIP liegt {gap:.0f} Min. zurück – Tageszahlen sind noch nicht '
                     'vergleichbar; abschließender Vergleich nach vollständiger Verarbeitung beider Bots.'
                     if gap > 1 else 'Beide Bots auf gleichem Marktzeitpunkt.'))
        body = cockpit_ui.table(('Bot / Datenquelle', 'Equity (virtuell)', 'Erkannte Signale',
                                 'Trades', 'Gewinn / Verlust', 'Tages-P&L', 'Tages-P&L %',
                                 'Gesamt-P&L (gebuchte Tage)'), rows, 'Noch keine Daten.')
        return (f'<div class="orb-cockpit"><h2>IEX vs. SIP – Vergleich</h2>'
                f'<p class="muted">{escape(note)}</p>{body}<div class="note">Reihe 1 (tatsächliche '
                'Alpaca-Paper-Fills von BOT 1) steht in IEX_REALTIME_PAPER_daily_performance.csv. '
                'Reihen 2 und 3 nutzen dasselbe Ausführungs- und Kostenmodell; einige Testtage '
                'beweisen keine langfristige Profitabilität.</div></div>')


def base_style():
    """The cockpit CSS, taken from an empty standard view."""
    from types import SimpleNamespace
    empty = SimpleNamespace(positions={}, closed=[], events=[], last_marks={}, report=lambda: dict(
        ending_equity_proxy_usd=0.0, ending_cash_usd=0.0, realized_pnl_usd=0.0,
        realized_pnl_10bp_stress_usd=0.0, data_degraded=False, entry_data_blocked=False,
        unreliable_virtual_trades=0, starting_cash_usd=0.0))
    now = pd.Timestamp.now(tz='UTC')
    return cockpit_ui.page_parts(cockpit_ui.render_html(
        cockpit_ui.snapshot(empty, now, None, 'WARTE_AUF_SIGNALFENSTER', 'live', 0)))[0]


class BotView:
    """Cockpit-like object handed to one bot's run loop."""

    def __init__(self, parent, bot_id):
        self.parent, self.bot_id = parent, bot_id

    def update(self, folder, *args, **kwargs):
        data = self.parent.cockpits[self.bot_id].update(folder, *args, **kwargs)
        if data is not None:
            self.parent.snapshots[self.bot_id] = data
            other = [b for b in (BOT1_ID, BOT2_ID) if b != self.bot_id]
            self.parent.refresh(read_files=other)
        return data


def serve_dual(dual, port=8765):
    """Browser link for the dual cockpit (same read-only server as V1.4.3)."""
    return cockpit_ui.open_browser_view(dual, port)


# ---------------------------------------------------------------- comparison log
def opening_stats(folder, feed):
    path = Path(folder) / f'session_bars_{feed}.csv'
    manifest = accounts.read_json(Path(folder) / 'manifest.json', {})
    if not path.exists() or path.stat().st_size <= 1:
        return {}
    bars = pd.read_csv(path)
    bars['timestamp'] = pd.to_datetime(bars.timestamp, utc=True)
    opening = pd.Timestamp(manifest['session_open_utc'])
    first = bars[(bars.timestamp >= opening) & (bars.timestamp < opening + pd.Timedelta(minutes=15))]
    return {symbol: dict(or_high=g.high.max(), or_low=g.low.min(), or_volume=g.volume.sum(),
                         or_bars=len(g)) for symbol, g in first.groupby('symbol')}


def first_signals(folder):
    found = {}
    for path in sorted((Path(folder) / 'signals').glob('*.json')):
        item = accounts.read_json(path)
        if isinstance(item, dict) and 'symbol' in item and item['symbol'] not in found:
            found[item['symbol']] = item
    return found


def eligible(folder):
    path = Path(folder) / 'universe_filter_results.csv'
    if not path.exists():
        return set()
    frame = pd.read_csv(path)
    return set(frame.loc[frame.result == 'PASS', 'symbol'])


def comparison_rows(day, iex_folder, sip_folder):
    sides = {}
    for name, folder, feed in (('iex', iex_folder, 'iex'), ('sip', sip_folder, 'sip')):
        sides[name] = dict(folder=folder, complete=bool(folder) and accounts.run_is_complete(folder),
                           stats=opening_stats(folder, feed) if folder else {},
                           signals=first_signals(folder) if folder else {},
                           eligible=eligible(folder) if folder else set())
    symbols = sorted(sides['iex']['eligible'] | sides['sip']['eligible'])
    complete = sides['iex']['complete'] and sides['sip']['complete']
    rows = []
    for symbol in symbols:
        row = dict(date=day, symbol=symbol,
                   same_universe=symbol in sides['iex']['eligible'] and symbol in sides['sip']['eligible'],
                   comparison_complete=complete)
        for name, side in sides.items():
            stats, signal = side['stats'].get(symbol, {}), side['signals'].get(symbol)
            decision = pd.Timestamp(signal['decision_time_utc']) if signal else None
            paper = signal.get('paper_order') if signal else None
            row.update({f'{name}_or_high': stats.get('or_high'), f'{name}_or_low': stats.get('or_low'),
                        f'{name}_or_volume': stats.get('or_volume'), f'{name}_or_bars': stats.get('or_bars'),
                        f'{name}_signal': signal is not None,
                        f'{name}_signal_time_ny': decision.tz_convert(NY).strftime('%H:%M') if decision is not None else None,
                        f'{name}_rvol': signal.get('rvol') if signal else None,
                        f'{name}_status': signal.get('status') if signal else None,
                        f'{name}_virtual_entry': (signal or {}).get('shadow_status') == 'VIRTUAL_ENTRY',
                        f'{name}_reason': (signal or {}).get('shadow_status') or (signal or {}).get('status')})
            if name == 'iex':
                row['iex_paper_order_status'] = paper.get('status') if isinstance(paper, dict) else None
        rows.append(row)
    return rows


def build_comparison(root):
    """IEX_SIP_comparison.csv and performance_series.csv from all recorded days."""
    root = Path(root)
    sip = accounts.read_json(root / BOT2_ID / 'account.json', {'days': {}})
    iex = accounts.read_json(root / (BOT1_ID + '_SIM') / 'account.json', {'days': {}})
    paper = accounts.read_json(root / BOT1_ID / 'paper_days.json', {})
    rows, series = [], []
    for day in sorted(set(sip['days']) | set(iex['days'])):
        rows += comparison_rows(day, iex['days'].get(day, {}).get('run_folder'),
                                sip['days'].get(day, {}).get('run_folder'))
    for label, source in (('IEX_SIMULATION', iex['days']), ('SIP_SIMULATION', sip['days'])):
        for day, entry in sorted(source.items()):
            series.append(dict(date=day, series=label, pnl_usd=entry['pnl_usd'],
                               return_pct=100 * entry['pnl_usd'] / entry['start_equity_usd'],
                               trades=entry['trades'], simulated=True))
    for day, entry in sorted(paper.items()):
        if entry.get('orders_enabled'):
            series.append(dict(date=day, series='IEX_ALPACA_PAPER', pnl_usd=entry['pnl_usd'],
                               return_pct=100 * entry['pnl_usd'] / entry['start_equity_usd'],
                               trades=None, simulated=False))
    accounts.write_csv(root / 'IEX_SIP_comparison.csv', pd.DataFrame(rows))
    accounts.write_csv(root / 'performance_series.csv', pd.DataFrame(series))
    return root / 'IEX_SIP_comparison.csv'


# ---------------------------------------------------------------- orchestration
def run_session(mode, trading_client, data_client, data_keys, project_root, c, s,
                approval=None, display=True, serve=True, shared=None, clock=None,
                bot2_process_options=None):
    """Central start of the live modes (V1.5 behaviour). Returns a dict of run folders.

    `shared`, `clock` and `bot2_process_options` exist for tests only."""
    info = resolve_mode(mode)
    root = Path(project_root) / 'v1_5'
    clock = clock or sim.SYSTEM_CLOCK
    day = clock.now().tz_convert(NY).strftime('%Y-%m-%d')
    print(f'{VERSION} · Modus {info["mode"]} · '
          f'{BOT1_NAME}: {"aktiv" if info["bot1_active"] else "deaktiviert"} · '
          f'{BOT2_NAME}: {"aktiv (SIMULIERT)" if info["bot2_active"] else "deaktiviert"}')
    dual = DualCockpit(root, info, display=display, latest_path=Path(project_root) / 'cockpit_latest.html')
    if serve:
        try:
            serve_dual(dual)
        except Exception as exc:
            print('Browser-Link nicht verfügbar:', type(exc).__name__)
    dual.refresh()
    calendar, universe = shared or prepare_shared(trading_client, day)
    results = {}
    if info['mode'] == 'SIP_ONLY':
        results[BOT2_ID] = bot2.run_sip_shadow_day(data_client, project_root, c, s, calendar, universe,
                                                   cockpit=dual.view(BOT2_ID),
                                                   base_clock=None if clock is sim.SYSTEM_CLOCK else clock)
    else:
        import iex_paper_bot_v16 as bot1   # only BOT 1 code paths load the order gateway
        child = None
        if info['bot2_active']:
            child = start_bot2_process(data_keys, project_root, c, s, calendar, universe, root,
                                       **(bot2_process_options or {}))
        factory = capture_gateway(dual, bot1.paper.PaperOrderGateway)
        try:
            results[BOT1_ID], _ = bot1.run_iex_day(trading_client, data_client, project_root, c, s,
                                                   universe, approval=approval, clock=clock,
                                                   cockpit=dual.view(BOT1_ID), gateway_factory=factory)
        finally:
            if child is not None:
                wait_for_bot2(child, dual)
    build_comparison(root)
    dual.refresh(read_files=(BOT1_ID, BOT2_ID))
    return results


def capture_gateway(dual, gateway_class):
    def factory(*args, **kwargs):
        dual.gateway = gateway_class(*args, **kwargs)
        return dual.gateway
    return factory


def start_bot2_process(data_keys, project_root, c, s, calendar, universe, root,
                       data_factory='sip_shadow_bot_v16:alpaca_market_data',
                       factory_kwargs=None, clock_factory=None):
    """BOT 2 in its own (spawned) process: own memory, no trading client."""
    context = multiprocessing.get_context('spawn')
    kwargs = factory_kwargs if factory_kwargs is not None else dict(
        api_key=data_keys[0], secret_key=data_keys[1])
    process = context.Process(target=bot2.child_main, name='SIP_DELAYED_SHADOW', daemon=True, args=(
        data_factory, kwargs, str(project_root), c, s, calendar, universe.to_dict('records'),
        str(root / BOT2_ID)), kwargs=dict(clock_factory=clock_factory))
    process.start()
    return process


def wait_for_bot2(process, dual, poll_seconds=30):
    """After BOT 1 finished: keep the cockpit current until BOT 2 has closed its day."""
    while process.is_alive():
        dual.refresh(read_files=(BOT2_ID,))
        process.join(poll_seconds)
    dual.refresh(read_files=(BOT2_ID,))


In [ ]:
%%writefile /content/control_panel_v16.py
"""Cockpit control panel (V1.6): the only place a user sets anything in normal use.

    MODE  [ OBSERVE ] [ DRY RUN ] [ PAPER ] [ BACKTEST ]      (exactly one)
          ☐ SIP-Vergleichsbot parallel          (DRY RUN / PAPER only; never orders)
          Freigabe PAPER-ORDERS-OK …            (PAPER only)
    BACKTEST  Startdatum · Enddatum · Startkapital · ☐ Trades vollständig laden
    [ DATENZUGANG PRÜFEN ] [ CACHE PRÜFEN ] [ ▶ START … ]
    progress · results: [ ERGEBNIS ] [ TRADES ] [ SIGNALE ] [ TAGE ] [ ANALYSIS-BUNDLE ]

The widgets only collect input; `RunRequest` + `validate_request` (plain Python,
tested without Colab) turn it into the settings the modules use. The central
config stays in the modules (Config, ScannerConfig, BacktestSettings): nothing
here is a second copy of a strategy parameter.
"""
from dataclasses import dataclass
from datetime import date, timedelta
from html import escape
from pathlib import Path
import json
import traceback

import pandas as pd

import bot_mode_v16 as bot_mode

NY = 'America/New_York'
VERSION = 'V1.6.1'


@dataclass(frozen=True)
class RunRequest:
    mode: str = bot_mode.DEFAULT_MODE
    sip_compare: bool = False
    approval: str = None
    start: date = None
    end: date = None
    initial_equity: float = 10000.0
    full_trades: bool = False


def validate_request(req, today=None):
    """Messages that block a start (empty list = OK)."""
    today = today or pd.Timestamp.now(tz=NY).date()
    errors = []
    try:
        bot_mode.plan(req.mode, req.sip_compare, req.approval)
    except ValueError as exc:
        errors.append(str(exc))
    if bot_mode.normalize(req.mode) == 'BACKTEST':
        if req.start is None or req.end is None:
            errors.append('Bitte Start- und Enddatum wählen.')
        else:
            if req.start > req.end:
                errors.append('Das Startdatum liegt nach dem Enddatum.')
            if req.end >= today:
                errors.append('Das Enddatum muss vor heute liegen (nur abgeschlossene Handelstage).')
            if req.start < date(2016, 1, 1):
                errors.append('SIP-Historie von Alpaca beginnt 2016.')
        if not req.initial_equity or req.initial_equity <= 0:
            errors.append('Startkapital muss größer als 0 sein.')
    return errors


def banner_html(mode, sip_compare=False):
    mode = bot_mode.normalize(mode)
    title, text = bot_mode.BANNERS[mode]
    colors = dict(OBSERVE='#344e75', DRY_RUN='#1f6f8b', PAPER='#b54708', BACKTEST='#5b3fa6')
    extra = ('<div style="font-size:13px;margin-top:4px">+ SIP-Vergleichsbot parallel: '
             'nur beobachten und vergleichen, <b>keine Orders</b></div>') if sip_compare else ''
    orders = ('Orders möglich: <b>nur BOT 1 (IEX) über das Alpaca-Paper-Konto</b>' if mode == 'PAPER'
              else 'Orders: <b>keine</b> (technisch gesperrt)')
    return (f'<div style="background:{colors[mode]};color:white;border-radius:10px;padding:10px 14px;'
            f'font:15px system-ui,sans-serif"><div style="font-size:20px;font-weight:800;letter-spacing:.5px">'
            f'{escape(title)}</div><div>{escape(text)}</div><div style="font-size:13px;margin-top:4px">'
            f'{orders}</div>{extra}</div>')


def hm(minutes):
    return f'{minutes // 60:02d}:{minutes % 60:02d}'


def schedule_html(c):
    """Session and strategy windows, derived from the running Config (display only)."""
    opening, closing = 9 * 60 + 30, 16 * 60
    return ('<div style="font:13px system-ui,sans-serif;color:#344">'
            'Session 09:30 – 16:00 NY (verkürzte Tage laut Börsenkalender) · '
            f'Opening Range 09:30 – {hm(opening + c.opening_minutes)} NY · '
            f'Einstiege ab {hm(opening + c.opening_minutes + 1)} bis vor '
            f'{hm(closing - c.entry_cutoff_minutes)} NY · Glattstellung {hm(closing - c.close_before_minutes)} NY '
            '(an verkürzten Tagen entsprechend früher) · alle Zeiten America/New_York · '
            'Werte aus der geprüften Strategie-Konfiguration, hier nicht änderbar.</div>')


def progress_html(info):
    phase = info.get('phase')
    if phase in ('download', 'research') and 'progress' in info:
        return (f'<b>HISTORICAL DATA DOWNLOAD</b> · Symbole {info.get("symbols_done")} / '
                f'{info.get("symbols_total")} · Zeitraum {escape(str(info.get("period", "")))} · '
                f'Fortschritt {info["progress"]:.0%} · aus Cache {info.get("cached_share", 0):.0%} · '
                f'geladen {info.get("downloaded_share", 0):.0%}')
    if phase == 'migration':
        return (f'<b>Alt-Cache wird einmalig übernommen</b> · Tag {info.get("days_done")} / '
                f'{info.get("days_total")} · {info.get("progress", 0):.0%}')
    if phase == 'daily_bars':
        return (f'<b>Tageskerzen (Liquiditätsfilter)</b> · Symbole {info.get("symbols_done")} / '
                f'{info.get("symbols_total")}')
    if phase == 'check_cache':
        missing = info.get('days_with_missing_bars', 0)
        return ('<b>HISTORICAL DATA</b> · Feed SIP · Bars: ' +
                ('READY' if not missing else f'{missing} Handelstage fehlen – werden jetzt geladen') +
                ' · Quotes: bei Bedarf (Entscheidungszeitpunkte) · Trades: bei Bedarf')
    if phase == 'day_done':
        equity = info.get('end_equity')
        return (f'<b>BACKTEST</b> · Tag {info.get("number")} / {info.get("days")} · {info.get("date")} · '
                f'{escape(str(info.get("status")))} · Trades {info.get("trades", 0)} · Equity '
                + (f'{equity:,.2f} USD' if isinstance(equity, (int, float)) else '—'))
    if phase == 'start':
        return (f'<b>BACKTEST {escape(info["run_id"])}</b> · {info["days"]} Handelstage + '
                f'{info["warmup"]} Warmup-Tage (RVOL)')
    if phase == 'research':
        return '<b>RESEARCH EXPORT</b> · Tabellen, Schema und Analysis-Bundle werden geschrieben …'
    if phase == 'done':
        return f'<b>BACKTEST COMPLETE</b> · Status {escape(info["status"])}'
    return escape(str(info))


def status_html(status):
    missing = status.get('days_with_missing_bars', 0)
    size = status.get('size_bytes', 0) / 1e6
    cached = status.get('cached_range')
    return ('<div style="font:13px system-ui,sans-serif"><b>HISTORICAL DATA CACHE</b> · Feed '
            f'{escape(status["feed"])} · gespeicherte Tage {escape(" – ".join(cached)) if cached else "—"} · '
            f'{size:,.1f} MB · Bars: ' +
            ('READY' if not missing else f'<span style="color:#b54708">Historical Data Missing: {missing} '
             'Handelstage (werden beim Start geladen)</span>') +
            ' · Quotes/Trades: bei Bedarf</div>')


def summary_html(summary, bundle=None):
    p, t, s, d = summary['performance'], summary['trades'], summary['signals'], summary['data_quality']

    def pct(v):
        return '—' if v is None else f'{100 * v:+.2f} %'

    def usd(v):
        return '—' if v is None else f'{v:,.2f} USD'
    tiles = [('Status', escape(summary['status'])), ('Handelstage', p['trading_days']),
             ('Startkapital', usd(p['initial_equity'])), ('Endkapital', usd(p['final_equity'])),
             ('Netto-PnL', usd(p['net_profit'])), ('Rendite', pct(p['return_pct'])),
             ('Max. Drawdown', pct(p['max_drawdown_pct'])), ('Trades', t['total']),
             ('Trefferquote', pct(t['win_rate']).replace('+', '')), ('Profitfaktor',
                                                                     '—' if t['profit_factor'] is None else f'{t["profit_factor"]:.2f}'),
             ('Signale gesamt / angenommen', f'{s["total"]} / {s["accepted"]}'),
             ('Datenqualität', escape(d['level'])), ('Tage mit Datenproblem', d['days_systemic_error'])]
    body = ''.join(f'<div style="background:white;border:1px solid #d9e1ed;border-radius:8px;padding:8px 10px">'
                   f'<div style="font-size:12px;color:#58677a">{escape(str(k))}</div>'
                   f'<div style="font-size:16px;font-weight:700">{v}</div></div>' for k, v in tiles)
    warn = ''.join(f'<li>{escape(w)}</li>' for w in d.get('warnings') or [])
    link = f'<div>Analysis-Bundle: <code>{escape(str(bundle))}</code></div>' if bundle else ''
    return (f'<div style="font:14px system-ui,sans-serif;background:#f5f8fc;padding:12px;border-radius:10px">'
            f'<div style="font-size:18px;font-weight:800">BACKTEST COMPLETE · {escape(summary["run_id"])}</div>'
            f'<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:8px;margin:10px 0">'
            f'{body}</div>{link}{"<ul>" + warn + "</ul>" if warn else ""}'
            f'<div style="font-size:12px;color:#58677a">{escape(summary["warning"])}</div></div>')


PHASE_NAMES = ('Cache prüfen', 'Historische Daten laden', 'Daten prüfen', 'Indikatoren vorbereiten',
               'Replay läuft', 'Research-Output erzeugen')


def clock_text(seconds):
    seconds = max(0, int(seconds))
    return f'{seconds // 3600:02d}:{seconds % 3600 // 60:02d}:{seconds % 60:02d}'


def local_time(wall):
    stamp = pd.Timestamp(wall, unit='s', tz='UTC').tz_convert('Europe/Berlin')
    return stamp.strftime('%H:%M:%S')


def running_html(snap, now=None):
    """Compact backtest view: day-by-day externally, never per minute."""
    import time as _time
    now = now or _time.time()
    status = snap.get('status') or 'CREATED'
    period = snap.get('period') or ('—', '—')

    def de(day):
        return pd.Timestamp(day).strftime('%d.%m.%Y') if day and day != '—' else '—'
    total, done = snap.get('days_total') or 0, snap.get('days_done') or 0
    share = done / total if total else 0.0
    download = snap.get('download') or {}
    if snap.get('phase') in (1, 2, 3) and download.get('progress') is not None:
        share_text = (f'Download {download["progress"]:.0%} · Symbole {download.get("symbols_done", "—")} / '
                      f'{download.get("symbols_total", "—")} · aus Cache {download.get("cached_share", 0):.0%}')
    else:
        share_text = f'{share:.0%}'
    equity = snap.get('equity')
    coverage = snap.get('coverage')
    signals = (snap.get('signals') or 0) + (snap.get('day_signals') or 0 if not snap.get('finished') else 0)
    trades = (snap.get('trades') or 0) + (snap.get('day_trades') or 0 if not snap.get('finished') else 0)
    sim_time = snap.get('sim_time')
    rows = [('Zeitraum', f'{de(period[0])} – {de(period[1])}'), ('Handelstage', f'{done} / {total}'),
            ('Aktueller Tag', de(snap.get('current_day')) + (
                f' · {pd.Timestamp(sim_time).tz_convert("America/New_York"):%H:%M} NY' if sim_time and
                not snap.get('finished') else '')),
            ('Universum', f'{snap.get("universe") or "—"} Aktien'), ('Signale', signals), ('Trades', trades),
            ('Equity', f'{equity:,.2f} USD' if equity is not None else '—'),
            ('Datenabdeckung', f'{100 * coverage:.1f} %' if coverage is not None else '—'),
            ('Laufzeit', clock_text(now - (snap.get('started_wall') or now)))]
    table = ''.join(f'<tr><td style="padding:2px 14px 2px 0;color:#58677a">{escape(str(k))}</td>'
                    f'<td style="font-weight:700">{escape(str(v))}</td></tr>' for k, v in rows)
    phase = snap.get('phase') or 0
    phases = ''.join(
        f'<li style="{"font-weight:800;color:#5b3fa6" if i + 1 == phase else "color:#58677a" if i + 1 < phase else "color:#9aa6b6"}">'
        f'{i + 1}. {escape(name)}{" ◀" if i + 1 == phase else ""}</li>' for i, name in enumerate(PHASE_NAMES))
    bar = int(round(share * 30)) if phase >= 5 else int(round((download.get('progress') or 0) * 30))
    lines = snap.get('day_lines') or []
    tail = escape('\n'.join(lines[-8:])) if lines else 'Noch kein Handelstag abgeschlossen.'
    title = {'RUNNING': 'BACKTEST RUNNING', 'PAUSED': 'BACKTEST PAUSED', 'COMPLETED': 'BACKTEST COMPLETE'}.get(
        status, f'BACKTEST {status}')
    message = snap.get('message') or ''
    error = f'<div style="color:#ba3042">{escape(snap["error"])}</div>' if snap.get('error') else ''
    pause = '<div style="color:#b54708">Pause angefordert – endet nach dem laufenden Handelstag.</div>' \
        if snap.get('pause_requested') and not snap.get('finished') else ''
    return (f'<div style="font:14px system-ui,sans-serif;background:#f5f8fc;border-radius:10px;padding:12px">'
            f'<div style="font-size:18px;font-weight:800">{escape(title)}'
            f'{" · " + escape(snap["run_id"]) if snap.get("run_id") else ""}</div>'
            f'<table style="margin:6px 0">{table}</table>'
            f'<div style="font-family:monospace;font-size:15px">{"█" * bar}{"░" * (30 - bar)} {escape(share_text)}</div>'
            f'<div style="margin-top:6px"><b>Phase:</b> {escape(PHASE_NAMES[phase - 1]) if phase else "Start"}'
            f'{" – " + escape(message) if message else ""}</div><ol style="margin:4px 0 4px 18px;padding:0">{phases}</ol>'
            f'{pause}{error}<div style="color:#58677a">Letzte Aktualisierung: {local_time(snap.get("last_update_wall") or now)} '
            f'(Berlin) · Status: {escape(status)}</div>'
            f'<pre style="font-size:12px;margin:8px 0 0;white-space:pre-wrap">{tail}</pre></div>')


def open_run_html(run):
    period = run.get('period') or {}
    state = 'unterbrochen (kein Lebenszeichen seit ' + clock_text(run['heartbeat_age_seconds']) + ')' \
        if run.get('interrupted') else run.get('status')
    done = len(run.get('records') or [])
    equity = run.get('current_equity')
    return ('<div style="font:14px system-ui,sans-serif;background:#fff4e5;border:1px solid #f0c27b;'
            'border-radius:10px;padding:10px 12px"><div style="font-weight:800">INCOMPLETE BACKTEST FOUND</div>'
            f'Lauf: <code>{escape(run["run_id"])}</code> · Status: {escape(str(state))}<br>'
            f'Zeitraum: {escape(str(period.get("start")))} – {escape(str(period.get("end")))} · '
            f'Abgeschlossen: {done} / {run.get("days_total")} Tage · letzter Tag: '
            f'{escape(str(run.get("last_completed_date") or "—"))} · Equity: '
            f'{f"{equity:,.2f} USD" if equity is not None else "—"}</div>')


def latest_run(project):
    index = Path(project) / 'backtests' / 'backtest_runs.csv'
    if not index.exists() or index.stat().st_size <= 1:
        return None
    frame = pd.read_csv(index)
    return Path(frame.iloc[-1]['path']) if len(frame) else None


class App:
    """Widget cockpit. Heavy imports happen on use, so the panel appears at once."""

    def __init__(self, project, broker, market_data, data_keys, config, scan_config,
                 config_source=None, display_now=True, background=True):
        import ipywidgets as w
        self.background = background     # backtest in its own thread; the UI never blocks it
        self.tracker, self.worker, self.renderer = None, None, None
        self.open_run = None
        self.w = w
        self.project, self.broker, self.market_data = Path(project), broker, market_data
        self.data_keys, self.config, self.scan_config = data_keys, config, scan_config
        self.config_source = config_source
        self.store = None
        self.last_folder = latest_run(project)
        self.running = False
        today = pd.Timestamp.now(tz=NY).date()
        self.mode = w.ToggleButtons(options=[(bot_mode.LABELS[m], m) for m in bot_mode.MODES],
                                    value=bot_mode.DEFAULT_MODE, description='MODE',
                                    style={'button_width': '120px'})
        self.banner = w.HTML()
        self.sip_compare = w.Checkbox(value=False, description='SIP-Vergleichsbot parallel (nur beobachten, nie Orders)',
                                      indent=False, layout=w.Layout(width='520px'))
        self.approval = w.Text(value='', placeholder='PAPER-ORDERS-OK <Paper-Kontonummer> <YYYY-MM-DD>',
                               description='Freigabe', layout=w.Layout(width='520px'))
        self.start_date = w.DatePicker(description='Startdatum', value=today - timedelta(days=30))
        self.end_date = w.DatePicker(description='Enddatum', value=today - timedelta(days=1))
        self.capital = w.BoundedFloatText(value=10000.0, min=100.0, max=1e9, step=1000.0,
                                          description='Startkapital $')
        self.full_trades = w.Checkbox(value=False, indent=False,
                                      description='Alle Trades der Signal-Aktien laden (FULL_MARKET_DATA, groß)')
        self.schedule = w.HTML(schedule_html(config))
        self.backtest_box = w.VBox([w.HTML('<b>BACKTEST</b> · historische SIP-Daten · keine Orders'),
                                    w.HBox([self.start_date, self.end_date]), self.capital,
                                    self.full_trades, self.schedule])
        self.check_button = w.Button(description='DATENZUGANG PRÜFEN', icon='plug', layout=w.Layout(width='190px'))
        self.cache_button = w.Button(description='CACHE PRÜFEN', icon='database', layout=w.Layout(width='150px'))
        self.start_button = w.Button(description='▶ START', button_style='success', layout=w.Layout(width='220px'))
        self.status = w.HTML()
        self.progress = w.FloatProgress(value=0, min=0, max=1, layout=w.Layout(width='520px', visibility='hidden'))
        self.result = w.HTML()
        names = (('ERGEBNIS', 'summary'), ('TRADES', 'trades'), ('SIGNALE', 'signals'), ('TAGE', 'daily'),
                 ('ANALYSIS-BUNDLE', 'bundle'))
        self.result_buttons = [w.Button(description=label, layout=w.Layout(width='150px')) for label, _ in names]
        for button, (_, kind) in zip(self.result_buttons, names):
            button.on_click(lambda _b, kind=kind: self.safe(self.show, kind))
        self.output = w.Output()
        self.stop_button = w.Button(description='⏸ PAUSE NACH TAG', button_style='warning',
                                    layout=w.Layout(width='190px', display='none'))
        self.stop_button.on_click(lambda _b: self.request_pause())
        self.open_box = w.HTML()
        self.resume_button = w.Button(description='▶ RESUME BACKTEST', button_style='success',
                                      layout=w.Layout(width='200px'))
        self.discard_button = w.Button(description='DISCARD', layout=w.Layout(width='120px'))
        self.resume_button.on_click(lambda _b: self.safe(self.resume))
        self.discard_button.on_click(lambda _b: self.safe(self.discard))
        self.open_panel = w.VBox([self.open_box, w.HBox([self.resume_button, self.discard_button])],
                                 layout=w.Layout(display='none'))
        self.mode.observe(lambda change: self.refresh(), names='value')
        self.sip_compare.observe(lambda change: self.refresh(), names='value')
        self.check_button.on_click(lambda _b: self.safe(self.check_access))
        self.cache_button.on_click(lambda _b: self.safe(self.check_cache))
        self.start_button.on_click(lambda _b: self.safe(self.start))
        self.panel = w.VBox([
            w.HTML(f'<div style="font:22px system-ui,sans-serif;font-weight:800">US-Aktien-Bot {VERSION} · Cockpit</div>'),
            self.open_panel, self.mode, self.banner, self.sip_compare, self.approval, self.backtest_box,
            w.HBox([self.check_button, self.cache_button, self.start_button, self.stop_button]),
            self.progress, self.status,
            self.result, w.HBox(self.result_buttons), self.output])
        self.refresh()
        self.find_open_run()
        if display_now:
            from IPython.display import display
            display(self.panel)

    # ------------------------------------------------------------ state
    def request(self):
        return RunRequest(mode=self.mode.value, sip_compare=bool(self.sip_compare.value),
                          approval=self.approval.value.strip() or None, start=self.start_date.value,
                          end=self.end_date.value, initial_equity=float(self.capital.value),
                          full_trades=bool(self.full_trades.value))

    def refresh(self):
        mode = self.mode.value
        live_order_mode = mode in ('DRY_RUN', 'PAPER')
        if not live_order_mode and self.sip_compare.value:
            self.sip_compare.value = False
        self.sip_compare.layout.display = None if live_order_mode else 'none'
        self.approval.layout.display = None if mode == 'PAPER' else 'none'
        self.backtest_box.layout.display = None if mode == 'BACKTEST' else 'none'
        self.full_trades.layout.display = None if mode == 'BACKTEST' else 'none'
        self.banner.value = banner_html(mode, self.sip_compare.value)
        self.start_button.description = f'▶ START {bot_mode.LABELS[mode]}'
        self.start_button.button_style = 'warning' if mode == 'PAPER' else 'success'
        has_run = self.last_folder is not None
        for button in self.result_buttons:
            button.disabled = not has_run or self.running

    def lock(self, locked):
        self.running = locked
        for widget in (self.mode, self.sip_compare, self.approval, self.start_date, self.end_date,
                       self.capital, self.full_trades, self.check_button, self.cache_button, self.start_button):
            widget.disabled = locked
        self.refresh()

    def safe(self, action, *args):
        """Run a button action; errors appear in the panel, never as credentials."""
        if self.running:
            return
        try:
            action(*args)
        except Exception as exc:
            self.status.value = (f'<span style="color:#ba3042"><b>{escape(type(exc).__name__)}</b>: '
                                 f'{escape(str(exc)[:600])}</span>')
            with self.output:
                traceback.print_exc(limit=3)
        finally:
            worker_alive = self.worker is not None and self.worker.is_alive()
            if self.running and not worker_alive:
                self.lock(False)

    def data_store(self):
        if self.store is None:
            import hist_data_v16 as hist
            self.store = hist.HistoricalDataStore(self.project / 'data' / 'historical', self.market_data,
                                                  hist.ReadOnlyBroker(self.broker), progress=self.on_progress)
            old = self.project / 'replay_cache_sip'
            if old.exists() and not self.store.manifest.get('migrations'):
                self.status.value = 'Übernehme vorhandenen replay_cache_sip/ (einmalig, mit Prüfung) …'
                result = self.store.migrate_replay_cache(old)
                self.status.value = (f'Alt-Cache übernommen: {result["days_accepted"]} Tage geprüft übernommen, '
                                     f'{result["days_rejected"]} abgelehnt (werden bei Bedarf neu geladen).')
        return self.store

    def on_progress(self, info):
        self.status.value = progress_html(info)
        if 'progress' in info:
            self.progress.layout.visibility = 'visible'
            self.progress.value = float(info['progress'])
        elif info.get('phase') == 'day_done' and info.get('days'):
            self.progress.layout.visibility = 'visible'
            self.progress.value = info['number'] / info['days']

    # ------------------------------------------------------------ actions
    def check_access(self):
        import data_access_v16 as access
        self.lock(True)
        self.status.value = 'Prüfe historische SIP-Daten für AAPL (Bars, Quotes, Trades) …'
        store = self.data_store()
        today = pd.Timestamp.now(tz=NY).date()
        sessions = store.sessions(today - timedelta(days=20), today)
        report = access.check_data_access(self.market_data, sessions,
                                          save_folder=self.project / 'data' / 'historical' / 'capabilities')
        self.status.value = access.format_html(report)
        with self.output:
            print(access.format_text(report))

    def period_sessions(self, req):
        store = self.data_store()
        days = [r['date'] for r in store.sessions(req.start, req.end)]
        return store, days

    def check_cache(self):
        import backtest_v16 as backtest
        req = self.request()
        if req.start is None or req.end is None:
            raise ValueError('Bitte Start- und Enddatum wählen.')
        self.lock(True)
        store = self.data_store()
        settings = backtest.BacktestSettings(str(req.start), str(req.end), req.initial_equity)
        plan = backtest.plan_days(store, settings, self.config)
        symbols = sorted(set(store.universe().symbol) | {'QQQ'})
        status = store.status(symbols, plan['warmup'] + plan['days'])
        self.status.value = status_html(status) + (
            f'<div style="font:13px system-ui">Zeitraum: {len(plan["days"])} Handelstage, Warmup ab '
            f'{plan["warmup_start"]} ({len(plan["warmup"])} Tage).</div>')

    def start(self):
        req = self.request()
        errors = validate_request(req)
        if errors:
            self.status.value = '<br>'.join(f'<span style="color:#ba3042">{escape(e)}</span>' for e in errors)
            return
        plan = bot_mode.plan(req.mode, req.sip_compare, req.approval)
        bot_mode.set_mode(plan.mode)
        self.lock(True)
        self.output.clear_output()
        if plan.kind == 'backtest':
            self.run_backtest(req)
        else:
            self.run_live(plan)

    def run_live(self, plan):
        import dual_bot_v16 as dual
        from dataclasses import replace
        day = pd.Timestamp.now(tz='UTC').tz_convert(NY).strftime('%Y-%m-%d')
        self.status.value = banner_html(plan.mode, plan.sip_compare) + '<div>Lauf gestartet – Cockpit unten.</div>'
        with self.output:
            folders = dual.run_session(plan.dual_mode, self.broker, self.market_data, self.data_keys,
                                       self.project, replace(self.config, test_date=day), self.scan_config,
                                       approval=plan.approval)
            print('Ergebnisordner:', {bot: str(folder) for bot, folder in folders.items()})
        self.status.value = f'Lauf beendet ({escape(plan.mode)}).'

    def run_backtest(self, req=None, resume=None):
        """Start the backtest core outside the widget callback (own thread in Colab)."""
        import threading
        import backtest_v16 as backtest
        settings = None if resume else backtest.BacktestSettings(
            str(req.start), str(req.end), float(req.initial_equity), full_trades=req.full_trades)
        self.tracker = backtest.BacktestProgress()
        self.tracker.set(status='CREATED', message='Starte …')
        self.result.value = ''
        self.stop_button.layout.display = None
        self.progress.layout.visibility = 'visible'

        def work():
            try:
                store = self.data_store()
                result = backtest.run_backtest(store, self.project, self.config, self.scan_config, settings,
                                               tracker=self.tracker, config_source=self.config_source,
                                               resume=resume)
                self.last_folder = result['folder']
                self.result.value = summary_html(result['summary'], result['bundle'])
            except Exception as exc:
                self.tracker.set(status='FAILED', finished=True, error=f'{type(exc).__name__}: {str(exc)[:400]}')
            finally:
                self.tracker.set(finished=True)
                self.render(force=True)
                self.stop_button.layout.display = 'none'
                self.open_panel.layout.display = 'none'
                self.lock(False)
                self.find_open_run()
        self.start_renderer()
        if self.background:
            self.worker = threading.Thread(target=work, name='backtest-core', daemon=False)
            self.worker.start()
        else:
            work()

    def start_renderer(self, interval=5.0):
        """Heartbeat for the eye: re-render the compact view every few seconds (own thread)."""
        import threading
        import time as _time
        tracker = self.tracker
        state = dict(last=0.0, seen=-1)

        def loop():
            while True:
                snap = tracker.snapshot()
                due = _time.time() - state['last'] >= interval or tracker.ui_updates_requested != state['seen']
                if due:
                    state['seen'] = tracker.ui_updates_requested
                    state['last'] = _time.time()
                    self.render(snapshot=snap)
                if snap.get('finished'):
                    return
                _time.sleep(1.0)
        if self.background:
            self.renderer = threading.Thread(target=loop, name='backtest-cockpit', daemon=True)
            self.renderer.start()

    def render(self, force=False, snapshot=None):
        if self.tracker is None:
            return
        snap = snapshot or self.tracker.snapshot()
        snap['pause_requested'] = self.tracker.pause_requested
        self.ui_renders = getattr(self, 'ui_renders', 0) + 1
        self.status.value = running_html(snap)
        total = snap.get('days_total') or 0
        if (snap.get('phase') or 0) >= 5 and total:
            self.progress.value = (snap.get('days_done') or 0) / total
        elif (snap.get('download') or {}).get('progress') is not None:
            self.progress.value = float(snap['download']['progress'])

    def keep_alive(self, poll_seconds=30, max_hours=24):
        """Keep a notebook cell busy while the backtest thread works (Colab counts an idle
        kernel as inactive). Returns when the run is finished, paused or after max_hours."""
        import time as _time
        deadline = _time.time() + max_hours * 3600
        while self.worker is not None and self.worker.is_alive() and _time.time() < deadline:
            self.worker.join(poll_seconds)
        return None if self.tracker is None else self.tracker.snapshot().get('status')

    def request_pause(self):
        if self.tracker is not None:
            self.tracker.pause_requested = True
            self.render(force=True)

    def find_open_run(self):
        import backtest_v16 as backtest
        try:
            runs = backtest.find_open_runs(self.project)
        except Exception:
            runs = []
        running_here = self.tracker is not None and not self.tracker.snapshot().get('finished')
        self.open_run = next((r for r in runs if not running_here or r['run_id'] != self.tracker.state.get('run_id')),
                             None)
        if self.open_run is None:
            self.open_panel.layout.display = 'none'
            return None
        self.open_box.value = open_run_html(self.open_run)
        self.open_panel.layout.display = None
        return self.open_run

    def resume(self):
        if self.open_run is None:
            raise ValueError('Kein unterbrochener Backtest gefunden.')
        bot_mode.set_mode('BACKTEST')
        self.mode.value = 'BACKTEST'
        self.lock(True)
        self.open_panel.layout.display = 'none'
        self.run_backtest(resume=self.open_run['folder'])

    def discard(self):
        import backtest_v16 as backtest
        if self.open_run is not None:
            backtest.discard_run(self.open_run['folder'])
            self.status.value = f'Lauf {escape(self.open_run["run_id"])} verworfen (Dateien bleiben erhalten).'
        self.find_open_run()

    def show(self, kind):
        folder = self.last_folder
        if folder is None:
            raise ValueError('Noch kein Backtest-Lauf vorhanden.')
        from IPython.display import display
        self.output.clear_output()
        with self.output:
            if kind == 'summary':
                summary = json.loads((folder / 'summary.json').read_text())
                bundle = next(folder.glob('*_ANALYSIS.zip'), None)
                self.result.value = summary_html(summary, bundle)
            elif kind == 'bundle':
                bundle = next(folder.glob('*_ANALYSIS.zip'), None)
                if bundle is None:
                    raise FileNotFoundError('Kein Analysis-Bundle im Laufordner.')
                print('Analysis-Bundle:', bundle)
                try:
                    from google.colab import files
                    files.download(str(bundle))
                except Exception:
                    print('Download-Dialog nur in Colab verfügbar; Datei liegt auf Google Drive.')
            else:
                frame = pd.read_csv(folder / f'{kind}.csv')
                print(f'{kind}.csv · {len(frame)} Zeilen · {folder}')
                display(frame.tail(200))


## Cockpit

Diese Zelle lädt die Module, prüft die Strategie-Konfiguration gegen den Scannerlauf und zeigt das Cockpit. Danach läuft die Bedienung nur über die Knöpfe.

- **Laufender Betrieb:** Solange ein Lauf aktiv ist, sind die Bedienelemente gesperrt.
- **Anzeige:** Die Live-Modi zeigen darunter das bekannte Drei-Bereiche-Cockpit; der Backtest zeigt Fortschritt und Ergebnis.
- **Unterbrochener Lauf:** Das Cockpit bietet ihn oben zur Fortsetzung an (**RESUME**) oder zum Verwerfen (**DISCARD**, die Dateien bleiben erhalten).


In [ ]:
from pathlib import Path
from google.colab import drive, userdata
from alpaca.trading.client import TradingClient
from alpaca.data.historical import StockHistoricalDataClient
import hashlib, importlib, json, sys
import pandas as pd

try:
    import pyarrow  # noqa: F401  (Parquet für den Datencache und den Research-Export)
except ImportError:
    raise RuntimeError('pyarrow fehlt in dieser Colab-Umgebung: bitte "%pip install pyarrow" ausführen.')

drive.mount('/content/drive')
project = Path('/content/drive/MyDrive/US_Aktien_Bot')
source = project / 'v0_3_runs' / PREVIOUS_SCANNER_RUN
prior = json.loads((source / 'manifest.json').read_text())
comparison = {'base_source.py': '/content/us_orb_test_v02.py',
              'scanner_source.py': '/content/us_orb_scanner_v03.py'}
for filename, digest in prior['code_hashes'].items():
    if hashlib.sha256((source / filename).read_bytes()).hexdigest() != digest:
        raise ValueError('Scannerquellcode wurde verändert: ' + filename)
    if hashlib.sha256(Path(comparison[filename]).read_bytes()).hexdigest() != digest:
        raise ValueError('Notebook-Regeln stimmen nicht mit dem Testlauf überein: ' + filename)

paper_key = (userdata.get('ALPACA_PAPER_API_KEY') or '').strip()
paper_secret = (userdata.get('ALPACA_PAPER_SECRET_KEY') or '').strip()
if not paper_key or not paper_secret:
    raise RuntimeError('Bitte die beiden Paper-Secrets freigeben.')
# Ausschließlich der Paper-Endpunkt. Backtests erhalten davon nur Kalender und Aktienliste.
broker = TradingClient(api_key=paper_key, secret_key=paper_secret,
                       paper=True, url_override='https://paper-api.alpaca.markets')
market_data = StockHistoricalDataClient(api_key=paper_key, secret_key=paper_secret)
data_keys = (paper_key, paper_secret)   # nur für den Marktdaten-Client des SIP-Vergleichsbots
del paper_key, paper_secret
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
modules = {}
for name in ('us_orb_test_v02', 'us_orb_scanner_v03', 'bot_mode_v16', 'orb_portfolio_v15', 'orb_sim_v16',
             'orb_cockpit_v16', 'orb_replay_v16', 'hist_data_v16', 'data_access_v16',
             'research_export_v16', 'backtest_v16', 'bot_accounts_v15', 'sip_shadow_bot_v16',
             'paper_orders_v16', 'iex_paper_bot_v16', 'dual_bot_v16', 'control_panel_v16'):
    modules[name] = importlib.reload(importlib.import_module(name))
base, scanner = modules['us_orb_test_v02'], modules['us_orb_scanner_v03']

config = base.Config(**prior['base_config'])
scan_config = scanner.ScannerConfig(**prior['scanner_config'])
app = modules['control_panel_v16'].App(
    project, broker, market_data, data_keys, config, scan_config,
    config_source=dict(scanner_run=PREVIOUS_SCANNER_RUN, code_hashes=prior['code_hashes']))


## Lange Backtests: Laufzeit aktiv halten (optional, empfohlen)

Nach **▶ START BACKTEST** rechnet der Backtest in einem eigenen Hintergrund-Thread. Das Cockpit aktualisiert sich alle paar Sekunden, auch ohne weitere Klicks.

Bei langen Läufen diese Zelle danach einmal ausführen: Sie wartet, bis der Lauf fertig oder pausiert ist, und hält die Colab-Laufzeit so beschäftigt. Sie kann jederzeit mit „Ausführung unterbrechen“ beendet werden, der Backtest läuft dabei weiter.

Geht die Laufzeit trotzdem verloren, zeigt das Cockpit beim nächsten Öffnen **INCOMPLETE BACKTEST FOUND** mit **▶ RESUME BACKTEST**. Es fehlt dann höchstens der Handelstag, der gerade lief.


In [ ]:
status = app.keep_alive()
print('Backtest-Status:', status)
latest = modules['control_panel_v16'].latest_run(project)
if latest is not None:
    print('Letzter Lauf:', latest, '· Analysis-Bundle:', next(latest.glob('*_ANALYSIS.zip'), None))
